# 研究 4：連続 T ゲートでの code switching の償却 — 本体（Google Colab 用）

**問い**：code switching（FSC → $\mathbb D_4$ 符号 → 論理 $T^\dagger$ → FSC）を 1 回だけでなく、同じ論理量子ビットに**続けて $K$ 回**行うと、$T$ ゲート 1 個あたりの誤りと時空間体積はどう変わるか。魔法状態蒸留との比較（研究 3）の結論は変わるか。

**このノートがすること**

1. $K$ 回連続の code switching を、研究 3 と同じ回路レベル雑音・同じ復号器・同じ Pauli proxy 法で標本化する（$K=1$ は研究 3 と 1 ショット単位で同一）。
2. 出力 $(T^\dagger)^K|+\rangle$ の不忠実度から **1 サイクルあたりの誤り** $\varepsilon_{\rm cycle}$ を求め、1 回だけの場合と比べる。サイクルの間の FSC ラウンド数（`GAP_LIST`）を短くした場合も比べる。
3. 回路から数えた体積で、$T$ ゲート 1 個あたりの費用を研究 3 の比較表に重ねる。

**使い方**：上から順に実行する。変更するのは**セル 2（パラメータ）だけ**。既定の `RUN_MODE = 'quick'` は数分で終わる動作確認。モンテカルロを走らせずに保存済みの試行データを解析するには `RUN_MODE = 'analysis_only'`。背景・理論・結果の読み方は同じフォルダの `研究4_方針と理論的考察.md` と `研究4_結果.md`。

**セルの並び**：1 インストール → 2 パラメータ → 出力先の準備 → 研究 3 のライブラリ（変更なし、読み飛ばしてよい）→ 研究 4 のライブラリ → 参照データ → 自己検査 → モンテカルロ → 集計・解析・図。

In [ ]:
# ==================================================================================================
# セル 1：必要なライブラリのインストール（Colab に無いものだけ）と import
# ==================================================================================================
import importlib.util, subprocess, sys
for _mod, _pkg in [('stim', 'stim'), ('pymatching', 'pymatching'), ('scipy', 'scipy')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])
import os, io, json, math, itertools
import numpy as np, pandas as pd, stim, pymatching
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats as _st
print('stim', stim.__version__, '| pymatching', pymatching.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)

In [ ]:
# ==================================================================================================
# セル 2：パラメータ（利用者が変更するのはこのセルだけ）
# ==================================================================================================

# ---- 実行モード -------------------------------------------------------------------------------
#   'quick'         : 数分の動作確認（d=3 だけ、少ないショット）
#   'pilot'         : 試行計算（リポジトリの results_pilot/ を作った設定。Colab の CPU 2 コアで 5〜6 時間）
#   'production'    : 本番（下の PLAN['production']。GPU は不要、CPU のコア数に比例して速くなる）
#   'analysis_only' : モンテカルロを走らせず、OUT_DIR にある CSV（無ければ埋め込みの試行データ）を解析する
RUN_MODE = 'quick'

# ---- 計算する点 -------------------------------------------------------------------------------
# 1 行 = (雑音シナリオ, 物理誤り率 p, サイズ d, 1 点あたりのショット数)。
# 各行について、下の K_LIST × GAP_LIST の全ての組を計算する（K=1 は GAP に依存しないので 1 回だけ）。
PLAN = {
    'quick': [('B', 1e-3, 3, 2000)],
    'pilot': [('B', 1e-3, 3, 40000), ('B', 1e-3, 5, 40000), ('A', 3e-4, 3, 40000), ('A', 3e-4, 5, 40000),
              ('B', 3e-4, 3, 200000), ('B', 3e-4, 5, 100000), ('B', 1e-3, 7, 10000), ('B', 1e-4, 3, 200000)],
    'production': [('B', 1e-3, 3, 200000), ('B', 1e-3, 5, 200000), ('B', 1e-3, 7, 100000),
                   ('B', 3e-4, 3, 1000000), ('B', 3e-4, 5, 1000000), ('B', 3e-4, 7, 300000),
                   ('B', 2e-4, 3, 1000000), ('B', 2e-4, 5, 2000000),
                   ('B', 1e-4, 3, 2000000), ('B', 1e-4, 5, 4000000),
                   ('A', 3e-4, 3, 200000), ('A', 3e-4, 5, 400000), ('A', 3e-4, 7, 200000),
                   ('A', 1e-4, 3, 500000), ('A', 1e-4, 5, 2000000), ('A', 1e-4, 7, 500000)],
}
SCENARIOS = {'A': 1.0, 'B': 0.0}   # 雑音シナリオ：アイドル雑音 p_idle = (この係数) × p。A＝アイドル雑音あり、B＝なし（研究 3 と同じ）
K_LIST = [1, 2, 3]                 # 連続して行う switch-in → T^dagger → switch-out の回数 K（1 = 研究 3 と同じ回路）
GAP_LIST = ['d', 1]                # サイクルの間に挟む FSC ラウンド数。'd' = d ラウンド、'half' = ceil(d/2)、整数 = その数（1 以上）
QUIET_WINDOW = False               # True：切り替えの窓の雑音を消した診断（研究 3 の quiet と同じ定義）も同じ点で計算する

# ---- 回路の定数（研究 3 と同じ値。変えると研究 3 と比べられなくなる）--------------------------
KAPPA = (1.0, 1.0, 1.0)            # (S0, S1, 最後の S4) のラウンド数 = ceil(kappa × d)
N_PRE_U = 2                        # U の直前に置く Z だけのラウンド数

# ---- 実行の制御 -------------------------------------------------------------------------------
WORKERS = 2                        # 並列プロセス数（Colab 無料版は 2）
BASE_SEED = 20261111               # 乱数の種。点とチャンクごとの seed はここから決まる（実行順に依存しない）
RESUME = True                      # True：OUT_DIR に CSV があれば続きから計算する（同じ点・同じチャンクは再計算しない）
OUT_DIR = 'results_study4'         # 出力先のフォルダ（CSV と図）
USE_GOOGLE_DRIVE = False           # True：Google Drive をマウントして DRIVE_DIR/OUT_DIR に保存する（ランタイムが切れても残る）
DRIVE_DIR = '/content/drive/MyDrive/CodeSwitching_study4'

# ---- 解析 -------------------------------------------------------------------------------------
CONFIDENCE = 0.95                  # 信頼区間の水準
TARGETS = [1e-4, 1e-6, 1e-8, 1e-10]    # T ゲート 1 個あたりの要求誤り（研究 3 と同じ 4 通り）
CONSUME_OVERHEAD = [0.0, 0.1, 0.3]     # 蒸留側に加える「魔法状態の消費（格子手術）の体積 ÷ 工場の体積」。0 = 研究 3 と同じ（消費を数えない）
RUN_SELFCHECK = True               # True：モンテカルロの前に 1〜2 分の自己検査（K=1 が研究 3 と一致、雑音なしで誤り 0、厳密計算との照合）

In [ ]:
# 出力先の準備（パラメータの検査を含む）
assert RUN_MODE in ('quick', 'pilot', 'production', 'analysis_only'), RUN_MODE
assert all(isinstance(k, int) and k >= 1 for k in K_LIST) and 1 in K_LIST, 'K_LIST には 1 を含める（1 サイクルあたりの誤りの基準）'
assert all(g in ('d', 'half') or (isinstance(g, int) and g >= 1) for g in GAP_LIST)
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = os.path.join(DRIVE_DIR, OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
print('RUN_MODE =', RUN_MODE, '| 出力先 =', os.path.abspath(OUT_DIR))

## 研究 3 のライブラリ（変更なし・読み飛ばしてよい）

研究 3 の `D4_noisy_simulation.ipynb` の物理ライブラリ（格子、回路、雑音モデル、Pauli proxy 法、復号器、厳密な参照シミュレータ）を**そのまま**埋め込んだもの。関数とクラスの定義だけで、実行しても何も出力しない。研究 4 で新しく書いたコードはこの次の節にある。

### 共通ライブラリ（F2 線形代数・ゲート／ガジェット・スケジューラ・Clifford+T 分解・計数ルール R1–R3）

In [ ]:
"""common_lib: F2 linear algebra, gate/gadget model, scheduler, decompositions, cost model rules."""
import itertools, math
import numpy as np
from collections import Counter, defaultdict

In [ ]:
# ---------------------------------------------------------------- F2 algebra
def f2_rref(M):
    """Row-reduce a uint8 matrix over F2. Returns (R, pivot_cols)."""
    M = (np.array(M, dtype=np.uint8) & 1).copy()
    rows, cols = M.shape
    piv = []
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        nz = np.nonzero(M[r:, c])[0]
        if len(nz) == 0:
            continue
        p = r + nz[0]
        if p != r:
            M[[r, p]] = M[[p, r]]
        others = np.nonzero(M[:, c])[0]
        others = others[others != r]
        if len(others):
            M[others] ^= M[r]
        piv.append(c)
        r += 1
    return M[:r], piv


def f2_rank(M):
    if len(M) == 0:
        return 0
    return len(f2_rref(M)[1])


def f2_nullspace(M, n=None):
    """Basis (rows) of {x : M x = 0} over F2."""
    M = np.array(M, dtype=np.uint8)
    if M.size == 0:
        return np.eye(n, dtype=np.uint8)
    n = M.shape[1]
    Rm, piv = f2_rref(M)
    free = [c for c in range(n) if c not in set(piv)]
    basis = []
    for f in free:
        v = np.zeros(n, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if Rm[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis, dtype=np.uint8).reshape(len(basis), n)


def f2_in_rowspace(Rref, piv, v):
    """Is v in the row space of the RREF matrix Rref (pivots piv)?"""
    v = np.array(v, dtype=np.uint8) & 1
    w = v.copy()
    for i, p in enumerate(piv):
        if w[p]:
            w ^= Rref[i]
    return not w.any()


def f2_solve(A, b):
    """Solve x A = b (x row vector, A rows = generators). Returns x or None."""
    A = np.array(A, dtype=np.uint8)
    m, n = A.shape
    aug = np.concatenate([A, np.eye(m, dtype=np.uint8)], axis=1)
    Rm, piv = f2_rref(aug)
    w = np.concatenate([np.array(b, dtype=np.uint8) & 1, np.zeros(m, dtype=np.uint8)])
    for i, p in enumerate(piv):
        if p >= n:
            break
        if w[p]:
            w ^= Rm[i]
    if w[:n].any():
        return None
    return w[n:]




import itertools
from collections import Counter, defaultdict

ARITY = {'INIT_X': 1, 'INIT_Z': 1, 'MEAS_X': 1, 'MEAS_Z': 1, 'H': 1, 'X': 1, 'Z': 1, 'S': 1, 'SDG': 1,
         'T': 1, 'TDG': 1, 'CNOT': 2, 'CZ': 2, 'CS': 2, 'CSDG': 2, 'CCZ': 3}
NONCLIFFORD = {'T', 'TDG', 'CS', 'CSDG', 'CCZ'}

In [ ]:
# ------------------------------------------------------------------ minimum-weight logical (MILP)
def min_weight_odd(Hc, lvec, cols):
    """min |x| over x in F2^cols with Hc x = 0 and lvec . x = 1 (exact MILP, HiGHS)."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    H = np.array(Hc, dtype=float)[:, cols]
    H = H[np.any(H, axis=1)]
    l = np.array(lvec, dtype=float)[cols]
    m, n = H.shape
    # variables: x (n, binary), y (m, integer), z (1, integer)
    nv = n + m + 1
    c = np.concatenate([np.ones(n), np.zeros(m + 1)])
    A1 = np.concatenate([H, -2 * np.eye(m), np.zeros((m, 1))], axis=1)
    A2 = np.concatenate([l[None, :], np.zeros((1, m)), -2 * np.ones((1, 1))], axis=1)
    cons = [LinearConstraint(A1, 0, 0), LinearConstraint(A2, 1, 1)]
    integrality = np.ones(nv)
    ub = np.concatenate([np.ones(n), np.full(m + 1, np.inf)])
    res = milp(c, constraints=cons, integrality=integrality, bounds=Bounds(np.zeros(nv), ub))
    assert res.status == 0, res.message
    x = np.round(res.x[:n]).astype(np.uint8)
    out = np.zeros(len(lvec), dtype=np.uint8)
    out[cols] = x
    return int(x.sum()), out

In [ ]:
# ------------------------------------------------------------------ gadgets
def pauli_check_gadget(anc, xq=(), zq=()):
    """Measure X(xq) Z(zq) with ancilla anc (standard CNOT circuit)."""
    g = [('INIT_X', anc)]
    g += [('CNOT', anc, q) for q in sorted(xq)]           # ancilla controls -> X on data
    g += [('CZ', anc, q) for q in sorted(zq)]             # controlled Z on data
    g += [('MEAS_X', anc)]
    return g


def z_check_gadget(anc, zq):
    g = [('INIT_Z', anc)] + [('CNOT', q, anc) for q in sorted(zq)] + [('MEAS_Z', anc)]
    return g


def clifford_x_gadget(anc, parts):
    """Measure a product of X-type Clifford operators S = S_1 S_2 ... with
    S_k = X(s_k) prod CZ(pairs_k): ancilla |+>, controlled-S (= CNOTs from anc and
    CCZ(anc,i,j) for each CZ pair; rightmost factor first), measure X.
    parts: list of (xqubits, czpairs) in operator order S_1 S_2 ..."""
    g = [('INIT_X', anc)]
    for xq, pairs in reversed(parts):
        g += [('CCZ', anc, i, j) for (i, j) in pairs]      # CZ part acts first within S_k
        g += [('CNOT', anc, q) for q in sorted(xq)]
    g += [('MEAS_X', anc)]
    return g

In [ ]:
# ------------------------------------------------------------------ scheduling
def asap_layers(gates):
    """Greedy ASAP layering of a gate list respecting qubit order (no reordering of gates that
    share a qubit).  Returns list of layers (lists of gates)."""
    ready = defaultdict(int)
    layers = []
    for gt in gates:
        qs = gt[1:]
        t = max(ready[q] for q in qs)
        while len(layers) <= t:
            layers.append([])
        layers[t].append(gt)
        for q in qs:
            ready[q] = t + 1
    return layers


def gadget_support(gad):
    s = set()
    for gt in gad:
        s |= set(gt[1:])
    return s


def disjoint_groups(gadgets):
    """Greedy colouring: partition gadgets so that gadgets in a group touch disjoint qubits.
    Measurements within a group are then exactly simultaneous and the sequential-group
    schedule is valid for commuting (on the relevant subspace) generators."""
    groups = []          # list of (used_qubits, [gadgets])
    order = sorted(range(len(gadgets)), key=lambda i: -len(gadget_support(gadgets[i])))
    for i in order:
        sup = gadget_support(gadgets[i])
        for used, members in groups:
            if not (used & sup):
                used |= sup
                members.append(gadgets[i])
                break
        else:
            groups.append((set(sup), [gadgets[i]]))
    return [m for _, m in groups]


def layer_duration(layer, dur):
    return max(dur[g[0]] for g in layer)


def circuit_time(layers, dur):
    return sum(layer_duration(l, dur) for l in layers)


def round_schedule(gadgets, dur):
    """Upper-bound schedule of one round: disjoint groups executed sequentially, each group
    ASAP-layered.  Returns dict with depth (layers), time (weighted by dur), groups."""
    groups = disjoint_groups(gadgets)
    depth = 0
    time = 0
    for grp in groups:
        gl = asap_layers([gt for gad in grp for gt in gad])
        depth += len(gl)
        time += circuit_time(gl, dur)
    return {'groups': len(groups), 'depth_ub': depth, 'time_ub': time}


def round_lower_bound(gadgets, dur):
    """Lower bound on the duration of one round: every qubit executes its gates sequentially,
    so time >= max_q sum of durations of gates acting on q."""
    busy = defaultdict(float)
    cnt = defaultdict(int)
    for gad in gadgets:
        for gt in gad:
            for q in gt[1:]:
                busy[q] += dur[gt[0]]
                cnt[q] += 1
    return {'depth_lb': max(cnt.values()), 'time_lb': max(busy.values())}


def gate_counts(gates):
    c = Counter(g[0] for g in gates)
    return dict(c)

In [ ]:
# ------------------------------------------------------------------ decompositions
# Explicit Clifford+T decompositions (verified numerically in the validation notebook).
def ccz_to_clifford_t(a, b, c):
    """CCZ with 7 T/T^dag and 6 CNOT (standard Toffoli decomposition, Nielsen & Chuang Fig. 4.9,
    with the target Hadamards removed)."""
    return [('CNOT', b, c), ('TDG', c), ('CNOT', a, c), ('T', c), ('CNOT', b, c), ('TDG', c),
            ('CNOT', a, c), ('T', b), ('T', c), ('CNOT', a, b), ('T', a), ('TDG', b), ('CNOT', a, b)]


def cs_to_clifford_t(a, b):
    """CS = diag(1,1,1,i) = T_a T_b CNOT(a,b) T^dag_b CNOT(a,b)."""
    return [('T', a), ('T', b), ('CNOT', a, b), ('TDG', b), ('CNOT', a, b)]


def csdg_to_clifford_t(a, b):
    return [('TDG', a), ('TDG', b), ('CNOT', a, b), ('T', b), ('CNOT', a, b)]


def decompose(gates):
    out = []
    for g in gates:
        if g[0] == 'CCZ':
            out += ccz_to_clifford_t(*g[1:])
        elif g[0] == 'CS':
            out += cs_to_clifford_t(*g[1:])
        elif g[0] == 'CSDG':
            out += csdg_to_clifford_t(*g[1:])
        else:
            out.append(g)
    return out


DUR_UNIT = {k: 1 for k in ARITY}            # every primitive = 1 time unit (ASSUMPTION, parametrised)

In [ ]:
# ------------------------------------------------- commuting-class scheduler
DIAG_GATES = {'CZ', 'CCZ', 'CS', 'CSDG', 'T', 'TDG', 'S', 'SDG', 'Z'}


def gate_roles(gt):
    """Per-qubit role of an entangling/1q unitary: 'Z' (diagonal in Z basis), 'X' (X-type)."""
    name = gt[0]
    if name in DIAG_GATES:
        return {q: 'Z' for q in gt[1:]}
    if name == 'CNOT':
        return {gt[1]: 'Z', gt[2]: 'X'}
    if name == 'X':
        return {gt[1]: 'X'}
    return {q: '?' for q in gt[1:]}      # H, INIT, MEAS: never shared between gadgets


def gadget_roles(gad):
    roles = {}
    for gt in gad:
        for q, r in gate_roles(gt).items():
            if roles.get(q, r) != r:
                roles[q] = '?'
            else:
                roles[q] = r
    return roles


def commuting_classes(gadgets, n_orders=6, seed=0):
    """Best (fewest classes) of several greedy colourings (different gadget orders)."""
    import random
    best = None
    orders = [list(range(len(gadgets)))]
    orders.append(sorted(orders[0], key=lambda i: tuple(sorted(set(gadget_roles(gadgets[i]).values())))))
    rnd = random.Random(seed)
    for _ in range(n_orders):
        o = orders[0][:]
        rnd.shuffle(o)
        orders.append(o)
    for o in orders:
        cl = _commuting_classes_greedy([gadgets[i] for i in o])
        if best is None or len(cl) < len(best):
            best = cl
    return best


def _commuting_classes_greedy(gadgets):
    """Greedy colouring of the gadget conflict graph.  Two gadgets are compatible if on every
    shared qubit both act with the same role ('Z' or 'X'); then all their gates commute and the
    joint circuit is equivalent to measuring them one after another (sufficient condition)."""
    roles = [gadget_roles(g) for g in gadgets]
    classes = []     # list of (role_map, [idx])
    for i, rm in enumerate(roles):
        placed = False
        for cmap, members in classes:
            ok = all(cmap.get(q, r) == r and r != '?' for q, r in rm.items() if q in cmap)
            if ok:
                for q, r in rm.items():
                    cmap[q] = r
                members.append(i)
                placed = True
                break
        if not placed:
            classes.append((dict(rm), [i]))
    return [[gadgets[i] for i in m] for _, m in classes]


def class_layers(gadgets):
    """Layering of one commuting class: init layer, greedy layering of the (mutually commuting)
    entangling gates, measurement layer."""
    inits = [g for gad in gadgets for g in gad if g[0].startswith('INIT')]
    meas = [g for gad in gadgets for g in gad if g[0].startswith('MEAS')]
    body = [g for gad in gadgets for g in gad if not (g[0].startswith('INIT') or g[0].startswith('MEAS'))]
    # greedy: most-constrained gates first
    body = sorted(body, key=lambda g: -len(g))
    layers = []
    busy = []
    for g in body:
        qs = set(g[1:])
        for li, b in enumerate(busy):
            if not (b & qs) and layers[li][0][0] == g[0]:     # keep one gate type per layer
                b |= qs
                layers[li].append(g)
                break
        else:
            busy.append(set(qs))
            layers.append([g])
    return ([inits] if inits else []) + layers + ([meas] if meas else [])


def round_schedule_classes(gadgets, dur):
    classes = commuting_classes(gadgets)
    depth = 0
    time = 0
    for c in classes:
        ls = class_layers(c)
        depth += len(ls)
        time += circuit_time(ls, dur)
    return {'classes': len(classes), 'depth_ub': depth, 'time_ub': time}


import math
from collections import Counter

SOURCE_LIT = 'literature'
SOURCE_COUNT = 'counted (explicit circuit)'
SOURCE_ASSUME = 'assumption (parameter)'
SOURCE_UNRES = 'unresolved (excluded / illustrative)'

DEFAULT_PARAMS = dict(
    kappa0=1.0,      # FSC |+_L> preparation rounds = ceil(kappa0*d)   [ASSUMPTION; Kobayashi: not specified]
    kappa_in=1.0,    # switch-in rounds            = ceil(kappa_in*d)  [ASSUMPTION; Kobayashi: O(d)]
    kappa_out=1.0,   # FSC EC rounds after return  = ceil(kappa_out*d) [ASSUMPTION; Kobayashi: O(d)]
    tau_rounds=1.0,  # Litinski: 1 time step = tau_rounds*d code cycles [LITERATURE: d cycles]
    t_inj_FT=6,      # Litinski App. A (Li's injection): 3 + 3 code cycles, d-independent [LITERATURE]
    t_inj_NL=1,      # noiseless-minimal injection: 1 round [ASSUMPTION]
)


def rounds_for(model, kappa, d):
    if model == 'NL_min':
        return 1
    return int(math.ceil(kappa * d))

In [ ]:
# ----------------------------------------------------------------- summaries
def summarize(rows):
    tot = Counter()
    for r in rows:
        for k, v in r['gates'].items():
            tot[k] += v
    peak = max(r['qubits'] for r in rows)
    rounds = sum(r['rounds'] for r in rows)
    tl = sum(r['time_lb'] for r in rows if r['time_lb'] is not None)
    tu = sum(r['time_ub'] for r in rows if r['time_ub'] is not None)
    v_round = sum(r['qubits'] * r['rounds'] for r in rows)
    v_tl = sum(r['qubits'] * r['time_lb'] for r in rows if r['time_lb'] is not None)
    v_tu = sum(r['qubits'] * r['time_ub'] for r in rows if r['time_ub'] is not None)
    nonclif = {k: v for k, v in tot.items() if k in NONCLIFFORD}
    tcount = 7 * tot.get('CCZ', 0) + 3 * (tot.get('CS', 0) + tot.get('CSDG', 0)) + tot.get('T', 0) + tot.get('TDG', 0)
    return dict(peak_qubits=peak, rounds=rounds, time_lb=tl, time_ub=tu, V_qubit_round=v_round,
                V_qubit_time_lb=v_tl, V_qubit_time_ub=v_tu, V_peak_x_rounds_UB=peak * rounds,
                gates=dict(tot), nonclifford=nonclif, T_count_if_decomposed=tcount)


def uncounted_costs(protocol, comparison):
    """Costs that are known to exist but are not included in the totals (listed, never summed as 0)."""
    out = [('classical decoding / control latency', 'both comparisons, both protocols')]
    if protocol == 'Kobayashi_D4':
        out.append(('JIT-decoder classical processing', 'not modelled'))
        if comparison == 'B':
            out.append(('FSC -> standard rotated surface-code patch conversion', 'UNRESOLVED: no protocol constructed'))
    else:
        out.append(('twist-based Z_m (x) Y_0 seam checks between |m> and |0>', 'O(d) measurement qubits, not constructed'))
    return out

In [ ]:
# ------------------------------------------------------------------ table helpers
def jsonify_columns(df, cols=('gates', 'nonclifford')):
    """Store dict-valued columns as JSON strings (same format as the saved CSV files)."""
    import json
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda g: g if isinstance(g, str) else json.dumps({k: float(v) for k, v in g.items()}))
    return df


def identify_polynomial(values, npts=4):
    """Interpolate a polynomial in d through the first npts points (degree <= npts-1) and report
    whether it reproduces ALL given points exactly.  values: dict d -> integer.  Returns (expr, ok)."""
    import sympy
    x = sympy.Symbol('d')
    ds = sorted(values)
    if len(ds) <= npts:
        return None, False
    p = sympy.interpolate([(dd, int(round(values[dd]))) for dd in ds[:npts]], x)
    ok = all(sympy.nsimplify(p.subs(x, dd) - int(round(values[dd]))) == 0 for dd in ds)
    return sympy.expand(p), ok

### D4 ライブラリ（境界付き D4 符号・transversal $U$・code switching・疎状態シミュレータ・資源モデル）

In [ ]:
"""d4_lib: D4 Clifford stabilizer code (Kobayashi-Zhu-Hsin) on the bounded square lattice, transversal U, code switching, resource model."""
import itertools, cmath
import numpy as np
from collections import Counter

R, G, B = 0, 1, 2
CNAME = "rgb"

In [ ]:
# ------------------------------------------------- phase-shift operators
class PSOp:
    """Operator  O |x> = (-1)^{f(x)} |x + s>  on n qubits (computational basis x in F2^n).

    f(x) = const + sum_{i in lin} x_i + sum_{(i,j) in quad} x_i x_j   (mod 2), i<j.
    X-type Clifford stabilizers of the D4 code, Pauli X/Z strings and products
    thereof all have this form.  Composition rule (A o B)|x> = A B |x>:
        s = s_A + s_B,   f(x) = f_B(x) + f_A(x + s_B).
    """
    __slots__ = ("s", "lin", "quad", "const")

    def __init__(self, s=frozenset(), lin=frozenset(), quad=frozenset(), const=0):
        self.s = frozenset(s)
        self.lin = frozenset(lin)
        self.quad = frozenset(tuple(sorted(p)) for p in quad)
        self.const = const & 1

    def shifted_phase(self, t):
        """Return (lin, quad, const) of x -> f(x + t)."""
        lin = set(self.lin)
        const = self.const
        for i in self.lin:
            if i in t:
                const ^= 1
        for (i, j) in self.quad:
            ti, tj = i in t, j in t
            if ti and tj:
                const ^= 1
            if tj:
                lin ^= {i}
            if ti:
                lin ^= {j}
        return frozenset(lin), self.quad, const

    def __matmul__(self, other):
        lA, qA, cA = self.shifted_phase(other.s)
        return PSOp(self.s ^ other.s, lA ^ other.lin, qA ^ other.quad, cA ^ other.const)

    def phase_at(self, x):
        """f(x) for x given as a set/frozenset of 1-positions."""
        v = self.const
        for i in self.lin:
            if i in x:
                v ^= 1
        for (i, j) in self.quad:
            if i in x and j in x:
                v ^= 1
        return v

    def apply(self, x):
        """O|x> = sign |x'> ; returns (x', sign_bit)."""
        return frozenset(x) ^ self.s, self.phase_at(x)

    def support(self):
        sup = set(self.s) | set(self.lin)
        for p in self.quad:
            sup |= set(p)
        return sup


def commutator_affine(A, B):
    """A B = (-1)^{g(x)} B A on |x>.  Returns g as (lin, const) -- quadratic terms cancel."""
    AB = A @ B
    BA = B @ A
    assert AB.s == BA.s
    q = AB.quad ^ BA.quad
    assert not q, "commutator has quadratic part (should not happen)"
    return AB.lin ^ BA.lin, AB.const ^ BA.const

In [ ]:
# ------------------------------------------------------------- the lattice
class D4Lattice:
    def __init__(self, L):
        assert L >= 1
        self.L = L
        self.edges = []          # (kind, x, y), kind 'h' or 'v'
        self.eidx = {}
        for y in range(L + 1):
            for x in range(L):
                self._add(('h', x, y))
        for x in range(L + 1):
            for y in range(L):
                self._add(('v', x, y))
        self.E = len(self.edges)
        self.n = 3 * self.E
        self.vertices = [(x, y) for y in range(L + 1) for x in range(L + 1)]
        self.plaquettes = [(x, y) for y in range(L) for x in range(L)]

    def _add(self, e):
        self.eidx[e] = len(self.edges)
        self.edges.append(e)

    def q(self, e, c):
        if not isinstance(e, int):
            e = self.eidx[e]
        return 3 * e + c

    # --- geometry --------------------------------------------------------
    def endpoints(self, e):
        k, x, y = self.edges[e] if isinstance(e, int) else e
        return ((x, y), (x + 1, y)) if k == 'h' else ((x, y), (x, y + 1))

    def incident(self, v):
        x, y = v
        out = []
        for e in [('h', x, y), ('h', x - 1, y), ('v', x, y), ('v', x, y - 1)]:
            if e in self.eidx:
                out.append(self.eidx[e])
        return out

    def plaq_edges(self, p):
        """(e01, e13, e02, e23) of plaquette p=(x,y)."""
        x, y = p
        ei = self.eidx
        return ei[('h', x, y)], ei[('v', x + 1, y)], ei[('v', x, y)], ei[('h', x, y + 1)]

    def plaq_boundary(self, p):
        return list(self.plaq_edges(p))

    def has_plaq(self, x, y):
        return 0 <= x < self.L and 0 <= y < self.L

    def boundary_of_edge(self, e):
        k, x, y = self.edges[e]
        L = self.L
        if k == 'v' and x == 0:
            return 'Lr'
        if k == 'h' and y == L:
            return 'Lb'
        if k == 'h' and y == 0:
            return 'Lrb'
        if k == 'v' and x == L:
            return 'Lrb'
        return None

    def boundary_edges(self, name):
        return [e for e in range(self.E) if self.boundary_of_edge(e) == name]

    # --- Z-type checks (rows of H_Z) --------------------------------------
    def z_checks(self, colours=(R, G, B)):
        """List of (label, set_of_qubits) Z-type stabilizer generators.
        Plaquettes for every colour + single-edge boundary conditions (Fig. 4 / eq. (7))."""
        rows = []
        for p in self.plaquettes:
            for c in colours:
                rows.append((('plaq', p, CNAME[c]), {self.q(e, c) for e in self.plaq_boundary(p)}))
        for e in range(self.E):
            bd = self.boundary_of_edge(e)
            if bd == 'Lr' and R in colours:
                rows.append((('bdZ', e, 'r'), {self.q(e, R)}))
            elif bd == 'Lb' and B in colours:
                rows.append((('bdZ', e, 'b'), {self.q(e, B)}))
            elif bd == 'Lrb':
                if R in colours and B in colours:
                    rows.append((('bdZ', e, 'rb'), {self.q(e, R), self.q(e, B)}))
                if G in colours:
                    rows.append((('bdZ', e, 'g'), {self.q(e, G)}))
        return rows

    # --- X-type Clifford stabilizers ----------------------------------------
    def cz_pairs(self, v, c):
        """CZ pairs (qubit, qubit) of the dressing of S^c_{X;v} (SM eq. for S_X, cup products
        evaluated with the hypercube formula; plaquettes outside the lattice are dropped)."""
        x, y = v
        pairs = []
        if c == R:      # int v u e' u e'' : v = vertex 0 of p(x,y);   CZ^{g,b}
            if self.has_plaq(x, y):
                e01, e13, e02, e23 = self.plaq_edges((x, y))
                pairs += [(self.q(e01, G), self.q(e13, B)), (self.q(e02, G), self.q(e23, B))]
        elif c == G:    # int e'' u v u e' : v = vertex 1 of p(x-1,y), vertex 2 of p(x,y-1); CZ^{r,b}
            if self.has_plaq(x - 1, y):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y))
                pairs.append((self.q(e01, R), self.q(e13, B)))
            if self.has_plaq(x, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x, y - 1))
                pairs.append((self.q(e02, R), self.q(e23, B)))
        else:           # int e' u e'' u v : v = vertex 3 of p(x-1,y-1); CZ^{r,g}
            if self.has_plaq(x - 1, y - 1):
                e01, e13, e02, e23 = self.plaq_edges((x - 1, y - 1))
                pairs += [(self.q(e01, R), self.q(e13, G)), (self.q(e02, R), self.q(e23, G))]
        return pairs

    def SX(self, v, c, dressed=True):
        s = {self.q(e, c) for e in self.incident(v)}
        quad = self.cz_pairs(v, c) if dressed else []
        # CZ applied first, then X (S = (prod X)(prod CZ)); X and CZ act on disjoint colours.
        return PSOp(s=s, quad=quad)

    def vertex_x_types(self, v, colours=(R, G, B)):
        """Allowed colour combinations at vertex v: the F2 span of colour sets whose
        X-part commutes with the boundary Z conditions.  Returns a list of colour tuples
        (a basis), derived automatically from the boundary Z checks."""
        bdrows = [rows for (lab, rows) in self.z_checks(colours) if lab[0] == 'bdZ']
        inc = self.incident(v)
        cand = []
        for k in range(1, len(colours) + 1):
            for combo in itertools.combinations(colours, k):
                s = {self.q(e, c) for e in inc for c in combo}
                if all(len(s & rw) % 2 == 0 for rw in bdrows):
                    cand.append(combo)
        # reduce to an F2 basis over colour space, preferring single colours
        basis, vecs = [], []
        for combo in sorted(cand, key=len):
            vec = np.zeros(3, dtype=np.uint8)
            vec[list(combo)] = 1
            if f2_rank(np.array(vecs + [vec])) > len(vecs):
                vecs.append(vec)
                basis.append(combo)
        return basis

    def x_generators(self, colours=(R, G, B), dressed=True):
        """List of (label, PSOp).  At a vertex where only a product of colours is allowed
        (L_rb), the generator is the ordered product S^r S^b (Fig. 4 shows S^r S^b there)."""
        gens = []
        for v in self.vertices:
            for combo in self.vertex_x_types(v, colours):
                op = PSOp()
                for c in combo:
                    op = op @ self.SX(v, c, dressed)
                gens.append((('X', v, ''.join(CNAME[c] for c in combo)), op))
        return gens

    def vec(self, qubits):
        v = np.zeros(self.n, dtype=np.uint8)
        v[list(qubits)] = 1
        return v

In [ ]:
# ---------------------------------------------------------- code data
class CodeData:
    """Linear-algebra data of a (D4 or folded-surface) code on a lattice."""

    def __init__(self, lat, colours=(R, G, B), dressed=True):
        self.lat = lat
        self.colours = tuple(colours)
        self.zlabels, zr = zip(*lat.z_checks(colours))
        self.HZ = np.array([lat.vec(r) for r in zr], dtype=np.uint8)
        self.xgens = lat.x_generators(colours, dressed)
        self.S = np.array([lat.vec(op.s) for _, op in self.xgens], dtype=np.uint8)
        self.cols = sorted({lat.q(e, c) for e in range(lat.E) for c in colours})
        self.HZr, self.HZpiv = f2_rref(self.HZ)
        self.Sr, self.Spiv = f2_rref(self.S)
        # logical representatives
        CZ = f2_nullspace(self.HZ[:, self.cols])
        self.t_log = None
        for row in CZ:
            t = np.zeros(lat.n, dtype=np.uint8)
            t[self.cols] = row
            if not f2_in_rowspace(self.Sr, self.Spiv, t):
                self.t_log = t
                break
        KS = f2_nullspace(self.S[:, self.cols])
        self.z_log = None
        for row in KS:
            u = np.zeros(lat.n, dtype=np.uint8)
            u[self.cols] = row
            if not f2_in_rowspace(self.HZr, self.HZpiv, u):
                self.z_log = u
                break
        self.k = len(self.cols) - f2_rank(self.HZ[:, self.cols]) - f2_rank(self.S[:, self.cols])


def distances(cd):
    """(d_Z, d_X_shift) : min weight of a logical Z-type Pauli, and min weight of an
    X-shift lying in C_Z but outside the stabilizer shift span."""
    dz, uz = min_weight_odd(cd.S, cd.t_log, cd.cols)
    dx, tx = min_weight_odd(cd.HZ, cd.z_log, cd.cols)
    return dz, dx, uz, tx

In [ ]:
# ------------------------------------------- transversal automorphism U = W V
class DiagPhase:
    """Diagonal unitary |x> -> omega^{w(x)} |x>, omega = exp(i pi/4), w integer mod 8.
    w(x) = sum_k c_k * prod_{i in mono_k} x_i  (monomials of degree <= 2)."""

    def __init__(self, terms=()):
        self.terms = list(terms)   # (coef mod 8, tuple_of_qubits)

    def value(self, x):
        v = 0
        for c, mono in self.terms:
            if all(i in x for i in mono):
                v += c
        return v % 8

    def support(self):
        s = set()
        for _, m in self.terms:
            s |= set(m)
        return s

    def terms_touching(self, qs):
        return DiagPhase([(c, m) for c, m in self.terms if set(m) & qs])


def W_operator(lat, cs_signs=(+1, -1), boundary_T=-1):
    """W of eq. (8):  prod_p CS^{r,b}_{e01,e13} (CS^{r,b}_{e02,e23})^dag  x  prod_{e in bdry_rb} (T^r_e)^dag.
    CS = diag(1,1,1,i) -> omega^{2 x y};  T^dag -> omega^{-x}."""
    terms = []
    for p in lat.plaquettes:
        e01, e13, e02, e23 = lat.plaq_edges(p)
        terms.append(((2 * cs_signs[0]) % 8, (lat.q(e01, R), lat.q(e13, B))))
        terms.append(((2 * cs_signs[1]) % 8, (lat.q(e02, R), lat.q(e23, B))))
    if boundary_T:
        for e in lat.boundary_edges('Lrb'):
            terms.append((boundary_T % 8, (lat.q(e, R),)))
    return DiagPhase(terms)


def V_map(lat, x):
    """V = prod_e CNOT^{(r,g)}_e CNOT^{(b,g)}_e :  x_g <- x_g + x_r + x_b  (set representation)."""
    x = set(x)
    out = set(x)
    for e in range(lat.E):
        qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
        if (qr in x) ^ (qb in x):
            out ^= {qg}
    return frozenset(out)


def V_support_map(lat, qs):
    """Qubits whose values determine (V x) restricted to qs."""
    out = set()
    for q in qs:
        e, c = divmod(q, 3)
        out.add(q)
        if c == G:
            out |= {3 * e + R, 3 * e + B}
    return out


def CZ_basis(cd):
    if not hasattr(cd, '_CZbasis'):
        nb = f2_nullspace(cd.HZ[:, cd.cols])
        full = np.zeros((nb.shape[0], cd.lat.n), dtype=np.uint8)
        full[:, cd.cols] = nb
        cd._CZbasis = full
    return cd._CZbasis


def project_CZ(cd, Q, functionals=()):
    """Exact image of C_Z under x -> (x|_Q, phi_1(x), ..., phi_k(x)) for F2-linear functionals
    phi_j (callables on a set of qubits, evaluated on basis vectors).  Returns a list of
    configurations (frozenset of Q-qubits) or, if functionals are given, (config, bits)."""
    Q = sorted(Q)
    Bs = CZ_basis(cd)
    P = Bs[:, Q]
    if functionals:
        extra = np.array([[phi(frozenset(np.nonzero(b)[0])) for phi in functionals] for b in Bs],
                         dtype=np.uint8).reshape(len(Bs), len(functionals))
        P = np.concatenate([P, extra], axis=1)
    Pr, piv = f2_rref(P)
    k = Pr.shape[0]
    assert k <= 24, f"local projection too large ({k})"
    out = []
    nq = len(Q)
    for bits in itertools.product((0, 1), repeat=k):
        v = np.zeros(P.shape[1], dtype=np.uint8)
        for b, row in zip(bits, Pr):
            if b:
                v ^= row
        cfg = frozenset(Q[i] for i in np.nonzero(v[:nq])[0])
        out.append((cfg, tuple(int(z) for z in v[nq:])) if functionals else cfg)
    return out


def check_conjugation(cd, Wop, gen_op, target_op):
    """Exhaustively check  U S U^dag == S'  on C_Z, where U = W V.
    Returns (ok, n_local_configs)."""
    lat = cd.lat
    As = V_map(lat, gen_op.s)
    if As != target_op.s:
        return False, 0
    # qubits on which the phase difference can depend
    Wt = Wop.terms_touching(set(As))
    Q = set(Wt.support()) | V_support_map(lat, gen_op.support()) | target_op.support() | set(As)
    configs = project_CZ(cd, Q)
    for x in configs:
        # omega^{ w(x+As) - w(x) + 4 f(A x) - 4 f'(x) }   (only W-terms touching As contribute)
        d = Wt.value(x ^ As) - Wt.value(x) + 4 * gen_op.phase_at(V_map(lat, x)) - 4 * target_op.phase_at(x)
        if d % 8:
            return False, len(configs)
    return True, len(configs)


def stabilizer_path_phase(cd, x, target):
    """Given x and target = x + (sum of generator shifts), apply generators to |x> and
    return the accumulated sign bit, i.e. psi(target) = (-1)^bit psi(x) for code states."""
    shifts = cd.S
    delta = cd.lat.vec(set(x) ^ set(target))
    coeff = f2_solve(shifts, delta)
    if coeff is None:
        return None
    bit = 0
    cur = frozenset(x)
    for i in np.nonzero(coeff)[0]:
        op = cd.xgens[i][1]
        cur, b = op.apply(cur)
        bit ^= b
    assert cur == frozenset(target)
    return bit


def logical_phases(cd, Wop):
    """Eigenphases (in units of pi/4) of U on |0_L>, |1_L>, assuming U preserves the code space
    (checked separately).  Uses x0 = 0 and x1 = t_log."""
    lat = cd.lat
    res = []
    for x in [frozenset(), frozenset(np.nonzero(cd.t_log)[0])]:
        Ax = V_map(lat, x)
        bit = stabilizer_path_phase(cd, x, Ax)
        assert bit is not None, "V does not map the coset to itself"
        res.append((Wop.value(Ax) + 4 * bit) % 8)
    return res

In [ ]:
# ------------------------------------------- per-configuration protocol phase
def lrb_vertices(lat):
    vs = set()
    for e in lat.boundary_edges('Lrb'):
        vs |= set(lat.endpoints(e))
    return vs


def cprime(lat, a):
    """0-cochain c' with  delta c' = a_r + a_b  and c' = 0 on L_rb vertices (BFS).
    a: set of qubits (configuration of r,b qubits).  Returns dict vertex -> bit.
    Raises if a_r + a_b is not exact relative to L_rb."""
    from collections import deque
    val = {v: 0 for v in lrb_vertices(lat)}
    dq = deque(val)
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            bit = val[u] ^ (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
            if w in val:
                if val[w] != bit:
                    raise ValueError("a_r + a_b not exact")
            else:
                val[w] = bit
                dq.append(w)
    return val


def g_phase_poly(lat, v):
    """CZ^{r,b} pairs of S^g_{X;v}  (f^g_v)."""
    return lat.cz_pairs(v, G)


def protocol_phase(lat, Wop, a, gverts):
    """Phase exponent (mod 8, units of pi/4) acquired by FSC configuration a through
    switch-in (S^g measurement) -> U -> Z^g measurement -> S^g correction -> Pauli frame,
    i.e.  F(a) = w(a) + 4 * sum_v c'_v(a) f^g_v(a)."""
    cp = cprime(lat, a)
    phi = 0
    for v in gverts:
        if cp[v]:
            for (i, j) in g_phase_poly(lat, v):
                if i in a and j in a:
                    phi ^= 1
    return (Wop.value(a) + 4 * phi) % 8


def check_protocol_invariance(lat, fsc, Wop, gverts, gen_op):
    """Exhaustive local proof that F(a + t) = F(a) (mod 8) for all a in C_FSC, t = shift of gen_op."""
    t = frozenset(gen_op.s)
    # c'(t) (linear in a)
    ct = cprime(lat, t)
    N2 = {v for v in gverts if ct[v]}
    N1 = {v for v in gverts if set(q for pr in g_phase_poly(lat, v) for q in pr) & t}
    Wt = Wop.terms_touching(set(t))
    Q = set(Wt.support()) | set(t)
    for v in N1 | N2:
        Q |= {q for pr in g_phase_poly(lat, v) for q in pr}
    # paths from a reference vertex w0 to each w in N1 (BFS on the lattice graph)
    w0 = min(N1 | N2 | {lat.endpoints(next(iter(t)) // 3)[0]})
    paths = {}
    from collections import deque
    prev = {w0: None}
    dq = deque([w0])
    while dq:
        u = dq.popleft()
        for e in lat.incident(u):
            p, q = lat.endpoints(e)
            w = q if p == u else p
            if w not in prev:
                prev[w] = (u, e)
                dq.append(w)
    for w in N1:
        es = []
        cur = w
        while prev[cur] is not None:
            u, e = prev[cur]
            es.append(e)
            cur = u
        paths[w] = es
        Q |= {lat.q(e, c) for e in es for c in (R, B)}
    beta_fn = lambda b: cprime(lat, b)[w0]          # c'_{w0}(a) is F2-linear in a
    configs = project_CZ(fsc, Q, functionals=[beta_fn])
    for a, (beta,) in configs:
        if True:
            cpa = {}
            for w in N1:
                bit = beta
                for e in paths[w]:
                    bit ^= (lat.q(e, R) in a) ^ (lat.q(e, B) in a)
                cpa[w] = bit
            at = a ^ t
            d = Wt.value(at) - Wt.value(a)
            phi = 0
            for w in N1:
                if cpa[w]:
                    for (i, j) in g_phase_poly(lat, w):
                        phi ^= ((i in at) and (j in at)) ^ ((i in a) and (j in a))
            for w in N2:
                for (i, j) in g_phase_poly(lat, w):
                    phi ^= ((i in at) and (j in at))
            if (d + 4 * phi) % 8:
                return False, len(configs)
    return True, len(configs)


OMEGA = cmath.exp(1j * cmath.pi / 4)


class SparseState:
    def __init__(self, amps):
        self.a = {k: v for k, v in amps.items() if abs(v) > 1e-14}

    def norm2(self):
        return sum(abs(v) ** 2 for v in self.a.values())

    def normalize(self):
        n = self.norm2() ** 0.5
        self.a = {k: v / n for k, v in self.a.items()}
        return self

    def apply_psop(self, op):
        out = {}
        for x, v in self.a.items():
            y, b = op.apply(x)
            out[y] = out.get(y, 0) + (-v if b else v)
        return SparseState(out)

    def apply_diag(self, D):
        return SparseState({x: v * OMEGA ** D.value(x) for x, v in self.a.items()})

    def apply_perm(self, f):
        out = {}
        for x, v in self.a.items():
            y = f(x)
            out[y] = out.get(y, 0) + v
        return SparseState(out)

    def project(self, op, sign):
        """(1 + sign*O)/2 |psi>  (unnormalised)."""
        Opsi = self.apply_psop(op).a
        out = {}
        for k in set(self.a) | set(Opsi):
            out[k] = 0.5 * (self.a.get(k, 0) + sign * Opsi.get(k, 0))
        return SparseState(out)

    def expect(self, op):
        Opsi = self.apply_psop(op).a
        return sum(np.conj(self.a.get(k, 0)) * v for k, v in Opsi.items())

    def measure_op(self, op, rng, forced=None):
        """Projective measurement of a Hermitian involution O; returns (outcome bit, state)."""
        p0 = self.project(op, +1)
        n0 = p0.norm2()
        if forced is None:
            bit = 0 if rng.random() < n0 else 1
        else:
            bit = forced
        st = p0 if bit == 0 else self.project(op, -1)
        prob = n0 if bit == 0 else 1 - n0
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def measure_z(self, q, rng, forced=None):
        p1 = sum(abs(v) ** 2 for x, v in self.a.items() if q in x)
        bit = (1 if rng.random() < p1 else 0) if forced is None else forced
        st = SparseState({x: v for x, v in self.a.items() if (q in x) == bool(bit)})
        prob = p1 if bit else 1 - p1
        return bit, st.normalize() if prob > 1e-12 else st, prob

    def overlap(self, other):
        return sum(np.conj(v) * other.a.get(k, 0) for k, v in self.a.items())


def coset_state(cd, rep, phases=True):
    """Code state supported on rep + span(shifts):  psi(rep)=1, psi(x+s_v)=(-1)^{f_v(x)} psi(x)."""
    amps = {frozenset(rep): 1.0}
    # breadth-first application of generators
    frontier = [frozenset(rep)]
    while frontier:
        new = []
        for x in frontier:
            for _, op in cd.xgens:
                y, b = op.apply(x)
                if y not in amps:
                    amps[y] = amps[x] * (-1 if b else 1)
                    new.append(y)
                else:
                    if phases:
                        assert abs(amps[y] - amps[x] * (-1 if b else 1)) < 1e-9, "inconsistent code state"
        frontier = new
    return SparseState(amps).normalize()




def frozen_qubits(lat):
    """Qubits fixed to |0> by single-qubit boundary Z stabilizers (L_r: r, L_b: b, L_rb: g)."""
    fz = set()
    for lab, rows in lat.z_checks():
        if lab[0] == 'bdZ' and len(rows) == 1:
            fz |= rows
    return fz


class D4Protocol:
    def __init__(self, L, prune_frozen=True):
        self.L = L
        self.lat = lat = D4Lattice(L)
        self.frozen = frozen_qubits(lat) if prune_frozen else set()
        self.prune = prune_frozen
        self.d4 = CodeData(lat)
        self.fsc = CodeData(lat, (R, B), dressed=False)
        # ancilla allocation: one ancilla per stabilizer generator label (D4 superset)
        self.anc = {}
        nxt = lat.n
        labels = [lab for lab, _ in self.d4.xgens] + list(self.d4.zlabels)
        labels += [lab for lab, _ in self.fsc.xgens] + list(self.fsc.zlabels)
        for lab in labels:
            key = self._anc_key(lab)
            if key not in self.anc and not self._is_frozen_check(lab):
                self.anc[key] = nxt
                nxt += 1
        self.gverts = [lab[1] for lab, op in self.d4.xgens if lab[2] == 'g']

    @staticmethod
    def _anc_key(lab):
        return lab

    def _is_frozen_check(self, lab):
        return lab[0] == 'bdZ' and lab[2] in ('r', 'b', 'g') and self.prune

    def data_qubits(self, colours):
        return {self.lat.q(e, c) for e in range(self.lat.E) for c in colours} - self.frozen

    # ---------------------------------------------------------- gadgets
    def _keep_pair(self, i, j):
        return i not in self.frozen and j not in self.frozen

    def x_gadget(self, lab, dressed):
        _, v, cols = lab
        parts = []
        for ch in cols:
            c = CNAME.index(ch)
            xq = {self.lat.q(e, c) for e in self.lat.incident(v)} - self.frozen
            pairs = [p for p in self.lat.cz_pairs(v, c) if self._keep_pair(*p)] if dressed else []
            parts.append((xq, pairs))
        return clifford_x_gadget(self.anc[lab], parts)

    def z_gadget(self, lab, rows):
        return z_check_gadget(self.anc[lab], set(rows) - self.frozen)

    def round_gadgets(self, code, which='all'):
        """Gadgets of one stabilizer round.  code: 'D4' or 'FSC'.
        which: 'all' | 'X' | 'Z' | 'g' (D4 S^g only)."""
        cd = self.d4 if code == 'D4' else self.fsc
        dressed = code == 'D4'
        gads = []
        if which in ('all', 'X', 'g'):
            for lab, op in cd.xgens:
                if which == 'g' and lab[2] != 'g':
                    continue
                gads.append(self.x_gadget(lab, dressed))
        if which in ('all', 'Z'):
            for lab, (l2, rows) in zip(cd.zlabels, self.lat.z_checks(cd.colours)):
                if self._is_frozen_check(lab):
                    continue
                gads.append(self.z_gadget(lab, rows))
        return gads

    # ---------------------------------------------------------- transversal U
    def U_gates(self):
        lat, fz = self.lat, self.frozen
        gates = []
        for e in range(lat.E):
            qr, qg, qb = lat.q(e, R), lat.q(e, G), lat.q(e, B)
            for ctrl in (qr, qb):
                if ctrl in fz:
                    continue
                if qg in fz:
                    continue   # L_rb: CNOT(r,g) CNOT(b,g) = identity on Z^r Z^b = +1 (g removed)
                gates.append(('CNOT', ctrl, qg))
        for p in lat.plaquettes:
            e01, e13, e02, e23 = lat.plaq_edges(p)
            for (a, b, name) in [(lat.q(e01, R), lat.q(e13, B), 'CS'), (lat.q(e02, R), lat.q(e23, B), 'CSDG')]:
                if a in fz or b in fz:
                    continue
                gates.append((name, a, b))
        for e in lat.boundary_edges('Lrb'):
            gates.append(('TDG', lat.q(e, R)))
        return gates

    # ---------------------------------------------------------- switch-out
    def switch_out_gates(self, worst_case=True):
        lat = self.lat
        gq = sorted(self.data_qubits((G,)))
        gates = [('MEAS_Z', q) for q in gq]
        cz = []
        for v in self.gverts:
            for p in lat.cz_pairs(v, G):
                if self._keep_pair(*p):
                    cz.append(('CZ',) + tuple(p))
        return gates, cz      # cz: worst-case correction (all vertices in kappa)

    def qubit_sets(self):
        fsc_data = self.data_qubits((R, B))
        d4_data = self.data_qubits((R, G, B))
        fsc_anc = {self.anc[l] for l, _ in self.fsc.xgens} | {self.anc[l] for l in self.fsc.zlabels
                                                              if not self._is_frozen_check(l)}
        d4_anc = {self.anc[l] for l, _ in self.d4.xgens} | {self.anc[l] for l in self.d4.zlabels
                                                            if not self._is_frozen_check(l)}
        return dict(fsc_data=fsc_data, d4_data=d4_data, fsc_anc=fsc_anc, d4_anc=d4_anc)

In [ ]:
# ----------------------------------------------------------------- Kobayashi (D4)
class D4CostData:
    """All d-dependent counts of the D4 protocol, computed once per L from explicit circuits."""

    def __init__(self, L, dur=None, decomposed=False):
        dur = dur or DUR_UNIT
        P = D4Protocol(L)
        self.L = L
        qs = P.qubit_sets()
        self.n = {k: len(v) for k, v in qs.items()}
        self.n_total = len(qs['d4_data'] | qs['d4_anc'])
        dec = decompose if decomposed else (lambda g: g)
        gad = {
            'D4': [dec(g) for g in P.round_gadgets('D4')],
            'D4g': [dec(g) for g in P.round_gadgets('D4', 'g')],
            'FSC': P.round_gadgets('FSC'),
        }
        self.round = {}
        for k, g in gad.items():
            ub = round_schedule_classes(g, dur)
            lb = round_lower_bound(g, dur)
            self.round[k] = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'], depth_lb=lb['depth_lb'],
                                 depth_ub=ub['depth_ub'], classes=ub['classes'],
                                 gates=Counter(gate_counts([x for gg in g for x in gg])),
                                 n_anc=len(g))
        self.n_anc_g = len(gad['D4g'])
        U = dec(P.U_gates())
        Ul = class_layers([U])              # all U gates mutually commute (validated)
        self.U = dict(gates=Counter(gate_counts(U)), time=circuit_time(Ul, dur), depth=len(Ul))
        meas, cz = P.switch_out_gates()
        czl = class_layers([cz]) if cz else []
        ml = [meas]
        self.out = dict(gates_worst=Counter(gate_counts(meas + cz)),
                        gates_expected=Counter({**gate_counts(meas), 'CZ': len(cz) / 2}),
                        time=circuit_time(ml + czl, dur), depth=len(ml) + len(czl))
        self.x_corr = dict(time=dur['X'], gates_worst=Counter({'X': self.n['fsc_data']}))
        self.init = Counter({'INIT_X': self.n['fsc_data'], 'INIT_Z': len(qs['d4_data']) - self.n['fsc_data']})


def d4_stages(cd, model, params=DEFAULT_PARAMS, comparison='A', data_only=False):
    d = cd.L
    n = cd.n
    fsc_q = n['fsc_data'] + (0 if data_only else n['fsc_anc'])
    d4_q = n['d4_data'] + (0 if data_only else n['d4_anc'])
    rows = []

    def add(stage, qubits, rounds, rkey, extra_time, gates, source, note=''):
        rt = cd.round[rkey] if rkey else dict(time_lb=0, time_ub=0)
        rows.append(dict(protocol='Kobayashi_D4', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * rt['time_lb'] + extra_time,
                         time_ub=rounds * rt['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    r0 = rounds_for(model, params['kappa0'], d)
    g0 = Counter(cd.init)
    for k, v in cd.round['FSC']['gates'].items():
        g0[k] += v * r0
    g0['X'] += cd.x_corr['gates_worst']['X']
    add('S0 FSC |+_L> preparation', fsc_q, r0, 'FSC', cd.x_corr['time'], g0,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'kappa0*d rounds (not specified in Kobayashi); X-correction worst case counted')
    if model == 'NL_min':
        r1, key = 1, 'D4g'
        q1 = n['d4_data'] + (0 if data_only else cd.n_anc_g)
        note1 = 'noiseless minimum: one round of S^g_X measurements only'
    else:
        r1, key = rounds_for(model, params['kappa_in'], d), 'D4'
        q1 = d4_q
        note1 = 'O(d) rounds of all D4 stabilizers (Kobayashi step 2); kappa_in*d assumed'
    g1 = Counter()
    for k, v in cd.round[key]['gates'].items():
        g1[k] += v * r1
    add('S1 switch-in (gauging) to D4', q1, r1, key, 0, g1,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT, note1)
    add('S2 transversal U = W V (logical T^dag)', n['d4_data'], 0, None, cd.U['time'], cd.U['gates'],
        SOURCE_COUNT, 'constant depth; gates from eq. (8)')
    add('S3 switch-out: Z^g meas + S^g correction', n['d4_data'], 0, None, cd.out['time'],
        cd.out['gates_worst'], SOURCE_COUNT, 'CZ correction counted worst case (expected = half)')
    r4 = 0 if model == 'NL_min' else rounds_for(model, params['kappa_out'], d)
    g4 = Counter()
    for k, v in cd.round['FSC']['gates'].items():
        g4[k] += v * r4
    add('S4 FSC error-correction rounds', fsc_q, r4, 'FSC', 0, g4,
        SOURCE_ASSUME if model == 'FT_ref' else SOURCE_COUNT,
        'O(d) rounds (Kobayashi step 4); kappa_out*d assumed; 0 in the noiseless minimum')
    add('S5 logical X frame: T^dag|+> -> T|+>', 0, 0, None, 0, Counter(), SOURCE_COUNT,
        'Pauli frame update, no physical cost')
    # NOTE (comparison B): the FSC -> standard rotated-patch conversion is UNRESOLVED.  It is deliberately
    # NOT added as a zero-cost stage; it is reported separately by uncounted_costs().
    return rows



def d4_conversion_illustrative(cd, model, params=DEFAULT_PARAMS, data_only=False):
    """ILLUSTRATIVE ONLY: conversion modelled as one lattice-surgery time step (tau) during which the
    FSC plus 2 tiles (target patch + ancilla strip) are active.  Not taken from literature."""
    d = cd.L
    r = rounds_for(model, params['tau_rounds'], d)
    q = cd.n['fsc_data'] + (0 if data_only else cd.n['fsc_anc']) + 2 * (d * d if data_only else 2 * d * d)
    return dict(protocol='Kobayashi_D4', comparison='B_illustrative', model=model, d=d,
                stage='S6 conversion (ILLUSTRATIVE: 1 tau lattice surgery)', qubits=q, rounds=r,
                time_lb=None, time_ub=None, gates={}, source=SOURCE_UNRES,
                note='illustrative assumption, not a validated protocol')

In [ ]:
# ---------------------------------------------------------- consistency checks of a code
def code_consistency(cd):
    """Exact F2 checks of a (possibly non-commuting) Clifford stabilizer code on its Z-subspace C_Z:
      shift_in_CZ : every X-type generator preserves C_Z (its X part commutes with all Z checks)
      noncommuting_pairs : generator pairs that do NOT commute on C_Z
      bad_squares : generators with S^2 != 1 on C_Z
      k : number of logical qubits = dim C_Z - rank(shifts)   (generators have independent shifts)"""
    lat = cd.lat
    HZ, S = cd.HZ, cd.S
    res = {}
    res['shift_in_CZ'] = not ((HZ.astype(int) @ S.T.astype(int)) % 2).any()
    supp = [op.support() for _, op in cd.xgens]
    bad = []
    for i in range(len(cd.xgens)):
        for j in range(i + 1, len(cd.xgens)):
            if not (supp[i] & supp[j]):
                continue
            lin, const = commutator_affine(cd.xgens[i][1], cd.xgens[j][1])
            if const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(lin)):
                bad.append((cd.xgens[i][0], cd.xgens[j][0]))
    res['noncommuting_pairs'] = bad
    sq = []
    for lab, op in cd.xgens:
        s2 = op @ op
        if s2.s or s2.quad or s2.const or not f2_in_rowspace(cd.HZr, cd.HZpiv, lat.vec(s2.lin)):
            sq.append(lab)
    res['bad_squares'] = sq
    res['n_qubits'] = len(cd.cols)
    res['dim_CZ'] = len(cd.cols) - f2_rank(HZ[:, cd.cols])
    res['rank_shifts'] = f2_rank(S[:, cd.cols])
    res['n_xgens'] = len(cd.xgens)
    res['k'] = res['dim_CZ'] - res['rank_shifts']
    res['shifts_independent'] = res['rank_shifts'] == res['n_xgens']
    return res


def U_target(lat, lab):
    """Image of generator `lab` under U predicted by SM eq. (permutingstabilizers):
    S^r -> S^r S^g, S^g -> S^g, S^b -> S^g S^b, and (boundary L_rb) S^r S^b -> S^r S^b."""
    _, v, cols = lab
    if cols == 'rb':
        return lat.SX(v, R) @ lat.SX(v, B)
    op = PSOp()
    for ch in cols:
        c = CNAME.index(ch)
        for cc in ([c] if c == G else [c, G]):
            op = op @ lat.SX(v, cc)
    return op

In [ ]:
# ---------------------------------------------------------- Knill-Laflamme distance lower bound
def _milp_binary(c_obj, A_eq_mod2, rhs_mod2, extra_ub=None, fixed_zero=()):
    """min c.x over binary x subject to A x = rhs (mod 2) (rows of A_eq_mod2), and optional
    linear constraints extra_ub = [(row, ub), ...] (row.x <= ub).
    Returns (value, x) if an optimum was found, (None, None) only if the problem is proven
    infeasible (scipy status 2).  Any other status (time/iteration limit, unbounded, numerical
    failure) raises RuntimeError, so that a solver failure is never mistaken for "no solution"."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    A = np.array(A_eq_mod2, dtype=float)
    m, n = A.shape
    nv = n + m
    c = np.concatenate([np.asarray(c_obj, float), np.zeros(m)])
    cons = [LinearConstraint(np.concatenate([A, -2 * np.eye(m)], axis=1), rhs_mod2, rhs_mod2)]
    for row, ub in (extra_ub or []):
        cons.append(LinearConstraint(np.concatenate([np.asarray(row, float), np.zeros(m)])[None, :], -np.inf, ub))
    ub_ = np.concatenate([np.ones(n), np.full(m, np.inf)])
    for i in fixed_zero:
        ub_[i] = 0
    res = milp(c, constraints=cons, integrality=np.ones(nv), bounds=Bounds(np.zeros(nv), ub_))
    if res.status == 2:
        return None, None
    if res.status != 0:
        raise RuntimeError(f'MILP solver did not finish (status {res.status}: {res.message})')
    return round(res.fun), np.round(res.x[:n]).astype(np.uint8)


def low_weight_stabilizer_shifts(cd, wmax, frozen=()):
    """Exact enumeration (MILP with no-good cuts) of all t in span(S) with 1 <= |t| <= wmax,
    where span(S) = C_Z  intersect  z_log^perp  (k = 1).  Frozen qubits are excluded
    (an X on a frozen qubit violates its single-qubit Z check, so t would leave C_Z).
    Returns (found, stop) where stop records why the enumeration ended:
    'infeasible' (no further t with 1 <= |t| <= wmax) or 'min weight > wmax'."""
    cols = [c for c in cd.cols if c not in set(frozen)]
    H = np.vstack([cd.HZ[:, cols], cd.z_log[cols][None, :]])
    H = H[np.any(H, axis=1)]
    rhs = np.zeros(len(H))
    n = len(cols)
    cuts = [(np.ones(n), wmax), (-np.ones(n), -1)]
    found = []
    while True:
        val, x = _milp_binary(np.ones(n), H, rhs, extra_ub=cuts)
        if val is None:
            stop = 'infeasible'
            break
        if val > wmax:
            stop = 'min weight > wmax'
            break
        sup = np.nonzero(x)[0]
        t = np.zeros(cd.lat.n, dtype=np.uint8)
        t[[cols[i] for i in sup]] = 1
        found.append(t)
        row = -np.ones(n)
        row[sup] = 1
        cuts.append((row, len(sup) - 1))                  # exclude exactly this vector
    return found, stop


def kl_distance_lower_bound(cd, wmax, frozen=()):
    r"""Rigorous lower-bound argument for the Knill-Laflamme distance of the (non-Pauli) code cd.

    For a Pauli E = X^t Z^u (up to phase):
      * t not in C_Z           -> E maps the code space to an orthogonal space: detectable;
      * t in the logical coset -> |t| >= X-shift distance (checked separately);
      * t in span(S)           -> X^t = S_c F_c with S_c a stabilizer and F_c = (-1)^{f_c} diagonal,
        so P E P = P F_c Z^u P.  Expanding (-1)^{f_c} in Z-type Paulis Z^v with v supported on
        Q_f = supp(f_c), a violation of KL requires some v + u to be a logical Z operator l.
        Hence |E| >= |t| + |u \ t| >= |t| + min_l |l \ (Q_f u t)|  =: LB_t.
    Returns (min over enumerated t of LB_t, ts, details, stop) where stop is the termination
    reason of the enumeration (see low_weight_stabilizer_shifts).  t = 0 is covered by d_Z."""
    lat = cd.lat
    ts, stop = low_weight_stabilizer_shifts(cd, wmax, frozen)
    cols = [c for c in cd.cols if c not in set(frozen)]
    Srows = cd.S[:, cols]
    Srows = Srows[np.any(Srows, axis=1)]
    best, details = None, []
    for t in ts:
        coef = f2_solve(cd.S, t)
        op = PSOp()
        for i in np.nonzero(coef)[0]:
            op = op @ cd.xgens[i][1]
        assert op.s == frozenset(np.nonzero(t)[0])
        Qf = {q for pr in op.quad for q in pr} | set(op.lin) | set(np.nonzero(t)[0])
        cost = np.array([0.0 if c in Qf else 1.0 for c in cols])
        A = np.vstack([Srows, cd.t_log[cols][None, :]])
        rhs = np.concatenate([np.zeros(len(Srows)), [1.0]])
        val, _ = _milp_binary(cost, A, rhs)
        assert val is not None                            # a logical Z always exists
        lb = int(t.sum()) + val
        details.append(dict(weight_t=int(t.sum()), Qf_size=len(Qf), LB=lb))
        best = lb if best is None else min(best, lb)
    return best, ts, details, stop

In [ ]:
# ---------------------------------------------------------- full resource tables (used by main + comparison)
def d4_resource_tables(D_LIST, dur=None, params=None, verbose=True):
    """Per-stage and summary tables of the Kobayashi protocol for every d in D_LIST, gate set
    ('primitive' | 'clifford_t'), counting variant (data_only), time model and comparison A/B.
    Returns (stage_df, sum_df, blocks)."""
    import pandas as pd, time
    dur = dur or DUR_UNIT
    params = params or DEFAULT_PARAMS
    records, summaries, blocks = [], [], {}
    for d in D_LIST:
        t0 = time.time()
        for gs in ['primitive', 'clifford_t']:
            cd = D4CostData(d, dur, decomposed=(gs == 'clifford_t'))
            for data_only in [False, True]:
                for model in ['FT_ref', 'NL_min']:
                    for comp in ['A', 'B']:
                        st = d4_stages(cd, model, params, comp, data_only=data_only)
                        for r in st:
                            r.update(gateset=gs, data_only=data_only)
                        records += st
                        s = summarize(st)
                        s.update(protocol='Kobayashi_D4', d=d, model=model, comparison=comp, gateset=gs,
                                 data_only=data_only)
                        summaries.append(s)
            blocks[f'{d}|{gs}'] = dict(d=d, gateset=gs, n=cd.n, n_total=cd.n_total, n_anc_g=cd.n_anc_g,
                                       round={k: {kk: (dict(vv) if isinstance(vv, Counter) else vv) for kk, vv in v.items()}
                                              for k, v in cd.round.items()},
                                       U_time=cd.U['time'], U_gates=dict(cd.U['gates']), out_time=cd.out['time'],
                                       out_gates=dict(cd.out['gates_worst']), xcorr_time=cd.x_corr['time'])
        if verbose:
            print(f'Kobayashi d={d}: {time.time() - t0:.1f}s')
    return jsonify_columns(pd.DataFrame(records)), jsonify_columns(pd.DataFrame(summaries)), blocks

### 雑音付き D4 code switching ライブラリ（回路・雑音モデル・Takada らの Pauli proxy 法による非 Clifford シミュレーション・復号器）

In [ ]:
"""noisy_d4_lib: circuit-level noisy simulation of the Kobayashi-Zhu-Hsin D4 code switching protocol.

Non-Clifford simulation method: Clifford-stabilizer simulation with Pauli proxies
(Y. Takada, S. D. Bartlett, D. J. Williamson, arXiv:2609.16929, Algorithm 2 (Sec. VII D) and Sec. VIII).
For every Monte Carlo shot the noisy non-Clifford circuit is mapped to a Clifford circuit (the
"Pauli-proxy circuit") with the same measurement statistics:
  * every diagonal third-level gate D (CCZ of the D4 gadgets, CS / CS^dag / T^dag of U) is removed;
  * the X component e of the sampled Pauli errors and of applied X corrections, propagated forward
    through the Clifford gates of the true circuit, is tracked ("continuing errors"); where e meets
    D the diagonal Clifford gate C_e = D^dag X^e D X^e (a product of S, Z, CZ) is inserted
    (= the controlled Pauli dressing Lambda(D_U(E)) of Takada Fig. 14 / Eq. (165));
  * Z components commute with every diagonal gate and need no propagation.
All Clifford gates, sampled Pauli errors and inserted dressings are executed by stim's TableauSimulator.

Requires (as globals, embedded before this code): numpy as np, stim, common_lib, d4_lib.
"""
import itertools
import zlib
import numpy as np

In [ ]:
# ------------------------------------------------------------------ gate algebra
DIAG_THETA = {                     # phase polynomials (units of pi/4) of the diagonal 3rd-level gates
    'CCZ': lambda x: 4 * x[0] * x[1] * x[2],
    'NCCZ': lambda x: 4 * x[0] * x[1] * x[2],      # virtual noiseless CCZ (deferred switch-out correction)
    'CS': lambda x: 2 * x[0] * x[1],
    'CSDG': lambda x: 6 * x[0] * x[1],
    'T': lambda x: x[0],
    'TDG': lambda x: 7 * x[0],
}
NONCLIFF = set(DIAG_THETA)
_DRESS = {}


def diag_clifford_gates(delta, k):
    """delta: dict bits -> phase exponent mod 8 (even).  Returns gates ('S'|'Z'|'SDG', i), ('CZ', i, j)
    (local indices) equal to diag(omega^delta) up to a global phase; asserts the result is exact."""
    c0 = delta[(0,) * k]
    a, gates = [], []
    for i in range(k):
        e = tuple(int(j == i) for j in range(k))
        ai = (delta[e] - c0) % 8
        assert ai % 2 == 0, 'dressing is not Clifford'
        a.append(ai // 2)
        if ai // 2:
            gates.append(({1: 'S', 2: 'Z', 3: 'SDG'}[ai // 2], i))
    b = {}
    for i, j in itertools.combinations(range(k), 2):
        e = tuple(int(m in (i, j)) for m in range(k))
        v = (delta[e] - c0 - 2 * a[i] - 2 * a[j]) % 8
        assert v in (0, 4), 'dressing has a non-Clifford quadratic term'
        b[(i, j)] = v // 4
        if v:
            gates.append(('CZ', i, j))
    for x in itertools.product((0, 1), repeat=k):
        pred = c0 + sum(2 * a[i] * x[i] for i in range(k)) + sum(4 * b[ij] * x[ij[0]] * x[ij[1]] for ij in b)
        assert (pred - delta[x]) % 8 == 0, 'dressing has a cubic term'
    return gates


def dressing(name, e):
    """C_e = D^dag X^e D X^e for diagonal gate `name` and X pattern e (tuple of bits), local indices."""
    key = (name, e)
    if key not in _DRESS:
        th = DIAG_THETA[name]
        k = len(e)
        delta = {x: (th(tuple(xi ^ ei for xi, ei in zip(x, e))) - th(x)) % 8
                 for x in itertools.product((0, 1), repeat=k)}
        _DRESS[key] = diag_clifford_gates(delta, k)
    return _DRESS[key]

In [ ]:
# ------------------------------------------------------------------ noise model
class NoiseModel:
    """Standard circuit-level Pauli noise (Takada et al. App. A 2) plus optional idling noise:
    after every k-qubit unitary a k-qubit depolarizing channel (probability p; p_ccz for CCZ, p_nc for
    the gates of U), preparation followed by a flip (p_reset), measurement preceded by a flip (p_meas),
    every idle qubit of a time step depolarized with p_idle."""

    def __init__(self, p, p_meas=None, p_reset=None, p_idle=None, p_ccz=None, p_nc=None):
        self.p = p
        self.p_meas = p if p_meas is None else p_meas
        self.p_reset = p if p_reset is None else p_reset
        self.p_idle = p if p_idle is None else p_idle
        self.p_ccz = p if p_ccz is None else p_ccz
        self.p_nc = p if p_nc is None else p_nc

    def locations(self, ops, idle):
        """Noise locations of one time step: list of (when, kind, qubits, prob), when in {'pre','post'}."""
        loc = []
        for op in ops:
            nm, qs = op[0], op[1:]
            if nm in ('NCCZ', 'MZ0'):
                continue
            if nm == 'FLIPX':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MZ':
                loc.append(('pre', 'X', qs, self.p_meas))
            elif nm == 'MX':
                loc.append(('pre', 'Z', qs, self.p_meas))
            elif nm == 'RZ':
                loc.append(('post', 'X', qs, self.p_reset))
            elif nm == 'RX':
                loc.append(('post', 'Z', qs, self.p_reset))
            elif nm == 'CCZ':
                loc.append(('post', 'D3', qs, self.p_ccz))
            elif nm in ('CS', 'CSDG', 'T', 'TDG'):
                loc.append(('post', 'D%d' % len(qs), qs, self.p_nc))
            else:
                loc.append(('post', 'D%d' % len(qs), qs, self.p))
        for q in idle:
            loc.append(('post', 'D1', (q,), self.p_idle))
        return [l for l in loc if l[3] > 0]


PAULI_CHOICES = {k: [''.join(p) for p in itertools.product('IXYZ', repeat=k) if set(p) != {'I'}]
                 for k in (1, 2, 3)}

In [ ]:
# ------------------------------------------------------------------ parallel FSC schedule
_FSC_SCHED = {}


def _fsc_checks(P):
    """[(type 'X'|'Z', ancilla, label, [data qubits])] of one FSC round (frozen qubits removed)."""
    lat, cd = P.lat, P.fsc
    out = []
    for lab, op in cd.xgens:
        _, v, cols = lab
        qs = []
        for ch in cols:
            cc = CNAME.index(ch)
            qs += sorted({lat.q(e, cc) for e in lat.incident(v)} - P.frozen)
        out.append(('X', P.anc[lab], lab, qs))
    for lab, (l2, rows) in zip(cd.zlabels, lat.z_checks(cd.colours)):
        if P._is_frozen_check(lab):
            continue
        out.append(('Z', P.anc[lab], lab, sorted(set(rows) - P.frozen)))
    return out


def _fsc_anneal(P, checks, T, seed, iters):
    """Assign a time slot 0..T-1 to every (check, data qubit) CNOT such that (i) no ancilla and no data qubit
    is used twice in a slot and (ii) for every X check A and Z check B the number of shared qubits on
    which B's CNOT precedes A's is even (so all checks can be measured at the same time).  Start: the
    direction-based order of the bulk (X stars E,N,S,W = 0,1,2,3; Z plaquettes E,S,N,W = 0,1,2,3, one of
    the 16 compatible bulk orders); simulated annealing fixes the boundary (weight-6 X^r X^b and
    weight-2 Z^r Z^b checks of L_rb)."""
    lat = P.lat
    XO, ZO = {'E': 0, 'W': 3, 'N': 1, 'S': 2}, {'S': 1, 'N': 2, 'W': 3, 'E': 0}
    rng = np.random.default_rng(seed)
    var, slot = [], []
    for ci, (typ, anc, lab, qs) in enumerate(checks):
        for q in qs:
            k, x, y = lat.edges[q // 3]
            if typ == 'X':
                d = ('E' if x == lab[1][0] else 'W') if k == 'h' else ('N' if y == lab[1][1] else 'S')
                s0 = XO[d]
            elif lab[0] == 'plaq':
                d = ('S' if y == lab[1][1] else 'N') if k == 'h' else ('W' if x == lab[1][0] else 'E')
                s0 = ZO[d]
            else:
                s0 = 0
            var.append((ci, q)); slot.append(s0)
    slot = np.array(slot)
    by_check, by_qubit = {}, {}
    for i, (ci, q) in enumerate(var):
        by_check.setdefault(ci, []).append(i); by_qubit.setdefault(q, []).append(i)
    pairs = {}
    for q, idx in by_qubit.items():
        xs = [i for i in idx if checks[var[i][0]][0] == 'X']
        zs = [i for i in idx if checks[var[i][0]][0] == 'Z']
        for a_ in xs:
            for b_ in zs:
                pairs.setdefault((var[a_][0], var[b_][0]), []).append((a_, b_))
    pairs = list(pairs.values())
    pv = {}
    for k, pl in enumerate(pairs):
        for a_, b_ in pl:
            pv.setdefault(a_, []).append(k); pv.setdefault(b_, []).append(k)
    bad = lambda k: sum(slot[b_] < slot[a_] for a_, b_ in pairs[k]) % 2
    conf = lambda i: (sum(slot[j] == slot[i] for j in by_check[var[i][0]] if j != i)
                      + sum(slot[j] == slot[i] for j in by_qubit[var[i][1]] if j != i))
    local = lambda i: conf(i) + sum(bad(k) for k in pv.get(i, []))
    total = lambda: sum(conf(i) for i in range(len(var))) // 2 + sum(bad(k) for k in range(len(pairs)))
    cost, temp = total(), 1.0
    for it in range(iters):
        if cost == 0:
            break
        i = int(rng.integers(len(var)))
        if local(i) == 0 and rng.random() < 0.9:
            continue
        old, before = slot[i], local(i)
        slot[i] = rng.integers(T)
        dlt = local(i) - before
        if dlt <= 0 or rng.random() < np.exp(-dlt / temp):
            cost += dlt
        else:
            slot[i] = old
        if it % 20000 == 0:
            cost, temp = total(), max(0.05, temp * 0.85)
    return var, slot, total()


def fsc_memory_circuit(P, sched, basis, rounds, p):
    """stim memory experiment of the FSC with the parallel schedule (validation): data in |+> (X) or |0>
    (Z), `rounds` rounds, final data measurement; observable = logical X (t_log) or a logical Z.  Noise:
    DEPOLARIZE2(p) after every CNOT, flips p on ancilla reset / before measurement / final data."""
    checks, T, layers = sched['checks'], sched['T'], sched['layers']
    data = sorted({q for typ, anc, lab, qs in checks for q in qs})
    idx = {q: i for i, q in enumerate(data)}
    t_log = [int(q) for q in np.nonzero(P.fsc.t_log)[0]]
    Hx = np.array([[1 if q in qs else 0 for q in data] for typ, anc, lab, qs in checks if typ == 'X'], dtype=np.uint8)
    tv = np.array([1 if q in t_log else 0 for q in data], dtype=np.uint8)
    zl = next([data[i] for i in np.nonzero(v)[0]] for v in f2_nullspace(Hx) if int(v.astype(int) @ tv.astype(int)) % 2)
    xa = [anc for typ, anc, lab, qs in checks if typ == 'X']
    za = [anc for typ, anc, lab, qs in checks if typ == 'Z']
    nX, nZ = len(xa), len(za)
    cir = stim.Circuit()
    cir.append('RX' if basis == 'X' else 'R', data)
    for r in range(rounds):
        cir.append('RX', xa); cir.append('R', za)
        if p:
            cir.append('Z_ERROR', xa, p); cir.append('X_ERROR', za, p)
        cir.append('TICK')
        for t in range(T):
            tg = []
            for typ, anc, q in layers[t]:
                tg += [anc, q] if typ == 'X' else [q, anc]
            if tg:
                cir.append('CX', tg)
                if p:
                    cir.append('DEPOLARIZE2', tg, p)
            cir.append('TICK')
        if p:
            cir.append('Z_ERROR', xa, p); cir.append('X_ERROR', za, p)
        cir.append('MX', xa); cir.append('M', za)
        nm = nX + nZ
        for k in range(nX):
            if r:
                cir.append('DETECTOR', [stim.target_rec(-nm + k), stim.target_rec(-2 * nm + k)])
            elif basis == 'X':
                cir.append('DETECTOR', [stim.target_rec(-nm + k)])
        for k in range(nZ):
            if r:
                cir.append('DETECTOR', [stim.target_rec(-nZ + k), stim.target_rec(-nZ - nm + k)])
            elif basis == 'Z':
                cir.append('DETECTOR', [stim.target_rec(-nZ + k)])
    if p:
        cir.append('Z_ERROR' if basis == 'X' else 'X_ERROR', data, p)
    cir.append('MX' if basis == 'X' else 'M', data)
    nd = len(data)
    off = 0 if basis == 'X' else nX
    for k, (anc, qs) in enumerate([(anc, qs) for typ, anc, lab, qs in checks if typ == basis]):
        cir.append('DETECTOR', [stim.target_rec(-nd + idx[q]) for q in qs] + [stim.target_rec(-nd - (nX + nZ) + off + k)])
    cir.append('OBSERVABLE_INCLUDE', [stim.target_rec(-nd + idx[q]) for q in (t_log if basis == 'X' else zl)], 0)
    return cir


def fsc_parallel_schedule(P, T=6, max_seeds=30, iters=3000000):
    """Measure all FSC checks at the same time in T CNOT layers (round = T + 2 time steps; T = 6 is the
    minimum because the X^r X^b checks of L_rb have weight 6; the bulk alone would need 4).  The slot
    assignment is searched (deterministic seeds) and ACCEPTED ONLY IF the stim memory experiments in both
    bases are deterministic without noise and have circuit-level distance >= L (no harmful hook errors).
    Cached per L.  Returns dict(T, checks, layers[t] = [(type, ancilla, data qubit)], seed, distance)."""
    L = P.L
    if L in _FSC_SCHED:
        return _FSC_SCHED[L]
    checks = _fsc_checks(P)
    for seed in range(max_seeds):
        var, slot, cost = _fsc_anneal(P, checks, T, seed, iters)
        if cost:
            continue
        layers = [[] for _ in range(T)]
        for i, (ci, q) in enumerate(var):
            layers[int(slot[i])].append((checks[ci][0], checks[ci][1], q))
        sch = dict(T=T, checks=checks, layers=layers, seed=seed)
        ok, dist = True, {}
        for basis in 'XZ':
            s0 = fsc_memory_circuit(P, sch, basis, 3, 0.0).compile_detector_sampler(seed=1).sample(20, append_observables=True)
            c1 = fsc_memory_circuit(P, sch, basis, max(L, 2), 1e-3)
            dist[basis] = len(c1.detector_error_model(decompose_errors=True).shortest_graphlike_error())
            ok &= (not s0.any()) and dist[basis] >= L
        if ok:
            sch['distance'] = dist
            _FSC_SCHED[L] = sch
            return sch
    raise RuntimeError(f'no valid parallel FSC schedule found for L={L}')

In [ ]:
# ------------------------------------------------------------------ the noisy protocol circuit
class D4SwitchingCircuit:
    """Time-step level description of the noisy KZH protocol on the bounded square lattice.

    Blocks (in order):
      ('S0', r)  r = 1..R0   FSC |+_L> preparation: data |+>, rounds of all FSC checks
      ('S1', r)  r = 1..Rin  switch-in: g |0>, rounds of all D4 stabilizers ([KZH] step 2)
      ('U',)                 transversal U = W V ([KZH] eq. (8), step 3)
      ('SO',)                switch-out: Z^g readout ([KZH] step 4; deferred-correction form)
      ('S4', r)  r = 1..Rout FSC rounds after switching back ([KZH] step 4)
      ('FIN',)               one noiseless FSC round (fictitious error correction, Takada Sec. VI D)
    Qubits are re-indexed compactly: data (non-frozen r,g,b) first, then ancillas.
    sequential=True measures the gadgets of a commuting class one after another (validation only)."""

    def __init__(self, L, R0, Rin, Rout, sequential=False, n_pre_u=2, fsc_schedule='classes'):
        self.n_pre_u = n_pre_u
        P = D4Protocol(L)
        lat = P.lat
        self.P, self.lat, self.L = P, lat, L
        self.R0, self.Rin, self.Rout = R0, Rin, Rout
        data = sorted(P.data_qubits((R, G, B)))
        gadgets = {'D4': P.round_gadgets('D4'), 'FSC': P.round_gadgets('FSC')}
        ancs = sorted({g[0][1] for k in gadgets for g in gadgets[k]})
        c = {q: i for i, q in enumerate(data + ancs)}
        self.c = c
        self.nq, self.nd = len(data) + len(ancs), len(data)
        self.data_rb = sorted(c[q] for q in P.data_qubits((R, B)))
        self.data_g = sorted(c[q] for q in P.data_qubits((G,)))
        self.data_all = sorted(c[q] for q in data)
        conv = {'INIT_X': 'RX', 'INIT_Z': 'RZ', 'MEAS_X': 'MX', 'MEAS_Z': 'MZ', 'CNOT': 'CX'}
        cv = lambda g: (conv.get(g[0], g[0]),) + tuple(c[q] for q in g[1:])
        # label of every ancilla, per code
        self.lab_of = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            labs = [l for l, _ in cd.xgens] + [l for l in cd.zlabels if not P._is_frozen_check(l)]
            for l in labs:
                self.lab_of[(code, c[P.anc[l]])] = l

        def round_moments(code, which='all'):
            gl = gadgets[code]
            if which != 'all':
                gl = [g for g in gl if g[0][0] == ('INIT_Z' if which == 'Z' else 'INIT_X')]
            classes = commuting_classes(gl)
            if sequential:
                classes = [[g] for cls in classes for g in cls]
            moms = []
            for cls in classes:
                for layer in class_layers(cls):
                    moms.append([cv(g) for g in layer])
            return moms
        # D4 rounds: Pauli Z checks first, then (after the real-time flux correction) the X-type
        # Clifford stabilizers, so that fluxes detected by the Z checks are removed before they meet CCZs
        if fsc_schedule == 'parallel' and not sequential:
            # all FSC checks at once: ancilla init, 6 CNOT layers (verified order, fsc_parallel_schedule), measure
            fs = fsc_parallel_schedule(P)
            moms = [[('RX' if typ == 'X' else 'RZ', c[anc]) for typ, anc, lab, qs in fs['checks']]]
            for t in range(fs['T']):
                moms.append([('CX', c[anc], c[q]) if typ == 'X' else ('CX', c[q], c[anc])
                             for (typ, anc, q) in fs['layers'][t]])
            moms.append([('MX' if typ == 'X' else 'MZ', c[anc]) for typ, anc, lab, qs in fs['checks']])
            fsc_moms = moms
        else:
            # default: X class then Z class (14 steps).  The verified 8-step parallel schedule gives a larger
            # output error with OUR flux decoder (its phenomenological error model does not represent the
            # hook errors that the interleaving splits across rounds), so it is used only as a sensitivity
            fsc_moms = round_moments('FSC')
        self.round_moms = {'FSC': fsc_moms, 'D4z': round_moments('D4', 'Z'), 'D4x': round_moments('D4', 'X')}
        self.anc = {code: sorted({c[g[0][1]] for g in gadgets[code]}) for code in gadgets}
        self.U_moms = [[cv(g) for g in layer] for layer in class_layers([P.U_gates()])]
        # ---- switch-out data: S^g generators, Gamma (g configuration -> kappa), CZ^{r,b} pairs
        self.gens_g = [(lab, op) for lab, op in P.d4.xgens if lab[2] == 'g']
        gpos = {q: i for i, q in enumerate(self.data_g)}
        Sg = np.zeros((len(self.gens_g), len(self.data_g)), dtype=np.uint8)
        for i, (lab, op) in enumerate(self.gens_g):
            for q in op.s:
                if q in c and c[q] in gpos:
                    Sg[i, gpos[c[q]]] = 1
        self.Sg = Sg
        self.Gamma = left_inverse_T(Sg)
        self.gpairs = [[(c[i], c[j]) for (i, j) in op.quad if i in c and j in c] for lab, op in self.gens_g]
        # physical switch-out correction CZ(Gamma(a_dec)) ([KZH] step 4): the CZ pairs of the selected S^g
        # (pairs occurring an even number of times cancel) are applied in a FIXED schedule: a greedy edge
        # colouring of the union of all pairs; colour l = time step l.  The number of steps does not depend
        # on the outcome (worst case, used for the time count); empty steps are still idle steps.
        allpairs = sorted({tuple(sorted(pr)) for prs in self.gpairs for pr in prs})
        colour, used = {}, {}
        for pr in allpairs:
            k = 0
            while any(k in used.get(q, ()) for q in pr):
                k += 1
            colour[pr] = k
            for q in pr:
                used.setdefault(q, set()).add(k)
        self.so_colour = colour
        self.so_steps = 1 + (max(colour.values()) + 1 if colour else 0)    # g readout + CZ layers
        self.network = [('NCCZ', self.data_g[e], i, j)
                        for v in range(len(self.gens_g)) for e in np.nonzero(self.Gamma[v])[0]
                        for (i, j) in self.gpairs[v]]
        # Pauli-frame strings Z^r Z^b for the S^g eigenvalues (protocol step 5 of the noiseless check V7)
        self.frame_strings = []
        for lab, op in self.gens_g:
            x0, y0 = lab[1]
            qs = [lat.q(lat.eidx[('v', x0, y)], cc) for y in range(y0) for cc in (R, B)]
            self.frame_strings.append(sorted(c[q] for q in qs if q in c))
        # ---- Z checks of the FSC (rb data, frozen checks removed) -> used for the S1 X frame
        self.zchecks = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            rows = []
            for lab, (l2, qs) in zip(cd.zlabels, lat.z_checks(cd.colours)):
                if P._is_frozen_check(lab):
                    continue
                rows.append((lab, sorted(c[q] for q in qs if q in c)))
            self.zchecks[code] = rows
        self.xgens = {}
        for code, cd in (('D4', P.d4), ('FSC', P.fsc)):
            self.xgens[code] = [(lab, sorted(c[q] for q in op.s if q in c),
                                 [(c[i], c[j]) for (i, j) in op.quad if i in c and j in c]) for lab, op in cd.xgens]
        self._logical()
        self.blocks = self._blocks()

    # ------------------------------------------------------------ logical Clifford stabilizer
    def _logical(self):
        """Ideal output = D_F |+_L>_FSC with F(a) = w(a) + 4 sum_v c'_v(a) f^g_v(a) (protocol phase,
        validated in V7/V7b/V8).  Its logical Clifford stabilizer is H = D_F X^t D_F^dag = X^t D_Delta,
        Delta(x) = F(x+t) - F(x) = c0 + sum 2 a_i x_i + sum 4 b_ij x_i x_j (mod 8)."""
        P, lat, c = self.P, self.lat, self.c
        F = {}

        def add(mono, coef):
            m = frozenset(mono)
            if any(q not in c for q in m):
                return                                   # frozen qubits are |0>
            m = frozenset(c[q] for q in m)
            F[m] = (F.get(m, 0) + coef) % 8
        for coef, mono in W_operator(lat).terms:
            add(mono, coef)
        # c'_v(a): path sums along a BFS tree from the L_rb vertices (linear in a)
        from collections import deque
        root = lrb_vertices(lat)
        path = {v: [] for v in root}
        dq = deque(root)
        while dq:
            u = dq.popleft()
            for e in lat.incident(u):
                p, q = lat.endpoints(e)
                w = q if p == u else p
                if w not in path:
                    path[w] = path[u] + [e]
                    dq.append(w)
        for v in P.gverts:
            lin = [lat.q(e, cc) for e in path[v] for cc in (R, B)]
            for (i, j) in lat.cz_pairs(v, G):
                for k in lin:
                    add({k, i, j}, 4)
        self.Fpoly = {m: v for m, v in F.items() if v}
        t = sorted(c[int(q)] for q in np.nonzero(P.fsc.t_log)[0])
        self.t_log = t
        tset = set(t)
        D = {}
        for m, coef in self.Fpoly.items():           # F(x+t): substitute x_i -> 1 - x_i for i in t
            fixed = [q for q in m if q not in tset]
            flip = [q for q in m if q in tset]
            for sub in itertools.product((0, 1), repeat=len(flip)):
                mono = frozenset(fixed + [q for q, s in zip(flip, sub) if s])
                sign = (-1) ** sum(sub)
                D[mono] = (D.get(mono, 0) + coef * sign) % 8
        for m, coef in self.Fpoly.items():
            D[m] = (D.get(m, 0) - coef) % 8
        D = {m: v for m, v in D.items() if v}
        self.Delta_c0 = D.pop(frozenset(), 0)
        self.Delta_a, self.Delta_b = {}, {}
        for m, v in D.items():
            if len(m) == 1:
                assert v % 2 == 0, 'logical stabilizer not Clifford (odd linear term)'
                self.Delta_a[next(iter(m))] = v // 2 % 4
            elif len(m) == 2:
                assert v % 4 == 0, 'logical stabilizer not Clifford (quadratic term)'
                self.Delta_b[tuple(sorted(m))] = v // 4 % 2
            else:
                raise AssertionError('logical stabilizer not Clifford (cubic term)')
        self.Delta_a = {q: a for q, a in self.Delta_a.items() if a}
        self.Delta_b = {ij: b for ij, b in self.Delta_b.items() if b}

    def logical_proxy(self, fx, sign=1):
        """stim.PauliString of the Pauli proxy of the dressed logical stabilizer X^e H X^e
        (e = X frame fx on the data): proxy(X^t D_Delta R_e) = X^t * prod_{a_i=2} Z_i * R_e, where
        R_e = D_Delta^dag X^e D_Delta X^e = phase * Z-string (Takada Secs. VI D, VII B)."""
        n = self.nq
        ps = stim.PauliString(n)
        for q in self.t_log:
            ps[q] = 'X'
        zs = stim.PauliString(n)
        for q, a in self.Delta_a.items():
            if a == 2:
                zs[q] = 'Z'
        R = stim.PauliString(n)
        ph = 1
        for q, a in self.Delta_a.items():
            if fx[q] and a % 2 == 1:
                ph *= 1j ** a
                R *= _single(n, q, 'Z')
            elif fx[q] and a == 2:
                ph *= -1
        for (i, j), b in self.Delta_b.items():
            if fx[i] and fx[j]:
                ph *= -1
                R *= _single(n, i, 'Z') * _single(n, j, 'Z')
            elif fx[i]:
                R *= _single(n, j, 'Z')
            elif fx[j]:
                R *= _single(n, i, 'Z')
        out = ps * zs * R
        return out * complex(ph * sign)

    # ------------------------------------------------------------ blocks
    def _blocks(self):
        B_ = []
        rb, g, alld = self.data_rb, self.data_g, self.data_all
        for r in range(1, self.R0 + 1):
            pre = [[('RX', q) for q in rb]] if r == 1 else []
            B_.append(dict(name=('S0', r), code='FSC', moms=pre + self.round_moms['FSC'], data=rb))
        for r in range(1, self.Rin + 1 + self.n_pre_u):
            pre = [[('RZ', q) for q in g]] if r == 1 else []
            B_.append(dict(name=('S1', r, 'z'), code='D4', moms=pre + self.round_moms['D4z'], data=alld))
            if r <= self.Rin:            # rounds Rin+1.. have only the Z checks (flux checks before U)
                B_.append(dict(name=('S1', r, 'x'), code='D4', moms=self.round_moms['D4x'], data=alld))
        B_.append(dict(name=('U',), code=None, moms=self.U_moms, data=alld))
        B_.append(dict(name=('SO',), code=None,
                       moms=[[('FLIPX', q) for q in g], list(self.network), [('MZ0', q) for q in g]], data=alld))
        for r in range(1, self.Rout + 1):
            B_.append(dict(name=('S4', r), code='FSC', moms=self.round_moms['FSC'], data=rb))
        B_.append(dict(name=('FIN',), code='FSC', moms=self.round_moms['FSC'], data=rb, noiseless=True))
        return B_


def _single(n, q, p):
    s = stim.PauliString(n)
    s[q] = p
    return s


def left_inverse_T(S):
    """Gamma (m x n) over F2 with Gamma @ S^T = I_m, for S (m x n) of full row rank."""
    m, n = S.shape
    A = (S.T % 2).astype(np.uint8)
    M = np.concatenate([A, np.eye(n, dtype=np.uint8)], axis=1)
    r = 0
    for col in range(m):
        nz = np.nonzero(M[r:, col])[0]
        assert len(nz), 'S^T is not of full column rank'
        p = r + nz[0]
        M[[r, p]] = M[[p, r]]
        for rr in range(n):
            if rr != r and M[rr, col]:
                M[rr] ^= M[r]
        r += 1
    Gm = M[:m, m:]
    assert not ((Gm.astype(int) @ A.astype(int)) % 2 - np.eye(m, dtype=int)).any()
    return Gm

In [ ]:
# ------------------------------------------------------------------ exact reference (validation only)
class TrueSim:
    """Exact sparse state-vector simulation of the TRUE (non-Clifford) circuit, used to validate the
    Pauli-proxy simulation.  Basis states are uint64 bit masks: data qubits 0..nd-1, ancillas get a
    slot nd+k while they are alive (reset ... measurement)."""
    OMEGA = np.exp(1j * np.pi / 4)

    def __init__(self, nd, keys, amps):
        self.nd = nd
        self.keys = np.asarray(keys, dtype=np.uint64)
        self.amps = np.asarray(amps, dtype=complex)
        self.slot = {}
        self.free = []
        self.nslots = 0

    def bit(self, q):
        if q < self.nd:
            return q
        if q not in self.slot:
            raise KeyError(q)
        return self.slot[q]

    def _alloc(self, q):
        if q < self.nd:
            return q
        if q in self.slot:
            return self.slot[q]
        b = self.free.pop() if self.free else self.nd + self.nslots
        if b == self.nd + self.nslots:
            self.nslots += 1
        self.slot[q] = b
        assert b < 64
        return b

    def _bits(self, b):
        return ((self.keys >> np.uint64(b)) & np.uint64(1)).astype(np.int64)

    def _merge(self):
        u, inv = np.unique(self.keys, return_inverse=True)
        re = np.bincount(inv, weights=self.amps.real, minlength=len(u))
        im = np.bincount(inv, weights=self.amps.imag, minlength=len(u))
        a = re + 1j * im
        keep = np.abs(a) > 1e-12
        self.keys, self.amps = u[keep], a[keep]

    def X(self, q):
        self.keys = self.keys ^ np.uint64(1 << self.bit(q))

    def Z(self, q):
        self.amps = self.amps * (1 - 2 * self._bits(self.bit(q)))

    def Y(self, q):
        self.Z(q)
        self.X(q)
        self.amps = self.amps * 1j

    def H(self, q):
        b = self.bit(q)
        m = np.uint64(1 << b)
        v = self._bits(b)
        k0 = self.keys & ~m
        a = self.amps / np.sqrt(2)
        self.keys = np.concatenate([k0, k0 | m])
        self.amps = np.concatenate([a, a * (1 - 2 * v)])
        self._merge()

    def diag(self, qs, theta):
        bits = [self._bits(self.bit(q)) for q in qs]
        ph = np.array([theta(t) for t in zip(*bits)], dtype=float) if len(self.keys) else np.zeros(0)
        self.amps = self.amps * self.OMEGA ** ph

    def CX(self, c, t):
        vc = self._bits(self.bit(c)).astype(np.uint64)
        self.keys = self.keys ^ (vc << np.uint64(self.bit(t)))

    def measure_z(self, q, forced):
        b = self.bit(q)
        v = self._bits(b)
        p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
        prob = p1 if forced else 1 - p1
        keep = v == forced
        self.keys, self.amps = self.keys[keep], self.amps[keep]
        if prob > 1e-12:
            self.amps = self.amps / np.sqrt(prob)
        return prob

    def release(self, q, outcome):
        if q >= self.nd:
            if outcome:
                self.X(q)
            self.free.append(self.slot.pop(q))

    def reset(self, q, basis, rng):
        b = self._alloc(q)
        if q < self.nd:                      # data reset: measure and flip to |0>
            v = self._bits(b)
            p1 = float(np.sum(np.abs(self.amps[v == 1]) ** 2))
            o = int(rng.random() < p1)
            self.measure_z(q, o)
            if o:
                self.X(q)
        if basis == 'X':
            self.H(q)

    def run_moment(self, mom, errs, outcomes, rng):
        """Execute a time step with the given sampled errors; measurements forced to `outcomes`
        (list in the order of the moment's measurements).  Returns list of probabilities."""
        for when, ps, qs in errs:
            if when == 'pre':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        probs = []
        k = 0
        for op in mom:
            nm, qs = op[0], op[1:]
            if nm in DIAG_THETA:
                self.diag(qs, DIAG_THETA[nm])
            elif nm == 'CX':
                self.CX(*qs)
            elif nm == 'CZ':
                self.diag(qs, lambda x: 4 * x[0] * x[1])
            elif nm in ('S', 'SDG'):
                self.diag(qs, (lambda x: 2 * x[0]) if nm == 'S' else (lambda x: 6 * x[0]))
            elif nm in ('X', 'Z', 'Y'):
                getattr(self, nm)(qs[0])
            elif nm in ('RX', 'RZ'):
                self.reset(qs[0], nm[1], rng)
            elif nm in ('MX', 'MZ', 'MZ0'):
                q = qs[0]
                if nm == 'MX':
                    self.H(q)
                o = outcomes[k]
                k += 1
                probs.append(self.measure_z(q, o))
                if q >= self.nd:
                    self.release(q, o)
            elif nm == 'FLIPX':
                pass
            else:
                raise ValueError(nm)
        for when, ps, qs in errs:
            if when == 'post':
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(self, pc)(q)
        return probs

    def expect_logical(self, circ):
        """<Psi| X^t D_Delta |Psi> on the data qubits (all ancillas released)."""
        assert not self.slot
        x = self.keys
        ph = np.full(len(x), circ.Delta_c0, dtype=float)
        bits = {}
        get = lambda q: bits.setdefault(q, ((x >> np.uint64(q)) & np.uint64(1)).astype(np.int64))
        for q, a in circ.Delta_a.items():
            ph += 2 * a * get(q)
        for (i, j), b in circ.Delta_b.items():
            ph += 4 * b * get(i) * get(j)
        Da = self.amps * self.OMEGA ** ph
        tm = np.uint64(sum(1 << q for q in circ.t_log))
        k2 = x ^ tm
        order = np.argsort(x)
        xs = x[order]
        pos = np.searchsorted(xs, k2)
        pos[pos >= len(xs)] = 0
        hit = xs[pos] == k2
        amp_t = np.zeros(len(x), dtype=complex)
        amp_t[hit] = self.amps[order][pos[hit]]
        return complex(np.sum(np.conj(amp_t) * Da))

In [ ]:
# ------------------------------------------------------------------ compiled protocol (fast Monte Carlo)
_KIND = {'X': 0, 'Z': 1, 'D1': 2, 'D2': 3, 'D3': 4}
_STIM1 = {'H': 'H', 'S': 'S', 'SDG': 'S_DAG', 'X': 'X', 'Y': 'Y', 'Z': 'Z'}     # single-qubit Cliffords -> stim


class CompiledMoment:
    __slots__ = ('ops', 'base', 'meas', 'cx_c', 'cx_t', 'resets', 'clears', 'nonc', 'loc_slice')


class CompiledProtocol:
    """Pre-compiled time steps of a D4SwitchingCircuit + flattened noise-location table."""

    def __init__(self, circ, noise):
        self.circ, self.noise = circ, noise
        locs = []
        self.blocks = []
        live = set()
        for blk in circ.blocks:
            moms = []
            for mom in blk['moms']:
                cm = CompiledMoment()
                cm.ops = mom
                lines, meas, cxc, cxt, rs, cl, nc = [], [], [], [], [], [], []
                for op in mom:
                    nm, qs = op[0], op[1:]
                    if nm in NONCLIFF:
                        nc.append((nm, qs))
                    elif nm == 'FLIPX':
                        pass
                    elif nm == 'CX':
                        lines.append('CX %d %d' % qs)
                        cxc.append(qs[0]); cxt.append(qs[1])
                    elif nm == 'CZ':
                        lines.append('CZ %d %d' % qs)
                    elif nm in ('RX', 'RZ'):
                        lines.append(('RX %d' if nm == 'RX' else 'R %d') % qs[0])
                        rs.append(qs[0])
                    elif nm in ('MX', 'MZ', 'MZ0'):
                        lines.append(('MX %d' if nm == 'MX' else 'M %d') % qs[0])
                        meas.append((qs[0], 'X' if nm == 'MX' else 'Z'))
                        if nm == 'MZ0':
                            cl.append(qs[0])
                    else:
                        lines.append('%s %d' % (_STIM1[nm], qs[0]))
                cm.base = stim.Circuit('\n'.join(lines)) if lines else None
                cm.meas = meas
                cm.cx_c, cm.cx_t = np.array(cxc, dtype=np.int64), np.array(cxt, dtype=np.int64)
                cm.resets, cm.clears = np.array(rs, dtype=np.int64), np.array(cl, dtype=np.int64)
                cm.nonc = nc
                touched = {q for op in mom for q in op[1:]}
                idle = [q for q in blk['data'] if q not in touched] + [q for q in live if q not in touched]
                if all(op[0] in ('NCCZ', 'FLIPX') for op in mom):
                    idle = []          # virtual moment of the deferred switch-out (no physical time step)
                ml = [] if blk.get('noiseless') else noise.locations(mom, idle)
                s0 = len(locs)
                locs += ml
                cm.loc_slice = (s0, len(locs))
                for op in mom:
                    if op[0] in ('RX', 'RZ') and op[1] >= circ.nd:
                        live.add(op[1])
                    if op[0] in ('MX', 'MZ'):
                        live.discard(op[1])
                moms.append(cm)
            self.blocks.append(dict(name=blk['name'], code=blk['code'], moms=moms))
        self.loc_when = np.array([0 if l[0] == 'pre' else 1 for l in locs], dtype=np.int8)
        self.loc_kind = np.array([_KIND[l[1]] for l in locs], dtype=np.int8)
        self.loc_p = np.array([l[3] for l in locs])
        self.loc_q = [l[2] for l in locs]
        self.n_loc = len(locs)


class FastShot:
    """One Monte Carlo shot of the Pauli-proxy circuit (same semantics as ProxyRun, compiled)."""
    _G1 = {'S': 's', 'SDG': 's_dag', 'Z': 'z'}

    def __init__(self, comp, rng, forced=None):
        self.comp, self.rng, self.circ = comp, rng, comp.circ
        self.sim = stim.TableauSimulator(seed=int(rng.integers(2 ** 62)))
        self.sim.set_num_qubits(comp.circ.nq)
        self.fx = np.zeros(comp.circ.nq, dtype=np.uint8)
        if forced is not None:                  # {location index: pauli string}
            self.errs = {i: (comp.loc_when[i], ps, comp.loc_q[i]) for i, ps in forced.items()}
            self.n_err = len(forced)
            self.n_dress = 0
            return
        hit = np.nonzero(rng.random(comp.n_loc) < comp.loc_p)[0]
        self.errs = {}
        for i in hit:
            k = comp.loc_kind[i]
            qs = comp.loc_q[i]
            if k == 0:
                ps = 'X'
            elif k == 1:
                ps = 'Z'
            else:
                ch = PAULI_CHOICES[len(qs)]
                ps = ch[rng.integers(len(ch))]
            self.errs[i] = (comp.loc_when[i], ps, qs)
        self.n_err = len(hit)
        self.n_dress = 0

    def _pauli(self, ps, qs):
        sim, fx = self.sim, self.fx
        for pc, q in zip(ps, qs):
            if pc == 'X':
                sim.x(q); fx[q] ^= 1
            elif pc == 'Y':
                sim.y(q); fx[q] ^= 1
            elif pc == 'Z':
                sim.z(q)

    def run_block(self, bi):
        """Run block bi; returns list of (qubit, basis, outcome) of its measurements."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        nmeas = 0
        meas = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs] if b > a and self.errs else []
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        if gt[0] == 'CZ':
                            sim.cz(qs[gt[1]], qs[gt[2]])
                        else:
                            getattr(sim, self._G1[gt[0]])(qs[gt[1]])
                        self.n_dress += 1
            if cm.base is not None:
                sim.do_circuit(cm.base)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            nmeas += len(cm.meas)
            meas += cm.meas
        if not nmeas:
            return []
        rec = sim.current_measurement_record()[-nmeas:]
        return [(q, bb, int(o)) for (q, bb), o in zip(meas, rec)]

    def apply_x(self, qubits):
        for q in qubits:
            self.sim.x(q)
            self.fx[q] ^= 1

    def init_frame(self, zrows, qubits):
        """X frame at the start of the first non-Clifford block (S1).  S0 is a Clifford circuit, so stim's
        state is the true state E|ideal>, |ideal> = FSC|+_L> (all checks +1); the X part of E is any
        solution x of H_Z x = s (s = current Z-check signs): solutions differ by X stabilizers / logical X
        of |ideal>, which act trivially on it, and Algorithm 2 is exact for any such representation."""
        s = []
        for lab, qs in zrows:
            ps = stim.PauliString(self.circ.nq)
            for q in qs:
                ps[q] = 'Z'
            v = self.sim.peek_observable_expectation(ps)
            assert v in (1, -1), 'Z check not deterministic at the end of S0'
            s.append(0 if v == 1 else 1)
        H = np.zeros((len(zrows), len(qubits)), dtype=np.uint8)
        pos = {q: i for i, q in enumerate(qubits)}
        for r, (lab, qs) in enumerate(zrows):
            for q in qs:
                H[r, pos[q]] = 1
        x = f2_solve(H.T, np.array(s, dtype=np.uint8))      # x H^T = s  <=>  H x = s
        assert x is not None
        self.fx[:] = 0
        for i in np.nonzero(x)[0]:
            self.fx[qubits[i]] = 1
        return np.array(s)

    _G = {'CX': 'cx', 'CZ': 'cz', 'S': 's', 'SDG': 's_dag', 'Z': 'z', 'X': 'x', 'Y': 'y'}

    def run_block_lockstep(self, bi, ts, rng):
        """Validation: same as run_block, but op by op, with the exact simulation `ts` (TrueSim) of the
        TRUE non-Clifford circuit run in lockstep with the same errors.  Before every measurement stim's
        peek value (+1/-1 deterministic, 0 random) is compared with the exact probability of the outcome.
        Returns list of (qubit, basis, outcome, peek, exact probability)."""
        blk = self.comp.blocks[bi]
        sim, fx = self.sim, self.fx
        out = []
        for cm in blk['moms']:
            a, b = cm.loc_slice
            errs = [self.errs[i] for i in range(a, b) if i in self.errs]
            terrs = [('pre' if w == 0 else 'post', ps, qs) for w, ps, qs in errs]
            for w, ps, qs in errs:
                if w == 0:
                    self._pauli(ps, qs)
            for nm, qs in cm.nonc:
                e = tuple(int(fx[q]) for q in qs)
                if any(e):
                    for gt in dressing(nm, e):
                        getattr(sim, self._G[gt[0]])(*[qs[i] for i in gt[1:]])
                        self.n_dress += 1
            outs, peeks = [], []
            for op in cm.ops:
                nm, qs = op[0], op[1:]
                if nm in NONCLIFF or nm == 'FLIPX':
                    continue
                if nm in ('RX', 'RZ'):
                    (sim.reset_x if nm == 'RX' else sim.reset_z)(qs[0])
                elif nm in ('MX', 'MZ', 'MZ0'):
                    q = qs[0]
                    if nm == 'MX':
                        peeks.append(sim.peek_x(q)); sim.h(q); o = sim.measure(q); sim.h(q)
                    else:
                        peeks.append(sim.peek_z(q)); o = sim.measure(q)
                    outs.append(int(o))
                else:
                    getattr(sim, self._G[nm])(*qs)
            if len(cm.cx_c):
                fx[cm.cx_t] ^= fx[cm.cx_c]
            if len(cm.resets):
                fx[cm.resets] = 0
            if len(cm.clears):
                fx[cm.clears] = 0
            for w, ps, qs in errs:
                if w == 1:
                    self._pauli(ps, qs)
            probs = ts.run_moment(cm.ops, terrs, outs, rng)
            out += [(q, bb, o, pk, pr) for (q, bb), o, pk, pr in zip(cm.meas, outs, peeks, probs)]
        return out

In [ ]:
# ------------------------------------------------------------------ decoders
def _wt(p):
    p = min(max(p, 1e-12), 0.5 - 1e-9)
    return float(np.log((1 - p) / p))


_XFRAC = {0: 1.0, 1: 0.0, 2: 2 / 3, 3: 8 / 15, 4: 32 / 63}       # prob. that a location puts X/Y on a qubit


class FluxDecoder:
    """Real-time ("just-in-time") decoding of the Pauli Z checks (non-Abelian fluxes = X errors).

    [KZH] step 2 pairs the fluxes "in real time ... whenever the observation time of the syndromes is
    comparable to the separation of the syndromes" with the decoder of [Dav] (not accessible here).
    Implementation used in this study (own construction, documented as such): after every round t the
    space-time matching graph of all rounds <= t (with an open time boundary after round t) is decoded
    by minimum-weight perfect matching (PyMatching); the space-like edges whose round is <= t - delay
    are committed and the corresponding X corrections are applied physically at once (feed-forward).
    Before the first non-Clifford block (end of S0) and before U (end of S1) everything is committed.
    The g readout of the switch-out is decoded together with the g fluxes of S1 and gives the
    corrected configuration used for kappa."""

    def __init__(self, comp, delay=0, gauge_w=0.05):
        circ = comp.circ
        self.comp, self.circ, self.delay, self.gauge_w = comp, circ, delay, gauge_w
        zr = {code: circ.zchecks[code] for code in ('D4', 'FSC')}
        g_rows = [(lab, qs) for lab, qs in zr['D4'] if lab[0] == 'plaq' and lab[2] == 'g']
        # flux rounds
        self.fr = []
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                self.fr.append(dict(block=bi, name=nm, rows=zr[blk['code']]))
            elif nm[0] == 'SO':
                self.fr.append(dict(block=bi, name=nm, rows=g_rows))
        off = 0
        for k, f in enumerate(self.fr):
            f['off'] = off
            f['idx'] = {lab: off + i for i, (lab, qs) in enumerate(f['rows'])}
            off += len(f['rows'])
        self.n_nodes = off
        # relation for the SO detectors: prod_{q in g-plaq} Z^g Z^r Z^b (pre-V) = sum of D4 rows
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        d4rows = zr['D4']
        H = np.zeros((len(d4rows), circ.nq), dtype=np.uint8)
        for i, (lab, qs) in enumerate(d4rows):
            H[i, qs] = 1
        self.so_rel = {}
        for lab, qs in g_rows:
            v = np.zeros(circ.nq, dtype=np.uint8)
            for q in qs:
                e = inv[q] // 3
                for cc in (R, G, B):
                    raw = lat.q(e, cc)
                    if raw in c:
                        v[c[raw]] ^= 1
            sol = f2_solve(H, v)
            assert sol is not None, 'SO relation not in the D4 Z-check span'
            self.so_rel[lab] = [d4rows[i][0] for i in np.nonzero(sol)[0]]
        # rows containing each qubit, per code
        self.rows_of = {code: {} for code in ('D4', 'FSC')}
        for code in ('D4', 'FSC'):
            for lab, qs in zr[code]:
                for q in qs:
                    self.rows_of[code].setdefault(q, []).append(lab)
        self._error_rates()
        self._build()

    def _error_rates(self):
        """Phenomenological edge probabilities estimated from the circuit-level noise locations."""
        comp = self.comp
        self.p_data = []           # per flux round: dict qubit -> prob of X/Y since the previous flux round
        self.p_meas = []           # per flux round: dict label -> prob that the check outcome is flipped
        prev_block = -1
        for f in self.fr:
            blk = comp.blocks[f['block']]
            pd, pa = {}, {}
            for bj in range(prev_block + 1, f['block'] + 1):
                for cm in comp.blocks[bj]['moms']:
                    a, b = cm.loc_slice
                    for i in range(a, b):
                        fr_ = _XFRAC[comp.loc_kind[i]] * comp.loc_p[i]
                        for q in comp.loc_q[i]:
                            if q < self.circ.nd:
                                pd[q] = pd.get(q, 0) + fr_
                            elif bj == f['block']:
                                pa[q] = pa.get(q, 0) + fr_
            prev_block = f['block']
            pm = {}
            if blk['code']:
                for lab, qs in f['rows']:
                    a = self.circ.P.anc[lab]
                    pm[lab] = pa.get(self.circ.c[a], 0)
            self.p_data.append(pd)
            self.p_meas.append(pm)

    def _build(self):
        """Matching graphs M_t for t = 0..len(fr)-1 (rounds <= t, open boundary after t)."""
        self.mt = []
        nfr = len(self.fr)
        for t in range(nfr):
            m = pymatching.Matching()
            look = {}

            def add(u, v, w, info):
                key = (u, v) if v == -1 else (min(u, v), max(u, v))
                if key in look and look[key][0] <= w:
                    return
                look[key] = (w, info)
            for k in range(t + 1):
                f = self.fr[k]
                code = self.comp.blocks[f['block']]['code'] or 'D4'
                rows = f['rows']
                rowset = {lab for lab, _ in rows}
                # space edges: X on a data qubit before round k
                qrows = {}
                for lab, qs in rows:
                    for q in qs:
                        qrows.setdefault(q, []).append(f['idx'][lab])
                for q, nodes in qrows.items():
                    if k == 0:
                        w = self.gauge_w           # random first-round outcomes of S0 (gauge choice)
                    else:
                        w = _wt(self.p_data[k].get(q, 1e-6))
                    if len(nodes) == 2:
                        add(nodes[0], nodes[1], w, ('s', q, k))
                    elif len(nodes) == 1:
                        add(nodes[0], -1, w, ('s', q, k))
                # time edges to the previous occurrence of the same label
                if k > 0:
                    for lab, qs in rows:
                        prev = self._prev(k, lab)
                        if prev is not None:
                            kp, node_p = prev
                            w = _wt(self.p_meas[kp].get(lab, 1e-6))
                            add(node_p, f['idx'][lab], w, ('t', lab, kp))
            # switch-out round: a measurement error of a check of the last D4 Z round enters the reference
            # of the g-readout detectors (hyper-edge with its next detector); represent it by an edge between
            # the two nodes known at this time and fix the stored reference when it is committed ('m')
            if self.fr[t]['name'][0] == 'SO':
                f = self.fr[t]
                kp = t - 1
                for lab, qs in f['rows']:
                    for l2 in self.so_rel[lab]:
                        if l2 != lab and l2 in self.fr[kp]['idx']:
                            add(self.fr[kp]['idx'][l2], f['idx'][lab], _wt(self.p_meas[kp].get(l2, 1e-6)),
                                ('m', l2, kp))
            # open time boundary: nodes whose label is measured again after t
            for k in range(t + 1):
                f = self.fr[k]
                for lab, qs in f['rows']:
                    if self._next(k, lab) is not None and self._next(k, lab) > t and self._latest(k, lab, t):
                        add(f['idx'][lab], -1, _wt(self.p_meas[k].get(lab, 1e-6) + 1e-9), ('f', lab, k))
            for (u, v), (w, info) in look.items():
                if v == -1:
                    m.add_boundary_edge(u, weight=w)
                else:
                    m.add_edge(u, v, weight=w)
            nn = self.fr[t]['off'] + len(self.fr[t]['rows'])
            self.mt.append((m, {k_: v[1] for k_, v in look.items()}, nn))

    def _prev(self, k, lab):
        for kp in range(k - 1, -1, -1):
            if lab in self.fr[kp]['idx']:
                return kp, self.fr[kp]['idx'][lab]
        return None

    def _next(self, k, lab):
        for kn in range(k + 1, len(self.fr)):
            if lab in self.fr[kn]['idx']:
                return kn
        return None

    def _latest(self, k, lab, t):
        """True if round k holds the latest occurrence of lab among rounds <= t."""
        for kk in range(k + 1, t + 1):
            if lab in self.fr[kk]['idx']:
                return False
        return True

    # ---------------------------------------------------------------- per shot
    def start(self):
        self.last = {}
        self.active = set()
        self.k = -1

    def push_round(self, k, outcomes_by_label=None, readout=None):
        """Add the detection events of flux round k.  outcomes_by_label: dict label -> bit (Z checks);
        readout: dict g-qubit -> bit (switch-out)."""
        f = self.fr[k]
        self.k = k
        if f['name'][0] == 'SO':
            par = {}
            for lab, qs in f['rows']:
                v = 0
                for q in qs:
                    v ^= readout[q]
                ref = 0
                for l2 in self.so_rel[lab]:
                    ref ^= self.last.get(l2, 0)
                par[lab] = v
                if v ^ ref:
                    self.active ^= {f['idx'][lab]}
            self.readout = dict(readout)
            return
        for lab, qs in f['rows']:
            m = outcomes_by_label[lab]
            if m ^ self.last.get(lab, 0):
                self.active ^= {f['idx'][lab]}
            self.last[lab] = m

    def decode(self, flush=False, min_age=None):
        """Decode with all rounds <= current; returns list of committed space corrections (qubit, k).
        Commits space-like edges with round k <= t - delay (flush: k <= t - min_age, default all)."""
        t = self.k
        m, look, nn = self.mt[t]
        if not self.active:
            return []
        syn = np.zeros(m.num_detectors, dtype=np.uint8)
        for u in self.active:
            syn[u] = 1
        edges = m.decode_to_edges_array(syn)
        out = []
        for u, v in edges:
            u, v = int(u), int(v)
            key = (u, -1) if v == -1 else ((v, -1) if u == -1 else (min(u, v), max(u, v)))
            info = look[key]
            if info[0] == 'm' and flush:
                self.active ^= {key[0], key[1]}
                self.last[info[1]] = self.last.get(info[1], 0) ^ 1     # it was a measurement error
                continue
            if info[0] != 's':
                continue
            q, k = info[1], info[2]
            lim = (t - (min_age or 0)) if flush else (t - self.delay)
            if k > lim:
                continue
            out.append((q, k))
            if key[1] == -1:
                self.active ^= {key[0]}
            else:
                self.active ^= {key[0], key[1]}
        return out

    def register_correction(self, q, code):
        """A physical X correction on data qubit q: flip the stored reference of every Z check of the
        current code containing q (as if it had been applied before that check was measured)."""
        for lab in self.rows_of[code].get(q, []):
            if lab in self.last:
                self.last[lab] ^= 1


class ChargeDecoder:
    """Decoding of the X-type checks (Abelian charges = Z errors) after the protocol: minimum-weight
    matching on the detector error model of the Pauli-proxy circuit (stim + PyMatching).  The DEM
    describes the Pauli errors of the circuit; the extra randomness produced by non-Abelian dressings
    is not in the model (it is present in the simulated data).  Output: predicted flip of the logical
    observable (proxy of the logical Clifford stabilizer, including the Pauli-frame strings)."""

    def __init__(self, comp, dem_noise=None, aligned=True):
        circ = comp.circ
        self.comp, self.circ = comp, circ
        nz = dem_noise or comp.noise
        # measurement bookkeeping
        meas_lab = []          # per absolute measurement: (block_name, kind, label or qubit)
        blk_meas = []
        for bi, blk in enumerate(comp.blocks):
            ids = []
            for cm in blk['moms']:
                for q, b in cm.meas:
                    if blk['code']:
                        lab = circ.lab_of[(blk['code'], q)]
                        meas_lab.append((blk['name'], b, lab))
                    else:
                        meas_lab.append((blk['name'], 'R', q))
                    ids.append(len(meas_lab) - 1)
            blk_meas.append(ids)
        self.n_meas = len(meas_lab)
        self.blk_meas = blk_meas
        idx = {}
        for i, (bn, b, lab) in enumerate(meas_lab):
            idx[(bn, lab)] = i
        self.meas_idx = idx
        sup = {code: {lab: set(xs) for lab, xs, quad in circ.xgens[code]} for code in ('D4', 'FSC')}
        names = [blk['name'] for blk in comp.blocks]
        S0 = [n for n in names if n[0] == 'S0']
        S1 = [n for n in names if n[0] == 'S1' and n[2] == 'x']
        S4 = [n for n in names if n[0] in ('S4', 'FIN')]
        dets = []
        fscl = [lab for lab, _, _ in circ.xgens['FSC']]
        d4l = [lab for lab, _, _ in circ.xgens['D4']]
        for j, n in enumerate(S0):
            for lab in fscl:
                dets.append([idx[(n, lab)]] + ([idx[(S0[j - 1], lab)]] if j else []))

        def span_solve(target, basis_labels, code):
            M = np.zeros((len(basis_labels), circ.nq), dtype=np.uint8)
            for i, l in enumerate(basis_labels):
                M[i, list(sup[code][l])] = 1
            v = np.zeros(circ.nq, dtype=np.uint8)
            v[list(target)] = 1
            sol = f2_solve(M, v)
            return None if sol is None else [basis_labels[i] for i in np.nonzero(sol)[0]]
        for j, n in enumerate(S1):
            for lab in d4l:
                if j == 0:
                    if not S0:
                        continue
                    combo = span_solve(sup['D4'][lab], fscl, 'FSC')
                    if combo is None:
                        continue                       # S^g: random first outcome (gauging)
                    dets.append([idx[(n, lab)]] + [idx[(S0[-1], l)] for l in combo])
                else:
                    dets.append([idx[(n, lab)], idx[(S1[j - 1], lab)]])
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}
        for j, n in enumerate(S4):
            first = {}
            for lab in fscl:
                if j == 0:
                    tgt = set(sup['FSC'][lab])
                    for q in list(sup['FSC'][lab]):
                        gq = lat.q(inv[q] // 3, G)
                        if gq in c:
                            tgt ^= {c[gq]}
                    combo = span_solve(tgt, d4l, 'D4')
                    assert combo is not None, 'S4 X check not determined by the last D4 round'
                    first[lab] = [idx[(n, lab)]] + [idx[(S1[-1], l)] for l in combo]
                else:
                    dets.append([idx[(n, lab)], idx[(S4[j - 1], lab)]])
            if j == 0:
                # detector basis across the switch-out: aligned=True uses, at a vertex with separate r and
                # b checks, D_r and D_r + D_b (S^g of the last D4 round cancels in D_r + D_b), so that a g
                # charge created during the last D4 round is a graph-like error, but an r charge then has
                # 4 symptoms; aligned=False (D_r, D_b) has the opposite property.  D4Experiment decodes
                # with both and keeps the lower-weight solution (no single basis makes all graph-like)
                for lab, d in first.items():
                    v, col = lab[1], lab[2]
                    if aligned and col == 'b' and ('X', v, 'r') in first:
                        dets.append(sorted(set(d) ^ set(first[('X', v, 'r')])))
                    else:
                        dets.append(d)
        self.dets = dets
        # logical observable: proxy logical (measured by MPP at the end of the DEM circuit) + frame terms
        tset = set(circ.t_log)
        self.cross = [len(set(fs) & tset) % 2 for fs in circ.frame_strings]
        self.obs_meas = [idx[(S1[-1], lab)] for (lab, op), cr in zip(circ.gens_g, self.cross) if cr]
        self._dem_circuit(nz)
        # sparse detector evaluation
        self.det_ptr = np.cumsum([0] + [len(d) for d in dets])
        self.det_idx = np.concatenate([np.array(d, dtype=np.int64) for d in dets])

    def _dem_circuit(self, nz):
        circ, comp = self.circ, self.comp
        lines = []
        for bi, blk in enumerate(comp.blocks):
            noiseless = blk['name'][0] == 'FIN'
            for cm in blk['moms']:
                pre, post = [], []
                touched = {q for op in cm.ops for q in op[1:]}
                a, b = cm.loc_slice
                for i in range(a, b):
                    k, p, qs = comp.loc_kind[i], comp.loc_p[i], comp.loc_q[i]
                    if k == 0:
                        ln = 'X_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 1:
                        ln = 'Z_ERROR(%.12g) %d' % (p, qs[0])
                    elif k == 2:
                        ln = 'DEPOLARIZE1(%.12g) %d' % (p, qs[0])
                    elif k == 3:
                        ln = 'DEPOLARIZE2(%.12g) %d %d' % (p, qs[0], qs[1])
                    else:
                        ln = 'DEPOLARIZE1(%.12g) %d %d %d' % (p * 48 / 63, *qs)
                    (pre if comp.loc_when[i] == 0 else post).append(ln)
                lines += pre
                if cm.base is not None:
                    lines.append(str(cm.base))
                lines += post
            lines.append('TICK')
        n = circ.nq
        lp = ['X%d' % q for q in circ.t_log if circ.Delta_a.get(q) != 2]
        lp += ['Y%d' % q for q in circ.t_log if circ.Delta_a.get(q) == 2]
        lp += ['Z%d' % q for q, a in circ.Delta_a.items() if a == 2 and q not in set(circ.t_log)]
        lines.append('MPP ' + '*'.join(lp))
        tot = self.n_meas + 1
        for d in self.dets:
            lines.append('DETECTOR ' + ' '.join('rec[%d]' % (i - tot) for i in d))
        lines.append('OBSERVABLE_INCLUDE(0) rec[-1] ' + ' '.join('rec[%d]' % (i - tot) for i in self.obs_meas))
        self.stim_circuit = stim.Circuit('\n'.join(lines))
        dem = self.stim_circuit.detector_error_model(decompose_errors=True, ignore_decomposition_failures=True,
                                                     approximate_disjoint_errors=True)
        self.matching = pymatching.Matching.from_detector_error_model(dem)
        # graph-like components of the DEM (for the flux-informed re-weighting)
        comp = {}
        for ins in dem.flattened():
            if ins.type != 'error':
                continue
            p = ins.args_copy()[0]
            cur_d, cur_o = [], 0
            parts = []
            for t in ins.targets_copy():
                if t.is_separator():
                    parts.append((tuple(sorted(cur_d)), cur_o)); cur_d, cur_o = [], 0
                elif t.is_relative_detector_id():
                    cur_d.append(t.val)
                elif t.is_logical_observable_id():
                    cur_o ^= 1
            parts.append((tuple(sorted(cur_d)), cur_o))
            for key in parts:
                if not key[0] or len(key[0]) > 2:
                    continue
                q0 = comp.get(key, 0.0)
                comp[key] = q0 * (1 - p) + p * (1 - q0)
        keys = list(comp)
        self.comp_keys = keys
        self.comp_index = {k[0]: i for i, k in enumerate(keys)}
        from scipy.sparse import csc_matrix
        rows, cols = [], []
        for j, (dd, o) in enumerate(keys):
            for dtt in dd:
                rows.append(dtt); cols.append(j)
        nd = self.stim_circuit.num_detectors
        self.H = csc_matrix((np.ones(len(rows), dtype=np.uint8), (rows, cols)), shape=(nd, len(keys)))
        self.O = csc_matrix((np.ones(sum(o for _, o in keys), dtype=np.uint8),
                             ([0] * sum(o for _, o in keys), [j for j, (dd, o) in enumerate(keys) if o])),
                            shape=(1, len(keys)))
        self.w0 = np.array([_wt(comp[k]) for k in keys])
        # detectors containing each measurement
        self.dets_of_meas = {}
        for j, d_ in enumerate(self.dets):
            for i in d_:
                self.dets_of_meas.setdefault(i, []).append(j)

    def detectors(self, outcomes):
        o = np.asarray(outcomes, dtype=np.uint8)[self.det_idx]
        return np.bitwise_xor.reduceat(o, self.det_ptr[:-1]) if len(o) else np.zeros(0, np.uint8)

    def flux_erasures(self, flux_events, w_erase=0.1):
        """Flux-informed re-weighting (our construction, following [KZH]'s description of cleaning up the
        Abelian charges after the fluxes have been paired): a flux on qubit q present during the X-type
        round ('S1', r, 'x') randomizes the outcomes of the D4 stabilizers whose CZ dressing contains q and
        leaves Z errors on the CZ partners with probability 1/2.  The corresponding time-like components
        and the space-like components of the partner Z errors get the weight w_erase.
        Scope (deliberately narrow): only measurements in at most two detectors, and the space-like part only
        between consecutive D4 X rounds (not for the last D4 round across U and the switch-out; fluxes
        corrected in the extra Z rounds before U are not attributed to the last D4 round).  A complete
        version (every detector set located in the DEM, the last round included) was tested in v6 and made
        the decoder WORSE (single-fault first-order coefficient at d = 3, scenario B: 4.2 -> 8.0; the
        attribution change alone is neutral): nearly free erased edges also open cheap wrong paths when the
        flux attribution is wrong.  See docs/methods.md sec. 12."""
        circ = self.circ
        if not hasattr(self, '_cz_of'):
            self._cz_of = {}
            for lab, xs, quad in circ.xgens['D4']:
                for (i, j) in quad:
                    self._cz_of.setdefault(i, []).append((lab, j))
                    self._cz_of.setdefault(j, []).append((lab, i))
            self._stars = {}
            for lab, xs, quad in circ.xgens['D4']:
                for q in xs:
                    self._stars.setdefault(q, []).append(lab)
        er = set()
        for q, r in flux_events:
            blk = ('S1', r, 'x')
            nxt = ('S1', r + 1, 'x')
            for lab, partner in self._cz_of.get(q, []):
                m = self.meas_idx.get((blk, lab))
                if m is None:
                    continue
                dd = tuple(sorted(self.dets_of_meas.get(m, [])))
                if 1 <= len(dd) <= 2 and dd in self.comp_index:
                    er.add(self.comp_index[dd])
                # space-like component: Z on the partner flips the two stars containing it
                st = self._stars.get(partner, [])
                for rr in (blk, nxt):
                    ds = []
                    for l2 in st:
                        m2 = self.meas_idx.get((rr, l2))
                        if m2 is not None:
                            ds += [j for j in self.dets_of_meas.get(m2, []) if self.dets[j][0] == m2]
                    dd = tuple(sorted(ds))
                    if 1 <= len(dd) <= 2 and dd in self.comp_index:
                        er.add(self.comp_index[dd])
        return er

    def predict(self, outcomes, erasures=None, w_erase=0.1):
        """Returns (predicted flip of the logical observable, weight of the matching)."""
        det = self.detectors(outcomes)
        if not erasures:
            pred, w = self.matching.decode(det, return_weight=True)
            return int(pred[0]), float(w)
        w = self.w0.copy()
        w[list(erasures)] = w_erase
        m = pymatching.Matching.from_check_matrix(self.H, weights=w, faults_matrix=self.O)
        pred, wt = m.decode(det, return_weight=True)
        return int(pred[0]), float(wt)


class D4Experiment:
    """Noisy KZH protocol + decoders.  run(shots) returns per-shot logical infidelity samples
    (0, 1/2 or 1; Takada Sec. VI D with the infidelity (1 - <H>)/2 of the output) and diagnostics."""

    def __init__(self, L, R0, Rin, Rout, noise, delay=0, pre_u_age=0, n_pre_u=2, flux_informed=True,
                 quiet_window=False, fsc_schedule='classes'):
        self.pre_u_age = pre_u_age
        self.flux_informed = flux_informed
        self.circ = D4SwitchingCircuit(L, R0, Rin, Rout, n_pre_u=n_pre_u, fsc_schedule=fsc_schedule)
        self.comp = CompiledProtocol(self.circ, noise)
        self.noise = noise
        self.flux = FluxDecoder(self.comp, delay=delay)
        self.charges = [ChargeDecoder(self.comp, aligned=a) for a in (True, False)]
        # diagnostic (not a physical model): no noise in the switching window = the last D4 round, the Z
        # rounds before U, U, the switch-out and the first FSC round, where the single faults that our
        # decoders fail to correct are located (validation N5).  It is NOT a bound on an optimal decoder (same
        # decoder, other noise); it shows how well the errors outside the window are suppressed.  The decoders keep
        # the priors of the full noise model (they are built before the window is silenced).
        self.quiet_window = quiet_window
        win = {('S1', Rin, 'z'), ('S1', Rin, 'x'), ('U',), ('SO',), ('S4', 1)}
        win |= {('S1', r, 'z') for r in range(Rin + 1, Rin + 1 + n_pre_u)}
        self.window = win
        if quiet_window:
            for blk in self.comp.blocks:
                if blk['name'] in win:
                    a, b = blk['moms'][0].loc_slice[0], blk['moms'][-1].loc_slice[1]
                    self.comp.loc_p[a:b] = 0.0
        self.gset = set(self.circ.data_g)

    def shot(self, rng, decode=True, forced=None, ts=None, init=None):
        """One Monte Carlo shot.  forced: {location index or dynamic fault id: Pauli string} = single-fault
        injection (all other noise off); dynamic fault ids are ('SOCZ', i, j) (after the physical switch-out
        CZ on i, j, if it is applied in this shot) and ('SOIDLE', layer, q).  ts: TrueSim run in lockstep
        (validation only; then the returned stats contain the lockstep record)."""
        circ, comp, fl = self.circ, self.comp, self.flux
        fstatic = {k: v for k, v in forced.items() if not isinstance(k, tuple)} if forced is not None else None
        fdyn = {k: v for k, v in forced.items() if isinstance(k, tuple)} if forced is not None else None
        sh = FastShot(comp, rng, fstatic)
        if init is not None:
            init(sh)               # validation: prepared initial state (see lockstep_full)
        lock = []
        fl.start()
        outs = []
        kfr = 0
        m_g_last = None
        flux_events = []
        stats = dict(n_err=sh.n_err, n_corr=0)
        names = [b['name'] for b in comp.blocks]
        last_S0 = max([i for i, n in enumerate(names) if n[0] == 'S0'], default=-1)
        last_S1 = max(i for i, n in enumerate(names) if n[0] == 'S1')          # Z-only round before U
        last_S1x = max(i for i, n in enumerate(names) if n[0] == 'S1' and n[2] == 'x')
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm == ('S1', 1, 'z'):
                sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
            if init is not None and nm[0] == 'S0':
                # validation start (lockstep_full): S0 replaced by the prepared state; its (noiseless,
                # ideal) outcomes are all 0
                res = [(q, bb, 0) for cm in comp.blocks[bi]['moms'] for (q, bb) in cm.meas]
            elif ts is None:
                res = sh.run_block(bi)
            else:
                r5 = sh.run_block_lockstep(bi, ts, rng)
                lock += r5
                res = [(q, bb, o) for q, bb, o, pk, pr in r5]
            outs += [o for (_, _, o) in res]
            if bi == last_S1x:
                gl = {circ.lab_of[('D4', q)]: o for (q, b, o) in res if b == 'X'}
                m_g_last = [gl[lab] for lab, _ in circ.gens_g]
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                code = blk['code']
                zl = {circ.lab_of[(code, q)]: o for (q, b, o) in res if b == 'Z'}
                fl.push_round(kfr, zl)
                kfr += 1
                if decode:
                    flush = bi in (last_S0, last_S1) or nm[0] == 'FIN'
                    # before U: corrections supported only by the newest round stay undecided (decoded
                    # after the switch-out with the S4 / g-readout data)
                    cc = fl.decode(flush=flush, min_age=self.pre_u_age if bi == last_S1 else 0)
                    corr = [q for q, k in cc]
                    if nm[0] == 'S1':
                        # the flux was present during the X-type round preceding this Z round
                        flux_events += [(q, nm[1] - 1) for q in corr if nm[1] >= 2]
                    sh.apply_x(corr)
                    if ts is not None:
                        for q in corr:
                            ts.X(q)
                    for q in corr:
                        fl.register_correction(q, code)
                    stats['n_corr'] += len(corr)
            elif nm[0] == 'SO':
                readout = {q: o for (q, b, o) in res}
                fl.push_round(kfr, readout=readout)
                kfr += 1
                a_dec = dict(readout)
                phys = []
                if decode:
                    for q, k in fl.decode(flush=True):
                        if q in self.gset:
                            a_dec[q] ^= 1
                        else:
                            phys.append(q)
                    for q in phys:
                        fl.register_correction(q, 'FSC')
                    stats['n_corr'] += len(phys)
                stats['so_cz'] = self._switch_out(sh, readout, a_dec, phys, rng, fdyn,
                                                  noisy=forced is None and not self.quiet_window, ts=ts)
        # Pauli frame of the S^g eigenvalues (last S1 round), then the logical Clifford stabilizer
        for v, m in enumerate(m_g_last):
            if m:
                for q in circ.frame_strings[v]:
                    sh.sim.z(q)
                    if ts is not None:
                        ts.Z(q)
        val = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
        if ts is not None:
            stats['lockstep'] = lock
            stats['H_true'] = ts.expect_logical(circ)
            stats['H_proxy'] = val
        if decode:
            sols = [ch.predict(outs, ch.flux_erasures(flux_events) if self.flux_informed else None)
                    for ch in self.charges]
            flip = min(sols, key=lambda s: s[1])[0]
        else:
            flip = 0
        val = val * (-1) ** flip
        stats['n_dress'] = sh.n_dress
        return (1 - val) / 2, stats

    def _switch_out(self, sh, readout, a_dec, phys, rng, fdyn=None, noisy=True, ts=None):
        """Physical switch-out after the Z^g readout ([KZH] step 4), in the order of the real protocol:
        (1) remove the virtual correction CZ(Gamma(a_raw)) of the deferred-measurement network (noiseless
            Clifford bookkeeping of the simulation, no physical operation);
        (2) the X corrections decided by the flux decoder (applied as a noiseless Pauli layer, like all
            feed-forward X corrections of this simulation);
        (3) the physical CZ(Gamma(a_dec)) in the fixed schedule circ.so_colour: in every layer each CZ is
            followed by DEPOLARIZE2(p) and every other rb data qubit by DEPOLARIZE1(p_idle) (exact Pauli
            propagation through the later layers, because they are applied in order).
        fdyn: forced dynamic faults (single-fault injection).  Returns the number of physical CZs."""
        circ, sim = self.circ, sh.sim
        ar = np.array([readout[q] for q in circ.data_g], dtype=np.uint8)
        ad = np.array([a_dec[q] for q in circ.data_g], dtype=np.uint8)
        kr = (circ.Gamma.astype(int) @ ar) % 2
        kd = (circ.Gamma.astype(int) @ ad) % 2

        def pairs_of(k):
            cnt = {}
            for v in np.nonzero(k)[0]:
                for pr in circ.gpairs[v]:
                    pr = tuple(sorted(pr))
                    cnt[pr] = cnt.get(pr, 0) ^ 1
            return sorted(pr for pr, c in cnt.items() if c)

        def pauli(ps, qs):
            sh._pauli(ps, qs)
            if ts is not None:
                for pc, q in zip(ps, qs):
                    if pc != 'I':
                        getattr(ts, pc)(q)

        def cz(i, j):
            sim.cz(i, j)
            if ts is not None:
                ts.diag((i, j), lambda x: 4 * x[0] * x[1])
        for (i, j) in pairs_of(kr):                       # (1)
            cz(i, j)
        sh.apply_x(phys)                                  # (2)
        if ts is not None:
            for q in phys:
                ts.X(q)
        gates = pairs_of(kd)                              # (3)
        nl = circ.so_steps - 1
        p, pi = self.noise.p, self.noise.p_idle
        for layer in range(nl):
            here = [pr for pr in gates if circ.so_colour[pr] == layer]
            busy = {q for pr in here for q in pr}
            for (i, j) in here:
                cz(i, j)
            for (i, j) in here:
                if noisy and p > 0 and rng.random() < p:
                    pauli(PAULI_CHOICES[2][rng.integers(15)], (i, j))
                if fdyn and ('SOCZ', i, j) in fdyn:
                    pauli(fdyn[('SOCZ', i, j)], (i, j))
            for q in circ.data_rb:
                if q in busy:
                    continue
                if noisy and pi > 0 and rng.random() < pi:
                    pauli('XYZ'[rng.integers(3)], (q,))
                if fdyn and ('SOIDLE', layer, q) in fdyn:
                    pauli(fdyn[('SOIDLE', layer, q)], (q,))
        return len(gates)

    def run(self, shots, rng, decode=True):
        inf = np.zeros(shots)
        st = []
        for s in range(shots):
            inf[s], d = self.shot(rng, decode)
            st.append(d)
        return inf, st

In [ ]:
# ------------------------------------------------------------------ applicability of Takada et al.
def equal_up_to_phase(A, B, tol=1e-12):
    """True iff A = e^{i phi} B for one global phase (exact operator comparison, NOT elementwise |.|)."""
    A, B = np.asarray(A, complex), np.asarray(B, complex)
    if A.shape != B.shape:
        return False
    k = np.unravel_index(np.argmax(np.abs(B)), B.shape)
    if abs(B[k]) < tol:
        return bool(np.allclose(A, 0, atol=tol))
    ph = A[k] / B[k]
    return bool(abs(abs(ph) - 1) < 1e-9 and np.allclose(A, ph * B, atol=1e-9))


def takada_conditions(L):
    """Algebraic checks of the sufficient conditions of Takada et al. (Definition 2 and Sec. VIII) for
    the D4 code switching protocol on the L x L lattice.  Returns dict name -> (passed, detail)."""
    P = D4Protocol(L)
    lat, d4, fsc = P.lat, P.d4, P.fsc
    res = {}
    # (a) admissible factorisation: every X-type Clifford stabilizer = X factors x CZ factors with disjoint
    #     supports (transversal, so every effectively measured operator stays Hermitian, Eq. (131))
    bad = 0
    for lab, op in d4.xgens:
        qs = list(op.s) + [q for pr in op.quad for q in pr]
        bad += len(qs) != len(set(qs))
    res['factors transversal (X and CZ factors on disjoint qubits)'] = (bad == 0, '%d generators' % len(d4.xgens))
    # (b) Pauli dressings: X/Y errors on the Clifford factors (CZ; S, CZ in the logical stabilizer; S,
    #     CZ dressings) give weight-1 Z dressings; all dressings commute (Eq. (91))
    I2 = np.eye(2); X = np.array([[0, 1], [1, 0]]); Y = np.array([[0, -1j], [1j, 0]]); Zm = np.diag([1, -1])
    CZm = np.diag([1, 1, 1, -1]); Sm = np.diag([1, 1j])
    ok = True
    for E in (X, Y):
        for k in (0, 1):                                                   # error on either qubit of the CZ
            Ek = np.kron(E, I2) if k == 0 else np.kron(I2, E)
            D = Ek @ CZm @ Ek.conj().T @ CZm.conj().T                          # group commutator [E, CZ]
            ok &= equal_up_to_phase(D, np.kron(I2, Zm) if k == 0 else np.kron(Zm, I2))
        D1 = E @ Sm @ E.conj().T @ Sm.conj().T                                 # [E, S]
        ok &= equal_up_to_phase(D1, Zm)
    for E in (Zm,):                                                        # Z errors commute: no dressing
        ok &= equal_up_to_phase(np.kron(E, I2) @ CZm @ np.kron(E, I2).conj().T @ CZm.conj().T, np.eye(4))
    res['dressings of CZ / S by X, Y are single-qubit Z on the partner / same qubit (exact up to a global '
        'phase); Z errors give no dressing'] = (bool(ok), '')
    # (c) effective commutation: group commutators of measured generators are Z-type Paulis in the
    #     persistent Z-stabilizer span (D4 Z checks), Eq. (58) / Lemma 1
    HZr, HZp = f2_rref(d4.HZ)
    nbad, npair = 0, 0
    for (l1, a), (l2, b) in itertools.combinations(d4.xgens, 2):
        lin, const = commutator_affine(a, b)
        npair += 1
        if lin and not f2_in_rowspace(HZr, HZp, lat.vec(lin)):
            nbad += 1
    res['D4 generators: commutators in the Z-stabilizer span'] = (nbad == 0, '%d pairs' % npair)
    # switch-in: S^g measured on FSC|+_L> (x) |0>_g: commutators with the FSC generators and the logical X
    # must lie in the span of the FSC Z checks and the single-qubit Z^g (current Z stabilizers)
    # The state before switch-in, FSC|+_L> (x) |0>_g, is stabilized by the FSC Z checks, the single-qubit
    # Z^g, and by the D4-dressed X-type operators S^r_v, S^b_v (their CZ^{g,.} dressing acts trivially
    # on |0>_g) and a dressed logical X = X^t D (D diagonal Clifford, also trivial on |0>_g).  Takada
    # Sec. VIII B: the generators may be chosen such that the new measurements (S^g) effectively commute
    # or anticommute with them.  Check this choice:
    zin = [rows for lab, rows in lat.z_checks((R, B))] + [{lat.q(e, G)} for e in range(lat.E)]
    Hin = np.array([lat.vec(r) for r in zin], dtype=np.uint8)
    Hir, Hip = f2_rref(Hin)
    gens_in = [op for lab, op in d4.xgens if lab[2] != 'g']
    nbad = 0
    for lab, op in d4.xgens:
        for g in gens_in:
            lin, const = commutator_affine(op, g)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
    # dressed logical X: find D = prod CZ^{c_k} over the CZ pairs of the D4 generators with
    # [X^t D, S] in the D4 Z-stabilizer span for every X-type generator S (linear system over F2)
    t = set(int(q) for q in np.nonzero(fsc.t_log)[0])
    pairs = sorted({pr for _, op in d4.xgens for pr in op.quad})
    rows_eq = []                          # for each generator: commutator linear part as function of c
    HZr2, HZp2 = f2_rref(d4.HZ)
    # basis of the complement of the Z span to express "lin in span" as linear equations
    Npre = f2_nullspace(d4.HZ)            # vectors orthogonal to the span: lin in span <=> N.lin = 0
    A_cols, b_vec = [], None
    base_list = []
    for lab, op in d4.xgens:
        lin0, _ = commutator_affine(PSOp(s=t), op)
        base_list.append(lat.vec(lin0) if lin0 else np.zeros(lat.n, dtype=np.uint8))
    for pr in pairs:
        col = []
        for lab, op in d4.xgens:
            lin, _ = commutator_affine(PSOp(quad=[pr]), op)
            col.append(lat.vec(lin) if lin else np.zeros(lat.n, dtype=np.uint8))
        A_cols.append(col)
    # equations: N @ (base_v + sum_k c_k col_k,v) = 0 for all v
    eqA = np.array([[int((Npre.astype(int) @ A_cols[k][v].astype(int) % 2)[i]) for k in range(len(pairs))]
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    eqb = np.array([int((Npre.astype(int) @ base_list[v].astype(int) % 2)[i])
                    for v in range(len(d4.xgens)) for i in range(Npre.shape[0])], dtype=np.uint8)
    sol = f2_solve(eqA.T, eqb) if eqA.size else np.zeros(0, dtype=np.uint8)
    logical_ok = sol is not None
    if logical_ok:
        D = [pr for pr, cbit in zip(pairs, sol) if cbit]
        Xbar = PSOp(s=t, quad=D)
        # dressing trivial on |0>_g: every CZ of D touches a g qubit
        triv = all(any(q % 3 == G for q in pr) for pr in D)
        for lab, op in d4.xgens:
            lin, const = commutator_affine(Xbar, op)
            if lin and not f2_in_rowspace(Hir, Hip, lat.vec(lin)):
                nbad += 1
        logical_ok = triv
    res['switch-in: S^g vs dressed generators of FSC|+_L>|0>_g (current Z span), dressed logical X exists'] = (
        nbad == 0 and logical_ok, 'logical dressing with %s CZ pairs' % (len(D) if logical_ok else '-'))
    # (d) primary-support exactness (Definition 1): X supports of the X-type generators (and the logical
    #     X) are linearly independent, so no product of them is a non-Pauli operator without X part
    Ssup = np.vstack([d4.S, d4.t_log[None, :]])
    rk = f2_rank(Ssup[:, d4.cols])
    res['primary-support exactness (independent X supports)'] = (rk == Ssup.shape[0], 'rank %d / %d' % (rk, Ssup.shape[0]))
    # (e) the transversal U and the deferred switch-out network consist of CNOTs and diagonal third-level
    #     gates only (Sec. VIII A, Eq. (178)): Pauli / diagonal-Clifford errors stay of that form
    names = {g[0] for g in P.U_gates()}
    res['U = CNOT + diagonal 3rd-level gates (Sec. VIII A)'] = (names <= {'CNOT', 'CS', 'CSDG', 'TDG'}, str(sorted(names)))
    # (f) logical Clifford stabilizer of the output: Clifford (no cubic / odd terms) and effectively
    #     commuting with every FSC X check (dressing = Z string in the FSC Z span with sign +-1)
    circ = D4SwitchingCircuit(L, 1, 1, 1)
    Hf = np.array([[1 if q in qs else 0 for q in range(circ.nq)] for lab, qs in circ.zchecks['FSC']], dtype=np.uint8)
    Hfr, Hfp = f2_rref(Hf)
    nbad = 0
    for lab, xs, quad in circ.xgens['FSC']:
        fx = np.zeros(circ.nq, dtype=np.uint8)
        fx[xs] = 1
        R_ = circ.logical_proxy(fx) * circ.logical_proxy(np.zeros(circ.nq, dtype=np.uint8))
        zpart = np.array([1 if R_[q] in (2, 3) else 0 for q in range(circ.nq)], dtype=np.uint8)
        xpart = np.array([1 if R_[q] in (1, 2) else 0 for q in range(circ.nq)], dtype=np.uint8)
        if xpart.any() or (zpart.any() and not f2_in_rowspace(Hfr, Hfp, zpart)) or R_.sign.imag != 0:
            nbad += 1
    res['output logical stabilizer: Clifford, effectively commutes with FSC X checks'] = (nbad == 0, '')
    return res


def lockstep_validation(L, Rin, Rout, noise, rng, sequential=False, corr_prob=0.05):
    """Run the Pauli-proxy shot (FastShot) and the exact simulation of the TRUE circuit (TrueSim) in
    lockstep from the start of S1 (initial state E|+_L>_FSC|0>_g with a random Pauli E on the FSC data),
    with the same sampled circuit-level errors and random feed-forward X corrections.  Returns dict with
    the number of measurements, the number of mismatches (deterministic vs random, probability of the
    outcome), and the final <H> from both simulations."""
    circ = D4SwitchingCircuit(L, 0, Rin, Rout, sequential=sequential)
    comp = CompiledProtocol(circ, noise)
    P, c, fsc = circ.P, circ.c, circ.P.fsc
    zero = coset_state(fsc, set())
    one = coset_state(fsc, set(int(q) for q in np.nonzero(fsc.t_log)[0]))
    amps = {}
    for st in (zero, one):
        for k, v in st.a.items():
            amps[k] = amps.get(k, 0) + v / np.sqrt(2)
    keys = [sum(1 << c[q] for q in k) for k in amps]
    ts = TrueSim(circ.nd, keys, list(amps.values()))
    sh = FastShot(comp, rng)
    n = circ.nq
    stabs = []
    for lab, xs, quad in circ.xgens['FSC']:
        ps = stim.PauliString(n)
        for q in xs:
            ps[q] = 'X'
        stabs.append(ps)
    for lab, qs in circ.zchecks['FSC']:
        ps = stim.PauliString(n)
        for q in qs:
            ps[q] = 'Z'
        stabs.append(ps)
    ps = stim.PauliString(n)
    for q in circ.t_log:
        ps[q] = 'X'
    stabs.append(ps)
    for q in range(n):
        if q not in circ.data_rb:
            ps = stim.PauliString(n)
            ps[q] = 'Z'
            stabs.append(ps)
    sh.sim.set_state_from_stabilizers(stabs, allow_redundant=True)
    for q in rng.choice(circ.data_rb, 2, replace=False):
        pc = 'XYZ'[rng.integers(3)]
        getattr(ts, pc)(int(q)); getattr(sh.sim, pc.lower())(int(q))
    sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
    nmeas = nbad = nrand = 0
    m_g = None
    for bi, blk in enumerate(comp.blocks):
        res = sh.run_block_lockstep(bi, ts, rng)
        for q, bb, o, pk, pr in res:
            nmeas += 1
            if pk == 0:
                nrand += 1
                nbad += abs(pr - 0.5) > 1e-9
            else:
                nbad += (o != (0 if pk == 1 else 1)) or abs(pr - 1) > 1e-9
        nm = blk['name']
        if nm[0] == 'S1' and nm[2] == 'x':
            gl = {circ.lab_of[('D4', q)]: o for q, bb, o, pk, pr in res if bb == 'X'}
            m_g = [gl[lab] for lab, _ in circ.gens_g]
        if corr_prob and nm[0] in ('S1', 'S4') and rng.random() < corr_prob * len(blk['moms']):
            q = int(rng.choice(circ.data_rb if nm[0] == 'S4' else circ.data_all))
            sh.apply_x([q]); ts.X(q)
    for v, m in enumerate(m_g):
        if m:
            for q in circ.frame_strings[v]:
                sh.sim.z(q); ts.Z(q)
    true = ts.expect_logical(circ)
    proxy = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
    return dict(n_meas=nmeas, n_random=nrand, n_mismatch=int(nbad), H_true=true, H_proxy=proxy,
                n_dress=sh.n_dress, agree=bool(nbad == 0 and abs(true - proxy) < 1e-9))


def lockstep_full(L, Rin, Rout, noise, rng, ex=None):
    """Validation of the adaptive protocol AS USED IN PRODUCTION (D4Experiment.shot, from S1 on): real-time
    flux decoding with X corrections, the switch-out with the decoded readout and the physical CZ layers
    with their noise, the Pauli frame and the charge decoder, run in lockstep on the Pauli-proxy circuit
    (stim) and on the exact sparse state vector of the TRUE non-Clifford circuit (TrueSim), with the same
    sampled errors and outcomes.  S0 (a Clifford circuit, simulated exactly by stim alone) is replaced by
    its ideal output E|+_L>_FSC|0>_g with a random 2-qubit Pauli E on the FSC data (noiseless S0 outcomes
    for the decoders; the exact state vector of S0's many parallel ancillas does not fit in memory).  Checks every measurement (proxy
    deterministic <-> exact probability 1, proxy random <-> exact 1/2) and the final <H> before the
    charge-decoder correction."""
    ex = ex or D4Experiment(L, 1, Rin, Rout, noise)
    circ = ex.circ
    P, c, fsc = circ.P, circ.c, circ.P.fsc
    zero = coset_state(fsc, set())
    one = coset_state(fsc, set(int(q) for q in np.nonzero(fsc.t_log)[0]))
    amps = {}
    for st_ in (zero, one):
        for k, v in st_.a.items():
            amps[k] = amps.get(k, 0) + v / np.sqrt(2)
    ts = TrueSim(circ.nd, [sum(1 << c[q] for q in k) for k in amps], list(amps.values()))
    n = circ.nq
    errs = [('XYZ'[rng.integers(3)], int(q)) for q in rng.choice(circ.data_rb, 2, replace=False)]

    def init(sh):
        stabs = []
        for lab, xs, quad in circ.xgens['FSC']:
            ps = stim.PauliString(n)
            for q in xs:
                ps[q] = 'X'
            stabs.append(ps)
        for lab, qs in circ.zchecks['FSC']:
            ps = stim.PauliString(n)
            for q in qs:
                ps[q] = 'Z'
            stabs.append(ps)
        ps = stim.PauliString(n)
        for q in circ.t_log:
            ps[q] = 'X'
        stabs.append(ps)
        for q in range(n):
            if q not in circ.data_rb:
                ps = stim.PauliString(n)
                ps[q] = 'Z'
                stabs.append(ps)
        sh.sim.set_state_from_stabilizers(stabs, allow_redundant=True)
        for pc, q in errs:
            getattr(ts, pc)(q)
            getattr(sh.sim, pc.lower())(q)
    inf, st = ex.shot(rng, ts=ts, init=init)
    nbad = nrand = 0
    for q, bb, o, pk, pr in st['lockstep']:
        if pk == 0:
            nrand += 1
            nbad += abs(pr - 0.5) > 1e-9
        else:
            nbad += (o != (0 if pk == 1 else 1)) or abs(pr - 1) > 1e-9
    ht, hp = complex(st['H_true']), float(st['H_proxy'])
    return dict(n_meas=len(st['lockstep']), n_random=nrand, n_mismatch=int(nbad), H_true=ht.real, H_proxy=hp,
                n_err=st['n_err'], n_corr=st['n_corr'], so_cz=st['so_cz'], infidelity=float(inf),
                agree=bool(nbad == 0 and abs(ht - hp) < 1e-9))


def single_fault_scan(ex, n_sample=None, seeds=2, rng_seed=1, keep_failures=True, only_window=False):
    """Single-fault injection over ALL noise locations of the protocol as simulated: every static
    location of the compiled circuit (gates incl. CCZ / CS / T^dag, resets, measurements, idling) with
    every Pauli of its channel, plus the dynamic switch-out faults (each Pauli after each possible physical
    CZ, idle faults of the CZ layers).  All other noise is off.  Every fault is run with `seeds` different
    measurement-randomness seeds (the outcomes of random measurements, hence the branches, differ); a
    fault 'fails' if any seed gives a nonzero output infidelity.  Not covered: other branches than the
    sampled ones.  only_window: restrict to the faults in the switching window (ex.window).
    Returns dict with the counts, the probability-weighted first-order coefficient
        c1 = sum_f (P(f) / p) * mean_seeds(infidelity)      (eps ~ c1 p for small p, this decoder),
    its split into the switching window (ex.window) and the rest, and (optionally) the failing faults."""
    comp, circ, nz = ex.comp, ex.circ, ex.noise
    p = nz.p
    names = [b['name'] for b in comp.blocks]
    blk_of = np.zeros(comp.n_loc, dtype=int)
    for bi, b in enumerate(comp.blocks):
        for cm in b['moms']:
            a_, b_ = cm.loc_slice
            blk_of[a_:b_] = bi
    faults = []                                   # (fault id, pauli, weight P(f)/p, block name)
    for i in range(comp.n_loc):
        k, qs, pl = comp.loc_kind[i], comp.loc_q[i], comp.loc_p[i]
        if pl == 0:
            continue
        chs = ['X'] if k == 0 else ['Z'] if k == 1 else PAULI_CHOICES[len(qs)]
        for ps in chs:
            faults.append((int(i), ps, pl / p / len(chs), names[blk_of[i]]))
    for pr in circ.so_colour:
        for ps in PAULI_CHOICES[2]:
            faults.append((('SOCZ',) + pr, ps, 1 / 15, ('SO',)))
    if nz.p_idle > 0:
        for layer in range(circ.so_steps - 1):
            for q in circ.data_rb:
                for ps in 'XYZ':
                    faults.append((('SOIDLE', layer, q), ps, nz.p_idle / p / 3, ('SO',)))
    if only_window:                               # exhaustive over the switching window only
        faults = [f for f in faults if f[3] in ex.window]
    r_ = np.random.default_rng(rng_seed)
    n_all = len(faults)
    if n_sample and n_sample < n_all:
        idx = r_.choice(n_all, n_sample, replace=False)
        faults = [faults[j] for j in idx]
        scale = n_all / n_sample                  # unbiased estimate of the full sum
    else:
        scale = 1.0
    c1 = c1_win = 0.0
    nfail = nfail_win = 0
    by_block, fails = {}, []
    for fid, ps, w, bn in faults:
        infs = [ex.shot(np.random.default_rng((rng_seed, s_, zlib.crc32(str(fid).encode()))), forced={fid: ps})[0]
                for s_ in range(seeds)]
        m = float(np.mean(infs))
        c1 += w * m * scale
        inwin = bn in ex.window
        if inwin:
            c1_win += w * m * scale
        if max(infs) > 0:
            nfail += 1
            nfail_win += inwin
            by_block[str(bn)] = by_block.get(str(bn), 0) + 1
            if keep_failures:
                fails.append(dict(fault=str(fid), pauli=ps, block=str(bn), in_window=inwin, infidelities=infs))
    return dict(n_faults_total=n_all, n_tested=len(faults), seeds=seeds, n_fail=nfail, n_fail_window=nfail_win,
                n_fail_outside=nfail - nfail_win, by_block=by_block, c1=c1, c1_window=c1_win,
                c1_outside=c1 - c1_win, failures=fails)


def block_steps(circ, blk):
    """Physical time steps of a block: the switch-out is the g readout plus the fixed CZ-layer schedule
    (its simulation moments FLIPX / NCCZ / MZ0 are a deferred-measurement representation, not time)."""
    return circ.so_steps if blk['name'][0] == 'SO' else len(blk['moms'])

## 研究 4 のライブラリ：連続サイクルの code switching

研究 3 のクラスを書き換えず、サブクラスとして K 回連続の回路・復号器・実験を定義する。`SeqSwitchingCircuit`（回路）、`SeqFluxDecoder`・`SeqChargeDecoder`（復号器）、`SeqExperiment`（1 ショットの実行）、`seq_counts`（体積の数え上げ）。K=1 は研究 3 と同一になる。

In [ ]:
# ==================================================================================================
# seq_lib : 研究 4 の拡張ライブラリ（連続サイクルの code switching）
#
# 研究 3 のライブラリ（D4SwitchingCircuit / CompiledProtocol / FluxDecoder / ChargeDecoder / D4Experiment）
# は一切書き換えず、サブクラスとして「K 回連続の switch-in → U → switch-out」を追加する。
#
#   S0 (FSC |+_L> の準備, R0 ラウンド)
#   [ S1 (switch-in, Rin ラウンド + Z だけ n_pre_u ラウンド) → U → SO (switch-out) → S4 (FSC ラウンド) ] × K
#   FIN (評価用の雑音なし FSC 1 ラウンド)
#
# サイクル c = 1..K-1 の後の FSC ラウンド数は Rgap（次のサイクルの前段を兼ねる）、最後のサイクルの後は Rout。
# K = 1, Rgap は無関係 → 研究 3 の回路と同一（乱数の使い方も同一で、同じ seed なら同じ結果になる）。
#
# 理想出力：1 サイクルは FSC の配位 a ごとに位相 omega^{F(a)} を掛ける（研究 3 で検証済み）ので、
# K サイクルの理想出力は D_{K F} |+_L> = (T_L^dagger)^K |+_L>。論理 Clifford 安定化子は Delta -> K * Delta。
# ==================================================================================================


class SeqSwitchingCircuit(D4SwitchingCircuit):
    """K 回連続の code switching の回路（時間ステップの列）。ブロック名の最後の要素がサイクル番号 c。
      ('S0', r) / ('S1', r, 'z', c) / ('S1', r, 'x', c) / ('U', c) / ('SO', c) / ('S4', r, c) / ('FIN',)"""

    def __init__(self, L, R0, Rin, Rgap, Rout, K, n_pre_u=2, fsc_schedule='classes'):
        assert K >= 1 and Rgap >= 1 and Rout >= 1
        self.K, self.Rgap = K, Rgap
        super().__init__(L, R0, Rin, Rout, n_pre_u=n_pre_u, fsc_schedule=fsc_schedule)

    def _logical(self):
        """理想出力 D_{K F}|+_L> の論理 Clifford 安定化子 X^t D_{K Delta}。Delta は F について線形なので
        K 倍するだけ。Pauli proxy の Z 部分（研究 3 では Delta の a_i = 2 の量子ビット）は 1 サイクルごとに
        1 回ずつ掛かるので、K サイクルでは「1 サイクルの集合」を K 回掛けたもの（K が偶数なら消える）。"""
        super()._logical()
        K = self.K
        self.proxy_z = sorted(q for q, a in self.Delta_a.items() if a == 2) if K % 2 else []
        self.Fpoly = {m: (K * v) % 8 for m, v in self.Fpoly.items() if (K * v) % 8}
        self.Delta_c0 = (K * self.Delta_c0) % 8
        self.Delta_a = {q: (K * a) % 4 for q, a in self.Delta_a.items() if (K * a) % 4}
        self.Delta_b = {ij: (K * b) % 2 for ij, b in self.Delta_b.items() if (K * b) % 2}

    def logical_proxy(self, fx, sign=1):
        """X^e H X^e の Pauli proxy：X^t * (proxy の Z 部分) * R_e。R_e = D^dag X^e D X^e（D = D_{K Delta}）は
        研究 3 と同じ式を K 倍した係数で計算する。"""
        n = self.nq
        ps = stim.PauliString(n)
        for q in self.t_log:
            ps[q] = 'X'
        zs = stim.PauliString(n)
        for q in self.proxy_z:
            zs[q] = 'Z'
        R_ = stim.PauliString(n)
        ph = 1
        for q, a in self.Delta_a.items():
            if fx[q] and a % 2 == 1:
                ph *= 1j ** a
                R_ *= _single(n, q, 'Z')
            elif fx[q] and a == 2:
                ph *= -1
        for (i, j), b in self.Delta_b.items():
            if fx[i] and fx[j]:
                ph *= -1
                R_ *= _single(n, i, 'Z') * _single(n, j, 'Z')
            elif fx[i]:
                R_ *= _single(n, j, 'Z')
            elif fx[j]:
                R_ *= _single(n, i, 'Z')
        return ps * zs * R_ * complex(ph * sign)

    def _blocks(self):
        B_ = []
        rb, g, alld = self.data_rb, self.data_g, self.data_all
        for r in range(1, self.R0 + 1):
            pre = [[('RX', q) for q in rb]] if r == 1 else []
            B_.append(dict(name=('S0', r), code='FSC', moms=pre + self.round_moms['FSC'], data=rb))
        for c in range(1, self.K + 1):
            for r in range(1, self.Rin + 1 + self.n_pre_u):
                pre = [[('RZ', q) for q in g]] if r == 1 else []
                B_.append(dict(name=('S1', r, 'z', c), code='D4', moms=pre + self.round_moms['D4z'], data=alld))
                if r <= self.Rin:
                    B_.append(dict(name=('S1', r, 'x', c), code='D4', moms=self.round_moms['D4x'], data=alld))
            B_.append(dict(name=('U', c), code=None, moms=self.U_moms, data=alld))
            B_.append(dict(name=('SO', c), code=None,
                           moms=[[('FLIPX', q) for q in g], list(self.network), [('MZ0', q) for q in g]], data=alld))
            for r in range(1, (self.Rout if c == self.K else self.Rgap) + 1):
                B_.append(dict(name=('S4', r, c), code='FSC', moms=self.round_moms['FSC'], data=rb))
        B_.append(dict(name=('FIN',), code='FSC', moms=self.round_moms['FSC'], data=rb, noiseless=True))
        return B_


class SeqFluxDecoder(FluxDecoder):
    """フラックス復号器（研究 3 と同じ）。違いは 1 点：緑の量子ビットは各サイクルの最初に |0> に初期化
    し直されるので、緑のプラケットの検出事象をサイクルをまたいでつながない。"""

    def _cyc(self, k):
        nm = self.fr[k]['name']
        return nm[-1] if nm[0] in ('S1', 'SO') else None

    @staticmethod
    def _is_g(lab):
        return lab[0] == 'plaq' and lab[2] == 'g'

    def _prev(self, k, lab):
        r = super()._prev(k, lab)
        if r is not None and self._is_g(lab) and self._cyc(r[0]) != self._cyc(k):
            return None
        return r

    def _next(self, k, lab):
        kn = super()._next(k, lab)
        if kn is not None and self._is_g(lab) and self._cyc(kn) != self._cyc(k):
            return None
        return kn

    def new_cycle(self):
        """次のサイクルの switch-in の直前：緑のプラケットの参照値を 0（|0> に初期化した直後の値）に戻す。"""
        for lab in list(self.last):
            if self._is_g(lab):
                del self.last[lab]


class SeqChargeDecoder(ChargeDecoder):
    """電荷復号器（研究 3 と同じ構成）。検出器の定義を K サイクルに一般化しただけ：
      * FSC ラウンド：同じ検査の 1 つ前のラウンドとの差
      * 各サイクルの最初の D4 ラウンド：直前の FSC ラウンド（サイクル 1 は S0、以降は前のサイクルの S4）との差
      * 各サイクルの最初の FSC ラウンド：そのサイクルの最後の D4 ラウンドとの差（研究 3 の aligned / 非 aligned の基底）
    論理観測量には各サイクルの S^g の固有値（Pauli フレーム）を含める。"""

    def __init__(self, comp, dem_noise=None, aligned=True):
        circ = comp.circ
        self.comp, self.circ = comp, circ
        nz = dem_noise or comp.noise
        meas_lab, blk_meas = [], []
        for bi, blk in enumerate(comp.blocks):
            ids = []
            for cm in blk['moms']:
                for q, b in cm.meas:
                    if blk['code']:
                        meas_lab.append((blk['name'], b, circ.lab_of[(blk['code'], q)]))
                    else:
                        meas_lab.append((blk['name'], 'R', q))
                    ids.append(len(meas_lab) - 1)
            blk_meas.append(ids)
        self.n_meas, self.blk_meas = len(meas_lab), blk_meas
        idx = {(bn, lab): i for i, (bn, b, lab) in enumerate(meas_lab)}
        self.meas_idx = idx
        sup = {code: {lab: set(xs) for lab, xs, quad in circ.xgens[code]} for code in ('D4', 'FSC')}
        names = [blk['name'] for blk in comp.blocks]
        fscl = [lab for lab, _, _ in circ.xgens['FSC']]
        d4l = [lab for lab, _, _ in circ.xgens['D4']]
        lat, c = circ.lat, circ.c
        inv = {v: k for k, v in c.items()}

        def span_solve(target, basis_labels, code):
            M = np.zeros((len(basis_labels), circ.nq), dtype=np.uint8)
            for i, l in enumerate(basis_labels):
                M[i, list(sup[code][l])] = 1
            v = np.zeros(circ.nq, dtype=np.uint8)
            v[list(target)] = 1
            sol = f2_solve(M, v)
            return None if sol is None else [basis_labels[i] for i in np.nonzero(sol)[0]]

        dets = []
        S0 = [n for n in names if n[0] == 'S0']
        for j, n in enumerate(S0):
            for lab in fscl:
                dets.append([idx[(n, lab)]] + ([idx[(S0[j - 1], lab)]] if j else []))
        prev_fsc = S0[-1] if S0 else None
        self.last_S1x = {}
        for cyc in range(1, circ.K + 1):
            S1 = [n for n in names if n[0] == 'S1' and n[2] == 'x' and n[3] == cyc]
            S4 = [n for n in names if n[0] == 'S4' and n[2] == cyc] + ([('FIN',)] if cyc == circ.K else [])
            self.last_S1x[cyc] = S1[-1]
            for j, n in enumerate(S1):
                for lab in d4l:
                    if j == 0:
                        if prev_fsc is None:
                            continue
                        combo = span_solve(sup['D4'][lab], fscl, 'FSC')
                        if combo is None:
                            continue                   # S^g：最初の結果はランダム（ゲージング）
                        dets.append([idx[(n, lab)]] + [idx[(prev_fsc, l)] for l in combo])
                    else:
                        dets.append([idx[(n, lab)], idx[(S1[j - 1], lab)]])
            for j, n in enumerate(S4):
                first = {}
                for lab in fscl:
                    if j == 0:
                        tgt = set(sup['FSC'][lab])
                        for q in list(sup['FSC'][lab]):
                            gq = lat.q(inv[q] // 3, G)
                            if gq in c:
                                tgt ^= {c[gq]}
                        combo = span_solve(tgt, d4l, 'D4')
                        assert combo is not None, 'S4 X check not determined by the last D4 round'
                        first[lab] = [idx[(n, lab)]] + [idx[(S1[-1], l)] for l in combo]
                    else:
                        dets.append([idx[(n, lab)], idx[(S4[j - 1], lab)]])
                if j == 0:
                    for lab, d in first.items():
                        v, col = lab[1], lab[2]
                        if aligned and col == 'b' and ('X', v, 'r') in first:
                            dets.append(sorted(set(d) ^ set(first[('X', v, 'r')])))
                        else:
                            dets.append(d)
            prev_fsc = S4[-1]
        self.dets = dets
        tset = set(circ.t_log)
        self.cross = [len(set(fs) & tset) % 2 for fs in circ.frame_strings]
        self.obs_meas = [idx[(self.last_S1x[cyc], lab)] for cyc in range(1, circ.K + 1)
                         for (lab, op), cr in zip(circ.gens_g, self.cross) if cr]
        saved = circ.Delta_a                 # 親クラスの _dem_circuit は「Delta_a == 2 の量子ビット」を proxy の
        circ.Delta_a = {q: 2 for q in circ.proxy_z}      # Z 部分として読むので、その集合だけを一時的に渡す
        try:
            self._dem_circuit(nz)
        finally:
            circ.Delta_a = saved
        self.det_ptr = np.cumsum([0] + [len(d) for d in dets])
        self.det_idx = np.concatenate([np.array(d, dtype=np.int64) for d in dets])

    def flux_erasures(self, flux_events, w_erase=0.1):
        """研究 3 の flux_erasures と同じ規則。flux_events の要素は (量子ビット, ラウンド r, サイクル c)。"""
        circ = self.circ
        if not hasattr(self, '_cz_of'):
            self._cz_of, self._stars = {}, {}
            for lab, xs, quad in circ.xgens['D4']:
                for (i, j) in quad:
                    self._cz_of.setdefault(i, []).append((lab, j))
                    self._cz_of.setdefault(j, []).append((lab, i))
                for q in xs:
                    self._stars.setdefault(q, []).append(lab)
        er = set()
        for q, r, cyc in flux_events:
            blk, nxt = ('S1', r, 'x', cyc), ('S1', r + 1, 'x', cyc)
            for lab, partner in self._cz_of.get(q, []):
                m = self.meas_idx.get((blk, lab))
                if m is None:
                    continue
                dd = tuple(sorted(self.dets_of_meas.get(m, [])))
                if 1 <= len(dd) <= 2 and dd in self.comp_index:
                    er.add(self.comp_index[dd])
                st = self._stars.get(partner, [])
                for rr in (blk, nxt):
                    ds = []
                    for l2 in st:
                        m2 = self.meas_idx.get((rr, l2))
                        if m2 is not None:
                            ds += [j for j in self.dets_of_meas.get(m2, []) if self.dets[j][0] == m2]
                    dd = tuple(sorted(ds))
                    if 1 <= len(dd) <= 2 and dd in self.comp_index:
                        er.add(self.comp_index[dd])
        return er


class SeqExperiment(D4Experiment):
    """K 回連続の code switching ＋復号器。shot() は出力 (T_L^dagger)^K|+_L> に対する不忠実度（0, 1/2, 1）を返す。
    1 サイクルの中の処理（実時間のフラックス復号、U の直前の確定、switch-out、Pauli フレーム）は研究 3 の
    D4Experiment.shot と同じ。違いは、(1) X フレームの初期化はサイクル 1 の直前だけ（以降は追跡を続ける）、
    (2) 次のサイクルの直前に緑の参照値を戻す、(3) Pauli フレームと電荷復号を全サイクルまとめて最後に行う。"""

    def __init__(self, L, R0, Rin, Rgap, Rout, K, noise, delay=0, pre_u_age=0, n_pre_u=2, flux_informed=True,
                 quiet_window=False, fsc_schedule='classes'):
        self.pre_u_age, self.flux_informed = pre_u_age, flux_informed
        self.circ = SeqSwitchingCircuit(L, R0, Rin, Rgap, Rout, K, n_pre_u=n_pre_u, fsc_schedule=fsc_schedule)
        self.comp = CompiledProtocol(self.circ, noise)
        self.noise, self.K = noise, K
        self.flux = SeqFluxDecoder(self.comp, delay=delay)
        self.charges = [SeqChargeDecoder(self.comp, aligned=a) for a in (True, False)]
        self.quiet_window = quiet_window
        win = set()
        for c in range(1, K + 1):                   # 切り替えの窓（研究 3 と同じ定義を各サイクルに）
            win |= {('S1', Rin, 'z', c), ('S1', Rin, 'x', c), ('U', c), ('SO', c), ('S4', 1, c)}
            win |= {('S1', r, 'z', c) for r in range(Rin + 1, Rin + 1 + n_pre_u)}
        self.window = win
        if quiet_window:
            for blk in self.comp.blocks:
                if blk['name'] in win:
                    a, b = blk['moms'][0].loc_slice[0], blk['moms'][-1].loc_slice[1]
                    self.comp.loc_p[a:b] = 0.0
        self.gset = set(self.circ.data_g)
        names = [b['name'] for b in self.comp.blocks]
        self._last_S0 = max([i for i, n in enumerate(names) if n[0] == 'S0'], default=-1)
        self._last_S1 = {max(i for i, n in enumerate(names) if n[0] == 'S1' and n[3] == c) for c in range(1, K + 1)}
        self._last_S1x = {max(i for i, n in enumerate(names) if n[0] == 'S1' and n[2] == 'x' and n[3] == c): c
                          for c in range(1, K + 1)}

    def shot(self, rng, decode=True, forced=None, ts=None, init=None):
        circ, comp, fl = self.circ, self.comp, self.flux
        fstatic = {k: v for k, v in forced.items() if not isinstance(k, tuple)} if forced is not None else None
        fdyn = {k: v for k, v in forced.items() if isinstance(k, tuple)} if forced is not None else None
        sh = FastShot(comp, rng, fstatic)
        if init is not None:
            init(sh)
        lock, outs, flux_events, m_g = [], [], [], {}
        fl.start()
        kfr = 0
        stats = dict(n_err=sh.n_err, n_corr=0, so_cz=0)
        for bi, blk in enumerate(comp.blocks):
            nm = blk['name']
            if nm[0] == 'S1' and nm[1] == 1 and nm[2] == 'z':
                if nm[3] == 1:
                    sh.init_frame(circ.zchecks['FSC'], circ.data_rb)
                else:
                    fl.new_cycle()
            if init is not None and nm[0] == 'S0':
                res = [(q, bb, 0) for cm in comp.blocks[bi]['moms'] for (q, bb) in cm.meas]
            elif ts is None:
                res = sh.run_block(bi)
            else:
                r5 = sh.run_block_lockstep(bi, ts, rng)
                lock += r5
                res = [(q, bb, o) for q, bb, o, pk, pr in r5]
            outs += [o for (_, _, o) in res]
            if bi in self._last_S1x:
                gl = {circ.lab_of[('D4', q)]: o for (q, b, o) in res if b == 'X'}
                m_g[self._last_S1x[bi]] = [gl[lab] for lab, _ in circ.gens_g]
            if nm[0] in ('S0', 'S4', 'FIN') or (nm[0] == 'S1' and nm[2] == 'z'):
                code = blk['code']
                zl = {circ.lab_of[(code, q)]: o for (q, b, o) in res if b == 'Z'}
                fl.push_round(kfr, zl)
                kfr += 1
                if decode:
                    flush = bi == self._last_S0 or bi in self._last_S1 or nm[0] == 'FIN'
                    cc = fl.decode(flush=flush, min_age=self.pre_u_age if bi in self._last_S1 else 0)
                    corr = [q for q, k in cc]
                    if nm[0] == 'S1':
                        flux_events += [(q, nm[1] - 1, nm[3]) for q in corr if nm[1] >= 2]
                    sh.apply_x(corr)
                    if ts is not None:
                        for q in corr:
                            ts.X(q)
                    for q in corr:
                        fl.register_correction(q, code)
                    stats['n_corr'] += len(corr)
            elif nm[0] == 'SO':
                readout = {q: o for (q, b, o) in res}
                fl.push_round(kfr, readout=readout)
                kfr += 1
                a_dec = dict(readout)
                phys = []
                if decode:
                    for q, k in fl.decode(flush=True):
                        if q in self.gset:
                            a_dec[q] ^= 1
                        else:
                            phys.append(q)
                    for q in phys:
                        fl.register_correction(q, 'FSC')
                    stats['n_corr'] += len(phys)
                stats['so_cz'] += self._switch_out(sh, readout, a_dec, phys, rng, fdyn,
                                                   noisy=forced is None and not self.quiet_window, ts=ts)
        for cyc in sorted(m_g):                      # 各サイクルの S^g の固有値の Pauli フレーム
            for v, m in enumerate(m_g[cyc]):
                if m:
                    for q in circ.frame_strings[v]:
                        sh.sim.z(q)
                        if ts is not None:
                            ts.Z(q)
        val = sh.sim.peek_observable_expectation(circ.logical_proxy(sh.fx))
        if ts is not None:
            stats['lockstep'] = lock
            stats['H_true'] = ts.expect_logical(circ)
            stats['H_proxy'] = val
        if decode:
            sols = [ch.predict(outs, ch.flux_erasures(flux_events) if self.flux_informed else None)
                    for ch in self.charges]
            flip = min(sols, key=lambda s: s[1])[0]
        else:
            flip = 0
        val = val * (-1) ** flip
        stats['n_dress'] = sh.n_dress
        return (1 - val) / 2, stats


def seq_counts(d, K=1, Rgap=None, R0=None, Rin=None, Rout=None, n_pre_u=2):
    """回路から数えた量子ビット数・時間ステップ数・時空間体積（研究 3 の kobayashi_counts と同じ数え方：
    FSC のブロックでは赤青のデータと FSC の補助、D4 のブロックでは全データと D4 の補助、U と switch-out では
    全データを「使用中」とする）を段階別に返す。"""
    R0 = d if R0 is None else R0
    Rin = d if Rin is None else Rin
    Rout = d if Rout is None else Rout
    Rgap = d if Rgap is None else Rgap
    circ = SeqSwitchingCircuit(d, R0, Rin, Rgap, Rout, K, n_pre_u=n_pre_u)
    out = dict(d=d, K=K, Rgap=Rgap, qubits=circ.nq, steps={}, volume={})
    for blk in circ.blocks:
        nm = blk['name']
        if nm[0] == 'FIN':
            continue
        if blk['code'] == 'FSC':
            active = len(set(circ.data_rb) | set(circ.anc['FSC']))
        elif blk['code'] == 'D4':
            active = len(set(circ.data_all) | set(circ.anc['D4']))
        else:
            active = len(circ.data_all)
        st = block_steps(circ, blk)
        key = 'S0' if nm[0] == 'S0' else ('gap' if (nm[0] == 'S4' and nm[2] < K) else nm[0])
        out['steps'][key] = out['steps'].get(key, 0) + st
        out['volume'][key] = out['volume'].get(key, 0) + st * active
    out['time_steps'] = sum(out['steps'].values())
    out['volume_total'] = sum(out['volume'].values())
    return out

## 参照データ

In [ ]:
# ==================================================================================================
# 参照データ（埋め込み）
#   STUDY3 : 研究 3 の比較表（3_魔法状態蒸留との比較/results_v13/cost_comparison.csv の一部の列）
#            sha256 = fe0722e7f7565e47d58a18034302c5e65cf7846dc707df2ff69873f59948f2b6
#            kobayashi_d_low = データで棄却されない最小のサイズ、ratio = その体積 ÷ 蒸留工場の体積
#   REF3   : 研究 3 の本番データの出力不忠実度（K=1 と同じ回路。比較の参考。停止規則が違うので推定には使わない）
#   EMBED_PILOT_MC : この研究の試行計算（results_pilot/study4_mc.csv）。RUN_MODE = "analysis_only" で使う
# ==================================================================================================
STUDY3 = pd.read_csv(io.StringIO('scenario,p,target,litinski_protocol,litinski_volume,kobayashi_status,kobayashi_d_low,kobayashi_volume_low,ratio_kobayashi_low_over_litinski\nA,0.0001,0.0001,"Litinski2019 (15-to-1)_{5,3,3}",56526.79187626232,unresolved,7,239063.9999999999,4.229215776535015\nA,0.0001,1e-06,"Litinski2019 (15-to-1)_{7,3,3}",87767.20958738509,unresolved,11,896200.0000000002,10.211102804945645\nA,0.0001,1e-08,"Litinski2019 (15-to-1)_{9,3,3}",124256.36275454954,unresolved,11,896200.0000000002,7.212507916156484\nA,0.0001,1e-10,"Litinski2019 (15-to-1)_{11,5,5}",372267.8446072078,unresolved,11,896200.0000000002,2.407406422506383\nA,0.0002,0.0001,"Litinski2019 (15-to-1)_{5,3,3}",56868.214929698,unresolved,9,497560.00000000006,8.749351471909167\nA,0.0002,1e-06,"Litinski2019 (15-to-1)_{9,3,3}",125481.88299060088,unresolved,11,896200.0000000002,7.142066875639165\nA,0.0002,1e-08,"Litinski2019 (15-to-1)_{11,5,5}",372750.8306625909,unresolved,11,896200.0000000002,2.404287063309668\nA,0.0002,1e-10,"Litinski2019 (15-to-1)_{15,7,7}",984273.443164155,unresolved,11,896200.0000000002,0.9105193340571864\nA,0.0003,0.0001,"Litinski2019 (15-to-1)_{7,3,3}",89560.30978164391,unresolved,11,896200.0000000002,10.006664807044732\nA,0.0003,1e-06,"Litinski2019 (15-to-1)_{9,3,3}",127466.46386370136,unresolved,11,896200.0000000002,7.030868926891218\nA,0.0003,1e-08,"Litinski2019 (15-to-1)_{13,5,5}",468954.4962627512,unresolved,11,896200.0000000002,1.9110596169609324\nA,0.0003,1e-10,"Litinski2019 (15-to-1)x(15-to-1)_{7,3,3;15,7,7;6}",3752104.5896285134,unresolved,11,896200.0000000002,0.2388526168692783\nA,0.0005,0.0001,"Litinski2019 (15-to-1)_{7,3,3}",93071.58648072714,unresolved,11,896200.0000000002,9.629147131660654\nA,0.0005,1e-06,"Litinski2019 (15-to-1)_{11,5,5}",375728.408334208,unresolved,11,896200.0000000002,2.385233536035519\nA,0.0005,1e-08,"Litinski2019 (15-to-1)_{15,7,7}",987642.054253026,unresolved,11,896200.0000000002,0.9074137701414656\nA,0.0005,1e-10,"Litinski2019 (15-to-1)x(15-to-1)_{9,3,3;19,7,7;6}",4925823.933670754,unresolved,11,896200.0000000002,0.1819391054304587\nA,0.001,0.0001,"Litinski2019 (15-to-1)_{11,5,5}",391872.8035107588,unresolved,11,896200.0000000002,2.286966566628284\nA,0.001,1e-06,"Litinski2019 (15-to-1)_{15,7,7}",998970.9923264446,unresolved,11,896200.0000000002,0.8971231466019778\nA,0.001,1e-08,"Litinski2019 (15-to-1)x(15-to-1)_{9,3,3;21,9,9;6}",7844942.178976283,unresolved,11,896200.0000000002,0.1142392103796166\nA,0.001,1e-10,"Litinski2019 (15-to-1)x(15-to-1)_{11,5,5;25,11,11;6}",15224985.923137063,unresolved,11,896200.0000000002,0.0588637654264143\nB,0.0001,0.0001,"GoSC 15-to-1 (11 tiles x 11 tau), d=3",39261.03072376387,data_certified,5,90375.9999999998,2.3019263206785125\nB,0.0001,1e-06,"Litinski2019 (15-to-1)_{7,3,3}",87618.54780619452,unresolved,7,239063.9999999999,2.7284633902948388\nB,0.0001,1e-08,"Litinski2019 (15-to-1)_{7,3,3}",87618.54780619452,unresolved,7,239063.9999999999,2.7284633902948388\nB,0.0001,1e-10,"Litinski2019 (15-to-1)_{11,3,3}",165559.78665298878,unresolved,9,497560.00000000006,3.0053191663195333\nB,0.0002,0.0001,"Litinski2019 (15-to-1)_{5,3,3}",56585.572233310544,unresolved,5,90375.9999999998,1.5971562437747633\nB,0.0002,1e-06,"Litinski2019 (15-to-1)_{7,3,3}",87859.62029944417,unresolved,11,896200.0000000002,10.20036277126581\nB,0.0002,1e-08,"Litinski2019 (15-to-1)_{9,3,3}",124389.07419012638,unresolved,11,896200.0000000002,7.204812848998098\nB,0.0002,1e-10,"Litinski2019 (15-to-1)_{13,5,5}",467888.97232910106,unresolved,11,896200.0000000002,1.9154116745663248\nB,0.0003,0.0001,"Litinski2019 (15-to-1)_{5,3,3}",56763.619639961566,unresolved,5,90375.9999999998,1.592146529295238\nB,0.0003,1e-06,"Litinski2019 (15-to-1)_{7,3,3}",88204.07574075514,unresolved,11,896200.0000000002,10.160528212257049\nB,0.0003,1e-08,"Litinski2019 (15-to-1)_{11,5,5}",373075.3815434044,unresolved,11,896200.0000000002,2.4021954927512\nB,0.0003,1e-10,"Litinski2019 (15-to-1)x(15-to-1)_{7,3,3;13,5,5;6}",2524288.6364958268,unresolved,11,896200.0000000002,0.3550307152054091\nB,0.0005,0.0001,"Litinski2019 (15-to-1)_{7,3,3}",89208.42723495633,unresolved,7,239063.9999999999,2.67983650659321\nB,0.0005,1e-06,"Litinski2019 (15-to-1)_{9,3,3}",126747.23685134496,unresolved,11,896200.0000000002,7.07076558245688\nB,0.0005,1e-08,"Litinski2019 (15-to-1)_{13,5,5}",469796.119942172,unresolved,11,896200.0000000002,1.907636019025264\nB,0.0005,1e-10,"Litinski2019 (15-to-1)x(15-to-1)_{7,3,3;15,7,7;6}",3752109.6362641496,unresolved,11,896200.0000000002,0.2388522956094419\nB,0.001,0.0001,"Litinski2019 (15-to-1)_{7,3,3}",93662.06076657082,unresolved,11,896200.0000000002,9.568442042221918\nB,0.001,1e-06,"Litinski2019 (15-to-1)_{13,5,5}",475935.5489742533,unresolved,11,896200.0000000002,1.883028073720297\nB,0.001,1e-08,"Litinski2019 (15-to-1)x(15-to-1)_{7,3,3;17,7,7;6}",4014871.081406037,unresolved,11,896200.0000000002,0.223220118860241\nB,0.001,1e-10,"Litinski2019 (15-to-1)x(15-to-1)_{7,3,3;21,9,9;4}",6559746.271210219,unresolved,11,896200.0000000002,0.1366211379140216\n'))
REF3 = pd.read_csv(io.StringIO('scenario,d,p,shots,n_half,n_one,eps\nA,3,0.0001,70800,8,192,0.0027683615819209\nA,3,0.0002,20400,10,190,0.0095588235294117\nA,3,0.0003,10800,5,196,0.0183796296296296\nA,3,0.0005,4000,8,215,0.05475\nA,3,0.001,1200,12,200,0.1716666666666666\nA,5,0.0001,199600,29,57,0.0003582164328657\nA,5,0.0002,72800,28,114,0.0017582417582417\nA,5,0.0003,37600,32,171,0.0049734042553191\nA,5,0.0005,7600,19,185,0.0255921052631578\nA,5,0.001,1200,17,219,0.1895833333333333\nA,7,0.0001,56900,3,5,0.0001142355008787\nA,7,0.0002,23500,12,16,0.0009361702127659\nA,7,0.0003,24500,17,57,0.0026734693877551\nA,7,0.0005,12100,23,178,0.0156611570247933\nA,7,0.001,1100,16,198,0.1872727272727272\nA,9,0.0001,23800,2,8,0.0003781512605042\nA,9,0.0002,9400,0,5,0.000531914893617\nA,9,0.0003,9960,9,22,0.0026606425702811\nA,9,0.0005,8560,16,113,0.0141355140186915\nA,9,0.001,880,11,192,0.2244318181818181\nB,3,0.0001,433200,3,197,0.000458217913204\nB,3,0.0002,217200,7,195,0.0009139042357274\nB,3,0.0003,121600,5,195,0.0016241776315789\nB,3,0.0005,57200,13,190,0.0034353146853146\nB,3,0.001,19200,12,189,0.01015625\nB,5,0.0001,414400,0,12,2.895752895752896e-05\nB,5,0.0002,297200,6,35,0.0001278600269179\nB,5,0.0003,115600,6,27,0.0002595155709342\nB,5,0.0005,101600,10,100,0.0010334645669291\nB,5,0.001,40400,26,174,0.0046287128712871\nB,7,0.0001,125800,1,1,1.192368839427663e-05\nB,7,0.0002,96500,1,6,6.735751295336788e-05\nB,7,0.0003,33100,4,4,0.0001812688821752\nB,7,0.0005,27300,0,11,0.0004029304029304\nB,7,0.001,25300,11,66,0.0028260869565217\nB,9,0.0001,48440,0,0,0.0\nB,9,0.0002,31040,0,4,0.0001288659793814\nB,9,0.0003,11520,1,2,0.0002170138888888\nB,9,0.0005,10800,1,3,0.000324074074074\nB,9,0.001,9640,5,31,0.0034751037344398\n'))
EMBED_PILOT_MC = 'run_id,point_id,chunk_id,seed_key,scenario,window,d,p,p_idle,K,gap,R0,Rin,Rgap,Rout,n_pre_u,shots,n0,n_half,n_one,sum_infidelity,mean_errors,mean_so_cz,seconds\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,0,20261111-287246149-0,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3950,2,48,49.0,2.91325,4.0165,12.96906566619873\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,1,20261111-287246149-1,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3957,0,43,43.0,2.95825,4.036,13.068977117538452\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,2,20261111-287246149-2,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3951,0,49,49.0,2.9035,4.0205,11.48726224899292\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,3,20261111-287246149-3,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3960,4,36,38.0,2.989,4.0065,12.822072982788086\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,4,20261111-287246149-4,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3959,3,38,39.5,2.9365,3.9775,11.988829612731934\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,5,20261111-287246149-5,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3953,5,42,44.5,2.91125,4.025,13.524627685546875\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,6,20261111-287246149-6,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3956,3,41,42.5,2.9045,3.963,12.136350870132446\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,7,20261111-287246149-7,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3960,4,36,38.0,2.93,4.023,11.0081787109375\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,8,20261111-287246149-8,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3958,4,38,40.0,2.94925,3.9965,12.25978422164917\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K1|g0,9,20261111-287246149-9,B,noisy,3,0.001,0.0,1,0,3,3,3,3,2,4000,3961,5,34,36.5,2.9045,4.0935,12.381903409957886\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,0,20261111-2594437393-0,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1954,6,40,43.0,5.074,8.096,10.237789630889893\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,1,20261111-2594437393-1,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1951,2,47,48.0,5.1155,8.076,10.984229326248169\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,2,20261111-2594437393-2,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1966,5,29,31.5,5.0455,7.93,9.553319215774536\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,3,20261111-2594437393-3,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1954,4,42,44.0,5.085,7.934,11.072781324386597\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,4,20261111-2594437393-4,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1955,2,43,44.0,5.1945,8.038,10.396265506744385\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,5,20261111-2594437393-5,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1951,1,48,48.5,5.0645,8.04,9.758470058441162\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,6,20261111-2594437393-6,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1954,2,44,45.0,5.0435,8.057,9.298441171646118\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,7,20261111-2594437393-7,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1957,1,42,42.5,5.162,8.057,9.71068024635315\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,8,20261111-2594437393-8,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1964,3,33,34.5,5.15,7.957,10.868411302566528\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,9,20261111-2594437393-9,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1957,1,42,42.5,5.1265,8.059,9.825258493423462\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,10,20261111-2594437393-10,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1940,1,59,59.5,5.16,7.991,9.718731880187988\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,11,20261111-2594437393-11,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1948,2,50,51.0,5.042,7.951,10.453017950057983\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,12,20261111-2594437393-12,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1959,5,36,38.5,5.111,7.979,9.547119617462158\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,13,20261111-2594437393-13,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1948,3,49,50.5,5.126,7.992,9.991023063659668\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,14,20261111-2594437393-14,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1973,3,24,25.5,5.1145,7.932,11.39089322090149\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,15,20261111-2594437393-15,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1958,3,39,40.5,5.087,7.99,10.12967038154602\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,16,20261111-2594437393-16,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1971,2,27,28.0,4.998,7.988,10.77712631225586\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,17,20261111-2594437393-17,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1952,2,46,47.0,5.0025,8.019,9.385026693344116\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,18,20261111-2594437393-18,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1973,0,27,27.0,5.0675,7.824,9.769144296646118\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g3,19,20261111-2594437393-19,B,noisy,3,0.001,0.0,2,3,3,3,3,3,2,2000,1947,4,49,51.0,5.094,8.068,10.579572916030884\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,0,20261111-1957533757-0,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1955,4,41,43.0,4.5925,8.027,9.559643983840942\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,1,20261111-1957533757-1,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1955,3,42,43.5,4.659,7.953,9.583370208740234\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,2,20261111-1957533757-2,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1958,3,39,40.5,4.6765,7.956,10.317213296890259\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,3,20261111-1957533757-3,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1963,4,33,35.0,4.6435,8.124,8.808956384658813\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,4,20261111-1957533757-4,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1939,4,57,59.0,4.677,8.058,10.336633443832397\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,5,20261111-1957533757-5,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1958,5,37,39.5,4.6465,8.025,9.791512489318848\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,6,20261111-1957533757-6,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1960,1,39,39.5,4.601,7.997,10.472139358520508\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,7,20261111-1957533757-7,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1954,2,44,45.0,4.6915,7.958,8.776320695877075\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,8,20261111-1957533757-8,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1959,3,38,39.5,4.649,7.97,9.063751459121704\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,9,20261111-1957533757-9,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1947,4,49,51.0,4.6445,8.109,8.84510350227356\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,10,20261111-1957533757-10,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1956,3,41,42.5,4.65,8.061,7.467664003372192\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,11,20261111-1957533757-11,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1956,1,43,43.5,4.6585,8.019,7.1196815967559814\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,12,20261111-1957533757-12,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1959,1,40,40.5,4.6105,7.998,7.303375959396362\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,13,20261111-1957533757-13,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1957,3,40,41.5,4.626,8.096,7.246519088745117\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,14,20261111-1957533757-14,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1959,4,37,39.0,4.6835,7.936,7.150564908981323\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,15,20261111-1957533757-15,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1953,3,44,45.5,4.5995,8.051,7.132821321487427\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,16,20261111-1957533757-16,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1960,0,40,40.0,4.712,8.13,6.634521484375\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,17,20261111-1957533757-17,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1970,0,30,30.0,4.6505,8.025,7.051669120788574\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,18,20261111-1957533757-18,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1958,1,41,41.5,4.5725,7.981,7.373063325881958\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K2|g1,19,20261111-1957533757-19,B,noisy,3,0.001,0.0,2,1,3,3,1,3,2,2000,1965,3,32,33.5,4.6255,7.945,7.18055534362793\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,0,20261111-572497524-0,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1286,0,47,47.0,7.357089272318079,11.932483120780194,7.407581090927124\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,1,20261111-572497524-1,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1281,2,50,51.0,7.271567891972993,12.168042010502626,7.676324129104614\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,2,20261111-572497524-2,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1292,1,40,40.5,7.337584396099024,11.996999249812452,7.351632356643677\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,3,20261111-572497524-3,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1296,2,35,36.0,7.250562640660165,11.789947486871718,7.720710277557373\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,4,20261111-572497524-4,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1299,1,33,33.5,7.146286571642911,11.992498124531133,7.096222400665283\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,5,20261111-572497524-5,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1295,3,35,36.5,7.41185296324081,12.17404351087772,7.304926872253418\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,6,20261111-572497524-6,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1275,1,57,57.5,7.2880720180045016,11.959489872468117,7.1318628787994385\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,7,20261111-572497524-7,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1280,2,51,52.0,7.290322580645161,12.109527381845462,6.988297939300537\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,8,20261111-572497524-8,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1290,2,41,42.0,7.2880720180045016,12.019504876219054,7.258272886276245\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,9,20261111-572497524-9,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1284,1,48,48.5,7.330832708177044,11.914478619654913,7.190066337585449\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,10,20261111-572497524-10,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1286,2,45,46.0,7.2250562640660165,11.96099024756189,7.113415002822876\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,11,20261111-572497524-11,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1276,3,54,55.5,7.351087771942986,12.01350337584396,6.773128271102905\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,12,20261111-572497524-12,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1299,2,32,33.0,7.452363090772693,12.088522130532633,7.252416610717773\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,13,20261111-572497524-13,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1285,1,47,47.5,7.177794448612153,12.024006001500375,7.050556898117065\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,14,20261111-572497524-14,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1295,1,37,37.5,7.210802700675169,12.163540885221305,7.504806995391846\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,15,20261111-572497524-15,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1294,1,38,38.5,7.351087771942986,12.14853713428357,7.072806358337402\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,16,20261111-572497524-16,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1291,4,38,40.0,7.228057014253563,11.783945986496624,6.843209266662598\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,17,20261111-572497524-17,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1284,7,42,45.5,7.348837209302325,12.12603150787697,7.416460752487183\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,18,20261111-572497524-18,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1298,6,29,32.0,7.385596399099775,12.138034508627157,7.5317277908325195\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,19,20261111-572497524-19,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1293,2,38,39.0,7.348837209302325,12.009002250562641,7.319109201431274\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,20,20261111-572497524-20,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1290,4,39,41.0,7.30907726931733,12.024006001500375,7.001860857009888\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,21,20261111-572497524-21,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1299,0,34,34.0,7.298574643660915,11.929482370592648,7.120044231414795\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,22,20261111-572497524-22,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1292,2,39,40.0,7.146286571642911,12.08252063015754,6.952863454818726\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,23,20261111-572497524-23,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1291,3,39,40.5,7.2685671417854465,11.91747936984246,7.653223752975464\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,24,20261111-572497524-24,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1287,0,46,46.0,7.294073518379595,11.962490622655665,6.784537315368652\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,25,20261111-572497524-25,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1294,1,38,38.5,7.354838709677419,11.981995498874719,7.061983346939087\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,26,20261111-572497524-26,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1287,2,44,45.0,7.246061515378845,11.815453863465866,7.031303405761719\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,27,20261111-572497524-27,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1296,0,37,37.0,7.196549137284321,11.941485371342836,7.316913843154907\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,28,20261111-572497524-28,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1291,4,38,40.0,7.285071267816954,12.033008252063016,7.106453895568848\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,29,20261111-572497524-29,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1287,3,43,44.5,7.290322580645161,12.0,6.79573392868042\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g3,30,20261111-572497524-30,B,noisy,3,0.001,0.0,3,3,3,3,3,3,2,1333,1298,1,34,34.5,7.395348837209302,12.012003000750187,7.2791664600372314\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,0,20261111-3423731544-0,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1291,1,41,41.5,6.232558139534884,11.845461365341336,6.612569570541382\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,1,20261111-3423731544-1,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1285,1,47,47.5,6.195048762190548,12.03450862715679,6.748072862625122\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,2,20261111-3423731544-2,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1281,2,50,51.0,6.306076519129783,11.98649662415604,6.428628921508789\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,3,20261111-3423731544-3,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1283,3,47,48.5,6.372093023255814,11.86046511627907,6.674243927001953\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,4,20261111-3423731544-4,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1285,3,45,46.5,6.301575393848462,11.977494373593398,6.278079032897949\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,5,20261111-3423731544-5,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1288,4,41,43.0,6.428357089272318,12.028507126781696,6.6008217334747314\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,6,20261111-3423731544-6,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1292,2,39,40.0,6.280570142535634,12.075018754688672,6.178745746612549\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,7,20261111-3423731544-7,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1286,3,44,45.5,6.383345836459115,12.12603150787697,6.0865724086761475\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,8,20261111-3423731544-8,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1275,4,54,56.0,6.486871717929483,11.897974493623407,6.316563367843628\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,9,20261111-3423731544-9,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1285,1,47,47.5,6.377344336084021,12.090022505626406,6.279538154602051\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,10,20261111-3423731544-10,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1282,2,49,50.0,6.378844711177795,11.88597149287322,6.063737392425537\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,11,20261111-3423731544-11,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1301,1,31,31.5,6.271567891972993,12.045011252813204,6.414388656616211\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,12,20261111-3423731544-12,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1300,2,31,32.0,6.423105776444111,11.933983495873969,6.644694805145264\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,13,20261111-3423731544-13,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1295,2,36,37.0,6.387096774193548,12.177044261065266,6.102229356765747\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,14,20261111-3423731544-14,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1296,1,36,36.5,6.370592648162041,12.13503375843961,6.064882278442383\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,15,20261111-3423731544-15,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1286,4,43,45.0,6.433608402100525,12.12603150787697,6.726909875869751\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,16,20261111-3423731544-16,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1295,0,38,38.0,6.2700675168792195,11.807951987996999,6.1857688426971436\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,17,20261111-3423731544-17,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1282,4,47,49.0,6.35258814703676,12.088522130532633,6.660871744155884\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,18,20261111-3423731544-18,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1288,3,42,43.5,6.388597149287322,12.150037509377345,6.285626649856567\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,19,20261111-3423731544-19,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1285,1,47,47.5,6.284321080270067,11.969992498124531,6.233523845672607\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,20,20261111-3423731544-20,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1291,1,41,41.5,6.559639909977494,11.968492123030758,6.188936948776245\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,21,20261111-3423731544-21,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1287,4,42,44.0,6.381845461365342,11.86496624156039,6.625460147857666\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,22,20261111-3423731544-22,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1288,4,41,43.0,6.421605401350337,11.809452363090772,6.252227067947388\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,23,20261111-3423731544-23,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1286,0,47,47.0,6.285821455363841,11.989497374343586,6.930261850357056\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,24,20261111-3423731544-24,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1291,2,40,41.0,6.40510127531883,11.927981995498875,6.912883520126343\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,25,20261111-3423731544-25,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1286,4,43,45.0,6.383345836459115,12.154538634658664,6.570104122161865\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,26,20261111-3423731544-26,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1289,3,41,42.5,6.4463615903976,11.857464366091524,6.752312183380127\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,27,20261111-3423731544-27,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1289,3,41,42.5,6.468867216804201,12.030007501875469,6.220655918121338\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,28,20261111-3423731544-28,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1286,4,43,45.0,6.326331582895724,12.010502625656414,6.291172742843628\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,29,20261111-3423731544-29,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1280,2,51,52.0,6.41635408852213,11.981995498874719,6.183306932449341\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.001|K3|g1,30,20261111-3423731544-30,B,noisy,3,0.001,0.0,3,1,3,3,1,3,2,1333,1281,1,51,51.5,6.428357089272318,12.190547636909228,6.467259407043457\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,0,20261111-3923594483-0,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,994,1,5,5.5,12.442,16.14,11.505189895629883\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,1,20261111-3923594483-1,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,994,0,6,6.0,12.825,16.128,12.163544178009033\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,2,20261111-3923594483-2,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,997,1,2,2.5,12.661,15.942,12.212944746017456\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,3,20261111-3923594483-3,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,990,0,10,10.0,12.541,15.948,12.6995849609375\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,4,20261111-3923594483-4,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,12.579,16.002,13.011579990386963\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,5,20261111-3923594483-5,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,994,0,6,6.0,12.677,16.204,13.596638917922974\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,6,20261111-3923594483-6,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,996,1,3,3.5,12.686,15.866,14.822075128555298\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,7,20261111-3923594483-7,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,0,5,5.0,12.543,16.076,14.967991590499878\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,8,20261111-3923594483-8,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,997,0,3,3.0,12.696,16.174,13.677106857299805\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,9,20261111-3923594483-9,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,1,4,4.5,12.703,16.132,11.983452081680298\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,10,20261111-3923594483-10,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,0,5,5.0,12.663,15.89,12.295887231826782\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,11,20261111-3923594483-11,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,992,2,6,7.0,12.701,16.036,12.551226615905762\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,12,20261111-3923594483-12,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,996,0,4,4.0,12.636,16.102,11.15707278251648\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,13,20261111-3923594483-13,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,994,0,6,6.0,12.531,16.02,11.525248765945435\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,14,20261111-3923594483-14,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,0,5,5.0,12.819,15.964,12.711896181106567\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,15,20261111-3923594483-15,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,992,0,8,8.0,12.743,15.854,14.067663431167603\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,16,20261111-3923594483-16,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,1,4,4.5,12.735,16.096,20.04297161102295\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,17,20261111-3923594483-17,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,998,2,0,1.0,12.514,16.088,20.09153938293457\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,18,20261111-3923594483-18,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,994,1,5,5.5,12.839,16.156,21.46477699279785\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,19,20261111-3923594483-19,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,992,0,8,8.0,12.657,15.886,20.95246410369873\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,20,20261111-3923594483-20,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,993,0,7,7.0,12.655,16.096,16.655065298080444\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,21,20261111-3923594483-21,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,997,0,3,3.0,12.679,15.962,15.495616912841797\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,22,20261111-3923594483-22,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,997,1,2,2.5,12.589,15.844,13.418215990066528\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,23,20261111-3923594483-23,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,1,4,4.5,12.599,15.996,13.601560831069946\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,24,20261111-3923594483-24,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,997,0,3,3.0,12.443,16.176,10.822978496551514\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,25,20261111-3923594483-25,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,993,2,5,6.0,12.767,16.076,9.89562201499939\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,26,20261111-3923594483-26,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,993,0,7,7.0,12.893,15.974,10.095325231552124\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,27,20261111-3923594483-27,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,989,2,9,10.0,12.719,15.996,10.985969066619873\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,28,20261111-3923594483-28,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,1,4,4.5,12.569,15.856,11.234853267669678\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,29,20261111-3923594483-29,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,12.576,16.016,9.682147979736328\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,30,20261111-3923594483-30,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,993,0,7,7.0,12.637,16.07,10.474914312362671\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,31,20261111-3923594483-31,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,997,0,3,3.0,12.576,15.726,10.06219744682312\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,32,20261111-3923594483-32,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,993,2,5,6.0,12.402,16.042,10.290657043457031\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,33,20261111-3923594483-33,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,1,4,4.5,12.655,16.086,10.33549952507019\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,34,20261111-3923594483-34,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,12.65,15.984,10.133275508880615\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,35,20261111-3923594483-35,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,992,1,7,7.5,12.65,15.89,9.85966968536377\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,36,20261111-3923594483-36,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,996,0,4,4.0,12.613,16.13,10.180455207824707\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,37,20261111-3923594483-37,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,1,4,4.5,12.72,15.888,10.537306070327759\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,38,20261111-3923594483-38,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,995,0,5,5.0,12.611,15.908,9.807517528533936\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K1|g0,39,20261111-3923594483-39,B,noisy,5,0.001,0.0,1,0,5,5,5,5,2,1000,992,2,6,7.0,12.633,16.024,9.766411304473877\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,0,20261111-2332170130-0,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,494,1,5,5.5,22.218,31.96,11.429619073867798\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,1,20261111-2332170130-1,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,499,1,0,0.5,22.248,32.148,11.128831386566162\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,2,20261111-2332170130-2,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,21.806,32.18,11.790242195129395\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,3,20261111-2332170130-3,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,21.89,32.052,12.258756637573242\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,4,20261111-2332170130-4,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,22.258,31.708,8.936520338058472\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,5,20261111-2332170130-5,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,1,2,2.5,22.116,31.612,10.801350116729736\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,6,20261111-2332170130-6,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,21.852,31.768,10.576577186584473\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,7,20261111-2332170130-7,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,1,3,3.5,21.822,32.256,10.754811763763428\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,8,20261111-2332170130-8,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,21.992,31.56,9.763111352920532\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,9,20261111-2332170130-9,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,22.182,32.268,8.600273132324219\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,10,20261111-2332170130-10,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,21.814,31.892,9.427713632583618\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,11,20261111-2332170130-11,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,492,0,8,8.0,22.294,31.824,8.61355185508728\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,12,20261111-2332170130-12,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,2,2,3.0,22.39,32.068,9.714309215545654\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,13,20261111-2332170130-13,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,1,4,4.5,21.958,32.236,9.619978427886963\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,14,20261111-2332170130-14,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,0,7,7.0,22.132,32.096,10.141875982284546\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,15,20261111-2332170130-15,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,22.36,31.8,9.78514289855957\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,16,20261111-2332170130-16,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,1,3,3.5,22.132,32.208,9.028535842895508\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,17,20261111-2332170130-17,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,0,7,7.0,21.932,31.988,8.962188005447388\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,18,20261111-2332170130-18,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,22.214,32.472,9.407081842422485\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,19,20261111-2332170130-19,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,1,2,2.5,21.84,32.12,9.717769622802734\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,20,20261111-2332170130-20,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,492,1,7,7.5,22.25,31.796,9.876884698867798\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,21,20261111-2332170130-21,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,21.94,32.14,8.8810875415802\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,22,20261111-2332170130-22,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,494,1,5,5.5,22.1,31.64,8.772115468978882\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,23,20261111-2332170130-23,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,1,2,2.5,22.338,32.044,9.034980058670044\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,24,20261111-2332170130-24,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,0,7,7.0,21.928,31.648,9.45190715789795\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,25,20261111-2332170130-25,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,0,4,4.0,22.062,32.016,9.563207387924194\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,26,20261111-2332170130-26,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,494,1,5,5.5,22.104,31.884,9.770155906677246\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,27,20261111-2332170130-27,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,22.304,31.596,9.411582231521606\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,28,20261111-2332170130-28,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,22.482,31.844,10.23119854927063\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,29,20261111-2332170130-29,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,492,1,7,7.5,22.416,31.836,9.687883615493774\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,30,20261111-2332170130-30,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,1,3,3.5,21.622,31.608,9.661097288131714\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,31,20261111-2332170130-31,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,1,4,4.5,21.972,32.204,9.054458618164062\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,32,20261111-2332170130-32,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,1,4,4.5,22.202,31.808,9.846804857254028\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,33,20261111-2332170130-33,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,0,7,7.0,21.722,31.944,8.885980606079102\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,34,20261111-2332170130-34,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,21.728,32.1,10.788671493530273\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,35,20261111-2332170130-35,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,21.926,31.956,9.146042346954346\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,36,20261111-2332170130-36,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,494,2,4,5.0,22.314,31.716,9.573642253875732\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,37,20261111-2332170130-37,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,1,6,6.5,22.04,32.452,9.992201805114746\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,38,20261111-2332170130-38,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,22.216,31.872,10.400100231170654\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,39,20261111-2332170130-39,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,22.286,32.336,9.846801042556763\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,40,20261111-2332170130-40,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,494,1,5,5.5,22.062,31.908,8.609122514724731\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,41,20261111-2332170130-41,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,21.808,31.852,9.195465803146362\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,42,20261111-2332170130-42,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,0,7,7.0,22.048,31.796,10.322142362594604\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,43,20261111-2332170130-43,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,21.968,31.668,8.905013084411621\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,44,20261111-2332170130-44,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,21.86,32.02,8.833975553512573\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,45,20261111-2332170130-45,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,492,1,7,7.5,21.92,32.172,9.355049848556519\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,46,20261111-2332170130-46,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,498,1,1,1.5,22.256,31.516,9.537389755249023\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,47,20261111-2332170130-47,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,494,1,5,5.5,22.286,31.232,9.631956815719604\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,48,20261111-2332170130-48,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,1,6,6.5,21.834,32.18,9.011334419250488\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,49,20261111-2332170130-49,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,0,7,7.0,21.744,31.924,9.069812774658203\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,50,20261111-2332170130-50,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,1,4,4.5,21.702,32.3,9.01302695274353\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,51,20261111-2332170130-51,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,21.912,32.228,10.13802170753479\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,52,20261111-2332170130-52,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,22.394,32.02,9.104877710342407\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,53,20261111-2332170130-53,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,494,0,6,6.0,22.16,32.316,8.997071981430054\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,54,20261111-2332170130-54,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,21.964,32.4,10.573339939117432\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,55,20261111-2332170130-55,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,1,2,2.5,21.898,31.96,9.88200306892395\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,56,20261111-2332170130-56,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,1,3,3.5,21.838,31.904,9.02564525604248\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,57,20261111-2332170130-57,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,1,3,3.5,22.156,31.712,9.503344058990479\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,58,20261111-2332170130-58,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,0,4,4.0,21.926,32.012,9.918544054031372\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,59,20261111-2332170130-59,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,0,4,4.0,22.11,31.972,8.58421802520752\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,60,20261111-2332170130-60,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,0,4,4.0,21.826,32.32,8.842278718948364\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,61,20261111-2332170130-61,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,21.954,32.328,9.587282180786133\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,62,20261111-2332170130-62,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,21.374,31.968,9.836169004440308\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,63,20261111-2332170130-63,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,0,4,4.0,22.23,31.896,9.54458999633789\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,64,20261111-2332170130-64,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,498,1,1,1.5,21.98,32.128,10.077603578567505\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,65,20261111-2332170130-65,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,22.228,31.564,9.466383218765259\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,66,20261111-2332170130-66,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,0,4,4.0,21.758,31.82,9.173199653625488\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,67,20261111-2332170130-67,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,491,3,6,7.5,21.998,31.94,10.02225136756897\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,68,20261111-2332170130-68,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,22.19,31.704,10.664614915847778\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,69,20261111-2332170130-69,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,22.25,31.748,9.345597505569458\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,70,20261111-2332170130-70,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,1,3,3.5,21.772,32.18,9.19713568687439\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,71,20261111-2332170130-71,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,22.262,31.68,9.455997705459595\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,72,20261111-2332170130-72,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,494,1,5,5.5,22.092,32.032,10.027647018432617\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,73,20261111-2332170130-73,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,495,0,5,5.0,21.99,31.28,10.446001052856445\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,74,20261111-2332170130-74,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,493,1,6,6.5,21.524,31.664,8.843361377716064\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,75,20261111-2332170130-75,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,496,0,4,4.0,21.882,32.224,9.52334475517273\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,76,20261111-2332170130-76,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,491,0,9,9.0,21.87,32.456,10.488717317581177\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,77,20261111-2332170130-77,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,21.844,32.252,9.902100086212158\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,78,20261111-2332170130-78,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,497,1,2,2.5,22.226,32.064,12.219889402389526\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g5,79,20261111-2332170130-79,B,noisy,5,0.001,0.0,2,5,5,5,5,5,2,500,491,3,6,7.5,21.428,31.908,11.564979076385498\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,0,20261111-2356138891-0,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.198,31.992,12.381074666976929\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,1,20261111-2356138891-1,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,491,1,8,8.5,19.606,31.964,12.495081663131714\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,2,20261111-2356138891-2,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,0,5,5.0,19.516,31.676,13.467001676559448\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,3,20261111-2356138891-3,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,0,6,6.0,19.374,31.656,13.681724071502686\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,4,20261111-2356138891-4,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.36,31.696,12.684715032577515\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,5,20261111-2356138891-5,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,19.592,32.124,10.505643367767334\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,6,20261111-2356138891-6,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,1,4,4.5,19.496,32.172,10.146586179733276\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,7,20261111-2356138891-7,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,19.134,31.86,9.74834680557251\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,8,20261111-2356138891-8,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,0,5,5.0,19.642,32.748,8.242084741592407\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,9,20261111-2356138891-9,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,492,1,7,7.5,19.584,32.308,9.373690843582153\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,10,20261111-2356138891-10,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,492,0,8,8.0,19.236,31.54,8.235902070999146\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,11,20261111-2356138891-11,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,492,1,7,7.5,19.49,32.236,8.971448183059692\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,12,20261111-2356138891-12,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.374,32.248,8.397473573684692\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,13,20261111-2356138891-13,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,1,5,5.5,19.838,31.996,8.388285160064697\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,14,20261111-2356138891-14,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,492,0,8,8.0,19.524,32.012,8.501930236816406\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,15,20261111-2356138891-15,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,491,3,6,7.5,19.51,31.812,8.075255632400513\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,16,20261111-2356138891-16,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,1,4,4.5,19.236,31.736,8.689165830612183\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,17,20261111-2356138891-17,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.304,31.844,8.12757420539856\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,18,20261111-2356138891-18,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.652,32.22,8.051035404205322\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,19,20261111-2356138891-19,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,1,2,2.5,19.666,32.42,8.115971803665161\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,20,20261111-2356138891-20,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.294,31.684,9.352284908294678\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,21,20261111-2356138891-21,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.298,31.86,8.827108383178711\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,22,20261111-2356138891-22,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.53,32.344,11.67309832572937\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,23,20261111-2356138891-23,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.258,31.592,10.64900517463684\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,24,20261111-2356138891-24,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,1,4,4.5,19.712,32.028,12.312076091766357\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,25,20261111-2356138891-25,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,493,0,7,7.0,19.812,32.164,12.183042764663696\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,26,20261111-2356138891-26,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.63,32.152,11.442913055419922\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,27,20261111-2356138891-27,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,19.392,32.056,10.784614324569702\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,28,20261111-2356138891-28,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,1,4,4.5,19.442,32.292,11.48676085472107\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,29,20261111-2356138891-29,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,0,5,5.0,19.602,31.952,11.091249704360962\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,30,20261111-2356138891-30,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,0,5,5.0,19.512,31.852,10.633432388305664\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,31,20261111-2356138891-31,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,1,3,3.5,19.3,32.436,10.878714561462402\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,32,20261111-2356138891-32,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.13,32.448,12.472250699996948\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,33,20261111-2356138891-33,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,19.684,32.052,10.383384466171265\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,34,20261111-2356138891-34,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,1,5,5.5,19.506,32.436,11.355305671691895\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,35,20261111-2356138891-35,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,489,2,9,10.0,19.336,31.808,10.96614933013916\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,36,20261111-2356138891-36,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,1,5,5.5,19.75,31.348,11.324859142303467\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,37,20261111-2356138891-37,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.81,31.86,11.305594205856323\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,38,20261111-2356138891-38,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,0,5,5.0,19.83,31.936,11.26899242401123\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,39,20261111-2356138891-39,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,20.002,31.508,11.565957069396973\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,40,20261111-2356138891-40,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.896,31.88,11.586030721664429\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,41,20261111-2356138891-41,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,1,5,5.5,19.174,31.896,11.408174991607666\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,42,20261111-2356138891-42,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,0,6,6.0,19.474,32.092,11.214660167694092\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,43,20261111-2356138891-43,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.764,31.624,12.655228614807129\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,44,20261111-2356138891-44,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,19.438,32.024,11.158961296081543\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,45,20261111-2356138891-45,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,0,6,6.0,19.396,32.388,12.137339353561401\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,46,20261111-2356138891-46,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,19.306,31.972,10.623711109161377\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,47,20261111-2356138891-47,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,1,3,3.5,19.27,31.752,10.51120138168335\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,48,20261111-2356138891-48,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,493,2,5,6.0,19.834,32.004,10.59170126914978\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,49,20261111-2356138891-49,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.622,32.004,10.935150861740112\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,50,20261111-2356138891-50,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.448,31.764,12.605536460876465\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,51,20261111-2356138891-51,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,0,5,5.0,19.672,31.792,11.889405488967896\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,52,20261111-2356138891-52,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.568,31.696,10.596487045288086\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,53,20261111-2356138891-53,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,1,1,1.5,19.452,31.824,12.052940845489502\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,54,20261111-2356138891-54,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.556,31.776,10.442872047424316\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,55,20261111-2356138891-55,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,1,5,5.5,19.706,31.876,12.20850658416748\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,56,20261111-2356138891-56,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,493,0,7,7.0,19.476,31.848,11.318750143051147\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,57,20261111-2356138891-57,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,1,3,3.5,19.528,32.24,13.120028734207153\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,58,20261111-2356138891-58,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,0,5,5.0,19.438,31.936,11.33482313156128\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,59,20261111-2356138891-59,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,2,4,5.0,19.436,32.26,11.89702844619751\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,60,20261111-2356138891-60,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.718,32.024,11.790786981582642\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,61,20261111-2356138891-61,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,492,1,7,7.5,19.278,31.884,13.12124228477478\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,62,20261111-2356138891-62,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,19.224,32.288,9.888558626174927\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,63,20261111-2356138891-63,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.604,31.984,11.120170831680298\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,64,20261111-2356138891-64,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,1,5,5.5,19.436,31.676,11.853282690048218\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,65,20261111-2356138891-65,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,491,1,8,8.5,19.422,32.188,11.355841398239136\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,66,20261111-2356138891-66,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,19.662,31.764,11.97373652458191\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,67,20261111-2356138891-67,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,19.722,32.292,12.535170316696167\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,68,20261111-2356138891-68,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,493,0,7,7.0,19.356,31.752,12.050703287124634\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,69,20261111-2356138891-69,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,1,4,4.5,19.376,32.348,11.864251852035522\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,70,20261111-2356138891-70,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,495,0,5,5.0,19.502,32.172,11.390150785446167\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,71,20261111-2356138891-71,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,492,3,5,6.5,19.444,31.924,11.312664031982422\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,72,20261111-2356138891-72,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.484,32.14,12.627409934997559\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,73,20261111-2356138891-73,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,19.412,32.176,11.410902500152588\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,74,20261111-2356138891-74,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,1,2,2.5,19.236,32.116,12.145476579666138\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,75,20261111-2356138891-75,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.384,31.956,11.23703908920288\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,76,20261111-2356138891-76,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,0,4,4.0,19.312,32.124,11.925524234771729\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,77,20261111-2356138891-77,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,497,0,3,3.0,19.402,31.436,11.976208209991455\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,78,20261111-2356138891-78,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,494,0,6,6.0,19.688,31.852,11.9696044921875\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K2|g1,79,20261111-2356138891-79,B,noisy,5,0.001,0.0,2,1,5,5,1,5,2,500,496,1,3,3.5,20.144,31.604,13.005461931228638\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,0,20261111-868118775-0,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.0990990990991,48.414414414414416,15.873866081237793\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,1,20261111-868118775-1,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.33033033033033,48.606606606606604,16.355743646621704\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,2,20261111-868118775-2,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.18018018018018,48.56456456456456,16.702611923217773\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,3,20261111-868118775-3,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.792792792792792,48.08408408408408,17.774353981018066\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,4,20261111-868118775-4,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,31.414414414414413,47.651651651651655,13.630668640136719\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,5,20261111-868118775-5,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.624624624624623,47.867867867867865,15.334267377853394\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,6,20261111-868118775-6,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.15015015015015,48.330330330330334,16.851322174072266\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,7,20261111-868118775-7,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.46846846846847,48.53453453453454,13.705454349517822\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,8,20261111-868118775-8,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.756756756756758,48.15615615615616,13.212895154953003\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,9,20261111-868118775-9,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,1,1,1.5,31.25825825825826,48.546546546546544,13.221676111221313\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,10,20261111-868118775-10,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,1,3,3.5,31.56156156156156,47.36336336336336,14.168889284133911\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,11,20261111-868118775-11,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.49249249249249,47.831831831831835,12.970404863357544\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,12,20261111-868118775-12,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.53153153153153,47.327327327327325,15.266406059265137\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,13,20261111-868118775-13,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.58858858858859,47.645645645645644,16.29499650001526\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,14,20261111-868118775-14,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.456456456456458,47.651651651651655,13.617280960083008\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,15,20261111-868118775-15,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,30.894894894894893,48.126126126126124,15.180659532546997\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,16,20261111-868118775-16,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,31.984984984984983,48.252252252252255,13.706674814224243\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,17,20261111-868118775-17,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,2,4,5.0,31.204204204204203,48.1981981981982,13.752757787704468\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,18,20261111-868118775-18,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,323,1,9,9.5,30.72972972972973,47.831831831831835,13.97545838356018\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,19,20261111-868118775-19,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,326,0,7,7.0,31.354354354354353,48.108108108108105,14.793137788772583\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,20,20261111-868118775-20,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,31.024024024024023,48.150150150150154,14.55689811706543\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,21,20261111-868118775-21,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.33933933933934,48.210210210210214,12.960596084594727\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,22,20261111-868118775-22,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,1,4,4.5,30.67267267267267,48.372372372372375,11.810934066772461\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,23,20261111-868118775-23,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.29129129129129,47.66366366366366,11.693335771560669\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,24,20261111-868118775-24,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.336336336336338,47.453453453453456,10.77173924446106\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,25,20261111-868118775-25,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.73873873873874,47.8978978978979,10.147387027740479\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,26,20261111-868118775-26,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.67867867867868,47.909909909909906,11.07341480255127\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,27,20261111-868118775-27,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.756756756756758,48.114114114114116,10.920605659484863\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,28,20261111-868118775-28,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.714714714714713,47.867867867867865,10.4421226978302\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,29,20261111-868118775-29,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.3003003003003,47.92192192192192,9.98165225982666\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,30,20261111-868118775-30,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.465465465465467,47.969969969969966,11.684146165847778\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,31,20261111-868118775-31,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,324,1,8,8.5,31.363363363363362,48.06006006006006,10.323065519332886\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,32,20261111-868118775-32,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,2,3,4.0,30.96096096096096,47.351351351351354,11.108718633651733\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,33,20261111-868118775-33,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.174174174174173,48.48048048048048,11.748929500579834\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,34,20261111-868118775-34,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.64264264264264,47.8018018018018,12.87423062324524\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,35,20261111-868118775-35,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,30.91891891891892,47.16516516516516,13.442780017852783\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,36,20261111-868118775-36,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.087087087087088,47.867867867867865,13.17267918586731\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,37,20261111-868118775-37,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.76876876876877,47.46546546546546,13.078607320785522\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,38,20261111-868118775-38,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.41141141141141,48.07807807807808,11.811272859573364\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,39,20261111-868118775-39,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,30.75975975975976,47.573573573573576,10.342244148254395\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,40,20261111-868118775-40,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,323,2,8,9.0,31.07207207207207,48.51651651651652,12.748213768005371\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,41,20261111-868118775-41,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.48048048048048,48.690690690690694,15.561951160430908\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,42,20261111-868118775-42,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.82882882882883,48.048048048048045,14.100088119506836\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,43,20261111-868118775-43,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.684684684684683,48.74474474474474,14.583231687545776\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,44,20261111-868118775-44,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,325,0,8,8.0,31.396396396396398,47.77777777777778,15.088985681533813\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,45,20261111-868118775-45,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.534534534534533,48.174174174174176,11.299406290054321\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,46,20261111-868118775-46,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,30.80780780780781,48.468468468468465,11.312599182128906\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,47,20261111-868118775-47,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,31.15015015015015,48.192192192192195,10.279758214950562\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,48,20261111-868118775-48,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.846846846846848,47.96396396396396,11.236592054367065\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,49,20261111-868118775-49,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.444444444444443,47.765765765765764,11.237647771835327\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,50,20261111-868118775-50,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,32.372372372372375,47.84384384384384,10.559887170791626\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,51,20261111-868118775-51,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,1,1,1.5,30.936936936936938,47.633633633633636,10.24594259262085\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,52,20261111-868118775-52,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,1,4,4.5,31.204204204204203,48.33633633633634,9.4488205909729\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,53,20261111-868118775-53,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,30.894894894894893,47.447447447447445,10.997072696685791\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,54,20261111-868118775-54,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.624624624624623,47.43543543543544,9.3312349319458\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,55,20261111-868118775-55,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.966966966966968,47.531531531531535,9.681162357330322\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,56,20261111-868118775-56,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,324,3,6,7.5,31.33933933933934,47.627627627627625,10.399789810180664\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,57,20261111-868118775-57,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,326,0,7,7.0,30.8018018018018,48.24024024024024,9.281296253204346\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,58,20261111-868118775-58,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,31.68168168168168,48.108108108108105,11.02126145362854\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,59,20261111-868118775-59,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,322,1,10,10.5,31.456456456456458,48.492492492492495,10.531956434249878\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,60,20261111-868118775-60,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,32.192192192192195,48.14414414414414,11.234362602233887\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,61,20261111-868118775-61,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.44744744744745,48.5945945945946,9.659010410308838\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,62,20261111-868118775-62,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,30.91891891891892,47.747747747747745,9.75969386100769\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,63,20261111-868118775-63,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.1021021021021,48.306306306306304,10.395683765411377\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,64,20261111-868118775-64,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,1,4,4.5,31.015015015015017,47.68168168168168,10.784199237823486\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,65,20261111-868118775-65,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,1,4,4.5,31.07807807807808,48.474474474474476,9.80334186553955\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,66,20261111-868118775-66,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.28228228228228,48.288288288288285,9.28678011894226\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,67,20261111-868118775-67,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,325,3,5,6.5,31.555555555555557,47.651651651651655,8.856576204299927\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,68,20261111-868118775-68,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.762762762762762,47.525525525525524,10.109998226165771\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,69,20261111-868118775-69,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,326,0,7,7.0,31.47747747747748,48.42042042042042,9.904109716415405\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,70,20261111-868118775-70,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,332,1,0,0.5,31.237237237237238,47.873873873873876,10.695009708404541\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,71,20261111-868118775-71,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.77177177177177,48.45645645645646,9.143997192382812\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,72,20261111-868118775-72,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.45945945945946,47.651651651651655,10.30697250366211\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,73,20261111-868118775-73,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.2012012012012,48.330330330330334,9.787976503372192\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,74,20261111-868118775-74,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.624624624624623,47.729729729729726,10.224586486816406\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,75,20261111-868118775-75,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,1,4,4.5,31.366366366366368,47.867867867867865,9.177169561386108\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,76,20261111-868118775-76,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.34834834834835,48.22222222222222,10.066069841384888\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,77,20261111-868118775-77,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.33033033033033,47.687687687687685,9.019830703735352\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,78,20261111-868118775-78,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.32132132132132,48.01801801801802,10.968732357025146\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,79,20261111-868118775-79,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.453453453453452,48.030030030030034,10.036172866821289\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,80,20261111-868118775-80,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.525525525525527,47.95795795795796,9.390169143676758\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,81,20261111-868118775-81,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,1,4,4.5,31.555555555555557,48.288288288288285,9.924487829208374\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,82,20261111-868118775-82,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,31.09009009009009,47.951951951951955,8.713101863861084\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,83,20261111-868118775-83,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,325,1,7,7.5,31.354354354354353,48.3003003003003,10.805391073226929\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,84,20261111-868118775-84,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,1,3,3.5,31.585585585585587,48.468468468468465,9.51225209236145\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,85,20261111-868118775-85,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.693693693693692,47.55555555555556,9.507816076278687\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,86,20261111-868118775-86,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,326,2,5,6.0,31.37237237237237,48.3963963963964,11.084674835205078\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,87,20261111-868118775-87,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,1,3,3.5,31.105105105105107,47.753753753753756,9.292591094970703\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,88,20261111-868118775-88,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,326,0,7,7.0,31.42942942942943,48.62462462462462,8.378427982330322\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,89,20261111-868118775-89,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,30.882882882882882,48.92492492492492,9.922720432281494\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,90,20261111-868118775-90,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,2,2,3.0,31.333333333333332,48.52252252252252,9.510143995285034\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,91,20261111-868118775-91,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.15915915915916,47.95795795795796,10.293696165084839\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,92,20261111-868118775-92,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.42942942942943,47.57957957957958,8.780183553695679\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,93,20261111-868118775-93,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,31.225225225225227,47.453453453453456,10.897246837615967\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,94,20261111-868118775-94,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,31.405405405405407,48.52252252252252,11.695677757263184\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,95,20261111-868118775-95,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,324,0,9,9.0,31.3003003003003,48.1981981981982,11.80932092666626\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,96,20261111-868118775-96,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,32.174174174174176,47.693693693693696,12.866589784622192\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,97,20261111-868118775-97,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.27027027027027,48.78078078078078,13.840538740158081\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,98,20261111-868118775-98,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.18018018018018,48.1021021021021,14.244097709655762\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,99,20261111-868118775-99,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,2,4,5.0,31.74774774774775,48.32432432432432,13.523575782775879\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,100,20261111-868118775-100,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,2,1,2.0,31.26126126126126,48.74474474474474,13.31916069984436\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,101,20261111-868118775-101,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.204204204204203,47.525525525525524,11.873183012008667\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,102,20261111-868118775-102,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,31.546546546546548,48.006006006006004,11.502514839172363\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,103,20261111-868118775-103,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,2,4,5.0,31.363363363363362,47.68168168168168,10.617497205734253\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,104,20261111-868118775-104,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,31.405405405405407,47.61561561561562,10.159457445144653\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,105,20261111-868118775-105,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,326,3,4,5.5,31.6006006006006,47.633633633633636,8.685392379760742\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,106,20261111-868118775-106,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,326,1,6,6.5,31.936936936936938,47.78378378378378,10.554849863052368\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,107,20261111-868118775-107,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,1,4,4.5,31.34834834834835,48.330330330330334,9.66744589805603\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,108,20261111-868118775-108,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.50750750750751,48.054054054054056,10.727455377578735\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,109,20261111-868118775-109,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.69069069069069,48.34234234234234,9.584440231323242\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,110,20261111-868118775-110,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,1,5,5.5,31.852852852852852,48.390390390390394,10.621371984481812\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,111,20261111-868118775-111,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.38738738738739,48.33633633633634,10.341859579086304\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,112,20261111-868118775-112,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,0,5,5.0,31.504504504504503,48.07807807807808,11.351921319961548\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,113,20261111-868118775-113,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,327,0,6,6.0,31.465465465465467,48.150150150150154,10.49004340171814\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,114,20261111-868118775-114,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.765765765765767,48.492492492492495,10.555176019668579\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,115,20261111-868118775-115,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,31.28228228228228,48.27627627627628,12.023114919662476\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,116,20261111-868118775-116,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.50750750750751,48.270270270270274,10.299518823623657\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,117,20261111-868118775-117,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,328,1,4,4.5,32.126126126126124,47.513513513513516,8.715368270874023\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,118,20261111-868118775-118,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,329,0,4,4.0,31.324324324324323,48.12012012012012,9.086054563522339\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,119,20261111-868118775-119,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,31.64864864864865,48.486486486486484,9.647409677505493\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g5,120,20261111-868118775-120,B,noisy,5,0.001,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,31.744744744744743,48.0,9.500919342041016\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,0,20261111-886289646-0,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.276276276276278,47.387387387387385,9.91686749458313\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,1,20261111-886289646-1,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,0,6,6.0,26.74774774774775,47.693693693693696,9.651168823242188\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,2,20261111-886289646-2,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.55255255255255,48.288288288288285,9.715152502059937\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,3,20261111-886289646-3,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.53753753753754,48.1021021021021,9.817561388015747\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,4,20261111-886289646-4,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,0,7,7.0,25.6996996996997,47.86186186186186,7.59900164604187\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,5,20261111-886289646-5,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.25825825825826,48.0960960960961,7.947280168533325\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,6,20261111-886289646-6,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.273273273273272,48.450450450450454,7.681477069854736\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,7,20261111-886289646-7,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.71171171171171,48.648648648648646,8.560251235961914\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,8,20261111-886289646-8,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,25.92792792792793,48.08408408408408,8.00883150100708\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,9,20261111-886289646-9,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,0,7,7.0,26.56756756756757,47.969969969969966,7.54373574256897\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,10,20261111-886289646-10,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,25.942942942942942,47.56156156156156,9.060231685638428\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,11,20261111-886289646-11,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.114114114114113,48.18018018018018,9.052806377410889\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,12,20261111-886289646-12,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,1,3,3.5,26.41741741741742,48.294294294294296,8.47195291519165\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,13,20261111-886289646-13,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,2,5,6.0,26.32732732732733,48.414414414414416,7.623639822006226\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,14,20261111-886289646-14,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.36936936936937,47.91591591591592,8.897824048995972\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,15,20261111-886289646-15,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,26.375375375375377,48.114114114114116,8.679259777069092\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,16,20261111-886289646-16,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,25.89189189189189,48.3963963963964,7.573649168014526\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,17,20261111-886289646-17,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,2,1,2.0,26.753753753753752,48.1021021021021,7.722748756408691\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,18,20261111-886289646-18,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,26.69069069069069,48.528528528528525,8.95722246170044\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,19,20261111-886289646-19,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.4024024024024,47.97597597597598,8.289111852645874\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,20,20261111-886289646-20,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,25.933933933933933,48.16216216216216,7.585019588470459\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,21,20261111-886289646-21,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,25.915915915915917,47.92192192192192,7.622145175933838\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,22,20261111-886289646-22,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,25.654654654654653,48.15615615615616,8.566218852996826\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,23,20261111-886289646-23,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.765765765765767,47.62162162162162,7.8616249561309814\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,24,20261111-886289646-24,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,324,0,9,9.0,26.666666666666668,47.78378378378378,8.653450965881348\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,25,20261111-886289646-25,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,325,0,8,8.0,26.31831831831832,47.8978978978979,7.5030858516693115\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,26,20261111-886289646-26,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.354354354354353,48.16216216216216,7.529284477233887\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,27,20261111-886289646-27,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.57057057057057,48.246246246246244,9.093205213546753\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,28,20261111-886289646-28,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,1,6,6.5,26.36036036036036,48.174174174174176,8.717551946640015\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,29,20261111-886289646-29,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.37237237237237,48.408408408408405,7.39996337890625\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,30,20261111-886289646-30,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,26.654654654654653,47.55555555555556,7.523425340652466\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,31,20261111-886289646-31,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.70870870870871,48.066066066066064,6.895991802215576\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,32,20261111-886289646-32,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.015015015015017,48.18018018018018,7.659783363342285\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,33,20261111-886289646-33,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,0,6,6.0,25.61861861861862,47.6996996996997,8.953873872756958\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,34,20261111-886289646-34,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,1,6,6.5,26.3003003003003,47.77777777777778,7.488011121749878\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,35,20261111-886289646-35,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,325,3,5,6.5,25.97897897897898,47.61561561561562,8.038318157196045\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,36,20261111-886289646-36,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.456456456456458,47.789789789789786,7.988689422607422\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,37,20261111-886289646-37,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,1,6,6.5,26.384384384384383,48.81681681681682,8.955445051193237\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,38,20261111-886289646-38,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,325,3,5,6.5,26.774774774774773,48.15615615615616,7.667466402053833\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,39,20261111-886289646-39,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.93093093093093,48.63663663663664,8.1031973361969\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,40,20261111-886289646-40,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.513513513513512,47.5015015015015,8.95742392539978\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,41,20261111-886289646-41,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,331,1,1,1.5,26.345345345345347,47.891891891891895,8.794927835464478\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,42,20261111-886289646-42,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,26.57957957957958,48.06006006006006,7.891526699066162\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,43,20261111-886289646-43,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,1,3,3.5,26.3003003003003,48.450450450450454,6.934652805328369\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,44,20261111-886289646-44,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,2,5,6.0,26.37237237237237,47.687687687687685,7.553605318069458\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,45,20261111-886289646-45,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,0,6,6.0,26.093093093093092,47.8978978978979,7.146336078643799\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,46,20261111-886289646-46,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.207207207207208,48.108108108108105,7.747055768966675\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,47,20261111-886289646-47,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.57957957957958,48.408408408408405,7.448126792907715\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,48,20261111-886289646-48,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,26.33933933933934,47.645645645645644,7.476334571838379\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,49,20261111-886289646-49,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.384384384384383,47.33933933933934,7.349456310272217\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,50,20261111-886289646-50,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.57957957957958,47.83783783783784,7.4920806884765625\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,51,20261111-886289646-51,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.83783783783784,47.945945945945944,7.548624277114868\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,52,20261111-886289646-52,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,1,3,3.5,26.663663663663662,47.867867867867865,7.827428817749023\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,53,20261111-886289646-53,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,1,3,3.5,26.69069069069069,47.63963963963964,8.308391094207764\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,54,20261111-886289646-54,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,325,0,8,8.0,26.276276276276278,47.47747747747748,8.793052911758423\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,55,20261111-886289646-55,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,0,6,6.0,26.603603603603602,47.909909909909906,7.447115659713745\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,56,20261111-886289646-56,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.15915915915916,47.987987987987985,7.076413631439209\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,57,20261111-886289646-57,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,27.003003003003002,48.0960960960961,8.241101026535034\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,58,20261111-886289646-58,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.204204204204203,47.891891891891895,7.524590253829956\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,59,20261111-886289646-59,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,323,0,10,10.0,26.255255255255257,48.14414414414414,8.222729444503784\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,60,20261111-886289646-60,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,325,2,6,7.0,26.273273273273272,47.6036036036036,8.05630898475647\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,61,20261111-886289646-61,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,0,7,7.0,26.13213213213213,48.006006006006004,7.777841091156006\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,62,20261111-886289646-62,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,324,1,8,8.5,26.33033033033033,47.993993993993996,7.531076908111572\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,63,20261111-886289646-63,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,2,3,4.0,26.435435435435437,47.83783783783784,8.689226865768433\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,64,20261111-886289646-64,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.306306306306308,47.531531531531535,7.272536516189575\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,65,20261111-886289646-65,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,331,1,1,1.5,26.873873873873872,47.6036036036036,8.530877590179443\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,66,20261111-886289646-66,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.57957957957958,47.831831831831835,7.78382682800293\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,67,20261111-886289646-67,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,0,6,6.0,26.72972972972973,47.771771771771775,8.034363508224487\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,68,20261111-886289646-68,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.843843843843842,47.97597597597598,8.166016578674316\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,69,20261111-886289646-69,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.09009009009009,47.753753753753756,7.208901405334473\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,70,20261111-886289646-70,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,0,7,7.0,26.633633633633632,48.288288288288285,8.092888593673706\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,71,20261111-886289646-71,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,25.984984984984983,48.4984984984985,7.594794034957886\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,72,20261111-886289646-72,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.525525525525527,48.030030030030034,7.40424919128418\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,73,20261111-886289646-73,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.40840840840841,48.57657657657658,8.573518753051758\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,74,20261111-886289646-74,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.07807807807808,48.12012012012012,8.319278955459595\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,75,20261111-886289646-75,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,0,6,6.0,26.11111111111111,48.08408408408408,7.923097610473633\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,76,20261111-886289646-76,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,26.19219219219219,47.687687687687685,8.032992362976074\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,77,20261111-886289646-77,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.33033033033033,48.306306306306304,7.198051691055298\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,78,20261111-886289646-78,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.237237237237238,48.012012012012015,8.549014329910278\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,79,20261111-886289646-79,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,1,2,2.5,26.195195195195197,48.33633633633634,8.748477458953857\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,80,20261111-886289646-80,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,25.93993993993994,48.51651651651652,7.996691942214966\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,81,20261111-886289646-81,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,0,6,6.0,26.53153153153153,47.55555555555556,7.647015571594238\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,82,20261111-886289646-82,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.16816816816817,48.0,7.6587910652160645\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,83,20261111-886289646-83,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.70870870870871,47.633633633633636,8.805968999862671\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,84,20261111-886289646-84,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.324324324324323,48.14414414414414,8.945338487625122\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,85,20261111-886289646-85,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,0,7,7.0,26.123123123123122,48.4984984984985,8.280961036682129\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,86,20261111-886289646-86,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,26.234234234234233,48.108108108108105,8.281516075134277\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,87,20261111-886289646-87,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,325,0,8,8.0,26.28828828828829,48.07807807807808,8.048417806625366\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,88,20261111-886289646-88,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.63063063063063,47.807807807807805,8.54621934890747\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,89,20261111-886289646-89,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.65165165165165,48.15615615615616,7.663919687271118\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,90,20261111-886289646-90,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,25.615615615615617,47.61561561561562,8.010982275009155\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,91,20261111-886289646-91,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.393393393393392,47.44144144144144,7.202219247817993\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,92,20261111-886289646-92,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.43243243243243,48.192192192192195,8.451729536056519\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,93,20261111-886289646-93,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,1,3,3.5,26.027027027027028,47.15915915915916,7.318989038467407\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,94,20261111-886289646-94,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,2,3,4.0,26.41741741741742,48.68468468468468,8.430357217788696\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,95,20261111-886289646-95,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,26.45045045045045,48.44444444444444,6.96847128868103\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,96,20261111-886289646-96,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,1,3,3.5,26.82882882882883,47.45945945945946,7.629350662231445\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,97,20261111-886289646-97,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,2,3,4.0,27.03903903903904,48.44444444444444,7.812272548675537\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,98,20261111-886289646-98,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,0,7,7.0,26.3003003003003,47.8018018018018,7.714512825012207\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,99,20261111-886289646-99,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,0,7,7.0,26.045045045045047,47.74174174174174,8.20500898361206\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,100,20261111-886289646-100,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,26.084084084084083,48.48048048048048,7.002197980880737\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,101,20261111-886289646-101,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,324,2,7,8.0,27.243243243243242,48.54054054054054,8.42858338356018\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,102,20261111-886289646-102,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.483483483483482,48.570570570570574,7.392404317855835\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,103,20261111-886289646-103,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.414414414414413,48.75675675675676,8.260679960250854\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,104,20261111-886289646-104,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.105105105105107,48.486486486486484,7.200623273849487\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,105,20261111-886289646-105,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.04804804804805,48.02402402402402,7.371683597564697\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,106,20261111-886289646-106,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.063063063063062,48.030030030030034,8.131535768508911\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,107,20261111-886289646-107,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,326,0,7,7.0,26.075075075075077,47.873873873873876,8.62827444076538\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,108,20261111-886289646-108,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,26.71171171171171,48.27627627627628,7.910292625427246\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,109,20261111-886289646-109,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,325,1,7,7.5,25.95195195195195,47.71771771771772,7.2598748207092285\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,110,20261111-886289646-110,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,26.24924924924925,47.81981981981982,7.511338472366333\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,111,20261111-886289646-111,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.84984984984985,48.210210210210214,7.184871196746826\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,112,20261111-886289646-112,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,330,0,3,3.0,26.246246246246248,48.06006006006006,8.167155027389526\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,113,20261111-886289646-113,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.56756756756757,48.15615615615616,8.594057083129883\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,114,20261111-886289646-114,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,26.255255255255257,47.74174174174174,7.060302019119263\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,115,20261111-886289646-115,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,329,0,4,4.0,25.915915915915917,48.34234234234234,7.525996685028076\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,116,20261111-886289646-116,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,0,5,5.0,26.4984984984985,47.95795795795796,7.561750411987305\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,117,20261111-886289646-117,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.426426426426428,48.22222222222222,8.387950420379639\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,118,20261111-886289646-118,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,1,5,5.5,26.234234234234233,47.97597597597598,7.8878631591796875\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,119,20261111-886289646-119,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,328,1,4,4.5,26.993993993993993,47.81981981981982,8.911768674850464\npilot-20261110T155331Z-d3f938,B|noisy|d5|p0.001|K3|g1,120,20261111-886289646-120,B,noisy,5,0.001,0.0,3,1,5,5,1,5,2,333,327,0,6,6.0,26.555555555555557,48.450450450450454,8.037850141525269\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,0,20261111-4066447750-0,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3922,3,75,76.5,3.71325,3.9795,11.109232187271118\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,1,20261111-4066447750-1,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3912,5,83,85.5,3.6725,4.069,11.213441133499146\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,2,20261111-4066447750-2,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3919,3,78,79.5,3.6605,4.006,11.625446557998657\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,3,20261111-4066447750-3,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3915,3,82,83.5,3.722,4.007,11.65985631942749\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,4,20261111-4066447750-4,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3904,2,94,95.0,3.71125,4.074,11.616785287857056\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,5,20261111-4066447750-5,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3916,4,80,82.0,3.702,4.019,11.142365455627441\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,6,20261111-4066447750-6,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3926,6,68,71.0,3.668,4.016,11.723653793334961\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,7,20261111-4066447750-7,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3934,2,64,65.0,3.6705,4.0215,11.389874458312988\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,8,20261111-4066447750-8,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3921,2,77,78.0,3.69225,4.001,11.61005711555481\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K1|g0,9,20261111-4066447750-9,A,noisy,3,0.0003,0.0003,1,0,3,3,3,3,2,4000,3919,3,78,79.5,3.709,3.9795,10.94593071937561\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,0,20261111-2044587986-0,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1914,3,83,84.5,6.7225,7.99,9.756708145141602\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,1,20261111-2044587986-1,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1922,2,76,77.0,6.6965,7.887,10.13747763633728\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,2,20261111-2044587986-2,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1933,5,62,64.5,6.697,7.955,9.648112058639526\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,3,20261111-2044587986-3,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1928,7,65,68.5,6.6765,8.144,9.533525943756104\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,4,20261111-2044587986-4,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1923,5,72,74.5,6.743,7.897,9.551620244979858\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,5,20261111-2044587986-5,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1917,4,79,81.0,6.838,8.04,9.896692514419556\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,6,20261111-2044587986-6,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1911,4,85,87.0,6.7795,7.934,9.64358115196228\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,7,20261111-2044587986-7,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1940,2,58,59.0,6.6115,7.98,9.566635131835938\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,8,20261111-2044587986-8,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1928,3,69,70.5,6.753,7.986,9.550256967544556\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,9,20261111-2044587986-9,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1924,6,70,73.0,6.6995,8.043,9.674359560012817\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,10,20261111-2044587986-10,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1919,0,81,81.0,6.7255,8.05,9.389496088027954\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,11,20261111-2044587986-11,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1922,2,76,77.0,6.7465,8.025,9.799603700637817\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,12,20261111-2044587986-12,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1914,2,84,85.0,6.742,7.964,10.050762176513672\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,13,20261111-2044587986-13,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1906,5,89,91.5,6.819,8.069,9.847306489944458\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,14,20261111-2044587986-14,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1928,5,67,69.5,6.649,7.913,9.734654903411865\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,15,20261111-2044587986-15,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1898,2,100,101.0,6.7545,8.034,9.363874197006226\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,16,20261111-2044587986-16,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1919,6,75,78.0,6.7495,7.994,10.245007038116455\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,17,20261111-2044587986-17,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1909,0,91,91.0,6.7285,8.007,9.377230882644653\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,18,20261111-2044587986-18,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1920,3,77,78.5,6.7735,7.973,9.80119013786316\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g3,19,20261111-2044587986-19,A,noisy,3,0.0003,0.0003,2,3,3,3,3,3,2,2000,1915,6,79,82.0,6.7545,8.039,9.845426797866821\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,0,20261111-2547224318-0,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1911,4,85,87.0,6.305,8.027,8.971588134765625\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,1,20261111-2547224318-1,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1923,3,74,75.5,6.245,8.055,8.893436193466187\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,2,20261111-2547224318-2,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1930,2,68,69.0,6.2865,7.865,9.191473960876465\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,3,20261111-2547224318-3,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1927,3,70,71.5,6.3565,7.922,9.010083436965942\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,4,20261111-2547224318-4,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1921,6,73,76.0,6.158,8.046,8.853328704833984\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,5,20261111-2547224318-5,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1916,2,82,83.0,6.283,8.096,9.274354219436646\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,6,20261111-2547224318-6,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1922,3,75,76.5,6.3955,7.938,8.877872228622437\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,7,20261111-2547224318-7,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1923,1,76,76.5,6.227,7.927,8.820792198181152\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,8,20261111-2547224318-8,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1905,3,92,93.5,6.241,7.981,9.157594203948975\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,9,20261111-2547224318-9,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1916,2,82,83.0,6.3145,7.991,8.616957426071167\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,10,20261111-2547224318-10,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1918,7,75,78.5,6.32,7.938,8.651918649673462\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,11,20261111-2547224318-11,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1913,0,87,87.0,6.3435,8.132,8.685640811920166\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,12,20261111-2547224318-12,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1923,2,75,76.0,6.3705,7.984,8.836175680160522\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,13,20261111-2547224318-13,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1914,3,83,84.5,6.247,8.099,9.231659173965454\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,14,20261111-2547224318-14,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1925,3,72,73.5,6.262,8.011,8.925628185272217\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,15,20261111-2547224318-15,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1911,2,87,88.0,6.3775,7.982,9.371208667755127\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,16,20261111-2547224318-16,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1933,2,65,66.0,6.238,8.131,9.488121509552002\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,17,20261111-2547224318-17,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1922,1,77,77.5,6.371,7.985,9.215045928955078\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,18,20261111-2547224318-18,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1913,4,83,85.0,6.345,7.963,8.852492094039917\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K2|g1,19,20261111-2547224318-19,A,noisy,3,0.0003,0.0003,2,1,3,3,1,3,2,2000,1918,1,81,81.5,6.2685,7.996,9.485708713531494\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,0,20261111-3244394679-0,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1255,2,76,77.0,9.684921230307577,11.945986496624156,9.791543006896973\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,1,20261111-3244394679-1,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1250,0,83,83.0,9.891222805701425,11.747936984246062,9.684162139892578\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,2,20261111-3244394679-2,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1260,1,72,72.5,9.81245311327832,11.920480120030007,9.406373977661133\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,3,20261111-3244394679-3,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1268,5,60,62.5,9.85371342835709,12.046511627906977,9.965418577194214\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,4,20261111-3244394679-4,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1257,4,72,74.0,9.767441860465116,11.911477869467367,9.140527725219727\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,5,20261111-3244394679-5,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1270,6,57,60.0,9.891222805701425,12.036009002250562,9.573557376861572\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,6,20261111-3244394679-6,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1251,5,77,79.5,9.794448612153039,12.08252063015754,9.256870746612549\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,7,20261111-3244394679-7,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1256,2,75,76.0,9.628657164291074,11.96549137284321,9.299818277359009\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,8,20261111-3244394679-8,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1247,3,83,84.5,9.65191297824456,11.888972243060765,8.822240591049194\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,9,20261111-3244394679-9,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1249,5,79,81.5,9.682670667666917,12.006001500375094,9.218898296356201\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,10,20261111-3244394679-10,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1262,3,68,69.5,9.737434358589647,12.037509377344335,9.314311265945435\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,11,20261111-3244394679-11,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1241,7,85,88.5,9.817704426106527,11.881470367591898,9.084460973739624\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,12,20261111-3244394679-12,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1261,3,69,70.5,9.71642910727682,11.909977494373594,9.009040594100952\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,13,20261111-3244394679-13,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1250,0,83,83.0,9.715678919729932,12.019504876219054,9.405182838439941\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,14,20261111-3244394679-14,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1258,4,71,73.0,9.807951987996999,12.085521380345087,9.072022199630737\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,15,20261111-3244394679-15,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1257,0,76,76.0,9.663165791447861,11.933983495873969,9.276435613632202\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,16,20261111-3244394679-16,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1254,5,74,76.5,9.667666916729182,11.930982745686421,8.880484104156494\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,17,20261111-3244394679-17,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1251,6,76,79.0,9.782445611402851,11.993998499624906,9.350136518478394\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,18,20261111-3244394679-18,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1259,3,71,72.5,9.785446361590397,11.981995498874719,8.695819854736328\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,19,20261111-3244394679-19,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1252,3,78,79.5,9.701425356339085,12.14853713428357,9.307058811187744\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,20,20261111-3244394679-20,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1245,3,85,86.5,9.708177044261065,12.078019504876218,9.154218196868896\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,21,20261111-3244394679-21,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1261,2,70,71.0,9.687171792948238,12.147036759189797,9.480573892593384\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,22,20261111-3244394679-22,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1257,5,71,73.5,9.687171792948238,12.018004501125281,9.194798469543457\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,23,20261111-3244394679-23,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1232,2,99,100.0,9.81695423855964,11.827456864216053,9.22602105140686\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,24,20261111-3244394679-24,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1254,2,77,78.0,9.656414103525881,11.944486121530383,9.473096370697021\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,25,20261111-3244394679-25,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1262,0,71,71.0,9.69767441860465,11.969992498124531,8.711782932281494\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,26,20261111-3244394679-26,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1260,3,70,71.5,9.719429857464366,12.132033008252064,9.338406801223755\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,27,20261111-3244394679-27,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1260,2,71,72.0,9.77569392348087,11.920480120030007,8.791934490203857\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,28,20261111-3244394679-28,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1258,2,73,74.0,9.687921980495124,11.932483120780194,9.153061866760254\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,29,20261111-3244394679-29,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1253,2,78,79.0,9.875468867216805,12.105026256564141,8.903612852096558\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g3,30,20261111-3244394679-30,A,noisy,3,0.0003,0.0003,3,3,3,3,3,3,2,1333,1245,4,84,86.0,9.801200300075019,12.031507876969242,9.418434381484985\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,0,20261111-795863451-0,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1262,3,68,69.5,8.793698424606152,12.03450862715679,8.249933004379272\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,1,20261111-795863451-1,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1259,2,72,73.0,8.952738184546137,11.953488372093023,8.879308700561523\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,2,20261111-795863451-2,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1253,5,75,77.5,8.783195798949738,12.08252063015754,8.860392808914185\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,3,20261111-795863451-3,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1240,3,90,91.5,8.956489122280571,12.058514628657164,8.566766738891602\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,4,20261111-795863451-4,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1263,5,65,67.5,9.04576144036009,12.015003750937735,8.13739800453186\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,5,20261111-795863451-5,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1262,4,67,69.0,8.89947486871718,12.01350337584396,8.479675769805908\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,6,20261111-795863451-6,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1253,1,79,79.5,8.939234808702176,12.060015003750937,8.765944242477417\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,7,20261111-795863451-7,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1257,4,72,74.0,8.762190547636909,12.019504876219054,8.126396179199219\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,8,20261111-795863451-8,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1252,3,78,79.5,8.858964741185297,12.108027006751687,8.625431537628174\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,9,20261111-795863451-9,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1244,2,87,88.0,9.009752438109528,12.123030757689422,8.534536600112915\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,10,20261111-795863451-10,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1238,1,94,94.5,8.881470367591898,12.069017254313579,8.479809522628784\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,11,20261111-795863451-11,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1255,5,73,75.5,8.792948237059266,11.854463615903976,7.872714519500732\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,12,20261111-795863451-12,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1260,4,69,71.0,8.907726931732933,12.00450112528132,8.304858922958374\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,13,20261111-795863451-13,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1259,2,72,73.0,9.057014253563391,11.963990997749438,8.25019383430481\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,14,20261111-795863451-14,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1259,2,72,73.0,8.970742685671418,11.866466616654163,8.612596988677979\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,15,20261111-795863451-15,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1246,5,82,84.5,8.876969242310578,11.893473368342086,8.050023794174194\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,16,20261111-795863451-16,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1250,2,81,82.0,9.05926481620405,11.924981245311328,8.533509254455566\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,17,20261111-795863451-17,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1247,4,82,84.0,8.889722430607652,11.948987246811702,7.7269837856292725\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,18,20261111-795863451-18,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1243,3,87,88.5,8.868717179294825,11.882970742685671,8.735873699188232\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,19,20261111-795863451-19,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1250,0,83,83.0,8.836459114778695,11.827456864216053,8.324938774108887\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,20,20261111-795863451-20,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1238,2,93,94.0,8.795948987246812,11.828957239309828,8.25911831855774\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,21,20261111-795863451-21,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1241,4,88,90.0,8.868717179294825,11.842460615153788,8.339640140533447\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,22,20261111-795863451-22,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1256,3,74,75.5,8.978244561140285,12.043510877719429,8.310264825820923\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,23,20261111-795863451-23,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1274,2,57,58.0,8.849962490622655,12.055513878469617,7.950028657913208\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,24,20261111-795863451-24,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1255,2,76,77.0,8.946736684171043,12.05251312828207,8.029139518737793\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,25,20261111-795863451-25,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1244,4,85,87.0,8.966241560390097,11.984996249062265,8.142053127288818\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,26,20261111-795863451-26,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1261,2,70,71.0,8.854463615903976,11.932483120780194,8.125694751739502\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,27,20261111-795863451-27,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1239,4,90,92.0,8.902475618904726,11.984996249062265,8.10577917098999\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,28,20261111-795863451-28,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1249,6,78,81.0,8.882220555138785,12.012003000750187,8.205357789993286\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,29,20261111-795863451-29,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1244,3,86,87.5,8.968492123030758,11.974493623405852,8.52663278579712\npilot-20261110T155331Z-d3f938,A|noisy|d3|p0.0003|K3|g1,30,20261111-795863451-30,A,noisy,3,0.0003,0.0003,3,1,3,3,1,3,2,1333,1245,5,83,85.5,9.003750937734434,12.165041260315078,8.203495979309082\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,0,20261111-3499869556-0,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,997,0,3,3.0,15.411,16.036,12.532771110534668\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,1,20261111-3499869556-1,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,989,4,7,9.0,15.489,15.876,13.668873071670532\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,2,20261111-3499869556-2,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,994,0,6,6.0,15.46,15.894,13.277339458465576\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,3,20261111-3499869556-3,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,994,2,4,5.0,15.708,15.812,12.624891757965088\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,4,20261111-3499869556-4,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,993,2,5,6.0,15.517,16.17,12.811820030212402\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,5,20261111-3499869556-5,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,1,3,3.5,15.503,15.95,12.174018383026123\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,6,20261111-3499869556-6,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,1,3,3.5,15.482,16.004,12.380168437957764\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,7,20261111-3499869556-7,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,1,3,3.5,15.279,15.972,11.711740493774414\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,8,20261111-3499869556-8,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,994,1,5,5.5,15.654,16.132,12.601781129837036\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,9,20261111-3499869556-9,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,992,0,8,8.0,15.465,16.22,12.580570936203003\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,10,20261111-3499869556-10,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,993,2,5,6.0,15.467,15.944,11.728381156921387\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,11,20261111-3499869556-11,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,997,0,3,3.0,15.447,16.07,12.934861421585083\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,12,20261111-3499869556-12,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,995,0,5,5.0,15.417,16.164,12.328897714614868\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,13,20261111-3499869556-13,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,991,1,8,8.5,15.659,16.14,12.69184398651123\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,14,20261111-3499869556-14,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,992,2,6,7.0,15.474,16.07,12.79176926612854\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,15,20261111-3499869556-15,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,989,1,10,10.5,15.578,16.148,12.098547458648682\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,16,20261111-3499869556-16,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,0,4,4.0,15.335,15.846,13.146825790405273\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,17,20261111-3499869556-17,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,993,2,5,6.0,15.599,15.874,13.363622665405273\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,18,20261111-3499869556-18,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,995,1,4,4.5,15.333,16.074,12.078251600265503\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,19,20261111-3499869556-19,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,997,0,3,3.0,15.45,16.17,12.372987747192383\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,20,20261111-3499869556-20,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,993,1,6,6.5,15.522,16.084,12.878057956695557\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,21,20261111-3499869556-21,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,0,4,4.0,15.547,16.034,13.392241716384888\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,22,20261111-3499869556-22,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,998,0,2,2.0,15.398,15.968,11.986685514450073\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,23,20261111-3499869556-23,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,991,0,9,9.0,15.411,15.864,13.083828449249268\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,24,20261111-3499869556-24,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,0,4,4.0,15.395,16.118,12.257999897003174\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,25,20261111-3499869556-25,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,992,1,7,7.5,15.459,15.972,11.893016815185547\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,26,20261111-3499869556-26,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,0,4,4.0,15.395,15.83,12.774820804595947\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,27,20261111-3499869556-27,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,993,1,6,6.5,15.598,16.156,12.245769262313843\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,28,20261111-3499869556-28,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,995,1,4,4.5,15.351,16.186,12.435004949569702\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,29,20261111-3499869556-29,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,994,1,5,5.5,15.264,16.068,12.811386585235596\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,30,20261111-3499869556-30,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,1,3,3.5,15.492,15.78,12.807244777679443\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,31,20261111-3499869556-31,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,995,1,4,4.5,15.309,15.976,12.334200620651245\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,32,20261111-3499869556-32,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,995,0,5,5.0,15.391,15.844,12.596741199493408\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,33,20261111-3499869556-33,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,998,1,1,1.5,15.433,16.086,12.702727556228638\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,34,20261111-3499869556-34,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,992,1,7,7.5,15.429,15.956,12.84596848487854\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,35,20261111-3499869556-35,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,996,0,4,4.0,15.351,16.058,12.584981441497803\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,36,20261111-3499869556-36,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,995,0,5,5.0,15.441,15.936,12.627549409866333\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,37,20261111-3499869556-37,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,993,1,6,6.5,15.341,16.206,12.962030172348022\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,38,20261111-3499869556-38,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,992,1,7,7.5,15.427,16.056,12.274587631225586\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K1|g0,39,20261111-3499869556-39,A,noisy,5,0.0003,0.0003,1,0,5,5,5,5,2,1000,999,0,1,1.0,15.482,15.966,12.637666702270508\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,0,20261111-2990841365-0,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,1,6,6.5,28.498,31.708,12.87131404876709\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,1,20261111-2990841365-1,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,490,1,9,9.5,28.118,32.328,11.640395164489746\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,2,20261111-2990841365-2,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,0,5,5.0,28.072,31.524,12.732632160186768\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,3,20261111-2990841365-3,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,0,5,5.0,27.906,31.716,11.414904594421387\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,4,20261111-2990841365-4,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,0,6,6.0,27.728,31.86,10.58081340789795\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,5,20261111-2990841365-5,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,1,6,6.5,28.284,32.396,10.339159965515137\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,6,20261111-2990841365-6,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,0,6,6.0,27.914,31.9,11.394092559814453\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,7,20261111-2990841365-7,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,1,6,6.5,27.804,31.928,9.649189949035645\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,8,20261111-2990841365-8,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,2,5,6.0,28.27,32.288,10.919069290161133\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,9,20261111-2990841365-9,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,0,4,4.0,28.03,32.44,10.974886178970337\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,10,20261111-2990841365-10,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,0,8,8.0,27.894,31.852,11.184261083602905\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,11,20261111-2990841365-11,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,497,0,3,3.0,28.134,32.112,10.562819719314575\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,12,20261111-2990841365-12,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,2,2,3.0,27.878,31.896,11.27372694015503\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,13,20261111-2990841365-13,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,1,4,4.5,27.776,32.212,10.72781777381897\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,14,20261111-2990841365-14,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,4,4,6.0,27.774,32.06,11.002647876739502\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,15,20261111-2990841365-15,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,1,4,4.5,27.91,32.16,11.425040006637573\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,16,20261111-2990841365-16,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,497,1,2,2.5,27.804,31.544,10.672491788864136\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,17,20261111-2990841365-17,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,497,2,1,2.0,27.998,32.344,10.768818140029907\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,18,20261111-2990841365-18,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,491,0,9,9.0,27.69,31.684,9.781730890274048\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,19,20261111-2990841365-19,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,497,0,3,3.0,28.068,32.344,11.824836015701294\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,20,20261111-2990841365-20,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,0,8,8.0,28.074,32.352,11.350634574890137\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,21,20261111-2990841365-21,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,0,5,5.0,27.42,31.852,9.746892929077148\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,22,20261111-2990841365-22,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,0,6,6.0,28.486,32.02,11.52924108505249\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,23,20261111-2990841365-23,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,1,7,7.5,28.154,31.984,10.696576595306396\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,24,20261111-2990841365-24,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,1,3,3.5,28.124,32.044,10.492676973342896\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,25,20261111-2990841365-25,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,491,1,8,8.5,28.14,32.06,11.666536092758179\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,26,20261111-2990841365-26,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,1,7,7.5,28.448,31.688,10.260724782943726\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,27,20261111-2990841365-27,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,2,5,6.0,27.722,32.148,11.252587080001831\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,28,20261111-2990841365-28,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,1,7,7.5,28.2,32.124,10.79241156578064\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,29,20261111-2990841365-29,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,0,7,7.0,27.626,32.168,12.081292152404785\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,30,20261111-2990841365-30,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,0,5,5.0,28.034,31.856,10.957301616668701\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,31,20261111-2990841365-31,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,1,4,4.5,28.546,31.828,10.834587812423706\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,32,20261111-2990841365-32,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,0,4,4.0,28.06,31.948,10.589572429656982\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,33,20261111-2990841365-33,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,497,1,2,2.5,28.118,32.06,11.130085468292236\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,34,20261111-2990841365-34,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,497,0,3,3.0,28.002,32.208,10.908194541931152\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,35,20261111-2990841365-35,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,1,6,6.5,27.892,31.796,10.775413274765015\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,36,20261111-2990841365-36,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,498,0,2,2.0,27.534,32.188,11.251194477081299\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,37,20261111-2990841365-37,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,1,6,6.5,28.01,32.188,10.937070369720459\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,38,20261111-2990841365-38,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,489,1,10,10.5,28.166,31.908,11.883763313293457\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,39,20261111-2990841365-39,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,1,4,4.5,28.046,32.004,11.00850510597229\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,40,20261111-2990841365-40,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,1,4,4.5,28.052,31.552,11.564801454544067\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,41,20261111-2990841365-41,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,1,7,7.5,28.314,32.136,10.572360277175903\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,42,20261111-2990841365-42,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,498,1,1,1.5,28.196,31.832,11.792936563491821\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,43,20261111-2990841365-43,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,497,1,2,2.5,28.218,31.84,11.42496633529663\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,44,20261111-2990841365-44,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,0,6,6.0,27.916,32.004,10.786950826644897\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,45,20261111-2990841365-45,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,1,7,7.5,27.944,31.988,11.830286264419556\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,46,20261111-2990841365-46,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,1,5,5.5,28.046,32.036,11.915796518325806\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,47,20261111-2990841365-47,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,0,4,4.0,28.016,31.832,11.120686292648315\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,48,20261111-2990841365-48,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,0,4,4.0,27.654,32.312,11.515395164489746\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,49,20261111-2990841365-49,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,0,5,5.0,28.082,31.856,11.059336423873901\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,50,20261111-2990841365-50,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,1,7,7.5,28.226,32.42,12.04762864112854\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,51,20261111-2990841365-51,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,497,0,3,3.0,28.09,32.08,10.657373428344727\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,52,20261111-2990841365-52,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,1,4,4.5,28.052,32.348,10.77738881111145\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,53,20261111-2990841365-53,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,2,6,7.0,27.94,32.288,11.2341890335083\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,54,20261111-2990841365-54,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,1,3,3.5,28.36,32.144,11.195938110351562\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,55,20261111-2990841365-55,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,1,4,4.5,28.402,32.108,10.868484497070312\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,56,20261111-2990841365-56,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,1,5,5.5,28.126,32.18,11.69422698020935\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,57,20261111-2990841365-57,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,1,3,3.5,28.152,32.464,10.741097211837769\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,58,20261111-2990841365-58,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,1,3,3.5,27.956,31.856,11.54828953742981\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,59,20261111-2990841365-59,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,1,7,7.5,27.832,32.348,10.967998027801514\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,60,20261111-2990841365-60,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,1,3,3.5,27.908,31.952,10.772205591201782\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,61,20261111-2990841365-61,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,3,4,5.5,28.058,31.872,11.328554391860962\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,62,20261111-2990841365-62,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,0,8,8.0,27.972,31.804,9.884280681610107\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,63,20261111-2990841365-63,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,0,5,5.0,27.988,31.968,11.40366506576538\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,64,20261111-2990841365-64,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,0,4,4.0,28.476,31.904,10.914299488067627\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,65,20261111-2990841365-65,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,3,1,2.5,27.934,31.888,11.31506896018982\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,66,20261111-2990841365-66,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,1,5,5.5,27.668,32.356,11.62367582321167\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,67,20261111-2990841365-67,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,1,3,3.5,27.8,31.896,12.031266450881958\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,68,20261111-2990841365-68,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,495,1,4,4.5,27.92,31.8,10.424180507659912\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,69,20261111-2990841365-69,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,0,7,7.0,27.962,31.98,11.374183893203735\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,70,20261111-2990841365-70,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,1,5,5.5,27.866,31.904,10.561956882476807\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,71,20261111-2990841365-71,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,488,3,9,10.5,28.188,31.98,10.997310638427734\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,72,20261111-2990841365-72,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,498,0,2,2.0,27.842,31.532,10.939165830612183\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,73,20261111-2990841365-73,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,1,5,5.5,27.89,31.876,11.432700395584106\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,74,20261111-2990841365-74,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,496,1,3,3.5,27.786,31.668,11.856979608535767\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,75,20261111-2990841365-75,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,0,7,7.0,28.364,32.292,11.738595962524414\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,76,20261111-2990841365-76,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,488,2,10,11.0,28.234,32.096,11.249436616897583\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,77,20261111-2990841365-77,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,493,1,6,6.5,28.172,32.028,11.034252882003784\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,78,20261111-2990841365-78,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,492,1,7,7.5,28.124,32.02,11.915269374847412\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g5,79,20261111-2990841365-79,A,noisy,5,0.0003,0.0003,2,5,5,5,5,5,2,500,494,0,6,6.0,28.038,31.96,11.09201717376709\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,0,20261111-3039386124-0,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,0,7,7.0,25.652,31.964,10.271013975143433\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,1,20261111-3039386124-1,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,2,4,5.0,25.56,31.66,10.48343014717102\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,2,20261111-3039386124-2,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,0,7,7.0,25.842,32.232,10.136655569076538\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,3,20261111-3039386124-3,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,0,5,5.0,25.772,32.192,11.182765007019043\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,4,20261111-3039386124-4,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,0,6,6.0,26.216,31.808,9.728536128997803\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,5,20261111-3039386124-5,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,0,5,5.0,25.704,32.008,9.545112609863281\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,6,20261111-3039386124-6,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,0,4,4.0,25.934,31.836,8.525543689727783\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,7,20261111-3039386124-7,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,1,4,4.5,25.618,31.848,9.446492195129395\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,8,20261111-3039386124-8,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,497,1,2,2.5,25.706,31.968,9.675320863723755\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,9,20261111-3039386124-9,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,0,4,4.0,26.198,32.12,9.41973352432251\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,10,20261111-3039386124-10,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,0,5,5.0,26.29,31.724,8.899165153503418\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,11,20261111-3039386124-11,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,2,4,5.0,26.102,32.096,9.242195844650269\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,12,20261111-3039386124-12,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,0,6,6.0,25.518,32.26,9.612678050994873\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,13,20261111-3039386124-13,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,497,1,2,2.5,26.04,31.816,9.783573389053345\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,14,20261111-3039386124-14,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,0,4,4.0,25.516,32.0,8.917126655578613\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,15,20261111-3039386124-15,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,491,2,7,8.0,25.55,31.836,8.998302936553955\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,16,20261111-3039386124-16,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,492,0,8,8.0,25.574,31.896,9.636125326156616\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,17,20261111-3039386124-17,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,0,5,5.0,25.7,31.892,9.68883752822876\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,18,20261111-3039386124-18,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,491,1,8,8.5,25.69,31.896,10.03244137763977\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,19,20261111-3039386124-19,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,498,1,1,1.5,25.758,31.932,9.542083501815796\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,20,20261111-3039386124-20,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,0,6,6.0,25.812,32.36,9.458429336547852\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,21,20261111-3039386124-21,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,492,1,7,7.5,25.662,31.908,9.749748945236206\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,22,20261111-3039386124-22,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,0,7,7.0,25.71,31.9,9.282835483551025\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,23,20261111-3039386124-23,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,1,6,6.5,25.79,31.78,10.090818166732788\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,24,20261111-3039386124-24,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,0,7,7.0,26.04,32.124,10.1081063747406\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,25,20261111-3039386124-25,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,0,4,4.0,25.498,31.236,8.801780223846436\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,26,20261111-3039386124-26,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,1,5,5.5,25.952,32.276,8.92525577545166\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,27,20261111-3039386124-27,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,0,6,6.0,25.71,32.216,9.387182235717773\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,28,20261111-3039386124-28,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,0,7,7.0,25.89,31.816,9.621958494186401\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,29,20261111-3039386124-29,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,0,7,7.0,25.558,31.216,9.73811149597168\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,30,20261111-3039386124-30,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,492,1,7,7.5,26.064,32.22,9.750328302383423\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,31,20261111-3039386124-31,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,492,0,8,8.0,26.114,32.212,9.488698959350586\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,32,20261111-3039386124-32,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,0,6,6.0,25.804,32.048,9.44236421585083\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,33,20261111-3039386124-33,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,2,2,3.0,25.718,31.652,9.507341384887695\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,34,20261111-3039386124-34,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,1,5,5.5,25.46,31.936,8.807065725326538\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,35,20261111-3039386124-35,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,1,4,4.5,25.832,32.0,8.935435771942139\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,36,20261111-3039386124-36,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,492,0,8,8.0,26.082,31.864,9.938408851623535\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,37,20261111-3039386124-37,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,1,4,4.5,25.582,32.536,8.990827322006226\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,38,20261111-3039386124-38,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,1,3,3.5,25.914,32.064,10.323111534118652\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,39,20261111-3039386124-39,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,1,4,4.5,25.858,32.044,8.732361793518066\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,40,20261111-3039386124-40,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,1,5,5.5,25.976,31.988,9.138806819915771\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,41,20261111-3039386124-41,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,499,0,1,1.0,25.738,31.908,9.738271474838257\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,42,20261111-3039386124-42,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,1,4,4.5,25.954,31.96,9.857044219970703\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,43,20261111-3039386124-43,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,1,6,6.5,26.052,31.868,9.268804550170898\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,44,20261111-3039386124-44,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,0,5,5.0,25.472,31.604,9.919782638549805\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,45,20261111-3039386124-45,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,0,7,7.0,26.056,32.212,9.481577396392822\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,46,20261111-3039386124-46,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,1,5,5.5,25.908,32.0,9.799511909484863\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,47,20261111-3039386124-47,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,491,0,9,9.0,25.994,32.024,9.567838668823242\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,48,20261111-3039386124-48,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,1,4,4.5,25.618,31.972,9.596189498901367\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,49,20261111-3039386124-49,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,491,1,8,8.5,26.374,31.944,8.890799522399902\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,50,20261111-3039386124-50,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,498,0,2,2.0,25.742,32.128,9.63627314567566\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,51,20261111-3039386124-51,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,0,5,5.0,25.232,31.42,9.93527865409851\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,52,20261111-3039386124-52,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,0,5,5.0,26.21,32.032,10.299635171890259\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,53,20261111-3039386124-53,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,497,1,2,2.5,25.698,32.232,8.447710990905762\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,54,20261111-3039386124-54,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,0,6,6.0,25.79,32.396,9.562473773956299\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,55,20261111-3039386124-55,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,498,2,0,1.0,25.688,31.76,9.137531042098999\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,56,20261111-3039386124-56,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,491,1,8,8.5,25.572,32.044,9.790911197662354\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,57,20261111-3039386124-57,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,0,6,6.0,25.256,31.692,9.499429941177368\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,58,20261111-3039386124-58,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,498,0,2,2.0,25.896,32.18,9.230523109436035\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,59,20261111-3039386124-59,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,0,4,4.0,25.904,32.244,8.907458543777466\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,60,20261111-3039386124-60,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,0,4,4.0,26.012,31.86,9.383879899978638\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,61,20261111-3039386124-61,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,491,2,7,8.0,25.942,32.004,10.36341404914856\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,62,20261111-3039386124-62,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,498,0,2,2.0,25.91,31.828,9.311168909072876\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,63,20261111-3039386124-63,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,498,1,1,1.5,25.592,32.02,9.836018085479736\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,64,20261111-3039386124-64,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,1,3,3.5,25.796,32.096,9.54210352897644\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,65,20261111-3039386124-65,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,0,4,4.0,26.138,32.18,9.864055633544922\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,66,20261111-3039386124-66,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,1,4,4.5,26.084,32.012,10.199459075927734\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,67,20261111-3039386124-67,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,1,4,4.5,25.706,32.328,9.246658325195312\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,68,20261111-3039386124-68,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,1,6,6.5,25.886,32.448,9.569709539413452\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,69,20261111-3039386124-69,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,0,4,4.0,25.954,32.344,10.38924264907837\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,70,20261111-3039386124-70,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,491,2,7,8.0,26.084,31.772,8.837880611419678\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,71,20261111-3039386124-71,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,497,1,2,2.5,25.772,32.172,9.69749665260315\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,72,20261111-3039386124-72,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,495,0,5,5.0,25.754,32.0,8.918126344680786\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,73,20261111-3039386124-73,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,492,0,8,8.0,25.822,32.452,9.186461687088013\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,74,20261111-3039386124-74,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,1,6,6.5,26.014,31.932,8.901468992233276\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,75,20261111-3039386124-75,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,493,0,7,7.0,26.384,31.612,9.428632020950317\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,76,20261111-3039386124-76,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,494,1,5,5.5,25.886,32.004,9.264350175857544\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,77,20261111-3039386124-77,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,496,2,2,3.0,25.774,32.236,9.509226083755493\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,78,20261111-3039386124-78,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,497,1,2,2.5,26.152,32.068,8.821282148361206\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K2|g1,79,20261111-3039386124-79,A,noisy,5,0.0003,0.0003,2,1,5,5,1,5,2,500,491,0,9,9.0,25.802,31.916,9.221736431121826\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,0,20261111-184088944-0,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.852852852852855,48.006006006006004,11.674773216247559\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,1,20261111-184088944-1,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.765765765765764,47.369369369369366,11.450339078903198\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,2,20261111-184088944-2,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,0,7,7.0,40.86186186186186,47.92192192192192,12.518954515457153\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,3,20261111-184088944-3,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,1,5,5.5,40.213213213213216,47.55555555555556,11.452234983444214\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,4,20261111-184088944-4,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,0,7,7.0,40.27927927927928,47.73573573573574,9.769455432891846\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,5,20261111-184088944-5,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,1,6,6.5,40.891891891891895,47.08108108108108,9.852519035339355\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,6,20261111-184088944-6,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,1,8,8.5,41.2012012012012,47.98198198198198,9.653481721878052\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,7,20261111-184088944-7,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,40.411411411411414,48.390390390390394,8.982796907424927\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,8,20261111-184088944-8,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.633633633633636,47.9039039039039,9.830323696136475\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,9,20261111-184088944-9,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.591591591591595,48.408408408408405,9.198107242584229\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,10,20261111-184088944-10,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,1,3,3.5,40.67567567567568,48.0960960960961,9.778956174850464\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,11,20261111-184088944-11,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,2,3,4.0,40.37837837837838,48.5045045045045,9.814609050750732\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,12,20261111-184088944-12,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,40.65765765765766,48.294294294294296,9.574944972991943\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,13,20261111-184088944-13,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,331,0,2,2.0,41.030030030030034,48.114114114114116,9.562116384506226\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,14,20261111-184088944-14,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.687687687687685,48.210210210210214,11.004165410995483\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,15,20261111-184088944-15,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,2,6,7.0,41.027027027027025,47.5975975975976,9.754846096038818\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,16,20261111-184088944-16,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,1,6,6.5,40.513513513513516,48.174174174174176,10.24767541885376\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,17,20261111-184088944-17,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,1,4,4.5,40.309309309309306,48.708708708708706,9.81675386428833\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,18,20261111-184088944-18,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,40.81681681681682,48.372372372372375,9.275359392166138\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,19,20261111-184088944-19,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,0,9,9.0,40.08108108108108,48.108108108108105,12.363813638687134\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,20,20261111-184088944-20,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,2,4,5.0,40.37837837837838,48.726726726726724,9.489728450775146\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,21,20261111-184088944-21,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,331,0,2,2.0,41.31531531531532,47.945945945945944,9.370629072189331\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,22,20261111-184088944-22,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.567567567567565,48.294294294294296,10.051696538925171\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,23,20261111-184088944-23,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,0,7,7.0,40.171171171171174,47.57957957957958,10.42780089378357\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,24,20261111-184088944-24,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.885885885885884,48.4984984984985,9.933380842208862\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,25,20261111-184088944-25,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,0,3,3.0,40.549549549549546,47.927927927927925,10.351154088973999\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,26,20261111-184088944-26,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.64264264264264,48.3963963963964,10.572197675704956\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,27,20261111-184088944-27,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,1,3,3.5,40.72372372372372,48.090090090090094,12.399959802627563\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,28,20261111-184088944-28,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,41.009009009009006,47.753753753753756,9.768030643463135\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,29,20261111-184088944-29,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,41.228228228228225,48.786786786786784,10.102866411209106\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,30,20261111-184088944-30,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,1,4,4.5,40.645645645645644,48.132132132132135,10.813095092773438\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,31,20261111-184088944-31,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,332,0,1,1.0,40.606606606606604,47.97597597597598,9.286113977432251\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,32,20261111-184088944-32,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,1,3,3.5,40.711711711711715,47.8018018018018,11.207373142242432\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,33,20261111-184088944-33,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.15615615615616,48.1021021021021,10.563769578933716\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,34,20261111-184088944-34,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,1,5,5.5,40.969969969969966,48.690690690690694,10.240984439849854\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,35,20261111-184088944-35,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,1,5,5.5,40.888888888888886,47.91591591591592,10.437585592269897\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,36,20261111-184088944-36,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,1,7,7.5,41.147147147147145,48.174174174174176,11.991228818893433\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,37,20261111-184088944-37,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,40.65465465465466,47.91591591591592,10.238818168640137\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,38,20261111-184088944-38,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,1,2,2.5,41.2972972972973,47.93993993993994,9.7178795337677\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,39,20261111-184088944-39,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.94294294294294,47.789789789789786,10.567213535308838\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,40,20261111-184088944-40,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,0,8,8.0,41.087087087087085,48.31831831831832,10.44519853591919\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,41,20261111-184088944-41,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,2,5,6.0,40.77777777777778,48.354354354354356,11.247076988220215\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,42,20261111-184088944-42,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,1,7,7.5,40.44444444444444,48.26426426426426,9.544204711914062\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,43,20261111-184088944-43,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,331,1,1,1.5,40.210210210210214,48.546546546546544,10.37671971321106\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,44,20261111-184088944-44,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,1,2,2.5,41.08108108108108,48.306306306306304,10.085432291030884\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,45,20261111-184088944-45,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.705705705705704,47.83783783783784,12.277942419052124\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,46,20261111-184088944-46,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,323,1,9,9.5,40.927927927927925,48.390390390390394,9.519338607788086\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,47,20261111-184088944-47,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,2,6,7.0,40.369369369369366,48.56456456456456,9.249138593673706\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,48,20261111-184088944-48,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,1,7,7.5,41.13813813813814,47.98198198198198,10.605757474899292\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,49,20261111-184088944-49,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,1,8,8.5,41.267267267267265,48.54054054054054,10.5820951461792\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,50,20261111-184088944-50,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,2,4,5.0,40.73573573573574,47.93993993993994,11.314563274383545\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,51,20261111-184088944-51,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,2,5,6.0,40.870870870870874,48.066066066066064,9.757644414901733\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,52,20261111-184088944-52,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,0,8,8.0,40.390390390390394,47.8978978978979,9.986267328262329\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,53,20261111-184088944-53,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,1,8,8.5,40.711711711711715,48.048048048048045,10.654478788375854\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,54,20261111-184088944-54,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,40.426426426426424,47.75975975975976,12.2511625289917\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,55,20261111-184088944-55,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,2,2,3.0,41.07807807807808,47.609609609609606,9.538445472717285\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,56,20261111-184088944-56,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.867867867867865,47.711711711711715,9.797643184661865\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,57,20261111-184088944-57,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,0,7,7.0,40.4984984984985,47.85585585585586,10.319836139678955\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,58,20261111-184088944-58,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,1,7,7.5,40.84384384384384,47.83783783783784,9.052067041397095\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,59,20261111-184088944-59,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,0,7,7.0,41.012012012012015,48.174174174174176,11.334314823150635\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,60,20261111-184088944-60,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,0,3,3.0,40.732732732732735,48.252252252252255,9.637065649032593\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,61,20261111-184088944-61,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,1,5,5.5,40.906906906906904,48.348348348348345,9.762988567352295\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,62,20261111-184088944-62,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,1,3,3.5,40.5045045045045,48.330330330330334,9.858398199081421\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,63,20261111-184088944-63,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,2,6,7.0,41.147147147147145,47.86186186186186,11.945590496063232\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,64,20261111-184088944-64,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,1,4,4.5,40.090090090090094,47.207207207207205,10.168664455413818\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,65,20261111-184088944-65,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.4024024024024,47.789789789789786,10.178323745727539\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,66,20261111-184088944-66,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,2,6,7.0,40.58258258258258,47.849849849849846,9.470948696136475\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,67,20261111-184088944-67,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,0,8,8.0,40.97897897897898,48.672672672672675,10.122584104537964\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,68,20261111-184088944-68,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,323,2,8,9.0,40.468468468468465,48.012012012012015,9.900463581085205\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,69,20261111-184088944-69,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,41.14114114114114,48.08408408408408,10.935900926589966\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,70,20261111-184088944-70,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,1,7,7.5,40.64264264264264,48.04204204204204,10.669869184494019\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,71,20261111-184088944-71,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,321,3,9,10.5,40.945945945945944,47.753753753753756,10.371464729309082\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,72,20261111-184088944-72,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,0,3,3.0,40.72372372372372,47.549549549549546,10.417215585708618\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,73,20261111-184088944-73,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,2,5,6.0,40.51951951951952,48.552552552552555,9.822326421737671\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,74,20261111-184088944-74,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,0,7,7.0,40.88288288288288,48.3003003003003,10.195143222808838\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,75,20261111-184088944-75,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.7987987987988,48.288288288288285,10.454282760620117\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,76,20261111-184088944-76,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,40.273273273273276,47.84384384384384,12.016605138778687\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,77,20261111-184088944-77,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,1,7,7.5,41.474474474474476,47.945945945945944,9.558746337890625\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,78,20261111-184088944-78,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,0,9,9.0,40.630630630630634,48.090090090090094,10.031030416488647\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,79,20261111-184088944-79,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.750750750750754,47.47747747747748,11.071835041046143\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,80,20261111-184088944-80,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,1,4,4.5,40.708708708708706,47.969969969969966,11.21709132194519\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,81,20261111-184088944-81,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,1,6,6.5,41.03603603603604,48.126126126126124,9.40935230255127\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,82,20261111-184088944-82,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,1,2,2.5,40.765765765765764,47.71771771771772,9.778725862503052\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,83,20261111-184088944-83,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,0,3,3.0,40.849849849849846,48.450450450450454,9.690322637557983\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,84,20261111-184088944-84,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,323,1,9,9.5,41.24324324324324,48.270270270270274,11.187070846557617\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,85,20261111-184088944-85,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,323,0,10,10.0,41.570570570570574,48.366366366366364,10.218693256378174\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,86,20261111-184088944-86,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,323,0,10,10.0,40.38438438438438,48.210210210210214,10.258872747421265\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,87,20261111-184088944-87,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,0,3,3.0,41.2042042042042,48.186186186186184,10.213402509689331\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,88,20261111-184088944-88,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,332,0,1,1.0,40.672672672672675,47.993993993993996,10.161987066268921\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,89,20261111-184088944-89,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,0,8,8.0,40.546546546546544,47.849849849849846,11.072383642196655\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,90,20261111-184088944-90,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,41.01501501501502,48.53453453453454,9.920402765274048\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,91,20261111-184088944-91,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.38138138138138,48.090090090090094,9.712945222854614\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,92,20261111-184088944-92,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,0,3,3.0,40.153153153153156,48.07807807807808,11.105224132537842\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,93,20261111-184088944-93,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,0,9,9.0,41.171171171171174,48.15615615615616,9.876571893692017\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,94,20261111-184088944-94,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,1,2,2.5,40.672672672672675,48.066066066066064,12.344804763793945\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,95,20261111-184088944-95,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,331,0,2,2.0,40.732732732732735,47.951951951951955,10.290257930755615\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,96,20261111-184088944-96,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,1,6,6.5,41.189189189189186,48.150150150150154,9.571153163909912\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,97,20261111-184088944-97,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,41.3033033033033,47.8018018018018,11.230153322219849\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,98,20261111-184088944-98,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,1,3,3.5,40.36336336336336,47.669669669669666,11.62430715560913\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,99,20261111-184088944-99,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,1,6,6.5,41.2012012012012,48.74474474474474,10.297611951828003\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,100,20261111-184088944-100,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.68168168168168,47.585585585585584,9.402336597442627\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,101,20261111-184088944-101,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.672672672672675,47.789789789789786,9.57340145111084\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,102,20261111-184088944-102,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,1,8,8.5,40.58258258258258,48.228228228228225,10.620992183685303\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,103,20261111-184088944-103,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,1,6,6.5,40.9009009009009,47.51951951951952,10.90983247756958\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,104,20261111-184088944-104,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,1,5,5.5,41.090090090090094,47.97597597597598,9.407114744186401\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,105,20261111-184088944-105,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,325,1,7,7.5,40.46246246246246,47.453453453453456,10.646276473999023\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,106,20261111-184088944-106,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,1,4,4.5,40.831831831831835,48.63663663663664,11.140357971191406\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,107,20261111-184088944-107,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,4,5,7.0,41.171171171171174,48.68468468468468,9.742696285247803\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,108,20261111-184088944-108,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,1,8,8.5,41.32432432432432,48.74474474474474,10.054487228393555\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,109,20261111-184088944-109,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,0,3,3.0,40.65465465465466,47.27927927927928,10.161578178405762\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,110,20261111-184088944-110,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,330,0,3,3.0,40.711711711711715,48.16216216216216,12.198158264160156\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,111,20261111-184088944-111,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,322,2,9,10.0,40.705705705705704,47.24324324324324,9.033284187316895\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,112,20261111-184088944-112,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,41.273273273273276,48.546546546546544,10.744800567626953\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,113,20261111-184088944-113,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,1,8,8.5,41.03603603603604,48.372372372372375,10.10467791557312\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,114,20261111-184088944-114,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,41.32132132132132,48.06006006006006,9.304640054702759\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,115,20261111-184088944-115,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,327,0,6,6.0,40.97597597597598,48.546546546546544,12.590471744537354\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,116,20261111-184088944-116,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,1,4,4.5,40.57657657657658,48.37837837837838,9.655877351760864\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,117,20261111-184088944-117,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,328,0,5,5.0,40.71771771771772,47.933933933933936,9.876328706741333\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,118,20261111-184088944-118,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,326,0,7,7.0,41.03603603603604,48.408408408408405,8.96729826927185\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,119,20261111-184088944-119,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,329,0,4,4.0,40.810810810810814,47.789789789789786,9.062803745269775\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g5,120,20261111-184088944-120,A,noisy,5,0.0003,0.0003,3,5,5,5,5,5,2,333,324,1,8,8.5,41.411411411411414,47.789789789789786,11.964538335800171\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,0,20261111-227884393-0,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,1,3,3.5,36.411411411411414,48.31831831831832,11.351066827774048\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,1,20261111-227884393-1,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,35.94294294294294,48.450450450450454,11.469638586044312\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,2,20261111-227884393-2,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,321,3,9,10.5,36.2972972972973,48.27627627627628,10.34778118133545\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,3,20261111-227884393-3,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,0,7,7.0,36.16516516516516,47.993993993993996,8.401660680770874\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,4,20261111-227884393-4,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,2,4,5.0,36.771771771771775,48.02402402402402,10.23945927619934\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,5,20261111-227884393-5,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,0,7,7.0,36.426426426426424,47.993993993993996,10.548474550247192\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,6,20261111-227884393-6,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,36.1981981981982,48.1021021021021,8.078389883041382\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,7,20261111-227884393-7,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,0,4,4.0,36.147147147147145,47.6996996996997,8.719414472579956\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,8,20261111-227884393-8,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,323,1,9,9.5,36.17717717717718,48.192192192192195,8.73046588897705\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,9,20261111-227884393-9,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,36.072072072072075,47.891891891891895,8.887649774551392\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,10,20261111-227884393-10,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,1,2,2.5,36.354354354354356,48.45645645645646,10.511093854904175\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,11,20261111-227884393-11,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,0,6,6.0,35.585585585585584,47.8978978978979,8.969674825668335\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,12,20261111-227884393-12,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,1,7,7.5,35.8048048048048,48.3003003003003,8.448201417922974\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,13,20261111-227884393-13,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,2,6,7.0,36.68468468468468,47.35735735735736,7.975208520889282\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,14,20261111-227884393-14,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,36.1021021021021,48.0960960960961,9.77228331565857\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,15,20261111-227884393-15,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,1,5,5.5,36.033033033033036,48.06006006006006,8.287064790725708\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,16,20261111-227884393-16,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,3,4,5.5,36.531531531531535,48.91891891891892,7.918501377105713\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,17,20261111-227884393-17,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,1,2,2.5,36.306306306306304,48.030030030030034,8.309144496917725\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,18,20261111-227884393-18,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,0,4,4.0,36.44444444444444,47.5015015015015,8.642220735549927\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,19,20261111-227884393-19,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,36.46246246246246,46.91891891891892,8.486695528030396\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,20,20261111-227884393-20,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,2,6,7.0,36.126126126126124,47.51951951951952,8.72007131576538\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,21,20261111-227884393-21,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,36.4054054054054,48.192192192192195,8.40470552444458\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,22,20261111-227884393-22,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,2,3,4.0,35.86186186186186,47.789789789789786,8.879685640335083\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,23,20261111-227884393-23,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,2,4,5.0,36.38138138138138,47.807807807807805,8.593076705932617\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,24,20261111-227884393-24,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,2,5,6.0,36.0960960960961,48.25825825825826,8.36014699935913\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,25,20261111-227884393-25,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,36.01501501501502,48.432432432432435,8.292732238769531\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,26,20261111-227884393-26,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,1,2,2.5,35.948948948948946,48.186186186186184,10.102024555206299\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,27,20261111-227884393-27,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,1,7,7.5,36.186186186186184,48.66066066066066,8.824856281280518\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,28,20261111-227884393-28,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,1,3,3.5,36.21921921921922,48.7027027027027,8.980311393737793\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,29,20261111-227884393-29,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,0,7,7.0,36.186186186186184,47.567567567567565,7.5202858448028564\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,30,20261111-227884393-30,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,324,2,7,8.0,36.090090090090094,47.933933933933936,8.158829689025879\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,31,20261111-227884393-31,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,1,5,5.5,35.67567567567568,47.6036036036036,8.585482358932495\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,32,20261111-227884393-32,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,324,0,9,9.0,36.087087087087085,47.41741741741742,9.148446559906006\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,33,20261111-227884393-33,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,0,6,6.0,36.5045045045045,47.831831831831835,7.8623716831207275\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,34,20261111-227884393-34,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,1,7,7.5,36.02402402402402,48.288288288288285,9.313014507293701\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,35,20261111-227884393-35,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,323,1,9,9.5,36.8948948948949,48.012012012012015,8.992835521697998\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,36,20261111-227884393-36,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,323,3,7,8.5,36.393393393393396,47.8978978978979,8.491001844406128\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,37,20261111-227884393-37,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,36.13513513513514,47.927927927927925,9.4483163356781\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,38,20261111-227884393-38,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,35.765765765765764,47.549549549549546,10.069277048110962\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,39,20261111-227884393-39,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,36.04504504504504,48.426426426426424,9.063197374343872\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,40,20261111-227884393-40,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,0,8,8.0,36.069069069069066,48.732732732732735,8.791724443435669\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,41,20261111-227884393-41,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,0,8,8.0,35.75975975975976,48.414414414414416,10.625966310501099\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,42,20261111-227884393-42,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,3,4,5.5,36.492492492492495,47.729729729729726,8.436337232589722\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,43,20261111-227884393-43,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,36.150150150150154,47.945945945945944,8.804269552230835\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,44,20261111-227884393-44,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,2,4,5.0,35.825825825825824,47.6996996996997,8.564432621002197\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,45,20261111-227884393-45,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,322,1,10,10.5,35.627627627627625,47.813813813813816,9.677696704864502\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,46,20261111-227884393-46,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,35.6996996996997,48.294294294294296,8.943216562271118\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,47,20261111-227884393-47,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,2,6,7.0,36.069069069069066,47.705705705705704,9.19741940498352\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,48,20261111-227884393-48,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,0,6,6.0,35.969969969969966,47.933933933933936,8.93989634513855\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,49,20261111-227884393-49,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,36.153153153153156,47.825825825825824,9.130422830581665\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,50,20261111-227884393-50,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,1,5,5.5,35.54054054054054,47.693693693693696,8.36983060836792\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,51,20261111-227884393-51,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,4,3,5.0,35.82282282282282,49.051051051051054,9.521543264389038\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,52,20261111-227884393-52,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,1,3,3.5,36.04204204204204,47.83783783783784,8.860281944274902\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,53,20261111-227884393-53,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,0,7,7.0,36.567567567567565,47.807807807807805,9.93468689918518\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,54,20261111-227884393-54,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,36.07507507507508,48.3003003003003,8.802587985992432\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,55,20261111-227884393-55,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,36.37537537537538,48.612612612612615,8.689899444580078\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,56,20261111-227884393-56,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,1,3,3.5,36.789789789789786,48.588588588588586,9.9770348072052\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,57,20261111-227884393-57,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,35.909909909909906,47.53753753753754,10.671286821365356\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,58,20261111-227884393-58,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,1,6,6.5,35.71471471471472,47.453453453453456,8.645187377929688\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,59,20261111-227884393-59,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,1,5,5.5,36.15915915915916,47.93993993993994,8.549167156219482\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,60,20261111-227884393-60,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,36.57957957957958,48.030030030030034,9.871874570846558\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,61,20261111-227884393-61,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,322,2,9,10.0,35.74474474474474,48.45645645645646,10.542025804519653\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,62,20261111-227884393-62,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,35.870870870870874,48.67867867867868,9.032915592193604\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,63,20261111-227884393-63,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,2,5,6.0,36.13813813813814,48.7027027027027,8.965699195861816\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,64,20261111-227884393-64,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,2,2,3.0,35.930930930930934,47.95795795795796,9.516295433044434\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,65,20261111-227884393-65,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,1,2,2.5,35.870870870870874,48.16216216216216,8.850362062454224\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,66,20261111-227884393-66,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,35.56456456456456,47.71771771771772,8.83097791671753\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,67,20261111-227884393-67,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,36.02102102102102,47.891891891891895,9.189789056777954\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,68,20261111-227884393-68,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,0,4,4.0,36.0,48.288288288288285,10.057259798049927\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,69,20261111-227884393-69,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,2,4,5.0,35.666666666666664,47.813813813813816,8.244383573532104\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,70,20261111-227884393-70,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,1,1,1.5,36.38138138138138,47.609609609609606,8.629131078720093\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,71,20261111-227884393-71,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,2,3,4.0,36.75675675675676,47.609609609609606,8.703638315200806\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,72,20261111-227884393-72,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,0,7,7.0,35.83483483483484,47.951951951951955,9.330122947692871\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,73,20261111-227884393-73,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,1,5,5.5,36.048048048048045,48.01801801801802,9.273406744003296\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,74,20261111-227884393-74,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,0,7,7.0,35.825825825825824,48.168168168168165,9.380403280258179\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,75,20261111-227884393-75,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,1,3,3.5,36.492492492492495,47.765765765765764,8.755756616592407\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,76,20261111-227884393-76,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,0,8,8.0,36.45945945945946,48.414414414414416,8.986528396606445\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,77,20261111-227884393-77,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,36.24324324324324,48.072072072072075,8.750402450561523\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,78,20261111-227884393-78,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,0,6,6.0,36.513513513513516,47.471471471471475,9.791158437728882\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,79,20261111-227884393-79,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,1,6,6.5,35.651651651651655,47.849849849849846,9.396031141281128\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,80,20261111-227884393-80,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,1,6,6.5,36.051051051051054,47.87987987987988,9.39221477508545\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,81,20261111-227884393-81,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,322,1,10,10.5,36.27927927927928,48.648648648648646,9.385457992553711\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,82,20261111-227884393-82,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,0,6,6.0,36.450450450450454,48.28228228228228,9.2395920753479\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,83,20261111-227884393-83,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,1,5,5.5,36.71471471471472,48.072072072072075,9.176628112792969\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,84,20261111-227884393-84,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,36.687687687687685,48.126126126126124,9.01763391494751\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,85,20261111-227884393-85,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,0,4,4.0,36.291291291291294,47.5975975975976,9.356192588806152\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,86,20261111-227884393-86,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,0,4,4.0,35.909909909909906,47.56156156156156,8.477947235107422\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,87,20261111-227884393-87,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,1,6,6.5,36.5015015015015,48.192192192192195,9.704294681549072\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,88,20261111-227884393-88,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,0,6,6.0,36.153153153153156,48.1981981981982,10.08310341835022\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,89,20261111-227884393-89,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,2,4,5.0,36.273273273273276,48.18018018018018,9.508399724960327\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,90,20261111-227884393-90,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,333,0,0,0.0,36.71771771771772,48.330330330330334,8.687279224395752\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,91,20261111-227884393-91,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,1,2,2.5,35.990990990990994,48.090090090090094,8.896143913269043\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,92,20261111-227884393-92,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,36.447447447447445,47.37537537537538,9.533655405044556\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,93,20261111-227884393-93,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,0,6,6.0,36.27627627627628,47.789789789789786,9.436654806137085\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,94,20261111-227884393-94,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,323,2,8,9.0,36.13513513513514,48.168168168168165,8.964117765426636\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,95,20261111-227884393-95,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,2,4,5.0,36.354354354354356,48.354354354354356,9.254011154174805\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,96,20261111-227884393-96,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,0,7,7.0,36.270270270270274,48.34234234234234,10.404706716537476\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,97,20261111-227884393-97,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,1,4,4.5,35.846846846846844,48.246246246246244,8.859379768371582\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,98,20261111-227884393-98,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,323,1,9,9.5,36.0960960960961,48.408408408408405,8.91494345664978\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,99,20261111-227884393-99,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,36.228228228228225,48.186186186186184,9.748254299163818\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,100,20261111-227884393-100,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,1,4,4.5,36.5015015015015,48.08408408408408,9.638203620910645\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,101,20261111-227884393-101,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,1,3,3.5,36.07807807807808,48.150150150150154,9.055341482162476\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,102,20261111-227884393-102,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,0,7,7.0,36.588588588588586,47.66366366366366,8.401618957519531\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,103,20261111-227884393-103,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,36.63963963963964,48.888888888888886,9.672236919403076\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,104,20261111-227884393-104,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,329,0,4,4.0,36.51651651651652,48.030030030030034,8.617199897766113\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,105,20261111-227884393-105,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,1,6,6.5,35.948948948948946,48.54054054054054,9.652390003204346\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,106,20261111-227884393-106,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,1,4,4.5,36.093093093093096,47.62162162162162,9.235034942626953\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,107,20261111-227884393-107,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,324,0,9,9.0,35.91891891891892,48.090090090090094,8.749553680419922\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,108,20261111-227884393-108,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,328,0,5,5.0,36.32132132132132,48.33633633633634,8.77188229560852\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,109,20261111-227884393-109,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,1,2,2.5,35.966966966966964,48.04204204204204,9.400466442108154\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,110,20261111-227884393-110,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,326,1,6,6.5,35.825825825825824,48.126126126126124,9.89313530921936\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,111,20261111-227884393-111,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,327,0,6,6.0,36.02102102102102,47.591591591591595,8.850955724716187\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,112,20261111-227884393-112,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,331,0,2,2.0,36.86486486486486,48.53453453453454,8.579022884368896\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,113,20261111-227884393-113,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,1,7,7.5,36.34534534534534,47.68168168168168,9.135202884674072\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,114,20261111-227884393-114,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,35.8048048048048,48.27627627627628,8.985820531845093\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,115,20261111-227884393-115,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,1,2,2.5,36.06006006006006,48.36036036036036,9.296191930770874\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,116,20261111-227884393-116,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,325,0,8,8.0,36.132132132132135,48.21621621621622,9.125111103057861\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,117,20261111-227884393-117,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,324,1,8,8.5,36.546546546546544,47.75975975975976,8.224029302597046\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,118,20261111-227884393-118,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,1,2,2.5,36.813813813813816,48.01801801801802,9.399453163146973\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,119,20261111-227884393-119,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,330,0,3,3.0,36.474474474474476,48.27627627627628,8.9643075466156\npilot-20261110T155331Z-d3f938,A|noisy|d5|p0.0003|K3|g1,120,20261111-227884393-120,A,noisy,5,0.0003,0.0003,3,1,5,5,1,5,2,333,333,0,0,0.0,36.1981981981982,48.64264264264264,9.230408906936646\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,0,20261111-2860497070-0,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3995,0,5,5.0,0.8905,3.9995,7.818910598754883\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,1,20261111-2860497070-1,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,1,3,3.5,0.89175,3.9715,7.709506988525391\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,2,20261111-2860497070-2,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,0,8,8.0,0.85175,3.9925,8.054871559143066\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,3,20261111-2860497070-3,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.89675,3.9525,8.331212043762207\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,4,20261111-2860497070-4,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3989,1,10,10.5,0.876,4.0255,7.269350290298462\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,5,20261111-2860497070-5,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3990,0,10,10.0,0.8695,3.986,7.745831727981567\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,6,20261111-2860497070-6,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.8545,4.081,7.7961585521698\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,7,20261111-2860497070-7,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3993,0,7,7.0,0.88225,4.0055,8.133849620819092\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,8,20261111-2860497070-8,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3995,1,4,4.5,0.8745,4.0135,7.780661344528198\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,9,20261111-2860497070-9,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.885,3.949,7.521380186080933\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,10,20261111-2860497070-10,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.8935,3.9905,8.075811386108398\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,11,20261111-2860497070-11,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3991,0,9,9.0,0.8945,3.966,7.84461236000061\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,12,20261111-2860497070-12,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3991,1,8,8.5,0.885,4.047,7.570972919464111\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,13,20261111-2860497070-13,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3987,1,12,12.5,0.877,3.9995,8.121232271194458\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,14,20261111-2860497070-14,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3993,0,7,7.0,0.8665,3.993,7.818941593170166\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,15,20261111-2860497070-15,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.89125,4.0205,7.798430442810059\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,16,20261111-2860497070-16,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.8625,4.0125,7.838435649871826\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,17,20261111-2860497070-17,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3995,0,5,5.0,0.865,3.9435,8.003373146057129\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,18,20261111-2860497070-18,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,0,8,8.0,0.9085,4.029,7.2183215618133545\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,19,20261111-2860497070-19,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3990,0,10,10.0,0.8785,3.9935,7.868528604507446\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,20,20261111-2860497070-20,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3990,0,10,10.0,0.8795,3.9865,7.9911487102508545\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,21,20261111-2860497070-21,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3993,0,7,7.0,0.88575,4.013,8.093457460403442\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,22,20261111-2860497070-22,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,1,7,7.5,0.899,4.0315,7.648454666137695\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,23,20261111-2860497070-23,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.87625,4.04,8.044704675674438\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,24,20261111-2860497070-24,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3988,0,12,12.0,0.871,4.039,8.141709566116333\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,25,20261111-2860497070-25,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.89825,4.014,7.915306091308594\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,26,20261111-2860497070-26,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3994,0,6,6.0,0.886,3.996,8.25007677078247\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,27,20261111-2860497070-27,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3994,0,6,6.0,0.873,3.9765,8.003501892089844\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,28,20261111-2860497070-28,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3989,1,10,10.5,0.8795,3.9765,8.045002222061157\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,29,20261111-2860497070-29,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3995,0,5,5.0,0.884,4.0085,7.652238845825195\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,30,20261111-2860497070-30,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3994,0,6,6.0,0.8695,4.0095,7.751109838485718\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,31,20261111-2860497070-31,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,1,7,7.5,0.875,4.047,7.42037558555603\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,32,20261111-2860497070-32,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3994,0,6,6.0,0.87575,4.0185,8.065385103225708\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,33,20261111-2860497070-33,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,0,8,8.0,0.88225,4.024,7.547063827514648\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,34,20261111-2860497070-34,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3994,0,6,6.0,0.88575,3.971,7.880238056182861\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,35,20261111-2860497070-35,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,0,8,8.0,0.87325,3.9505,7.837316036224365\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,36,20261111-2860497070-36,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,1,7,7.5,0.873,4.01,7.8408496379852295\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,37,20261111-2860497070-37,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3993,0,7,7.0,0.87925,4.0245,7.599180459976196\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,38,20261111-2860497070-38,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3994,0,6,6.0,0.9115,3.963,8.055644512176514\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,39,20261111-2860497070-39,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.86725,4.021,7.955959796905518\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,40,20261111-2860497070-40,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3993,0,7,7.0,0.89875,3.999,7.922944068908691\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,41,20261111-2860497070-41,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,1,3,3.5,0.854,4.0255,7.91977858543396\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,42,20261111-2860497070-42,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3995,1,4,4.5,0.88175,3.958,7.809992074966431\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,43,20261111-2860497070-43,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.8825,4.0085,7.607861280441284\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,44,20261111-2860497070-44,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.90025,4.01,7.8098814487457275\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,45,20261111-2860497070-45,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.88025,4.0215,7.806738376617432\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,46,20261111-2860497070-46,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.874,3.9925,7.719263553619385\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,47,20261111-2860497070-47,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,0,8,8.0,0.894,3.9975,7.915992498397827\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,48,20261111-2860497070-48,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3994,0,6,6.0,0.894,3.992,7.743818521499634\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K1|g0,49,20261111-2860497070-49,B,noisy,3,0.0003,0.0,1,0,3,3,3,3,2,4000,3992,0,8,8.0,0.8735,4.0185,7.521291255950928\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,0,20261111-566445818-0,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,1.512,8.133,6.32403039932251\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,1,20261111-566445818-1,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.544,8.07,6.4270546436309814\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,2,20261111-566445818-2,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,1,8,8.5,1.5365,7.967,6.157444477081299\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,3,20261111-566445818-3,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.515,7.912,6.095598936080933\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,4,20261111-566445818-4,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.5525,8.069,6.159789562225342\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,5,20261111-566445818-5,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,1,4,4.5,1.5775,7.921,6.026008605957031\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,6,20261111-566445818-6,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.502,8.037,6.212379455566406\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,7,20261111-566445818-7,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1990,1,9,9.5,1.5005,8.026,5.780450344085693\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,8,20261111-566445818-8,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.585,8.013,6.3145811557769775\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,9,20261111-566445818-9,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,1.53,7.938,6.073984146118164\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,10,20261111-566445818-10,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.567,8.043,5.917781591415405\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,11,20261111-566445818-11,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,1.5475,7.961,6.1063902378082275\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,12,20261111-566445818-12,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,2,4,5.0,1.513,7.983,6.078561067581177\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,13,20261111-566445818-13,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,1,4,4.5,1.5175,7.947,6.112785577774048\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,14,20261111-566445818-14,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5185,8.007,6.264230728149414\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,15,20261111-566445818-15,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,1,4,4.5,1.542,7.991,6.119701147079468\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,16,20261111-566445818-16,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.5575,7.937,6.131728649139404\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,17,20261111-566445818-17,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.53,8.171,6.165550708770752\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,18,20261111-566445818-18,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5345,7.984,6.042015075683594\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,19,20261111-566445818-19,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.4985,7.928,6.12881326675415\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,20,20261111-566445818-20,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.5705,8.045,6.132165908813477\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,21,20261111-566445818-21,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.5375,8.045,5.6627655029296875\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,22,20261111-566445818-22,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.516,8.024,5.690351247787476\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,23,20261111-566445818-23,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,1,6,6.5,1.494,7.958,6.029378890991211\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,24,20261111-566445818-24,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.514,7.952,6.044477224349976\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,25,20261111-566445818-25,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.526,8.08,5.978834390640259\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,26,20261111-566445818-26,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5355,8.016,6.213000059127808\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,27,20261111-566445818-27,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1990,0,10,10.0,1.534,8.017,6.13296365737915\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,28,20261111-566445818-28,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1989,0,11,11.0,1.54,8.065,6.132657527923584\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,29,20261111-566445818-29,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,1.503,8.012,5.512394189834595\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,30,20261111-566445818-30,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,1.5325,8.088,6.006241321563721\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,31,20261111-566445818-31,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.5395,7.937,6.044550895690918\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,32,20261111-566445818-32,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.542,8.032,6.037694215774536\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,33,20261111-566445818-33,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1990,0,10,10.0,1.5355,7.919,5.68192982673645\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,34,20261111-566445818-34,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1996,1,3,3.5,1.49,8.019,6.0379958152771\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,35,20261111-566445818-35,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,1.521,8.065,6.067835807800293\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,36,20261111-566445818-36,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,1,7,7.5,1.5565,8.068,5.727057933807373\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,37,20261111-566445818-37,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,1.5375,8.047,5.739959955215454\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,38,20261111-566445818-38,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.521,8.057,6.116492748260498\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,39,20261111-566445818-39,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,1.547,7.968,6.049830675125122\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,40,20261111-566445818-40,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.489,7.966,5.7710325717926025\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,41,20261111-566445818-41,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.527,7.914,6.119549751281738\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,42,20261111-566445818-42,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.535,7.937,5.974917650222778\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,43,20261111-566445818-43,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5415,7.975,6.048820972442627\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,44,20261111-566445818-44,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1988,0,12,12.0,1.5195,7.999,5.909474849700928\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,45,20261111-566445818-45,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.5025,7.862,5.955388784408569\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,46,20261111-566445818-46,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1982,0,18,18.0,1.562,7.955,5.95582127571106\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,47,20261111-566445818-47,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.545,8.017,6.143770456314087\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,48,20261111-566445818-48,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,2,4,5.0,1.57,7.983,6.109620571136475\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,49,20261111-566445818-49,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.5455,8.046,5.634535551071167\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,50,20261111-566445818-50,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.504,8.027,6.066555500030518\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,51,20261111-566445818-51,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1989,0,11,11.0,1.5295,7.992,5.947862863540649\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,52,20261111-566445818-52,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,1,4,4.5,1.553,7.84,6.123211860656738\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,53,20261111-566445818-53,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,1.5445,7.949,5.970396041870117\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,54,20261111-566445818-54,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.5705,7.919,6.0790674686431885\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,55,20261111-566445818-55,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1990,0,10,10.0,1.551,8.009,5.904058456420898\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,56,20261111-566445818-56,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,1.5735,8.058,6.11841893196106\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,57,20261111-566445818-57,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,1,5,5.5,1.582,8.037,6.160881042480469\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,58,20261111-566445818-58,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.525,7.951,5.929548501968384\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,59,20261111-566445818-59,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.518,7.931,6.152113914489746\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,60,20261111-566445818-60,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,1.6035,8.008,6.312178134918213\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,61,20261111-566445818-61,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.486,7.98,6.224003314971924\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,62,20261111-566445818-62,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,1.5905,7.943,5.707672595977783\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,63,20261111-566445818-63,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,1.479,8.128,6.267596960067749\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,64,20261111-566445818-64,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,1,4,4.5,1.55,8.095,6.241697788238525\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,65,20261111-566445818-65,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,1,8,8.5,1.531,7.881,5.950519561767578\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,66,20261111-566445818-66,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,1.5535,7.982,5.949481010437012\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,67,20261111-566445818-67,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,1,4,4.5,1.5085,7.962,6.020495891571045\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,68,20261111-566445818-68,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,1,7,7.5,1.4945,8.061,5.700117111206055\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,69,20261111-566445818-69,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5665,8.051,5.825221300125122\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,70,20261111-566445818-70,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5175,7.987,5.688619613647461\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,71,20261111-566445818-71,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1988,1,11,11.5,1.5705,7.967,6.188045978546143\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,72,20261111-566445818-72,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1988,0,12,12.0,1.5235,7.889,5.712603569030762\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,73,20261111-566445818-73,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,1.505,8.056,6.09133243560791\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,74,20261111-566445818-74,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,1.558,7.973,5.870749235153198\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,75,20261111-566445818-75,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5815,8.028,5.867406606674194\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,76,20261111-566445818-76,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.5615,7.97,6.125073194503784\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,77,20261111-566445818-77,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,1.556,8.032,5.856847524642944\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,78,20261111-566445818-78,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,1,8,8.5,1.5375,8.006,5.938607215881348\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,79,20261111-566445818-79,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.52,8.054,6.204966306686401\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,80,20261111-566445818-80,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,1,6,6.5,1.5595,7.993,6.1762940883636475\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,81,20261111-566445818-81,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.4645,8.093,6.188152551651001\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,82,20261111-566445818-82,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1990,0,10,10.0,1.5205,7.995,5.855984687805176\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,83,20261111-566445818-83,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1990,0,10,10.0,1.57,7.956,5.90692663192749\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,84,20261111-566445818-84,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.4735,8.016,6.330408334732056\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,85,20261111-566445818-85,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.536,7.926,6.007981777191162\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,86,20261111-566445818-86,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.529,7.961,5.760218381881714\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,87,20261111-566445818-87,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,1,7,7.5,1.555,8.001,6.246745824813843\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,88,20261111-566445818-88,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,1.54,8.08,5.612112760543823\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,89,20261111-566445818-89,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,1.5065,8.051,5.937669992446899\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,90,20261111-566445818-90,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,0,8,8.0,1.517,7.974,5.873631715774536\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,91,20261111-566445818-91,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,1.5575,7.933,5.808405637741089\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,92,20261111-566445818-92,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1991,0,9,9.0,1.567,8.066,5.766998767852783\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,93,20261111-566445818-93,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1990,2,8,9.0,1.508,8.055,5.560531854629517\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,94,20261111-566445818-94,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1992,1,7,7.5,1.52,7.99,5.909715414047241\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,95,20261111-566445818-95,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,1,5,5.5,1.558,7.882,5.716024875640869\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,96,20261111-566445818-96,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,1.519,8.058,5.698893308639526\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,97,20261111-566445818-97,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1995,2,3,4.0,1.5015,7.936,6.027272462844849\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,98,20261111-566445818-98,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5175,7.954,5.583374261856079\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g3,99,20261111-566445818-99,B,noisy,3,0.0003,0.0,2,3,3,3,3,3,2,2000,1993,0,7,7.0,1.5295,8.01,6.0213398933410645\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,0,20261111-3486328790-0,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.434,7.896,5.97868800163269\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,1,20261111-3486328790-1,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,1,6,6.5,1.416,7.996,5.934747934341431\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,2,20261111-3486328790-2,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1996,1,3,3.5,1.3615,7.911,5.962861776351929\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,3,20261111-3486328790-3,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.3995,8.118,6.15906548500061\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,4,20261111-3486328790-4,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.396,7.972,5.664469003677368\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,5,20261111-3486328790-5,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1988,0,12,12.0,1.432,8.124,5.781094551086426\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,6,20261111-3486328790-6,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1992,0,8,8.0,1.388,8.025,5.553454160690308\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,7,20261111-3486328790-7,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.386,7.893,5.891092300415039\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,8,20261111-3486328790-8,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,1.376,8.017,5.2241880893707275\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,9,20261111-3486328790-9,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.3905,7.99,5.652656316757202\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,10,20261111-3486328790-10,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.4035,8.07,5.584993600845337\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,11,20261111-3486328790-11,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,1.4085,7.932,5.338878631591797\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,12,20261111-3486328790-12,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,1.388,7.985,5.7403833866119385\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,13,20261111-3486328790-13,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.355,8.083,5.603891849517822\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,14,20261111-3486328790-14,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.3585,8.129,5.770366430282593\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,15,20261111-3486328790-15,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.394,8.101,5.456515789031982\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,16,20261111-3486328790-16,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.38,7.992,5.140546560287476\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,17,20261111-3486328790-17,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.41,8.052,5.933665990829468\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,18,20261111-3486328790-18,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,1.397,7.987,5.713924169540405\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,19,20261111-3486328790-19,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1987,1,12,12.5,1.376,7.949,5.496263027191162\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,20,20261111-3486328790-20,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1987,0,13,13.0,1.3925,7.99,5.367618560791016\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,21,20261111-3486328790-21,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,1,9,9.5,1.3805,8.018,5.65630578994751\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,22,20261111-3486328790-22,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.4135,7.988,5.718964099884033\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,23,20261111-3486328790-23,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.4035,8.009,5.791912078857422\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,24,20261111-3486328790-24,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1992,0,8,8.0,1.414,7.884,5.818499803543091\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,25,20261111-3486328790-25,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.4015,7.97,5.513797760009766\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,26,20261111-3486328790-26,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.4065,7.906,5.650439023971558\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,27,20261111-3486328790-27,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.3845,7.928,5.51611590385437\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,28,20261111-3486328790-28,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.3935,8.013,6.007763147354126\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,29,20261111-3486328790-29,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1992,0,8,8.0,1.4085,8.014,5.491595506668091\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,30,20261111-3486328790-30,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.4205,8.001,5.807028293609619\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,31,20261111-3486328790-31,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.4255,8.003,5.5221452713012695\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,32,20261111-3486328790-32,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.411,7.995,5.453128337860107\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,33,20261111-3486328790-33,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.356,7.994,5.8588221073150635\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,34,20261111-3486328790-34,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.4235,7.968,5.6293768882751465\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,35,20261111-3486328790-35,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,1,4,4.5,1.3875,7.966,5.892314195632935\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,36,20261111-3486328790-36,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.387,8.003,5.67444920539856\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,37,20261111-3486328790-37,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.3495,7.871,5.69675350189209\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,38,20261111-3486328790-38,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.3485,8.082,5.836132764816284\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,39,20261111-3486328790-39,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1992,0,8,8.0,1.4325,8.011,5.830450534820557\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,40,20261111-3486328790-40,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.4,7.95,5.959287166595459\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,41,20261111-3486328790-41,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.388,7.934,5.385045051574707\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,42,20261111-3486328790-42,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.438,7.91,5.746212005615234\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,43,20261111-3486328790-43,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,2,3,4.0,1.3495,7.988,5.835313558578491\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,44,20261111-3486328790-44,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.4505,7.96,5.941756963729858\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,45,20261111-3486328790-45,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.4405,8.021,5.828014135360718\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,46,20261111-3486328790-46,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.355,7.937,5.450617551803589\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,47,20261111-3486328790-47,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.4025,7.88,5.906944751739502\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,48,20261111-3486328790-48,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1992,0,8,8.0,1.406,7.975,5.48894190788269\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,49,20261111-3486328790-49,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.43,7.985,5.387928485870361\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,50,20261111-3486328790-50,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1992,0,8,8.0,1.398,8.02,5.25423526763916\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,51,20261111-3486328790-51,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.376,7.93,5.397735834121704\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,52,20261111-3486328790-52,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,1,9,9.5,1.357,8.17,5.47667932510376\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,53,20261111-3486328790-53,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.408,8.017,5.5815513134002686\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,54,20261111-3486328790-54,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.4075,7.931,5.338016748428345\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,55,20261111-3486328790-55,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.4065,7.986,5.445545673370361\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,56,20261111-3486328790-56,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1988,0,12,12.0,1.394,8.045,5.763339042663574\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,57,20261111-3486328790-57,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1985,2,13,14.0,1.382,8.023,5.805793523788452\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,58,20261111-3486328790-58,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.3885,8.036,5.616051197052002\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,59,20261111-3486328790-59,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1988,0,12,12.0,1.383,8.014,5.906102657318115\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,60,20261111-3486328790-60,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.416,8.065,5.79623007774353\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,61,20261111-3486328790-61,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.3815,7.931,5.092336416244507\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,62,20261111-3486328790-62,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.438,8.077,5.623378038406372\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,63,20261111-3486328790-63,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,1.38,8.009,5.687716722488403\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,64,20261111-3486328790-64,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1988,0,12,12.0,1.388,8.066,5.737629652023315\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,65,20261111-3486328790-65,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.3825,8.049,5.757137298583984\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,66,20261111-3486328790-66,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.3645,7.997,5.540915012359619\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,67,20261111-3486328790-67,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.364,7.964,5.517275333404541\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,68,20261111-3486328790-68,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1992,0,8,8.0,1.4385,8.058,5.846928358078003\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,69,20261111-3486328790-69,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1988,0,12,12.0,1.4045,7.951,5.8661932945251465\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,70,20261111-3486328790-70,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.353,8.002,5.615880250930786\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,71,20261111-3486328790-71,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.4395,8.042,5.533170461654663\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,72,20261111-3486328790-72,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.408,7.912,5.424368381500244\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,73,20261111-3486328790-73,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.3915,8.032,5.297145128250122\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,74,20261111-3486328790-74,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.363,8.01,5.7863335609436035\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,75,20261111-3486328790-75,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.3845,7.974,5.426255702972412\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,76,20261111-3486328790-76,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.364,8.099,5.271979093551636\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,77,20261111-3486328790-77,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.3555,7.929,5.411561727523804\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,78,20261111-3486328790-78,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.368,8.059,5.583532094955444\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,79,20261111-3486328790-79,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.381,7.957,5.6957995891571045\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,80,20261111-3486328790-80,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.3925,8.074,5.476589202880859\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,81,20261111-3486328790-81,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.374,7.963,5.312336444854736\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,82,20261111-3486328790-82,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1997,2,1,2.0,1.407,8.078,5.628346920013428\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,83,20261111-3486328790-83,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,1.4185,7.981,5.537291526794434\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,84,20261111-3486328790-84,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1994,0,6,6.0,1.3935,8.066,5.444971323013306\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,85,20261111-3486328790-85,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,1.3685,8.093,5.789154291152954\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,86,20261111-3486328790-86,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.4175,7.925,5.664546489715576\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,87,20261111-3486328790-87,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.4065,8.091,5.925833702087402\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,88,20261111-3486328790-88,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.3815,8.004,5.348616123199463\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,89,20261111-3486328790-89,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1993,0,7,7.0,1.386,8.023,5.661465406417847\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,90,20261111-3486328790-90,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,1.363,7.963,5.300989627838135\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,91,20261111-3486328790-91,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.4145,8.086,5.439099073410034\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,92,20261111-3486328790-92,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1989,0,11,11.0,1.3915,7.925,5.580563068389893\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,93,20261111-3486328790-93,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,2,7,8.0,1.3745,7.868,5.760518312454224\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,94,20261111-3486328790-94,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1991,0,9,9.0,1.422,7.998,5.41577935218811\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,95,20261111-3486328790-95,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1989,0,11,11.0,1.3925,7.951,5.8133111000061035\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,96,20261111-3486328790-96,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,1.399,8.092,5.444828271865845\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,97,20261111-3486328790-97,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1990,0,10,10.0,1.412,7.924,5.716810941696167\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,98,20261111-3486328790-98,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,1.4175,8.104,5.239228248596191\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K2|g1,99,20261111-3486328790-99,B,noisy,3,0.0003,0.0,2,1,3,3,1,3,2,2000,1987,0,13,13.0,1.433,7.954,5.32344913482666\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,0,20261111-2575245727-0,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,1,6,6.5,2.116279069767442,11.978994748687171,5.729693412780762\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,1,20261111-2575245727-1,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.186046511627907,12.123030757689422,5.2478461265563965\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,2,20261111-2575245727-2,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.2310577644411103,12.0735183795949,5.735071659088135\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,3,20261111-2575245727-3,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.2535633908477117,11.836459114778695,5.687559366226196\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,4,20261111-2575245727-4,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.205551387846962,11.984996249062265,5.449325084686279\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,5,20261111-2575245727-5,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,1,6,6.5,2.2280570142535634,12.0960240060015,5.284512519836426\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,6,20261111-2575245727-6,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,1,5,5.5,2.186796699174794,12.007501875468867,5.259096384048462\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,7,20261111-2575245727-7,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.1792948237059266,11.938484621155288,5.239400386810303\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,8,20261111-2575245727-8,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.1927981995498875,11.923480870217555,5.542673826217651\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,9,20261111-2575245727-9,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.245311327831958,11.906976744186046,5.4129722118377686\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,10,20261111-2575245727-10,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.213053263315829,11.920480120030007,5.283431053161621\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,11,20261111-2575245727-11,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.1627906976744184,11.911477869467367,5.323668956756592\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,12,20261111-2575245727-12,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.1417854463615904,12.111027756939235,5.556356430053711\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,13,20261111-2575245727-13,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,0,11,11.0,2.27006751687922,12.031507876969242,5.611911058425903\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,14,20261111-2575245727-14,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,2,7,8.0,2.1252813203300827,11.981995498874719,5.455593585968018\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,15,20261111-2575245727-15,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,2.1530382595648914,11.99549887471868,4.899501085281372\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,16,20261111-2575245727-16,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.1125281320330083,12.028507126781696,5.644733428955078\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,17,20261111-2575245727-17,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,1,10,10.5,2.173293323330833,12.040510127531883,5.711683034896851\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,18,20261111-2575245727-18,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.1492873218304576,12.055513878469617,5.5638203620910645\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,19,20261111-2575245727-19,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.207801950487622,12.055513878469617,5.359705686569214\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,20,20261111-2575245727-20,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.207801950487622,12.0,5.3733696937561035\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,21,20261111-2575245727-21,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.2175543885971494,11.867966991747936,5.33794093132019\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,22,20261111-2575245727-22,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.1890472618154537,12.08252063015754,5.153528690338135\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,23,20261111-2575245727-23,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,2,8,9.0,2.229557389347337,11.855963990997749,5.695669889450073\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,24,20261111-2575245727-24,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.213053263315829,12.021005251312829,5.314338445663452\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,25,20261111-2575245727-25,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,1,6,6.5,2.168792198049512,12.0,5.360789775848389\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,26,20261111-2575245727-26,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.105026256564141,11.930982745686421,5.099971055984497\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,27,20261111-2575245727-27,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.140285071267817,12.022505626406602,5.746241092681885\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,28,20261111-2575245727-28,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1319,0,14,14.0,2.1875468867216803,11.948987246811702,5.80227255821228\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,29,20261111-2575245727-29,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.107276819204801,11.978994748687171,5.363309383392334\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,30,20261111-2575245727-30,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.2205551387846962,11.896474118529632,5.857987880706787\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,31,20261111-2575245727-31,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1320,0,13,13.0,2.229557389347337,11.977494373593398,5.668383836746216\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,32,20261111-2575245727-32,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.2280570142535634,11.929482370592648,5.340573072433472\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,33,20261111-2575245727-33,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.196549137284321,12.042010502625656,5.661050319671631\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,34,20261111-2575245727-34,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.201800450112528,12.05251312828207,5.654849529266357\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,35,20261111-2575245727-35,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1319,1,13,13.5,2.1950487621905475,11.941485371342836,5.500144004821777\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,36,20261111-2575245727-36,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.2010502625656416,12.11402850712678,5.384350061416626\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,37,20261111-2575245727-37,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.264066016504126,11.998499624906227,5.579654932022095\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,38,20261111-2575245727-38,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.13728432108027,12.108027006751687,5.884489297866821\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,39,20261111-2575245727-39,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.13278319579895,11.858964741185297,5.572939872741699\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,40,20261111-2575245727-40,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.2040510127531885,11.846961740435109,5.083495140075684\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,41,20261111-2575245727-41,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.154538634658665,11.827456864216053,5.613001346588135\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,42,20261111-2575245727-42,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1321,0,12,12.0,2.2213053263315827,11.983495873968492,5.5278520584106445\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,43,20261111-2575245727-43,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,1,5,5.5,2.1335333833458363,12.117029257314329,5.747693061828613\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,44,20261111-2575245727-44,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.2393098274568644,12.009002250562641,5.742388963699341\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,45,20261111-2575245727-45,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,0,11,11.0,2.162040510127532,11.974493623405852,5.325545787811279\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,46,20261111-2575245727-46,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,0,10,10.0,2.136534133533383,12.058514628657164,5.532435178756714\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,47,20261111-2575245727-47,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.2250562640660165,11.972993248312077,5.729771614074707\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,48,20261111-2575245727-48,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,0,10,10.0,2.1710427606901725,11.891972993248313,5.457556247711182\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,49,20261111-2575245727-49,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.1530382595648914,11.866466616654163,5.557720422744751\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,50,20261111-2575245727-50,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.1890472618154537,12.007501875468867,5.593709945678711\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,51,20261111-2575245727-51,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.250562640660165,12.019504876219054,5.435843467712402\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,52,20261111-2575245727-52,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,2,8,9.0,2.2280570142535634,11.971492873218304,5.602701902389526\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,53,20261111-2575245727-53,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.1252813203300827,11.963990997749438,5.19248628616333\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,54,20261111-2575245727-54,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.1635408852213054,12.009002250562641,5.759420871734619\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,55,20261111-2575245727-55,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,2.183795948987247,11.944486121530383,5.571678161621094\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,56,20261111-2575245727-56,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,0,10,10.0,2.2138034508627156,11.993998499624906,5.598176717758179\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,57,20261111-2575245727-57,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.266316579144786,12.117029257314329,5.627677917480469\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,58,20261111-2575245727-58,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.1410352588147035,12.031507876969242,5.664448499679565\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,59,20261111-2575245727-59,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.151537884471118,12.081020255063766,5.335598468780518\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,60,20261111-2575245727-60,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.164291072768192,12.061515378844712,5.754783868789673\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,61,20261111-2575245727-61,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.201800450112528,11.867966991747936,5.110849142074585\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,62,20261111-2575245727-62,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.1680420105026257,11.977494373593398,5.542896270751953\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,63,20261111-2575245727-63,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1319,1,13,13.5,2.213053263315829,12.168042010502626,5.176214218139648\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,64,20261111-2575245727-64,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.2010502625656416,11.963990997749438,5.640315055847168\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,65,20261111-2575245727-65,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.193548387096774,12.015003750937735,5.411133766174316\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,66,20261111-2575245727-66,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.177794448612153,12.10052513128282,5.274232625961304\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,67,20261111-2575245727-67,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,1,6,6.5,2.143285821455364,11.993998499624906,5.391912460327148\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,68,20261111-2575245727-68,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.2213053263315827,12.090022505626406,5.595535516738892\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,69,20261111-2575245727-69,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.186046511627907,11.996999249812452,5.82289457321167\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,70,20261111-2575245727-70,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1319,2,12,13.0,2.148537134283571,11.94748687171793,5.379255294799805\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,71,20261111-2575245727-71,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1320,0,13,13.0,2.240810202550638,12.11402850712678,5.49890661239624\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,72,20261111-2575245727-72,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.237809452363091,12.055513878469617,5.343058109283447\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,73,20261111-2575245727-73,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,1,4,4.5,2.208552138034509,12.0960240060015,5.79695987701416\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,74,20261111-2575245727-74,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.2115528882220556,11.90847711927982,5.3764495849609375\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,75,20261111-2575245727-75,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.1492873218304576,12.132033008252064,5.3173909187316895\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,76,20261111-2575245727-76,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,1,7,7.5,2.1747936984246063,12.060015003750937,5.43917989730835\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,77,20261111-2575245727-77,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.0960240060015005,12.067516879219806,5.592411994934082\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,78,20261111-2575245727-78,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.1747936984246063,11.977494373593398,5.876485109329224\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,79,20261111-2575245727-79,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.1785446361590397,12.057014253563391,5.699639558792114\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,80,20261111-2575245727-80,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.1747936984246063,12.031507876969242,5.888113737106323\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,81,20261111-2575245727-81,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,1,9,9.5,2.136534133533383,11.953488372093023,5.602046966552734\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,82,20261111-2575245727-82,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,1,7,7.5,2.1605401350337585,11.993998499624906,5.407986164093018\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,83,20261111-2575245727-83,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.164291072768192,12.103525881470368,5.57181453704834\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,84,20261111-2575245727-84,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.190547636909227,12.018004501125281,5.723536968231201\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,85,20261111-2575245727-85,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,1,6,6.5,2.129782445611403,12.060015003750937,5.363795042037964\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,86,20261111-2575245727-86,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.2580645161290325,11.950487621905477,5.767502069473267\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,87,20261111-2575245727-87,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.1920480120030006,11.918979744936234,5.146246671676636\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,88,20261111-2575245727-88,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.2175543885971494,12.129032258064516,5.436990022659302\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,89,20261111-2575245727-89,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,0,11,11.0,2.1665416354088523,11.783945986496624,5.34871244430542\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,90,20261111-2575245727-90,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.177794448612153,11.909977494373594,5.4560706615448\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,91,20261111-2575245727-91,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.207801950487622,11.936984246061515,5.101264953613281\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,92,20261111-2575245727-92,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,0,11,11.0,2.1657914478619653,12.133533383345837,5.526686906814575\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,93,20261111-2575245727-93,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1318,0,15,15.0,2.2325581395348837,11.962490622655665,5.190038681030273\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,94,20261111-2575245727-94,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.172543135783946,12.061515378844712,5.786997079849243\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,95,20261111-2575245727-95,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.150787696924231,12.024006001500375,5.101235628128052\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,96,20261111-2575245727-96,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.1125281320330083,12.180045011252814,5.414026737213135\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,97,20261111-2575245727-97,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.1672918229557387,11.911477869467367,5.54346489906311\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,98,20261111-2575245727-98,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.2168042010502624,12.168042010502626,5.839200019836426\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,99,20261111-2575245727-99,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,0,10,10.0,2.158289572393098,12.08702175543886,5.315902471542358\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,100,20261111-2575245727-100,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.240810202550638,12.031507876969242,5.414099931716919\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,101,20261111-2575245727-101,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,1,9,9.5,2.191297824456114,11.966991747936984,5.642847537994385\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,102,20261111-2575245727-102,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,1,8,8.5,2.240810202550638,11.858964741185297,6.000916004180908\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,103,20261111-2575245727-103,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.1747936984246063,11.897974493623407,5.3792502880096436\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,104,20261111-2575245727-104,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.2273068267066765,12.081020255063766,5.366949796676636\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,105,20261111-2575245727-105,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,1,3,3.5,2.1177794448612155,12.015003750937735,5.579644680023193\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,106,20261111-2575245727-106,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.168792198049512,12.015003750937735,5.1883790493011475\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,107,20261111-2575245727-107,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.2063015753938484,12.03450862715679,5.545731544494629\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,108,20261111-2575245727-108,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.259564891222806,12.00450112528132,5.90332818031311\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,109,20261111-2575245727-109,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,1,9,9.5,2.162040510127532,12.045011252813204,5.305131435394287\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,110,20261111-2575245727-110,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,1,10,10.5,2.294073518379595,11.972993248312077,5.690165758132935\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,111,20261111-2575245727-111,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.1530382595648914,12.099024756189047,5.225431203842163\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,112,20261111-2575245727-112,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,0,11,11.0,2.223555888972243,12.105026256564141,5.397202014923096\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,113,20261111-2575245727-113,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.1920480120030006,12.015003750937735,5.48089861869812\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,114,20261111-2575245727-114,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.1890472618154537,11.9039759939985,5.073133945465088\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,115,20261111-2575245727-115,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,1,7,7.5,2.213053263315829,12.055513878469617,5.377250909805298\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,116,20261111-2575245727-116,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,0,11,11.0,2.202550637659415,12.117029257314329,5.528856515884399\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,117,20261111-2575245727-117,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.2438109527381846,12.090022505626406,5.360640525817871\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,118,20261111-2575245727-118,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1322,2,9,10.0,2.1635408852213054,12.234058514628657,5.072455883026123\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,119,20261111-2575245727-119,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1323,1,9,9.5,2.121530382595649,12.063015753938485,5.580425262451172\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,120,20261111-2575245727-120,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,2,6,7.0,2.1815453863465866,11.9039759939985,5.404069662094116\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,121,20261111-2575245727-121,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.1335333833458363,12.084021005251312,5.468722820281982\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,122,20261111-2575245727-122,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.1890472618154537,11.945986496624156,5.57202935218811\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,123,20261111-2575245727-123,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,2,7,8.0,2.2220555138784697,11.983495873968492,5.681085109710693\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,124,20261111-2575245727-124,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.1125281320330083,11.87846961740435,5.862280368804932\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,125,20261111-2575245727-125,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.1920480120030006,11.876969242310578,5.776940107345581\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,126,20261111-2575245727-126,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,1,7,7.5,2.193548387096774,11.971492873218304,5.462886571884155\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,127,20261111-2575245727-127,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.2198049512378093,11.884471117779444,5.795478820800781\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,128,20261111-2575245727-128,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.173293323330833,11.8304576144036,5.544792652130127\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,129,20261111-2575245727-129,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.2325581395348837,12.268567141785446,5.75964093208313\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,130,20261111-2575245727-130,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.2115528882220556,11.968492123030758,5.6010355949401855\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,131,20261111-2575245727-131,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.2363090772693175,12.046511627906977,5.250393390655518\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,132,20261111-2575245727-132,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.1597899474868716,12.13953488372093,5.724492311477661\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,133,20261111-2575245727-133,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.1807951987997,11.887471867966992,5.700003623962402\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,134,20261111-2575245727-134,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.1897974493623407,12.129032258064516,5.421645879745483\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,135,20261111-2575245727-135,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.204801200300075,12.069017254313579,5.578112602233887\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,136,20261111-2575245727-136,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.231807951987997,11.974493623405852,5.693094491958618\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,137,20261111-2575245727-137,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.1792948237059266,11.978994748687171,5.5686352252960205\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,138,20261111-2575245727-138,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.1717929482370595,12.057014253563391,5.328244686126709\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,139,20261111-2575245727-139,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.1717929482370595,12.10052513128282,5.6372106075286865\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,140,20261111-2575245727-140,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.2168042010502624,11.927981995498875,5.7264485359191895\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,141,20261111-2575245727-141,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.205551387846962,11.920480120030007,5.597248554229736\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,142,20261111-2575245727-142,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1324,0,9,9.0,2.1852963240810204,12.042010502625656,5.536139726638794\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,143,20261111-2575245727-143,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.222805701425356,11.815453863465866,5.749332904815674\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,144,20261111-2575245727-144,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.1335333833458363,12.130532633158289,5.836148977279663\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,145,20261111-2575245727-145,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1326,0,7,7.0,2.17629407351838,11.875468867216805,5.574428081512451\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,146,20261111-2575245727-146,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1325,0,8,8.0,2.1447861965491373,12.196549137284322,5.905497312545776\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,147,20261111-2575245727-147,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,2.180045011252813,12.063015753938485,5.675809144973755\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,148,20261111-2575245727-148,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,2.204801200300075,12.231057764441111,5.3434739112854\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,149,20261111-2575245727-149,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,2.202550637659415,12.13953488372093,5.38208270072937\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g3,150,20261111-2575245727-150,B,noisy,3,0.0003,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,2.186046511627907,11.972993248312077,5.866256237030029\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,0,20261111-2003910835-0,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8927231807951987,11.881470367591898,5.1522276401519775\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,1,20261111-2003910835-1,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9287321830457616,11.990997749437359,5.354342222213745\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,2,20261111-2003910835-2,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.8732183045761441,11.984996249062265,5.258784055709839\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,3,20261111-2003910835-3,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8994748687171792,11.936984246061515,4.973349332809448\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,4,20261111-2003910835-4,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1318,1,14,14.5,1.9219804951237809,11.79144786196549,5.039587497711182\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,5,20261111-2003910835-5,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,0,9,9.0,1.9167291822955739,12.10052513128282,4.891988277435303\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,6,20261111-2003910835-6,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,0,9,9.0,1.918979744936234,12.207051762940734,5.1688313484191895\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,7,20261111-2003910835-7,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,1,8,8.5,1.9497374343585896,12.05251312828207,5.129420518875122\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,8,20261111-2003910835-8,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.8657164291072768,12.109527381845462,5.047767400741577\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,9,20261111-2003910835-9,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9332333083270818,12.061515378844712,4.892023086547852\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,10,20261111-2003910835-10,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9077269317329333,12.097524381095274,5.295605421066284\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,11,20261111-2003910835-11,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8987246811702925,11.96099024756189,5.1316587924957275\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,12,20261111-2003910835-12,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.8964741185296323,11.86496624156039,5.245136260986328\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,13,20261111-2003910835-13,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.9167291822955739,12.015003750937735,5.199030876159668\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,14,20261111-2003910835-14,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.9684921230307577,12.115528882220556,5.369854211807251\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,15,20261111-2003910835-15,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.940735183795949,11.9264816204051,5.422486066818237\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,16,20261111-2003910835-16,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.967741935483871,11.924981245311328,4.90532660484314\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,17,20261111-2003910835-17,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.8904726181545386,12.021005251312829,4.933398246765137\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,18,20261111-2003910835-18,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,0,11,11.0,1.9422355588897224,12.078019504876218,4.95675802230835\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,19,20261111-2003910835-19,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1321,2,10,11.0,1.9062265566391599,11.998499624906227,5.271944761276245\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,20,20261111-2003910835-20,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9939984996249063,12.009002250562641,4.732086658477783\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,21,20261111-2003910835-21,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.8964741185296323,11.935483870967742,4.908374309539795\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,22,20261111-2003910835-22,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,1,9,9.5,1.859714928732183,11.953488372093023,4.712664604187012\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,23,20261111-2003910835-23,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.9339834958739686,11.897974493623407,4.823314666748047\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,24,20261111-2003910835-24,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.935483870967742,12.049512378094523,4.865339517593384\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,25,20261111-2003910835-25,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8642160540135033,12.0,4.696493625640869\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,26,20261111-2003910835-26,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,1,5,5.5,1.876219054763691,11.924981245311328,4.56212592124939\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,27,20261111-2003910835-27,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,1,6,6.5,1.8844711177794449,12.123030757689422,4.828946828842163\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,28,20261111-2003910835-28,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,0,9,9.0,1.8807201800450113,12.024006001500375,5.012864351272583\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,29,20261111-2003910835-29,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,1,5,5.5,1.9519879969992497,11.990997749437359,4.705562353134155\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,30,20261111-2003910835-30,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.9047261815453864,11.918979744936234,4.976252317428589\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,31,20261111-2003910835-31,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1321,0,12,12.0,1.8267066766691673,11.948987246811702,4.8227317333221436\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,32,20261111-2003910835-32,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,1,6,6.5,1.945986496624156,12.070517629407352,4.675049066543579\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,33,20261111-2003910835-33,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9174793698424606,12.010502625656414,5.091231822967529\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,34,20261111-2003910835-34,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,1,8,8.5,1.8852213053263316,12.018004501125281,4.75774621963501\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,35,20261111-2003910835-35,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8469617404351089,11.972993248312077,4.973543167114258\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,36,20261111-2003910835-36,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,2,7,8.0,1.9317329332333084,12.08252063015754,4.6960039138793945\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,37,20261111-2003910835-37,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8912228057014253,12.069017254313579,4.87227201461792\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,38,20261111-2003910835-38,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8747186796699176,11.927981995498875,5.1668009757995605\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,39,20261111-2003910835-39,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.9002250562640661,12.078019504876218,5.003131866455078\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,40,20261111-2003910835-40,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8709677419354838,12.03450862715679,5.0046916007995605\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,41,20261111-2003910835-41,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,1,10,10.5,1.870217554388597,11.942985746436609,5.477421760559082\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,42,20261111-2003910835-42,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,1,7,7.5,1.9294823705926483,11.735933983495874,5.111926317214966\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,43,20261111-2003910835-43,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,0,11,11.0,1.9054763690922731,12.015003750937735,5.277678489685059\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,44,20261111-2003910835-44,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.924231057764441,12.060015003750937,5.047240257263184\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,45,20261111-2003910835-45,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,1,6,6.5,1.8657164291072768,12.046511627906977,4.891982793807983\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,46,20261111-2003910835-46,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.8732183045761441,12.012003000750187,5.021751403808594\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,47,20261111-2003910835-47,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.9129782445611403,12.033008252063016,5.006307125091553\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,48,20261111-2003910835-48,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,1.8927231807951987,12.13953488372093,5.016139984130859\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,49,20261111-2003910835-49,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.927231807951988,12.037509377344335,5.018245697021484\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,50,20261111-2003910835-50,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,0,9,9.0,1.8687171792948236,11.950487621905477,5.404155969619751\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,51,20261111-2003910835-51,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,1.935483870967742,12.093023255813954,4.885985851287842\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,52,20261111-2003910835-52,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.8934733683420855,12.08702175543886,4.939807891845703\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,53,20261111-2003910835-53,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8559639909977494,12.015003750937735,5.001870632171631\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,54,20261111-2003910835-54,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.9039759939984997,11.953488372093023,5.190319538116455\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,55,20261111-2003910835-55,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,1,7,7.5,1.9092273068267067,12.028507126781696,5.0236732959747314\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,56,20261111-2003910835-56,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.9204801200300075,12.046511627906977,5.07248854637146\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,57,20261111-2003910835-57,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,0,11,11.0,1.8657164291072768,12.079519879969993,4.795387268066406\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,58,20261111-2003910835-58,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.881470367591898,12.001500375093773,5.170168876647949\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,59,20261111-2003910835-59,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.935483870967742,12.05251312828207,5.064436912536621\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,60,20261111-2003910835-60,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.9422355588897224,12.0,5.0737597942352295\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,61,20261111-2003910835-61,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8957239309827456,11.771942985746437,4.96236252784729\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,62,20261111-2003910835-62,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1320,1,12,12.5,1.8942235558889722,11.977494373593398,5.0834925174713135\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,63,20261111-2003910835-63,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,1,6,6.5,1.8889722430607652,12.009002250562641,5.041517734527588\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,64,20261111-2003910835-64,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,0,11,11.0,1.8642160540135033,12.027006751687923,5.112991809844971\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,65,20261111-2003910835-65,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.9204801200300075,11.987996999249813,5.227947950363159\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,66,20261111-2003910835-66,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.90847711927982,11.944486121530383,5.119054079055786\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,67,20261111-2003910835-67,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,1,10,10.5,1.8927231807951987,11.936984246061515,4.893694639205933\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,68,20261111-2003910835-68,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.90847711927982,12.124531132783195,4.91009259223938\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,69,20261111-2003910835-69,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.859714928732183,11.971492873218304,4.800115585327148\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,70,20261111-2003910835-70,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.8897224306076519,12.033008252063016,5.534181118011475\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,71,20261111-2003910835-71,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.8874718679669917,12.123030757689422,5.251009941101074\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,72,20261111-2003910835-72,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8447111777944487,11.834958739684922,4.707890272140503\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,73,20261111-2003910835-73,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.8784696174043511,12.108027006751687,4.641572713851929\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,74,20261111-2003910835-74,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.9309827456864217,12.157539384846212,4.583505868911743\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,75,20261111-2003910835-75,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9474868717179294,11.978994748687171,4.891463279724121\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,76,20261111-2003910835-76,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,1,6,6.5,1.832708177044261,12.01350337584396,5.2538228034973145\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,77,20261111-2003910835-77,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.897224306076519,11.987996999249813,5.3053624629974365\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,78,20261111-2003910835-78,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,1,8,8.5,1.8237059264816204,12.147036759189797,4.74289608001709\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,79,20261111-2003910835-79,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.9489872468117029,12.112528132033008,5.167434453964233\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,80,20261111-2003910835-80,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.9129782445611403,11.936984246061515,5.300218343734741\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,81,20261111-2003910835-81,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,1,6,6.5,1.8619654913728432,12.03900975243811,5.136759281158447\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,82,20261111-2003910835-82,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,1,4,4.5,1.9392348087021756,11.9264816204051,4.977080583572388\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,83,20261111-2003910835-83,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,0,11,11.0,1.930232558139535,12.033008252063016,4.9123125076293945\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,84,20261111-2003910835-84,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,1.9722430607651913,12.015003750937735,5.098618984222412\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,85,20261111-2003910835-85,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.9444861215303826,11.875468867216805,5.090898752212524\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,86,20261111-2003910835-86,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,1,10,10.5,1.9332333083270818,12.142535633908476,5.092528581619263\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,87,20261111-2003910835-87,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8994748687171792,12.118529632408102,5.0213141441345215\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,88,20261111-2003910835-88,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.8934733683420855,11.945986496624156,5.033141851425171\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,89,20261111-2003910835-89,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,0,11,11.0,1.840960240060015,11.950487621905477,4.844429016113281\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,90,20261111-2003910835-90,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.9497374343585896,11.969992498124531,5.179811716079712\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,91,20261111-2003910835-91,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.881470367591898,12.0735183795949,4.864232301712036\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,92,20261111-2003910835-92,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,1,8,8.5,1.913728432108027,11.8304576144036,5.301811218261719\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,93,20261111-2003910835-93,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9122280570142536,12.076519129782445,5.334399223327637\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,94,20261111-2003910835-94,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.9452363090772693,12.054013503375844,5.182029724121094\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,95,20261111-2003910835-95,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.870217554388597,11.881470367591898,5.1633477210998535\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,96,20261111-2003910835-96,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,1,7,7.5,1.9114778694673669,12.111027756939235,5.225353002548218\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,97,20261111-2003910835-97,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,1,7,7.5,1.9159789947486872,12.031507876969242,5.03298282623291\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,98,20261111-2003910835-98,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,1.9669917479369843,12.021005251312829,5.110564708709717\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,99,20261111-2003910835-99,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.8904726181545386,11.971492873218304,4.869463682174683\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,100,20261111-2003910835-100,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,0,9,9.0,1.9219804951237809,12.08702175543886,5.185238838195801\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,101,20261111-2003910835-101,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.8837209302325582,12.246061515378845,4.704251050949097\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,102,20261111-2003910835-102,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8469617404351089,11.87396849212303,5.198888063430786\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,103,20261111-2003910835-103,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.8912228057014253,11.920480120030007,5.029951810836792\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,104,20261111-2003910835-104,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.9197299324831207,12.11402850712678,5.151479005813599\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,105,20261111-2003910835-105,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.876219054763691,12.0,4.78880763053894\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,106,20261111-2003910835-106,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8897224306076519,12.033008252063016,5.243213415145874\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,107,20261111-2003910835-107,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.9309827456864217,12.001500375093773,4.974461555480957\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,108,20261111-2003910835-108,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.9062265566391599,11.938484621155288,5.04578423500061\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,109,20261111-2003910835-109,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.8852213053263316,12.040510127531883,4.722384929656982\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,110,20261111-2003910835-110,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,1,8,8.5,1.9414853713428357,11.893473368342086,5.003432750701904\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,111,20261111-2003910835-111,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.9437359339834959,12.09152288072018,4.952194690704346\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,112,20261111-2003910835-112,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.9414853713428357,12.097524381095274,5.085693359375\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,113,20261111-2003910835-113,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.8717179294823707,12.108027006751687,4.606440544128418\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,114,20261111-2003910835-114,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8469617404351089,11.992498124531133,4.975840091705322\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,115,20261111-2003910835-115,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.8964741185296323,11.810952738184547,5.069756031036377\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,116,20261111-2003910835-116,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.8777194298574644,12.153038259564891,5.283205270767212\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,117,20261111-2003910835-117,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.9587396849212304,11.99549887471868,4.828366279602051\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,118,20261111-2003910835-118,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.9024756189047263,11.82595648912228,4.9569642543792725\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,119,20261111-2003910835-119,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.9077269317329333,11.94748687171793,4.773338317871094\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,120,20261111-2003910835-120,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,1,3,3.5,1.86271567891973,12.12603150787697,4.90934419631958\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,121,20261111-2003910835-121,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,1,6,6.5,1.9047261815453864,11.956489122280571,5.0307936668396\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,122,20261111-2003910835-122,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8792198049512379,11.900975243810953,5.0259108543396\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,123,20261111-2003910835-123,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,0,9,9.0,1.9579894973743437,12.055513878469617,5.141810417175293\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,124,20261111-2003910835-124,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.927231807951988,12.084021005251312,4.967898368835449\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,125,20261111-2003910835-125,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9872468117029258,12.180045011252814,5.206986904144287\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,126,20261111-2003910835-126,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.8657164291072768,12.021005251312829,5.120602130889893\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,127,20261111-2003910835-127,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8732183045761441,12.00450112528132,5.222325801849365\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,128,20261111-2003910835-128,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.9377344336084021,12.058514628657164,5.129695892333984\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,129,20261111-2003910835-129,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,1,5,5.5,1.9482370592648162,11.95198799699925,5.188173294067383\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,130,20261111-2003910835-130,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8447111777944487,11.87396849212303,5.499757766723633\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,131,20261111-2003910835-131,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,1,4,4.5,1.8664666166541635,11.854463615903976,5.239685297012329\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,132,20261111-2003910835-132,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.8694673668417103,12.085521380345087,5.18538761138916\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,133,20261111-2003910835-133,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,1,6,6.5,1.9024756189047263,12.031507876969242,5.0065367221832275\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,134,20261111-2003910835-134,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,1,8,8.5,1.9129782445611403,12.024006001500375,5.187825679779053\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,135,20261111-2003910835-135,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.9474868717179294,11.876969242310578,4.930751800537109\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,136,20261111-2003910835-136,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.8589647411852963,12.057014253563391,5.026446104049683\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,137,20261111-2003910835-137,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.8799699924981246,12.028507126781696,5.255722284317017\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,138,20261111-2003910835-138,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.9099774943735934,11.933983495873969,5.411977767944336\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,139,20261111-2003910835-139,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1322,2,9,10.0,1.870217554388597,11.950487621905477,5.117068529129028\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,140,20261111-2003910835-140,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.9362340585146287,11.9039759939985,5.152541160583496\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,141,20261111-2003910835-141,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,1.8792198049512379,12.196549137284322,4.935654878616333\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,142,20261111-2003910835-142,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1327,0,6,6.0,1.8994748687171792,11.87396849212303,5.209001064300537\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,143,20261111-2003910835-143,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,2.000750187546887,12.010502625656414,4.979758977890015\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,144,20261111-2003910835-144,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,1.8979744936234058,11.882970742685671,4.855443716049194\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,145,20261111-2003910835-145,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1323,0,10,10.0,1.9332333083270818,11.983495873968492,4.701758146286011\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,146,20261111-2003910835-146,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,0,9,9.0,1.8559639909977494,12.04801200300075,4.892815113067627\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,147,20261111-2003910835-147,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1325,0,8,8.0,1.9069767441860466,12.006001500375094,5.1727283000946045\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,148,20261111-2003910835-148,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1324,1,8,8.5,1.8724681170292574,12.019504876219054,4.887797117233276\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,149,20261111-2003910835-149,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,1.9069767441860466,11.91297824456114,5.079887390136719\npilot-20261110T155331Z-d3f938,B|noisy|d3|p0.0003|K3|g1,150,20261111-2003910835-150,B,noisy,3,0.0003,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,1.9092273068267067,12.105026256564141,5.086156129837036\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,0,20261111-2290445404-0,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,998,1,1,1.5,3.887,16.194,8.072701215744019\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,1,20261111-2290445404-1,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.838,15.734,8.17970323562622\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,2,20261111-2290445404-2,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.796,16.048,7.715702533721924\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,3,20261111-2290445404-3,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.793,15.744,8.024473190307617\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,4,20261111-2290445404-4,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.813,15.812,6.814172983169556\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,5,20261111-2290445404-5,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.714,16.124,6.9324047565460205\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,6,20261111-2290445404-6,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.681,15.748,6.660045862197876\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,7,20261111-2290445404-7,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.841,15.91,7.0187907218933105\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,8,20261111-2290445404-8,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.89,15.904,7.024786710739136\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,9,20261111-2290445404-9,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.819,16.146,7.47344183921814\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,10,20261111-2290445404-10,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.695,15.952,7.465728759765625\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,11,20261111-2290445404-11,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.898,15.792,7.078505992889404\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,12,20261111-2290445404-12,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.801,15.886,7.111605882644653\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,13,20261111-2290445404-13,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.744,15.874,7.3791351318359375\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,14,20261111-2290445404-14,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.825,15.878,7.055369138717651\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,15,20261111-2290445404-15,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.766,15.908,6.909124851226807\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,16,20261111-2290445404-16,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.871,16.1,7.026978015899658\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,17,20261111-2290445404-17,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.841,16.076,7.070211172103882\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,18,20261111-2290445404-18,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.773,15.996,7.385433197021484\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,19,20261111-2290445404-19,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.84,15.82,7.60038161277771\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,20,20261111-2290445404-20,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,1,0,0.5,3.844,16.012,7.35217547416687\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,21,20261111-2290445404-21,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.789,16.042,7.195161819458008\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,22,20261111-2290445404-22,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.803,16.03,7.248080492019653\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,23,20261111-2290445404-23,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.755,16.112,7.259382486343384\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,24,20261111-2290445404-24,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.85,15.802,7.2482287883758545\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,25,20261111-2290445404-25,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.728,15.974,7.4093098640441895\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,26,20261111-2290445404-26,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.749,16.096,6.970953464508057\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,27,20261111-2290445404-27,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.931,15.984,7.437504291534424\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,28,20261111-2290445404-28,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.855,16.008,7.277537107467651\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,29,20261111-2290445404-29,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.782,15.85,7.327716827392578\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,30,20261111-2290445404-30,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.716,16.092,7.326812505722046\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,31,20261111-2290445404-31,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.72,16.0,7.312854766845703\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,32,20261111-2290445404-32,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.746,15.916,7.472602367401123\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,33,20261111-2290445404-33,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.929,15.994,7.769472360610962\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,34,20261111-2290445404-34,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.76,16.032,7.191892623901367\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,35,20261111-2290445404-35,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.859,16.084,7.404188632965088\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,36,20261111-2290445404-36,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.735,16.05,7.146816253662109\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,37,20261111-2290445404-37,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.797,15.85,7.356400966644287\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,38,20261111-2290445404-38,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.821,15.924,7.6984782218933105\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,39,20261111-2290445404-39,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.655,16.102,7.093303680419922\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,40,20261111-2290445404-40,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.803,16.108,7.139472246170044\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,41,20261111-2290445404-41,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.71,15.954,7.3224711418151855\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,42,20261111-2290445404-42,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.746,15.95,7.2660627365112305\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,43,20261111-2290445404-43,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.692,15.906,7.4531214237213135\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,44,20261111-2290445404-44,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,1,0,0.5,3.901,15.774,7.200998067855835\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,45,20261111-2290445404-45,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.757,16.204,7.22837233543396\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,46,20261111-2290445404-46,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.904,16.006,7.515225410461426\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,47,20261111-2290445404-47,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.843,15.954,7.384262800216675\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,48,20261111-2290445404-48,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.851,16.05,7.406765460968018\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,49,20261111-2290445404-49,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.771,16.042,7.496799945831299\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,50,20261111-2290445404-50,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.923,16.308,7.379516839981079\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,51,20261111-2290445404-51,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.808,15.878,7.2511985301971436\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,52,20261111-2290445404-52,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.805,15.952,7.407778263092041\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,53,20261111-2290445404-53,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.749,15.754,6.938579082489014\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,54,20261111-2290445404-54,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,997,0,3,3.0,3.753,15.996,7.18989634513855\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,55,20261111-2290445404-55,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,998,1,1,1.5,3.717,15.918,6.936414003372192\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,56,20261111-2290445404-56,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.82,15.856,7.243245601654053\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,57,20261111-2290445404-57,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.778,16.09,6.977188348770142\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,58,20261111-2290445404-58,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.75,16.286,6.980522632598877\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,59,20261111-2290445404-59,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.891,16.132,7.1075122356414795\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,60,20261111-2290445404-60,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.797,16.032,7.058981895446777\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,61,20261111-2290445404-61,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.86,16.028,6.7974631786346436\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,62,20261111-2290445404-62,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,998,0,2,2.0,3.875,16.178,7.531041860580444\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,63,20261111-2290445404-63,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.788,15.91,7.198283433914185\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,64,20261111-2290445404-64,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.912,16.004,7.22240424156189\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,65,20261111-2290445404-65,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.783,16.06,7.116389751434326\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,66,20261111-2290445404-66,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.778,16.01,7.109858512878418\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,67,20261111-2290445404-67,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,998,1,1,1.5,3.817,16.188,7.425292491912842\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,68,20261111-2290445404-68,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.672,15.93,7.113734245300293\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,69,20261111-2290445404-69,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.814,15.994,7.459876537322998\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,70,20261111-2290445404-70,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.538,16.046,7.02775239944458\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,71,20261111-2290445404-71,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.736,16.356,7.441778182983398\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,72,20261111-2290445404-72,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.808,15.948,7.0770580768585205\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,73,20261111-2290445404-73,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.696,16.024,7.615697622299194\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,74,20261111-2290445404-74,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.875,15.952,7.164355754852295\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,75,20261111-2290445404-75,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.779,16.228,7.9148268699646\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,76,20261111-2290445404-76,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.666,15.944,7.457556962966919\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,77,20261111-2290445404-77,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.802,16.168,7.516315698623657\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,78,20261111-2290445404-78,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.769,16.008,7.226604461669922\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,79,20261111-2290445404-79,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.785,16.058,7.5729217529296875\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,80,20261111-2290445404-80,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.754,16.084,7.145162105560303\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,81,20261111-2290445404-81,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.675,15.966,7.263296604156494\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,82,20261111-2290445404-82,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.873,15.816,7.6061179637908936\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,83,20261111-2290445404-83,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.901,15.9,7.81494402885437\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,84,20261111-2290445404-84,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.761,15.806,7.369414567947388\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,85,20261111-2290445404-85,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.694,15.916,7.194235563278198\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,86,20261111-2290445404-86,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.767,15.992,6.826742649078369\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,87,20261111-2290445404-87,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.916,15.852,7.722692012786865\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,88,20261111-2290445404-88,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,1,0,0.5,3.793,15.78,7.378196954727173\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,89,20261111-2290445404-89,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,998,0,2,2.0,3.788,15.952,7.353666305541992\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,90,20261111-2290445404-90,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.757,16.128,7.470633506774902\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,91,20261111-2290445404-91,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.851,16.2,7.405519247055054\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,92,20261111-2290445404-92,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,999,0,1,1.0,3.801,16.026,7.287441730499268\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,93,20261111-2290445404-93,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.824,15.888,7.175437927246094\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,94,20261111-2290445404-94,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.827,15.842,7.3154284954071045\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,95,20261111-2290445404-95,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,998,0,2,2.0,3.737,16.066,7.405336618423462\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,96,20261111-2290445404-96,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.719,15.978,7.435558795928955\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,97,20261111-2290445404-97,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,998,0,2,2.0,3.856,15.854,6.959142684936523\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,98,20261111-2290445404-98,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.778,16.104,7.141542434692383\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K1|g0,99,20261111-2290445404-99,B,noisy,5,0.0003,0.0,1,0,5,5,5,5,2,1000,1000,0,0,0.0,3.746,16.322,7.5856688022613525\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,0,20261111-3931780925-0,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.776,32.24,8.267813682556152\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,1,20261111-3931780925-1,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.722,32.008,8.072062015533447\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,2,20261111-3931780925-2,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.728,31.9,8.093756198883057\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,3,20261111-3931780925-3,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.654,31.596,7.473881959915161\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,4,20261111-3931780925-4,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.354,31.628,6.851177215576172\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,5,20261111-3931780925-5,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.492,31.768,6.817846298217773\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,6,20261111-3931780925-6,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.804,32.128,6.556659460067749\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,7,20261111-3931780925-7,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.532,31.86,6.63730788230896\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,8,20261111-3931780925-8,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.638,31.476,7.285852909088135\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,9,20261111-3931780925-9,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.488,31.992,6.767145395278931\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,10,20261111-3931780925-10,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.608,31.908,6.522852897644043\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,11,20261111-3931780925-11,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.69,32.312,6.715202331542969\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,12,20261111-3931780925-12,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.79,31.504,6.876845836639404\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,13,20261111-3931780925-13,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.466,31.468,6.8028035163879395\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,14,20261111-3931780925-14,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.49,31.948,6.6818528175354\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,15,20261111-3931780925-15,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.52,31.688,6.979002475738525\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,16,20261111-3931780925-16,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.828,32.488,7.094382286071777\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,17,20261111-3931780925-17,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.684,31.82,6.874663591384888\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,18,20261111-3931780925-18,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,498,1,1,1.5,6.596,31.808,6.554651975631714\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,19,20261111-3931780925-19,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.48,32.168,7.053311824798584\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,20,20261111-3931780925-20,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.478,32.172,6.925292491912842\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,21,20261111-3931780925-21,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.584,31.968,6.350243330001831\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,22,20261111-3931780925-22,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,6.464,31.988,7.458010911941528\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,23,20261111-3931780925-23,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.728,32.376,7.293254852294922\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,24,20261111-3931780925-24,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.698,31.848,7.075968265533447\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,25,20261111-3931780925-25,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.646,31.84,6.710347414016724\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,26,20261111-3931780925-26,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.46,31.928,7.055563688278198\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,27,20261111-3931780925-27,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.508,31.804,7.476810693740845\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,28,20261111-3931780925-28,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.686,31.576,6.662773370742798\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,29,20261111-3931780925-29,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.428,31.864,6.928194522857666\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,30,20261111-3931780925-30,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.502,32.252,6.963069200515747\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,31,20261111-3931780925-31,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.516,32.24,7.597445011138916\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,32,20261111-3931780925-32,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.874,31.696,7.091270208358765\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,33,20261111-3931780925-33,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.714,32.04,5.866250038146973\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,34,20261111-3931780925-34,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.612,32.452,7.170129060745239\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,35,20261111-3931780925-35,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.802,32.108,7.068406581878662\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,36,20261111-3931780925-36,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.566,31.756,6.366162061691284\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,37,20261111-3931780925-37,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.662,32.14,6.598254919052124\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,38,20261111-3931780925-38,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.79,31.988,6.943902015686035\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,39,20261111-3931780925-39,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.518,31.852,6.727850675582886\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,40,20261111-3931780925-40,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.67,31.86,7.487214088439941\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,41,20261111-3931780925-41,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.612,31.888,6.824660778045654\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,42,20261111-3931780925-42,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.482,32.136,6.723877429962158\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,43,20261111-3931780925-43,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,497,0,3,3.0,6.694,31.904,7.782424211502075\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,44,20261111-3931780925-44,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.65,31.812,7.087271690368652\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,45,20261111-3931780925-45,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.61,31.86,6.645967960357666\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,46,20261111-3931780925-46,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.596,32.1,6.64980411529541\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,47,20261111-3931780925-47,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.64,32.06,6.918226718902588\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,48,20261111-3931780925-48,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.422,31.9,6.713040828704834\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,49,20261111-3931780925-49,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.548,32.212,7.507701873779297\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,50,20261111-3931780925-50,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,6.492,32.276,6.557466506958008\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,51,20261111-3931780925-51,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.65,32.324,6.916667699813843\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,52,20261111-3931780925-52,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.732,32.288,7.367409706115723\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,53,20261111-3931780925-53,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.554,31.556,7.4178571701049805\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,54,20261111-3931780925-54,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.52,31.892,7.4134838581085205\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,55,20261111-3931780925-55,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.634,31.692,6.620019435882568\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,56,20261111-3931780925-56,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.554,31.908,7.241787433624268\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,57,20261111-3931780925-57,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.608,31.84,7.437766075134277\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,58,20261111-3931780925-58,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.344,32.16,7.298555374145508\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,59,20261111-3931780925-59,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.57,32.16,6.725777626037598\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,60,20261111-3931780925-60,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.608,31.808,7.476972579956055\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,61,20261111-3931780925-61,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.562,31.68,7.0363311767578125\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,62,20261111-3931780925-62,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.436,31.976,6.9561607837677\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,63,20261111-3931780925-63,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.53,31.98,6.750688552856445\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,64,20261111-3931780925-64,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.636,31.768,7.236429691314697\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,65,20261111-3931780925-65,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.646,32.0,7.026747703552246\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,66,20261111-3931780925-66,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.572,31.872,6.997568130493164\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,67,20261111-3931780925-67,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.61,31.912,7.279232501983643\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,68,20261111-3931780925-68,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.846,32.388,7.17871356010437\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,69,20261111-3931780925-69,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.558,32.056,7.068670749664307\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,70,20261111-3931780925-70,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,6.548,32.288,7.654592752456665\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,71,20261111-3931780925-71,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.4,31.788,7.074922323226929\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,72,20261111-3931780925-72,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.634,31.924,6.909647226333618\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,73,20261111-3931780925-73,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.624,31.832,6.771674871444702\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,74,20261111-3931780925-74,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.404,31.732,7.322138786315918\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,75,20261111-3931780925-75,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.51,32.128,6.645061016082764\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,76,20261111-3931780925-76,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.788,32.048,6.773011207580566\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,77,20261111-3931780925-77,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.384,31.688,7.287890434265137\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,78,20261111-3931780925-78,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.476,32.48,7.241959095001221\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,79,20261111-3931780925-79,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.692,32.044,7.237383604049683\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,80,20261111-3931780925-80,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.682,32.192,7.441757917404175\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,81,20261111-3931780925-81,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.628,31.984,6.750817060470581\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,82,20261111-3931780925-82,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.74,31.9,7.463382005691528\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,83,20261111-3931780925-83,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.574,31.888,7.287089109420776\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,84,20261111-3931780925-84,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.502,31.752,7.479578256607056\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,85,20261111-3931780925-85,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.622,32.092,6.776135683059692\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,86,20261111-3931780925-86,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.47,32.04,6.904729843139648\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,87,20261111-3931780925-87,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.608,32.46,6.994660139083862\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,88,20261111-3931780925-88,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,6.8,32.16,7.395516395568848\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,89,20261111-3931780925-89,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.522,32.172,6.634286403656006\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,90,20261111-3931780925-90,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.668,31.692,6.661369800567627\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,91,20261111-3931780925-91,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.386,31.692,6.748186349868774\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,92,20261111-3931780925-92,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,1,0,0.5,6.65,32.28,6.307433366775513\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,93,20261111-3931780925-93,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.556,32.308,7.3553221225738525\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,94,20261111-3931780925-94,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,1,0,0.5,6.64,31.816,7.134768009185791\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,95,20261111-3931780925-95,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.582,31.432,6.440534353256226\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,96,20261111-3931780925-96,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.688,31.432,6.614844083786011\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,97,20261111-3931780925-97,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,6.704,31.796,6.972809076309204\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,98,20261111-3931780925-98,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.714,32.276,6.646863698959351\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,99,20261111-3931780925-99,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,1,0,0.5,6.52,31.848,6.732110023498535\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,100,20261111-3931780925-100,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.634,31.552,6.842963457107544\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,101,20261111-3931780925-101,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.654,31.688,7.271191596984863\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,102,20261111-3931780925-102,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.316,32.104,6.6399922370910645\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,103,20261111-3931780925-103,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.544,32.112,6.70044732093811\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,104,20261111-3931780925-104,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.634,32.328,7.630959510803223\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,105,20261111-3931780925-105,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.576,31.876,7.456236839294434\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,106,20261111-3931780925-106,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.586,31.984,6.421560525894165\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,107,20261111-3931780925-107,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.706,31.58,6.41099214553833\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,108,20261111-3931780925-108,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.448,31.708,7.599046945571899\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,109,20261111-3931780925-109,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.72,32.46,7.252898216247559\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,110,20261111-3931780925-110,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.846,32.2,7.6355180740356445\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,111,20261111-3931780925-111,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.808,32.328,6.175518274307251\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,112,20261111-3931780925-112,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.644,31.688,7.3098039627075195\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,113,20261111-3931780925-113,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.738,32.496,6.862998008728027\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,114,20261111-3931780925-114,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.846,32.064,6.487069845199585\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,115,20261111-3931780925-115,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,1,0,0.5,6.47,31.76,7.377779006958008\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,116,20261111-3931780925-116,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,1,0,0.5,6.686,31.908,7.289151191711426\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,117,20261111-3931780925-117,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.396,32.104,6.189708948135376\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,118,20261111-3931780925-118,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.806,31.784,6.6020355224609375\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,119,20261111-3931780925-119,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.722,32.9,6.29045557975769\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,120,20261111-3931780925-120,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.674,31.448,7.317552804946899\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,121,20261111-3931780925-121,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.78,32.208,7.443725109100342\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,122,20261111-3931780925-122,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.638,32.036,6.88509464263916\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,123,20261111-3931780925-123,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.642,32.244,6.630499362945557\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,124,20261111-3931780925-124,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.494,31.324,7.22214674949646\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,125,20261111-3931780925-125,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.402,31.648,6.742631196975708\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,126,20261111-3931780925-126,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.49,31.992,7.459304571151733\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,127,20261111-3931780925-127,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.612,31.904,7.0255560874938965\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,128,20261111-3931780925-128,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.682,32.264,7.331534147262573\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,129,20261111-3931780925-129,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.616,32.16,6.67849588394165\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,130,20261111-3931780925-130,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.696,32.296,7.379000425338745\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,131,20261111-3931780925-131,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.45,32.132,6.923985958099365\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,132,20261111-3931780925-132,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.522,32.18,6.954057216644287\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,133,20261111-3931780925-133,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.51,32.132,7.300685405731201\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,134,20261111-3931780925-134,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.526,31.644,7.68369197845459\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,135,20261111-3931780925-135,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.722,31.98,7.25827956199646\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,136,20261111-3931780925-136,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.574,32.124,6.801140785217285\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,137,20261111-3931780925-137,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.652,31.544,7.171456575393677\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,138,20261111-3931780925-138,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.578,31.968,6.684352159500122\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,139,20261111-3931780925-139,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.618,31.484,7.016063928604126\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,140,20261111-3931780925-140,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.456,31.644,7.061192035675049\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,141,20261111-3931780925-141,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.622,32.052,6.9182350635528564\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,142,20261111-3931780925-142,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.572,32.068,7.695133924484253\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,143,20261111-3931780925-143,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.4,32.34,7.402154922485352\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,144,20261111-3931780925-144,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.574,32.08,6.7624671459198\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,145,20261111-3931780925-145,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.7,31.936,6.666365146636963\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,146,20261111-3931780925-146,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.576,31.828,7.216511487960815\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,147,20261111-3931780925-147,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.558,31.8,7.489016771316528\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,148,20261111-3931780925-148,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.488,31.864,6.472329139709473\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,149,20261111-3931780925-149,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.416,31.928,7.485337018966675\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,150,20261111-3931780925-150,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.708,31.856,7.13498067855835\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,151,20261111-3931780925-151,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.536,32.168,6.543823003768921\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,152,20261111-3931780925-152,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.678,32.112,6.485358715057373\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,153,20261111-3931780925-153,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.752,32.132,7.7551867961883545\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,154,20261111-3931780925-154,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,498,1,1,1.5,6.54,31.948,6.86263632774353\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,155,20261111-3931780925-155,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.566,32.248,7.741402864456177\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,156,20261111-3931780925-156,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.66,32.5,7.067101001739502\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,157,20261111-3931780925-157,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.482,32.188,7.434675931930542\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,158,20261111-3931780925-158,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.768,31.952,7.207745790481567\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,159,20261111-3931780925-159,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.83,31.832,7.569357633590698\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,160,20261111-3931780925-160,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.614,32.068,6.814608573913574\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,161,20261111-3931780925-161,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.582,32.004,7.2034385204315186\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,162,20261111-3931780925-162,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,498,0,2,2.0,6.722,31.684,7.449023485183716\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,163,20261111-3931780925-163,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.572,31.844,7.488788843154907\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,164,20261111-3931780925-164,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.666,31.728,6.646541595458984\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,165,20261111-3931780925-165,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.55,32.076,7.433035373687744\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,166,20261111-3931780925-166,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.34,32.392,7.442997217178345\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,167,20261111-3931780925-167,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.58,31.864,7.410959005355835\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,168,20261111-3931780925-168,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.544,32.376,7.153644323348999\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,169,20261111-3931780925-169,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.472,31.744,7.335916757583618\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,170,20261111-3931780925-170,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.674,32.34,7.362401962280273\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,171,20261111-3931780925-171,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.508,31.572,7.890371561050415\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,172,20261111-3931780925-172,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,1,0,0.5,6.574,32.152,7.078495740890503\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,173,20261111-3931780925-173,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.718,32.176,6.964867830276489\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,174,20261111-3931780925-174,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.638,32.42,6.705446720123291\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,175,20261111-3931780925-175,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.766,32.128,7.087181806564331\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,176,20261111-3931780925-176,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.714,32.084,7.640592575073242\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,177,20261111-3931780925-177,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.688,31.496,6.579764127731323\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,178,20261111-3931780925-178,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.664,31.756,7.703983545303345\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,179,20261111-3931780925-179,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.63,31.94,6.395822763442993\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,180,20261111-3931780925-180,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.524,31.988,7.556106805801392\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,181,20261111-3931780925-181,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.594,31.892,6.782381296157837\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,182,20261111-3931780925-182,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.554,31.92,7.517385959625244\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,183,20261111-3931780925-183,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.616,31.744,6.624355792999268\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,184,20261111-3931780925-184,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.65,32.012,7.102147102355957\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,185,20261111-3931780925-185,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.618,31.932,7.293285369873047\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,186,20261111-3931780925-186,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.442,31.588,6.552007436752319\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,187,20261111-3931780925-187,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.724,31.828,7.077123165130615\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,188,20261111-3931780925-188,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.692,32.128,7.527024984359741\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,189,20261111-3931780925-189,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.548,32.04,7.102772951126099\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,190,20261111-3931780925-190,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.692,32.62,6.708091974258423\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,191,20261111-3931780925-191,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,0,1,1.0,6.872,32.052,7.226014852523804\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,192,20261111-3931780925-192,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.472,31.788,7.001163005828857\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,193,20261111-3931780925-193,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.442,32.028,7.072378158569336\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,194,20261111-3931780925-194,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,499,1,0,0.5,6.682,32.028,7.099781036376953\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,195,20261111-3931780925-195,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.698,31.836,6.892290115356445\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,196,20261111-3931780925-196,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.568,31.72,7.141850709915161\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,197,20261111-3931780925-197,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.59,31.8,6.595491170883179\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,198,20261111-3931780925-198,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.654,32.008,6.7464165687561035\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g5,199,20261111-3931780925-199,B,noisy,5,0.0003,0.0,2,5,5,5,5,5,2,500,500,0,0,0.0,6.726,31.792,7.414920330047607\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,0,20261111-3979866916-0,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.888,31.856,6.585886716842651\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,1,20261111-3979866916-1,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.768,32.348,6.859105587005615\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,2,20261111-3979866916-2,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.908,31.5,7.608453989028931\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,3,20261111-3979866916-3,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.828,32.024,7.060256004333496\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,4,20261111-3979866916-4,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.768,32.152,5.984544038772583\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,5,20261111-3979866916-5,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.79,32.184,6.1183459758758545\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,6,20261111-3979866916-6,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.892,32.636,6.5309157371521\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,7,20261111-3979866916-7,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.814,31.996,5.987119674682617\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,8,20261111-3979866916-8,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.828,31.952,6.062564849853516\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,9,20261111-3979866916-9,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.812,32.084,6.208365201950073\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,10,20261111-3979866916-10,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.646,32.356,6.379622220993042\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,11,20261111-3979866916-11,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.64,31.64,6.198296785354614\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,12,20261111-3979866916-12,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.754,31.98,6.207146883010864\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,13,20261111-3979866916-13,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,6.038,31.812,5.696314334869385\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,14,20261111-3979866916-14,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,6.01,32.192,6.398479700088501\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,15,20261111-3979866916-15,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.85,32.4,6.043849229812622\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,16,20261111-3979866916-16,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.718,31.848,6.259376525878906\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,17,20261111-3979866916-17,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.758,32.004,5.979466915130615\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,18,20261111-3979866916-18,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.808,31.888,6.141736745834351\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,19,20261111-3979866916-19,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.806,31.552,5.954997301101685\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,20,20261111-3979866916-20,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.696,31.568,5.841290235519409\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,21,20261111-3979866916-21,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.872,31.432,6.925656795501709\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,22,20261111-3979866916-22,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.58,32.4,6.209194660186768\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,23,20261111-3979866916-23,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,6.02,31.92,6.946888208389282\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,24,20261111-3979866916-24,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.928,31.968,5.727325439453125\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,25,20261111-3979866916-25,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,6.002,31.948,7.068076133728027\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,26,20261111-3979866916-26,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.73,32.28,5.707555532455444\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,27,20261111-3979866916-27,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.896,32.028,5.937211036682129\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,28,20261111-3979866916-28,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.696,31.9,6.22456955909729\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,29,20261111-3979866916-29,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.872,31.848,6.505689859390259\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,30,20261111-3979866916-30,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.762,31.92,5.616447687149048\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,31,20261111-3979866916-31,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.724,32.012,6.051604747772217\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,32,20261111-3979866916-32,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.732,32.064,6.019459247589111\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,33,20261111-3979866916-33,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.648,31.904,5.7975380420684814\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,34,20261111-3979866916-34,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.796,32.088,6.075074195861816\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,35,20261111-3979866916-35,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.952,31.824,5.675727844238281\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,36,20261111-3979866916-36,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.904,31.844,6.24041748046875\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,37,20261111-3979866916-37,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.98,32.172,5.767902851104736\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,38,20261111-3979866916-38,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.82,31.508,5.928134202957153\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,39,20261111-3979866916-39,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.77,31.476,6.25426459312439\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,40,20261111-3979866916-40,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.762,32.332,6.093379020690918\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,41,20261111-3979866916-41,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.784,31.84,5.9087607860565186\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,42,20261111-3979866916-42,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.08,32.044,5.900509357452393\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,43,20261111-3979866916-43,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.048,31.888,6.170018911361694\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,44,20261111-3979866916-44,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.918,32.06,6.024203062057495\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,45,20261111-3979866916-45,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.932,32.104,6.049640417098999\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,46,20261111-3979866916-46,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,6.028,31.408,5.971561431884766\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,47,20261111-3979866916-47,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.902,32.064,6.311633586883545\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,48,20261111-3979866916-48,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.782,32.124,5.846954584121704\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,49,20261111-3979866916-49,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.77,31.796,6.970016002655029\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,50,20261111-3979866916-50,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.858,32.096,5.9711291790008545\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,51,20261111-3979866916-51,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.798,31.612,6.423147201538086\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,52,20261111-3979866916-52,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.848,31.976,6.023897647857666\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,53,20261111-3979866916-53,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.768,32.036,6.226925373077393\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,54,20261111-3979866916-54,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.84,32.14,6.544549465179443\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,55,20261111-3979866916-55,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.896,31.792,6.3653318881988525\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,56,20261111-3979866916-56,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.94,31.808,6.242238759994507\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,57,20261111-3979866916-57,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.926,31.976,6.399588108062744\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,58,20261111-3979866916-58,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.712,32.008,5.778264284133911\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,59,20261111-3979866916-59,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.64,31.624,6.413873910903931\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,60,20261111-3979866916-60,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.89,32.124,6.378625869750977\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,61,20261111-3979866916-61,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.774,32.472,6.2303667068481445\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,62,20261111-3979866916-62,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.888,32.216,6.070200681686401\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,63,20261111-3979866916-63,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.874,31.944,6.389926433563232\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,64,20261111-3979866916-64,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.128,32.216,6.142452716827393\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,65,20261111-3979866916-65,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.814,32.04,6.004236459732056\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,66,20261111-3979866916-66,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.882,32.08,6.176432847976685\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,67,20261111-3979866916-67,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.858,32.076,6.174957036972046\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,68,20261111-3979866916-68,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.896,32.064,6.101511240005493\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,69,20261111-3979866916-69,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.92,31.876,5.904014348983765\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,70,20261111-3979866916-70,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.84,31.704,6.550935983657837\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,71,20261111-3979866916-71,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.858,32.2,6.028302431106567\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,72,20261111-3979866916-72,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.964,31.788,6.301231622695923\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,73,20261111-3979866916-73,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,1,0,0.5,5.904,31.892,6.525681257247925\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,74,20261111-3979866916-74,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.764,32.216,6.644241809844971\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,75,20261111-3979866916-75,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.944,31.888,5.601123571395874\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,76,20261111-3979866916-76,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.832,31.872,6.730357646942139\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,77,20261111-3979866916-77,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.874,32.448,5.633349657058716\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,78,20261111-3979866916-78,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,6.048,31.972,6.609177827835083\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,79,20261111-3979866916-79,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.87,31.768,6.486271858215332\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,80,20261111-3979866916-80,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.902,32.072,6.551347494125366\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,81,20261111-3979866916-81,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.886,31.812,6.083593845367432\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,82,20261111-3979866916-82,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.828,31.56,6.17024040222168\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,83,20261111-3979866916-83,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.908,31.916,6.305089950561523\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,84,20261111-3979866916-84,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.698,32.484,6.728582143783569\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,85,20261111-3979866916-85,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.766,32.06,6.040525913238525\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,86,20261111-3979866916-86,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.85,31.972,6.599004745483398\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,87,20261111-3979866916-87,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.918,31.976,6.126966714859009\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,88,20261111-3979866916-88,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.73,31.992,6.747774362564087\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,89,20261111-3979866916-89,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.912,32.504,5.919658184051514\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,90,20261111-3979866916-90,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.78,31.86,6.1759562492370605\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,91,20261111-3979866916-91,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.946,31.848,6.027482986450195\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,92,20261111-3979866916-92,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.776,32.088,6.1706461906433105\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,93,20261111-3979866916-93,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.816,32.004,5.87360405921936\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,94,20261111-3979866916-94,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,6.082,31.896,5.963275194168091\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,95,20261111-3979866916-95,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.878,31.828,6.1739842891693115\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,96,20261111-3979866916-96,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.112,32.58,6.031517267227173\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,97,20261111-3979866916-97,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.772,32.22,5.733139991760254\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,98,20261111-3979866916-98,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.792,31.952,6.1591057777404785\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,99,20261111-3979866916-99,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.034,32.152,6.259339094161987\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,100,20261111-3979866916-100,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.764,32.104,6.429629802703857\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,101,20261111-3979866916-101,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,1,0,0.5,5.774,31.94,6.00184178352356\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,102,20261111-3979866916-102,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.786,32.056,5.957229375839233\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,103,20261111-3979866916-103,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.838,32.396,6.6215362548828125\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,104,20261111-3979866916-104,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.778,32.256,5.967406272888184\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,105,20261111-3979866916-105,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.824,31.608,6.630725145339966\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,106,20261111-3979866916-106,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.992,31.824,6.071119070053101\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,107,20261111-3979866916-107,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.728,32.04,5.9193785190582275\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,108,20261111-3979866916-108,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.69,32.308,5.7815117835998535\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,109,20261111-3979866916-109,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.82,31.84,6.0008885860443115\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,110,20261111-3979866916-110,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.764,32.296,5.995512962341309\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,111,20261111-3979866916-111,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,1,0,0.5,6.032,32.072,6.287250757217407\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,112,20261111-3979866916-112,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.836,31.856,6.257159233093262\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,113,20261111-3979866916-113,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.18,31.852,6.381898880004883\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,114,20261111-3979866916-114,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.71,31.5,6.027958154678345\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,115,20261111-3979866916-115,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.9,32.156,6.468250513076782\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,116,20261111-3979866916-116,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.832,31.744,6.08010196685791\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,117,20261111-3979866916-117,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.812,31.8,6.277835130691528\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,118,20261111-3979866916-118,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.934,31.916,6.070090055465698\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,119,20261111-3979866916-119,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.864,32.152,6.088813066482544\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,120,20261111-3979866916-120,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.742,31.872,6.094584941864014\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,121,20261111-3979866916-121,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.912,31.9,6.272791624069214\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,122,20261111-3979866916-122,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.986,32.5,6.381856203079224\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,123,20261111-3979866916-123,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.648,32.452,5.991103887557983\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,124,20261111-3979866916-124,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.8,32.116,6.3207995891571045\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,125,20261111-3979866916-125,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.77,32.42,6.22237753868103\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,126,20261111-3979866916-126,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.846,31.564,5.745825290679932\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,127,20261111-3979866916-127,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.784,32.292,6.195537567138672\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,128,20261111-3979866916-128,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.734,32.112,5.882641553878784\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,129,20261111-3979866916-129,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.786,31.908,5.822774410247803\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,130,20261111-3979866916-130,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.854,32.012,6.472124099731445\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,131,20261111-3979866916-131,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.724,31.876,6.404799461364746\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,132,20261111-3979866916-132,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.742,31.924,5.614783525466919\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,133,20261111-3979866916-133,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,6.052,31.916,6.051871299743652\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,134,20261111-3979866916-134,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.742,31.9,6.142378330230713\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,135,20261111-3979866916-135,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.738,32.36,6.528822660446167\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,136,20261111-3979866916-136,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.896,31.684,5.605223655700684\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,137,20261111-3979866916-137,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.85,32.276,6.525100946426392\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,138,20261111-3979866916-138,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.93,32.532,6.304720878601074\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,139,20261111-3979866916-139,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.804,32.28,6.244315147399902\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,140,20261111-3979866916-140,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.926,32.128,6.833268404006958\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,141,20261111-3979866916-141,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.958,31.904,6.364734172821045\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,142,20261111-3979866916-142,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.896,32.12,6.231747388839722\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,143,20261111-3979866916-143,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.994,31.28,6.308285713195801\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,144,20261111-3979866916-144,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.026,31.684,6.37331748008728\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,145,20261111-3979866916-145,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.84,32.348,6.300961256027222\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,146,20261111-3979866916-146,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.96,32.232,6.150299549102783\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,147,20261111-3979866916-147,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.012,32.048,6.00854754447937\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,148,20261111-3979866916-148,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.726,32.536,6.60491681098938\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,149,20261111-3979866916-149,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.782,32.296,6.376656770706177\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,150,20261111-3979866916-150,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.906,32.004,5.602312326431274\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,151,20261111-3979866916-151,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.756,32.028,6.398372650146484\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,152,20261111-3979866916-152,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.996,32.192,6.613718748092651\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,153,20261111-3979866916-153,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.956,31.664,6.279647350311279\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,154,20261111-3979866916-154,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.778,31.96,6.430783748626709\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,155,20261111-3979866916-155,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.762,32.048,6.212440490722656\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,156,20261111-3979866916-156,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.682,32.124,6.326422691345215\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,157,20261111-3979866916-157,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.82,32.064,6.312492370605469\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,158,20261111-3979866916-158,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.016,31.9,5.946542024612427\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,159,20261111-3979866916-159,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.726,31.968,6.030074596405029\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,160,20261111-3979866916-160,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.764,31.952,5.9446210861206055\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,161,20261111-3979866916-161,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.61,32.432,5.880131959915161\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,162,20261111-3979866916-162,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.822,31.776,6.125849723815918\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,163,20261111-3979866916-163,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.93,32.004,6.183676242828369\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,164,20261111-3979866916-164,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.812,31.64,5.949048757553101\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,165,20261111-3979866916-165,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.956,32.216,6.047621011734009\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,166,20261111-3979866916-166,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.972,31.884,6.167131662368774\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,167,20261111-3979866916-167,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.784,32.372,6.243314743041992\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,168,20261111-3979866916-168,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.832,32.312,6.22721004486084\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,169,20261111-3979866916-169,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.864,31.86,5.89464807510376\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,170,20261111-3979866916-170,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.884,31.664,6.043526649475098\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,171,20261111-3979866916-171,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.862,31.992,6.095217704772949\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,172,20261111-3979866916-172,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.776,31.532,6.430714130401611\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,173,20261111-3979866916-173,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,1,0,0.5,5.892,32.192,6.103708982467651\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,174,20261111-3979866916-174,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,6.01,31.98,6.180805444717407\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,175,20261111-3979866916-175,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.828,32.084,5.844898462295532\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,176,20261111-3979866916-176,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.916,31.924,6.54670786857605\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,177,20261111-3979866916-177,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.716,31.972,6.094283580780029\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,178,20261111-3979866916-178,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.836,32.38,6.403414011001587\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,179,20261111-3979866916-179,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.99,32.156,6.093429088592529\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,180,20261111-3979866916-180,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.854,32.0,5.973846435546875\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,181,20261111-3979866916-181,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.956,32.244,6.2787251472473145\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,182,20261111-3979866916-182,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.848,32.256,6.155644655227661\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,183,20261111-3979866916-183,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.936,31.828,6.158912897109985\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,184,20261111-3979866916-184,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.902,31.788,6.547903776168823\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,185,20261111-3979866916-185,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.808,32.12,5.676066875457764\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,186,20261111-3979866916-186,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.978,32.092,6.032556772232056\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,187,20261111-3979866916-187,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.874,31.788,6.3853676319122314\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,188,20261111-3979866916-188,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.934,32.196,5.920562744140625\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,189,20261111-3979866916-189,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.926,31.448,6.833309173583984\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,190,20261111-3979866916-190,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.806,31.948,5.488795042037964\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,191,20261111-3979866916-191,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.76,31.868,6.20986533164978\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,192,20261111-3979866916-192,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.998,32.14,6.4872353076934814\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,193,20261111-3979866916-193,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,498,0,2,2.0,5.888,32.104,6.065994024276733\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,194,20261111-3979866916-194,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.846,32.176,6.157743453979492\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,195,20261111-3979866916-195,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.876,31.716,6.138789415359497\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,196,20261111-3979866916-196,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.824,32.092,6.607863426208496\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,197,20261111-3979866916-197,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,500,0,0,0.0,5.844,32.032,5.911397933959961\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,198,20261111-3979866916-198,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.716,32.38,6.165956974029541\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K2|g1,199,20261111-3979866916-199,B,noisy,5,0.0003,0.0,2,1,5,5,1,5,2,500,499,0,1,1.0,5.638,32.248,5.873128414154053\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,0,20261111-1390825560-0,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.363363363363364,47.98198198198198,9.89745044708252\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,1,20261111-1390825560-1,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.192192192192191,47.933933933933936,8.539167642593384\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,2,20261111-1390825560-2,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.402402402402402,48.066066066066064,9.489346504211426\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,3,20261111-1390825560-3,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.627627627627628,48.210210210210214,8.298195123672485\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,4,20261111-1390825560-4,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.318318318318319,47.489489489489486,7.564043283462524\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,5,20261111-1390825560-5,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.117117117117116,47.32132132132132,7.80222225189209\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,6,20261111-1390825560-6,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.183183183183184,47.693693693693696,7.258007764816284\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,7,20261111-1390825560-7,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.306306306306306,48.31831831831832,7.276149034500122\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,8,20261111-1390825560-8,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.312312312312311,47.73573573573574,7.138617992401123\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,9,20261111-1390825560-9,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.14114114114114,47.36336336336336,6.660900831222534\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,10,20261111-1390825560-10,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.216216216216216,47.351351351351354,7.0010058879852295\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,11,20261111-1390825560-11,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.537537537537538,47.747747747747745,7.265660762786865\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,12,20261111-1390825560-12,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.234234234234235,48.12012012012012,7.035490274429321\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,13,20261111-1390825560-13,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.30930930930931,47.651651651651655,6.733605146408081\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,14,20261111-1390825560-14,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.273273273273274,48.13813813813814,6.99778938293457\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,15,20261111-1390825560-15,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,1,0,0.5,9.507507507507508,48.18018018018018,7.252330541610718\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,16,20261111-1390825560-16,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.282282282282281,48.1981981981982,6.964083194732666\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,17,20261111-1390825560-17,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.651651651651651,47.807807807807805,7.123706340789795\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,18,20261111-1390825560-18,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.492492492492492,47.97597597597598,7.175266265869141\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,19,20261111-1390825560-19,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.411411411411411,47.47747747747748,6.79278826713562\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,20,20261111-1390825560-20,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.558558558558559,48.792792792792795,7.343685150146484\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,21,20261111-1390825560-21,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.285285285285285,47.98198198198198,7.364262580871582\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,22,20261111-1390825560-22,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.66066066066066,48.84084084084084,7.176482677459717\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,23,20261111-1390825560-23,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.468468468468469,47.85585585585586,6.759092569351196\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,24,20261111-1390825560-24,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.15915915915916,47.65765765765766,7.72158145904541\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,25,20261111-1390825560-25,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.381381381381381,48.186186186186184,7.6361613273620605\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,26,20261111-1390825560-26,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.174174174174174,48.06006006006006,7.0975422859191895\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,27,20261111-1390825560-27,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.534534534534535,47.4954954954955,7.202268362045288\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,28,20261111-1390825560-28,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.534534534534535,48.066066066066064,7.104827165603638\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,29,20261111-1390825560-29,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.24024024024024,48.28228228228228,7.01195764541626\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,30,20261111-1390825560-30,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.18018018018018,47.71771771771772,7.634320497512817\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,31,20261111-1390825560-31,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.456456456456456,47.945945945945944,6.605942487716675\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,32,20261111-1390825560-32,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.453453453453454,48.474474474474476,6.454871416091919\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,33,20261111-1390825560-33,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.402402402402402,48.612612612612615,6.787879228591919\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,34,20261111-1390825560-34,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.582582582582583,47.7957957957958,7.6332478523254395\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,35,20261111-1390825560-35,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.567567567567568,47.72372372372372,6.996943712234497\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,36,20261111-1390825560-36,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.57957957957958,48.3003003003003,6.649776220321655\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,37,20261111-1390825560-37,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.861861861861861,47.549549549549546,7.762826204299927\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,38,20261111-1390825560-38,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.492492492492492,47.807807807807805,7.5577757358551025\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,39,20261111-1390825560-39,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.66066066066066,48.108108108108105,7.129488229751587\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,40,20261111-1390825560-40,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.402402402402402,47.46546546546546,6.997873306274414\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,41,20261111-1390825560-41,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.723723723723724,48.588588588588586,6.981841087341309\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,42,20261111-1390825560-42,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.51051051051051,48.12012012012012,7.081909894943237\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,43,20261111-1390825560-43,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.198198198198199,48.006006006006004,7.802727222442627\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,44,20261111-1390825560-44,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.30930930930931,48.03603603603604,6.8813700675964355\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,45,20261111-1390825560-45,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.702702702702704,47.627627627627625,7.604983806610107\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,46,20261111-1390825560-46,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.408408408408409,48.42042042042042,7.181053400039673\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,47,20261111-1390825560-47,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.438438438438439,48.15615615615616,7.450469255447388\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,48,20261111-1390825560-48,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.591591591591591,47.747747747747745,7.554235458374023\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,49,20261111-1390825560-49,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.393393393393394,47.93993993993994,7.396171569824219\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,50,20261111-1390825560-50,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.474474474474475,47.807807807807805,6.930202007293701\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,51,20261111-1390825560-51,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.843843843843844,48.33633633633634,7.449273586273193\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,52,20261111-1390825560-52,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,1,0,0.5,9.528528528528529,48.108108108108105,6.758168935775757\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,53,20261111-1390825560-53,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.252252252252251,48.55855855855856,5.948781490325928\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,54,20261111-1390825560-54,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.807807807807809,48.546546546546544,6.8000569343566895\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,55,20261111-1390825560-55,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.033033033033034,48.150150150150154,8.001959323883057\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,56,20261111-1390825560-56,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.492492492492492,48.174174174174176,6.3991358280181885\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,57,20261111-1390825560-57,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.432432432432432,48.52252252252252,6.903964519500732\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,58,20261111-1390825560-58,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.237237237237236,47.68168168168168,6.854835748672485\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,59,20261111-1390825560-59,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.093093093093094,47.96396396396396,7.237536191940308\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,60,20261111-1390825560-60,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.381381381381381,48.252252252252255,6.698394060134888\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,61,20261111-1390825560-61,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.552552552552553,49.471471471471475,7.469221830368042\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,62,20261111-1390825560-62,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.405405405405405,48.246246246246244,7.82790470123291\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,63,20261111-1390825560-63,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.015015015015015,48.108108108108105,7.92520546913147\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,64,20261111-1390825560-64,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.42942942942943,47.765765765765764,6.630901575088501\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,65,20261111-1390825560-65,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.354354354354355,48.090090090090094,6.593657970428467\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,66,20261111-1390825560-66,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.342342342342342,48.090090090090094,7.337119817733765\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,67,20261111-1390825560-67,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.441441441441441,47.825825825825824,6.597738742828369\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,68,20261111-1390825560-68,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.35135135135135,48.03603603603604,7.285110950469971\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,69,20261111-1390825560-69,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.366366366366366,48.72072072072072,6.321915149688721\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,70,20261111-1390825560-70,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.33033033033033,47.3033033033033,6.96479606628418\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,71,20261111-1390825560-71,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.318318318318319,48.474474474474476,6.7786335945129395\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,72,20261111-1390825560-72,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.05105105105105,47.945945945945944,7.900682687759399\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,73,20261111-1390825560-73,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.237237237237236,47.729729729729726,7.209413528442383\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,74,20261111-1390825560-74,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.51051051051051,48.072072072072075,8.010613203048706\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,75,20261111-1390825560-75,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,1,0,0.5,9.36936936936937,47.93993993993994,6.896687269210815\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,76,20261111-1390825560-76,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.465465465465465,48.0,8.16405463218689\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,77,20261111-1390825560-77,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.093093093093094,47.4954954954955,7.14338493347168\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,78,20261111-1390825560-78,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.342342342342342,48.43843843843844,7.79755711555481\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,79,20261111-1390825560-79,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.693693693693694,47.2972972972973,6.812530517578125\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,80,20261111-1390825560-80,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.495495495495495,48.6006006006006,6.9907941818237305\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,81,20261111-1390825560-81,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.795795795795796,48.126126126126124,7.931075811386108\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,82,20261111-1390825560-82,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.285285285285285,48.12012012012012,6.729281663894653\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,83,20261111-1390825560-83,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.195195195195195,47.95795795795796,8.09877347946167\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,84,20261111-1390825560-84,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.264264264264265,48.348348348348345,6.943691253662109\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,85,20261111-1390825560-85,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.312312312312311,47.74174174174174,7.720334053039551\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,86,20261111-1390825560-86,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.294294294294295,48.588588588588586,7.014795780181885\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,87,20261111-1390825560-87,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.45945945945946,48.1021021021021,7.601866960525513\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,88,20261111-1390825560-88,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.57057057057057,48.2042042042042,6.8522398471832275\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,89,20261111-1390825560-89,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.621621621621621,48.510510510510514,7.638211488723755\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,90,20261111-1390825560-90,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.345345345345345,47.97597597597598,7.342300653457642\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,91,20261111-1390825560-91,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.402402402402402,47.909909909909906,7.210260629653931\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,92,20261111-1390825560-92,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.60960960960961,49.051051051051054,6.59880518913269\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,93,20261111-1390825560-93,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.05105105105105,47.849849849849846,7.294434547424316\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,94,20261111-1390825560-94,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.3003003003003,47.61561561561562,7.49622654914856\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,95,20261111-1390825560-95,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.18918918918919,47.807807807807805,7.809669494628906\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,96,20261111-1390825560-96,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.3993993993994,47.909909909909906,7.3996498584747314\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,97,20261111-1390825560-97,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.498498498498499,47.669669669669666,7.757674217224121\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,98,20261111-1390825560-98,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.312312312312311,48.14414414414414,6.877136945724487\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,99,20261111-1390825560-99,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,1,1,1.5,9.213213213213214,48.810810810810814,7.805994749069214\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,100,20261111-1390825560-100,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.084084084084084,47.813813813813816,7.717437505722046\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,101,20261111-1390825560-101,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.468468468468469,47.6036036036036,7.9203102588653564\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,102,20261111-1390825560-102,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.3993993993994,47.591591591591595,6.977700233459473\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,103,20261111-1390825560-103,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.6006006006006,47.951951951951955,7.138976097106934\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,104,20261111-1390825560-104,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.222222222222221,47.711711711711715,7.668635606765747\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,105,20261111-1390825560-105,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.597597597597598,47.693693693693696,7.734902620315552\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,106,20261111-1390825560-106,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.45045045045045,48.132132132132135,8.370099067687988\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,107,20261111-1390825560-107,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,8.91891891891892,47.753753753753756,6.9423911571502686\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,108,20261111-1390825560-108,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.696696696696696,48.108108108108105,7.307580232620239\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,109,20261111-1390825560-109,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.3003003003003,48.04204204204204,7.626811504364014\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,110,20261111-1390825560-110,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.318318318318319,48.52252252252252,7.485488653182983\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,111,20261111-1390825560-111,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.27927927927928,47.9039039039039,7.058456182479858\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,112,20261111-1390825560-112,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.186186186186186,48.01801801801802,8.05548620223999\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,113,20261111-1390825560-113,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.492492492492492,48.18018018018018,7.2114036083221436\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,114,20261111-1390825560-114,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.486486486486486,47.12312312312312,7.578507423400879\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,115,20261111-1390825560-115,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.387387387387387,48.37837837837838,7.4799275398254395\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,116,20261111-1390825560-116,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,1,0,0.5,9.285285285285285,46.86486486486486,8.07695984840393\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,117,20261111-1390825560-117,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.18018018018018,48.34234234234234,6.7282609939575195\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,118,20261111-1390825560-118,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.114114114114114,48.03603603603604,7.4130284786224365\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,119,20261111-1390825560-119,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.7987987987988,47.66366366366366,6.999323129653931\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,120,20261111-1390825560-120,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.378378378378379,47.53753753753754,6.822120189666748\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,121,20261111-1390825560-121,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.195195195195195,48.15615615615616,6.591106414794922\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,122,20261111-1390825560-122,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.372372372372372,47.72372372372372,7.6002302169799805\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,123,20261111-1390825560-123,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.35135135135135,47.41741741741742,7.559039831161499\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,124,20261111-1390825560-124,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.366366366366366,48.768768768768766,7.4634130001068115\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,125,20261111-1390825560-125,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.63063063063063,47.753753753753756,7.034979343414307\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,126,20261111-1390825560-126,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.126126126126126,48.27627627627628,7.8409423828125\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,127,20261111-1390825560-127,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.702702702702704,48.468468468468465,7.321638822555542\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,128,20261111-1390825560-128,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.42042042042042,47.927927927927925,7.61937403678894\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,129,20261111-1390825560-129,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.21921921921922,47.27927927927928,6.788419246673584\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,130,20261111-1390825560-130,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.417417417417417,48.3963963963964,7.051411390304565\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,131,20261111-1390825560-131,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.483483483483484,48.426426426426424,8.114017963409424\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,132,20261111-1390825560-132,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.402402402402402,48.0,7.059285402297974\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,133,20261111-1390825560-133,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.438438438438439,48.510510510510514,7.608433961868286\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,134,20261111-1390825560-134,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.084084084084084,47.693693693693696,7.395852565765381\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,135,20261111-1390825560-135,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.531531531531531,48.53453453453454,7.171071767807007\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,136,20261111-1390825560-136,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.81081081081081,47.78378378378378,7.371554851531982\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,137,20261111-1390825560-137,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.222222222222221,48.246246246246244,7.1528942584991455\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,138,20261111-1390825560-138,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.342342342342342,48.02402402402402,7.140009641647339\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,139,20261111-1390825560-139,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.501501501501501,47.969969969969966,7.706238508224487\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,140,20261111-1390825560-140,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.495495495495495,47.92192192192192,7.043285131454468\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,141,20261111-1390825560-141,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.3993993993994,47.2012012012012,7.508406400680542\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,142,20261111-1390825560-142,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.42042042042042,48.04204204204204,6.6773622035980225\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,143,20261111-1390825560-143,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.3003003003003,48.252252252252255,7.292157173156738\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,144,20261111-1390825560-144,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.393393393393394,47.06306306306306,7.8308398723602295\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,145,20261111-1390825560-145,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.3993993993994,48.372372372372375,7.796524286270142\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,146,20261111-1390825560-146,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.387387387387387,48.27627627627628,7.110592603683472\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,147,20261111-1390825560-147,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.66966966966967,47.951951951951955,7.135622501373291\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,148,20261111-1390825560-148,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.24024024024024,48.450450450450454,7.256152391433716\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,149,20261111-1390825560-149,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.273273273273274,48.168168168168165,6.988861799240112\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,150,20261111-1390825560-150,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.354354354354355,47.8978978978979,7.494760513305664\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,151,20261111-1390825560-151,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.21021021021021,48.054054054054056,7.334684610366821\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,152,20261111-1390825560-152,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.234234234234235,48.13813813813814,7.387938976287842\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,153,20261111-1390825560-153,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.393393393393394,48.306306306306304,7.292668581008911\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,154,20261111-1390825560-154,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.042042042042041,47.651651651651655,7.351018667221069\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,155,20261111-1390825560-155,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.396396396396396,47.951951951951955,6.8192808628082275\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,156,20261111-1390825560-156,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.225225225225225,47.84384384384384,7.172939777374268\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,157,20261111-1390825560-157,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.312312312312311,47.5975975975976,6.935443162918091\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,158,20261111-1390825560-158,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.342342342342342,47.61561561561562,6.968881368637085\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,159,20261111-1390825560-159,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.132132132132131,47.327327327327325,7.939872741699219\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,160,20261111-1390825560-160,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.333333333333334,48.090090090090094,7.426504373550415\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,161,20261111-1390825560-161,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.477477477477477,47.633633633633636,7.986072778701782\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,162,20261111-1390825560-162,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.588588588588589,48.15615615615616,7.136866331100464\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,163,20261111-1390825560-163,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.32132132132132,47.909909909909906,8.4275541305542\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,164,20261111-1390825560-164,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.438438438438439,48.13813813813814,7.038145065307617\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,165,20261111-1390825560-165,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.345345345345345,47.849849849849846,7.82084059715271\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,166,20261111-1390825560-166,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.156156156156156,47.97597597597598,7.008153915405273\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,167,20261111-1390825560-167,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.702702702702704,48.270270270270274,7.084007978439331\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,168,20261111-1390825560-168,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.504504504504505,47.77777777777778,8.727329969406128\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,169,20261111-1390825560-169,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.24024024024024,47.873873873873876,6.651702165603638\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,170,20261111-1390825560-170,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.495495495495495,48.68468468468468,7.087809324264526\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,171,20261111-1390825560-171,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.66066066066066,48.270270270270274,8.051060914993286\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,172,20261111-1390825560-172,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.726726726726726,48.792792792792795,7.513962507247925\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,173,20261111-1390825560-173,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.492492492492492,48.01801801801802,8.798847436904907\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,174,20261111-1390825560-174,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.2012012012012,47.147147147147145,6.953228950500488\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,175,20261111-1390825560-175,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,1,1,1.5,9.537537537537538,48.012012012012015,7.6946752071380615\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,176,20261111-1390825560-176,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.30930930930931,47.4954954954955,8.079692840576172\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,177,20261111-1390825560-177,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.45045045045045,47.37537537537538,6.478360414505005\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,178,20261111-1390825560-178,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.45945945945946,47.86186186186186,8.576271533966064\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,179,20261111-1390825560-179,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.516516516516516,48.63663663663664,8.049017906188965\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,180,20261111-1390825560-180,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.444444444444445,47.91591591591592,8.101844310760498\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,181,20261111-1390825560-181,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.33033033033033,47.591591591591595,7.86462664604187\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,182,20261111-1390825560-182,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.441441441441441,48.288288288288285,7.822291135787964\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,183,20261111-1390825560-183,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.675675675675675,47.153153153153156,8.50467824935913\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,184,20261111-1390825560-184,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.438438438438439,48.15615615615616,7.731641530990601\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,185,20261111-1390825560-185,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.606606606606606,48.648648648648646,7.494013547897339\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,186,20261111-1390825560-186,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.408408408408409,48.090090090090094,7.523892879486084\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,187,20261111-1390825560-187,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.303303303303304,48.150150150150154,7.6237757205963135\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,188,20261111-1390825560-188,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.54054054054054,47.68168168168168,7.716070175170898\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,189,20261111-1390825560-189,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.54054054054054,48.906906906906904,7.603375434875488\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,190,20261111-1390825560-190,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.663663663663664,47.73573573573574,7.248776435852051\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,191,20261111-1390825560-191,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.465465465465465,47.969969969969966,8.361902713775635\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,192,20261111-1390825560-192,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.324324324324325,47.687687687687685,7.865566730499268\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,193,20261111-1390825560-193,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.75075075075075,48.006006006006004,7.687984466552734\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,194,20261111-1390825560-194,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.576576576576576,48.03603603603604,6.884233474731445\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,195,20261111-1390825560-195,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,1,0,0.5,9.594594594594595,47.891891891891895,8.137009143829346\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,196,20261111-1390825560-196,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.462462462462462,47.43543543543544,7.121613025665283\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,197,20261111-1390825560-197,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.3993993993994,47.21921921921922,7.40792989730835\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,198,20261111-1390825560-198,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.504504504504505,48.57657657657658,7.186833381652832\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,199,20261111-1390825560-199,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.294294294294295,47.4954954954955,7.221415758132935\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,200,20261111-1390825560-200,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.267267267267266,47.867867867867865,7.1069300174713135\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,201,20261111-1390825560-201,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.063063063063064,48.474474474474476,7.224323034286499\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,202,20261111-1390825560-202,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.543543543543544,48.02402402402402,6.686743497848511\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,203,20261111-1390825560-203,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.324324324324325,48.31831831831832,6.539942026138306\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,204,20261111-1390825560-204,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.414414414414415,48.43843843843844,7.405964612960815\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,205,20261111-1390825560-205,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.192192192192191,47.747747747747745,7.728238344192505\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,206,20261111-1390825560-206,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.381381381381381,47.933933933933936,6.615153074264526\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,207,20261111-1390825560-207,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.48948948948949,47.747747747747745,6.904991626739502\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,208,20261111-1390825560-208,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.681681681681681,47.507507507507505,7.5473244190216064\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,209,20261111-1390825560-209,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.333333333333334,48.426426426426424,7.038379192352295\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,210,20261111-1390825560-210,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.285285285285285,48.732732732732735,8.260884284973145\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,211,20261111-1390825560-211,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.276276276276276,47.249249249249246,6.813119649887085\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,212,20261111-1390825560-212,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.225225225225225,47.95795795795796,7.523991823196411\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,213,20261111-1390825560-213,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.45045045045045,48.51651651651652,7.048977375030518\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,214,20261111-1390825560-214,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.234234234234235,47.765765765765764,6.96502685546875\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,215,20261111-1390825560-215,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.447447447447447,48.090090090090094,6.927973747253418\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,216,20261111-1390825560-216,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.345345345345345,48.006006006006004,7.059996843338013\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,217,20261111-1390825560-217,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.276276276276276,48.390390390390394,6.656637668609619\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,218,20261111-1390825560-218,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.177177177177176,48.210210210210214,7.813104867935181\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,219,20261111-1390825560-219,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.57057057057057,48.432432432432435,6.745591878890991\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,220,20261111-1390825560-220,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.285285285285285,47.73573573573574,7.460338592529297\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,221,20261111-1390825560-221,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.432432432432432,47.51951951951952,7.402083158493042\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,222,20261111-1390825560-222,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.381381381381381,48.31831831831832,7.5732526779174805\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,223,20261111-1390825560-223,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.30930930930931,48.006006006006004,6.773287534713745\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,224,20261111-1390825560-224,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.45045045045045,47.513513513513516,7.300949811935425\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,225,20261111-1390825560-225,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.333333333333334,48.306306306306304,7.362738370895386\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,226,20261111-1390825560-226,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,8.97897897897898,48.0,7.120489120483398\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,227,20261111-1390825560-227,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.21921921921922,47.831831831831835,7.039717197418213\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,228,20261111-1390825560-228,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.702702702702704,49.06306306306306,8.296343088150024\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,229,20261111-1390825560-229,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.27927927927928,47.849849849849846,6.886492729187012\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,230,20261111-1390825560-230,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.333333333333334,48.67867867867868,6.87531590461731\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,231,20261111-1390825560-231,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.228228228228229,47.831831831831835,7.584740400314331\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,232,20261111-1390825560-232,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.507507507507508,48.132132132132135,7.783350706100464\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,233,20261111-1390825560-233,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.51951951951952,48.966966966966964,7.1689534187316895\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,234,20261111-1390825560-234,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.54054054054054,47.747747747747745,7.914114475250244\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,235,20261111-1390825560-235,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.654654654654655,47.747747747747745,6.693390607833862\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,236,20261111-1390825560-236,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.57057057057057,47.73573573573574,7.484359264373779\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,237,20261111-1390825560-237,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.366366366366366,48.51651651651652,7.501010417938232\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,238,20261111-1390825560-238,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.675675675675675,47.01501501501502,7.971445798873901\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,239,20261111-1390825560-239,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.63963963963964,47.993993993993996,7.36950159072876\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,240,20261111-1390825560-240,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.45945945945946,48.126126126126124,7.53314208984375\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,241,20261111-1390825560-241,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,330,0,3,3.0,9.318318318318319,47.38138138138138,7.240233898162842\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,242,20261111-1390825560-242,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.54054054054054,48.126126126126124,7.948533773422241\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,243,20261111-1390825560-243,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.546546546546546,48.294294294294296,6.7992682456970215\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,244,20261111-1390825560-244,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.225225225225225,47.73573573573574,7.755983352661133\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,245,20261111-1390825560-245,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.24924924924925,48.37837837837838,8.081726312637329\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,246,20261111-1390825560-246,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.516516516516516,47.98198198198198,6.788172006607056\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,247,20261111-1390825560-247,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.315315315315315,47.525525525525524,6.660139322280884\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,248,20261111-1390825560-248,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.354354354354355,47.17717717717718,7.092382907867432\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,249,20261111-1390825560-249,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.312312312312311,48.5045045045045,7.1729490756988525\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,250,20261111-1390825560-250,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.387387387387387,48.330330330330334,7.030777454376221\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,251,20261111-1390825560-251,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.465465465465465,48.054054054054056,6.908743619918823\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,252,20261111-1390825560-252,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.498498498498499,48.348348348348345,7.7693586349487305\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,253,20261111-1390825560-253,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.372372372372372,47.489489489489486,7.0312159061431885\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,254,20261111-1390825560-254,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.48048048048048,47.951951951951955,6.965228080749512\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,255,20261111-1390825560-255,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.597597597597598,48.372372372372375,7.078578233718872\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,256,20261111-1390825560-256,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.36036036036036,47.66366366366366,6.905628681182861\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,257,20261111-1390825560-257,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.42042042042042,47.74174174174174,7.53029465675354\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,258,20261111-1390825560-258,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.528528528528529,48.186186186186184,7.531669855117798\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,259,20261111-1390825560-259,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.615615615615615,47.513513513513516,7.607122898101807\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,260,20261111-1390825560-260,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.315315315315315,48.4024024024024,7.24930477142334\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,261,20261111-1390825560-261,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.387387387387387,48.52252252252252,7.052006244659424\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,262,20261111-1390825560-262,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.621621621621621,48.1021021021021,7.465281248092651\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,263,20261111-1390825560-263,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.285285285285285,48.82282282282282,7.351894378662109\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,264,20261111-1390825560-264,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.45045045045045,48.408408408408405,6.953459978103638\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,265,20261111-1390825560-265,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.573573573573574,48.132132132132135,7.295113563537598\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,266,20261111-1390825560-266,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.435435435435435,48.06006006006006,7.1725499629974365\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,267,20261111-1390825560-267,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.42942942942943,47.51951951951952,7.442487716674805\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,268,20261111-1390825560-268,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.198198198198199,48.510510510510514,7.064673185348511\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,269,20261111-1390825560-269,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.687687687687689,47.765765765765764,7.90583872795105\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,270,20261111-1390825560-270,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.378378378378379,47.87987987987988,7.689589500427246\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,271,20261111-1390825560-271,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.228228228228229,47.705705705705704,6.834711313247681\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,272,20261111-1390825560-272,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.396396396396396,49.34534534534534,7.548318147659302\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,273,20261111-1390825560-273,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.468468468468469,47.873873873873876,6.860379219055176\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,274,20261111-1390825560-274,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.495495495495495,47.813813813813816,8.121262073516846\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,275,20261111-1390825560-275,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.435435435435435,47.669669669669666,6.71100926399231\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,276,20261111-1390825560-276,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.483483483483484,47.78378378378378,6.831305742263794\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,277,20261111-1390825560-277,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.48948948948949,48.030030030030034,6.839932203292847\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,278,20261111-1390825560-278,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.348348348348349,48.030030030030034,7.378270149230957\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,279,20261111-1390825560-279,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.432432432432432,47.813813813813816,7.235960960388184\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,280,20261111-1390825560-280,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.165165165165165,47.4954954954955,7.164392709732056\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,281,20261111-1390825560-281,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.528528528528529,47.74174174174174,7.483544588088989\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,282,20261111-1390825560-282,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.561561561561561,47.48348348348348,7.090497016906738\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,283,20261111-1390825560-283,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.441441441441441,48.270270270270274,7.5738677978515625\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,284,20261111-1390825560-284,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.33933933933934,48.372372372372375,7.455154895782471\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,285,20261111-1390825560-285,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.57057057057057,47.66366366366366,7.793245077133179\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,286,20261111-1390825560-286,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.312312312312311,47.97597597597598,6.6291491985321045\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,287,20261111-1390825560-287,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.486486486486486,47.531531531531535,7.2945473194122314\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,288,20261111-1390825560-288,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.252252252252251,48.174174174174176,7.159029722213745\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,289,20261111-1390825560-289,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.36936936936937,47.789789789789786,6.93312668800354\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,290,20261111-1390825560-290,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.48948948948949,48.192192192192195,7.819456100463867\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,291,20261111-1390825560-291,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.273273273273274,48.37837837837838,7.385768890380859\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,292,20261111-1390825560-292,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.645645645645645,47.77777777777778,6.70870304107666\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,293,20261111-1390825560-293,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.18018018018018,48.108108108108105,6.548493146896362\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,294,20261111-1390825560-294,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,332,0,1,1.0,9.183183183183184,47.93993993993994,8.116268396377563\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,295,20261111-1390825560-295,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.474474474474475,47.41741741741742,6.7606987953186035\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,296,20261111-1390825560-296,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.576576576576576,47.87987987987988,7.508915901184082\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,297,20261111-1390825560-297,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.582582582582583,48.03603603603604,7.070380687713623\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,298,20261111-1390825560-298,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.72072072072072,48.114114114114116,7.771981954574585\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,299,20261111-1390825560-299,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,333,0,0,0.0,9.666666666666666,47.987987987987985,7.260838985443115\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g5,300,20261111-1390825560-300,B,noisy,5,0.0003,0.0,3,5,5,5,5,5,2,333,331,0,2,2.0,9.207207207207206,47.61561561561562,7.056676626205444\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,0,20261111-1435210817-0,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.681681681681682,47.927927927927925,7.163832187652588\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,1,20261111-1435210817-1,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.960960960960961,48.306306306306304,7.457102060317993\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,2,20261111-1435210817-2,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.813813813813814,47.591591591591595,8.177445411682129\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,3,20261111-1435210817-3,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.11111111111111,47.609609609609606,7.109242677688599\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,4,20261111-1435210817-4,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.936936936936937,48.13813813813814,5.831148147583008\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,5,20261111-1435210817-5,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.963963963963964,48.51651651651652,5.646321535110474\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,6,20261111-1435210817-6,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.018018018018019,48.38438438438438,6.8905229568481445\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,7,20261111-1435210817-7,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.918918918918919,47.96396396396396,5.814542293548584\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,8,20261111-1435210817-8,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.921921921921922,47.753753753753756,6.3390467166900635\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,9,20261111-1435210817-9,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.801801801801802,48.64264264264264,6.151311635971069\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,10,20261111-1435210817-10,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,1,0,0.5,8.03003003003003,48.08408408408408,6.625946283340454\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,11,20261111-1435210817-11,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.912912912912913,47.92192192192192,6.339022159576416\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,12,20261111-1435210817-12,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.936936936936937,47.63963963963964,6.486378908157349\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,13,20261111-1435210817-13,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.8108108108108105,47.825825825825824,6.205747127532959\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,14,20261111-1435210817-14,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.00900900900901,47.46546546546546,6.444221496582031\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,15,20261111-1435210817-15,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.96996996996997,47.729729729729726,6.608302593231201\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,16,20261111-1435210817-16,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.7057057057057055,47.21921921921922,5.96050238609314\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,17,20261111-1435210817-17,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.78978978978979,47.42342342342342,6.380524396896362\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,18,20261111-1435210817-18,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.816816816816817,47.891891891891895,6.049774646759033\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,19,20261111-1435210817-19,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.681681681681682,47.891891891891895,6.364757061004639\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,20,20261111-1435210817-20,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.72972972972973,48.25825825825826,5.794909715652466\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,21,20261111-1435210817-21,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.12012012012012,48.768768768768766,6.4131646156311035\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,22,20261111-1435210817-22,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.861861861861862,48.066066066066064,6.020157814025879\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,23,20261111-1435210817-23,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.882882882882883,48.24024024024024,6.900030612945557\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,24,20261111-1435210817-24,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.774774774774775,47.87987987987988,5.978950500488281\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,25,20261111-1435210817-25,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.078078078078079,47.77777777777778,5.854738473892212\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,26,20261111-1435210817-26,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.06006006006006,47.387387387387385,6.166884183883667\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,27,20261111-1435210817-27,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.867867867867868,47.8978978978979,5.9138524532318115\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,28,20261111-1435210817-28,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.003003003003004,47.489489489489486,5.876018285751343\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,29,20261111-1435210817-29,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.756756756756757,47.6036036036036,5.760543346405029\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,30,20261111-1435210817-30,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.894894894894895,48.288288288288285,6.011399984359741\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,31,20261111-1435210817-31,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.09009009009009,48.246246246246244,5.853395938873291\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,32,20261111-1435210817-32,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.84984984984985,47.933933933933936,6.367605686187744\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,33,20261111-1435210817-33,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.063063063063064,48.28228228228228,6.061662435531616\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,34,20261111-1435210817-34,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.138138138138139,48.14414414414414,7.1153411865234375\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,35,20261111-1435210817-35,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,7.948948948948949,47.63963963963964,5.759721040725708\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,36,20261111-1435210817-36,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.918918918918919,48.3003003003003,6.091151237487793\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,37,20261111-1435210817-37,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.966966966966967,47.7957957957958,5.752254009246826\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,38,20261111-1435210817-38,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.303303303303304,48.95495495495496,6.741619348526001\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,39,20261111-1435210817-39,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.774774774774775,48.030030030030034,6.797725200653076\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,40,20261111-1435210817-40,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.054054054054054,47.95795795795796,6.473047971725464\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,41,20261111-1435210817-41,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.963963963963964,48.168168168168165,6.431877613067627\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,42,20261111-1435210817-42,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.7057057057057055,47.9039039039039,6.848066806793213\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,43,20261111-1435210817-43,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.861861861861862,47.92192192192192,6.25981593132019\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,44,20261111-1435210817-44,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.873873873873874,49.11711711711712,6.887166500091553\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,45,20261111-1435210817-45,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.093093093093094,48.186186186186184,6.0821692943573\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,46,20261111-1435210817-46,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.0,48.426426426426424,6.037739515304565\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,47,20261111-1435210817-47,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.8558558558558556,48.65465465465466,6.0877366065979\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,48,20261111-1435210817-48,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.675675675675675,47.68168168168168,6.251255989074707\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,49,20261111-1435210817-49,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.093093093093094,48.16216216216216,6.165472984313965\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,50,20261111-1435210817-50,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.903903903903904,48.3003003003003,6.200128555297852\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,51,20261111-1435210817-51,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.741741741741742,47.885885885885884,6.2223005294799805\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,52,20261111-1435210817-52,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.936936936936937,48.45645645645646,6.9836814403533936\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,53,20261111-1435210817-53,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.09009009009009,47.471471471471475,6.244770526885986\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,54,20261111-1435210817-54,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.147147147147146,47.651651651651655,6.008347511291504\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,55,20261111-1435210817-55,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.723723723723723,48.090090090090094,6.029672384262085\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,56,20261111-1435210817-56,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.870870870870871,47.813813813813816,6.26821756362915\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,57,20261111-1435210817-57,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.96996996996997,48.25825825825826,6.993690729141235\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,58,20261111-1435210817-58,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,1,0,0.5,8.123123123123124,48.414414414414416,5.94573187828064\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,59,20261111-1435210817-59,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,7.945945945945946,48.294294294294296,5.986470460891724\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,60,20261111-1435210817-60,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.03903903903904,48.048048048048045,6.1684675216674805\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,61,20261111-1435210817-61,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.087087087087086,48.030030030030034,6.090419292449951\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,62,20261111-1435210817-62,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.714714714714715,47.32132132132132,5.9113929271698\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,63,20261111-1435210817-63,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.957957957957958,47.5975975975976,6.198912620544434\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,64,20261111-1435210817-64,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.03003003003003,47.7957957957958,6.023622751235962\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,65,20261111-1435210817-65,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,7.792792792792793,48.1981981981982,6.543131113052368\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,66,20261111-1435210817-66,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.894894894894895,47.393393393393396,6.304426670074463\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,67,20261111-1435210817-67,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.915915915915916,48.0,6.427325010299683\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,68,20261111-1435210817-68,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.936936936936937,48.006006006006004,6.217164039611816\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,69,20261111-1435210817-69,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.918918918918919,47.97597597597598,6.100924015045166\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,70,20261111-1435210817-70,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.726726726726727,48.54054054054054,5.8799920082092285\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,71,20261111-1435210817-71,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.228228228228229,47.18318318318318,5.654138803482056\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,72,20261111-1435210817-72,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.894894894894895,47.633633633633636,6.126310110092163\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,73,20261111-1435210817-73,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,1,0,0.5,7.945945945945946,47.91591591591592,6.421196937561035\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,74,20261111-1435210817-74,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.870870870870871,48.18018018018018,5.984277009963989\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,75,20261111-1435210817-75,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.684684684684685,47.129129129129126,6.075861692428589\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,76,20261111-1435210817-76,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.915915915915916,48.192192192192195,5.278474807739258\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,77,20261111-1435210817-77,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.642642642642643,47.609609609609606,5.784836292266846\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,78,20261111-1435210817-78,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.042042042042041,47.993993993993996,5.690584421157837\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,79,20261111-1435210817-79,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.942942942942943,47.873873873873876,6.311928033828735\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,80,20261111-1435210817-80,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.066066066066066,47.1951951951952,5.907764196395874\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,81,20261111-1435210817-81,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.105105105105105,47.8978978978979,5.463308572769165\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,82,20261111-1435210817-82,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.078078078078079,48.82282282282282,6.570663690567017\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,83,20261111-1435210817-83,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.801801801801802,47.74174174174174,6.761851072311401\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,84,20261111-1435210817-84,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.8108108108108105,48.132132132132135,5.9387736320495605\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,85,20261111-1435210817-85,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.873873873873874,47.249249249249246,5.670055866241455\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,86,20261111-1435210817-86,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.05105105105105,48.52252252252252,6.184868574142456\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,87,20261111-1435210817-87,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.972972972972973,47.45945945945946,5.932926177978516\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,88,20261111-1435210817-88,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.930930930930931,47.729729729729726,5.925831317901611\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,89,20261111-1435210817-89,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.063063063063064,47.85585585585586,6.905566692352295\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,90,20261111-1435210817-90,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,7.7657657657657655,47.825825825825824,6.0309975147247314\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,91,20261111-1435210817-91,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.546546546546547,48.006006006006004,5.474515914916992\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,92,20261111-1435210817-92,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.7207207207207205,48.114114114114116,5.513630628585815\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,93,20261111-1435210817-93,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.084084084084084,47.41741741741742,6.800613880157471\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,94,20261111-1435210817-94,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.02102102102102,47.98198198198198,6.120821714401245\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,95,20261111-1435210817-95,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.17117117117117,47.753753753753756,5.680523157119751\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,96,20261111-1435210817-96,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.114114114114114,47.885885885885884,5.727141618728638\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,97,20261111-1435210817-97,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.870870870870871,47.729729729729726,6.6848695278167725\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,98,20261111-1435210817-98,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.876876876876877,47.56156156156156,6.420476913452148\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,99,20261111-1435210817-99,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.621621621621622,48.02402402402402,5.752001523971558\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,100,20261111-1435210817-100,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.963963963963964,48.36036036036036,6.130305051803589\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,101,20261111-1435210817-101,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.708708708708708,47.969969969969966,7.09449028968811\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,102,20261111-1435210817-102,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.846846846846847,47.969969969969966,5.653120994567871\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,103,20261111-1435210817-103,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.165165165165165,47.93993993993994,6.200519800186157\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,104,20261111-1435210817-104,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.042042042042041,47.86186186186186,7.149380445480347\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,105,20261111-1435210817-105,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.981981981981982,48.44444444444444,5.431758642196655\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,106,20261111-1435210817-106,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.822822822822823,47.987987987987985,6.9645349979400635\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,107,20261111-1435210817-107,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.08108108108108,48.18018018018018,6.747343301773071\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,108,20261111-1435210817-108,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.864864864864865,47.693693693693696,5.6918861865997314\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,109,20261111-1435210817-109,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.12912912912913,48.03603603603604,6.387407064437866\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,110,20261111-1435210817-110,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.012012012012011,47.885885885885884,6.873795509338379\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,111,20261111-1435210817-111,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.924924924924925,47.933933933933936,6.613170862197876\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,112,20261111-1435210817-112,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.915915915915916,47.789789789789786,5.707283973693848\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,113,20261111-1435210817-113,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.948948948948949,48.88288288288288,5.75362491607666\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,114,20261111-1435210817-114,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.156156156156156,47.507507507507505,5.508510589599609\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,115,20261111-1435210817-115,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.786786786786787,47.891891891891895,6.480304718017578\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,116,20261111-1435210817-116,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.15015015015015,48.26426426426426,6.535129070281982\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,117,20261111-1435210817-117,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.747747747747748,47.87987987987988,5.663823366165161\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,118,20261111-1435210817-118,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.771771771771772,48.08408408408408,6.046660661697388\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,119,20261111-1435210817-119,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.990990990990991,48.306306306306304,6.300623893737793\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,120,20261111-1435210817-120,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.972972972972973,48.43843843843844,6.338923454284668\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,121,20261111-1435210817-121,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.036036036036036,47.729729729729726,5.997659683227539\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,122,20261111-1435210817-122,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.066066066066066,47.21921921921922,6.032977342605591\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,123,20261111-1435210817-123,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.696696696696697,48.13813813813814,5.783832788467407\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,124,20261111-1435210817-124,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.84984984984985,48.28228228228228,5.867901086807251\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,125,20261111-1435210817-125,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.873873873873874,48.210210210210214,5.959468841552734\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,126,20261111-1435210817-126,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.90990990990991,47.831831831831835,6.668876886367798\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,127,20261111-1435210817-127,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.930930930930931,47.87987987987988,6.024033546447754\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,128,20261111-1435210817-128,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.954954954954955,48.168168168168165,6.500552415847778\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,129,20261111-1435210817-129,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.05105105105105,48.07807807807808,5.4298882484436035\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,130,20261111-1435210817-130,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.03903903903904,48.414414414414416,6.534560918807983\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,131,20261111-1435210817-131,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.054054054054054,48.28228228228228,6.600236654281616\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,132,20261111-1435210817-132,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.84984984984985,47.74174174174174,6.381595849990845\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,133,20261111-1435210817-133,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.897897897897898,47.38138138138138,5.59025239944458\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,134,20261111-1435210817-134,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.948948948948949,48.03603603603604,5.866800308227539\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,135,20261111-1435210817-135,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.7807807807807805,47.45945945945946,6.110360145568848\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,136,20261111-1435210817-136,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.837837837837838,48.4984984984985,5.801138639450073\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,137,20261111-1435210817-137,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,7.936936936936937,47.71771771771772,5.7994067668914795\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,138,20261111-1435210817-138,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.042042042042041,47.92192192192192,5.680469989776611\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,139,20261111-1435210817-139,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,8.027027027027026,48.114114114114116,5.839953899383545\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,140,20261111-1435210817-140,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.12012012012012,48.33633633633634,6.969447135925293\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,141,20261111-1435210817-141,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.813813813813814,48.22222222222222,5.9130332469940186\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,142,20261111-1435210817-142,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.777777777777778,48.030030030030034,6.355084419250488\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,143,20261111-1435210817-143,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.876876876876877,47.789789789789786,6.01257061958313\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,144,20261111-1435210817-144,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.831831831831832,48.92492492492492,5.643906354904175\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,145,20261111-1435210817-145,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.987987987987988,48.13813813813814,6.723002672195435\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,146,20261111-1435210817-146,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.975975975975976,48.07807807807808,6.315439701080322\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,147,20261111-1435210817-147,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.954954954954955,47.951951951951955,6.667566299438477\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,148,20261111-1435210817-148,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.858858858858859,47.849849849849846,5.975733757019043\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,149,20261111-1435210817-149,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,8.00900900900901,48.414414414414416,6.336838006973267\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,150,20261111-1435210817-150,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.726726726726727,48.02402402402402,5.859517335891724\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,151,20261111-1435210817-151,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.768768768768769,48.38438438438438,5.288894414901733\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,152,20261111-1435210817-152,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.711711711711712,47.91591591591592,5.906203508377075\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,153,20261111-1435210817-153,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.057057057057056,47.93993993993994,6.9325525760650635\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,154,20261111-1435210817-154,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.12912912912913,48.38438438438438,5.610744953155518\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,155,20261111-1435210817-155,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.726726726726727,47.867867867867865,6.0003886222839355\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,156,20261111-1435210817-156,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.768768768768769,48.0,6.013848543167114\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,157,20261111-1435210817-157,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.777777777777778,48.048048048048045,6.282328128814697\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,158,20261111-1435210817-158,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.165165165165165,47.705705705705704,6.004547119140625\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,159,20261111-1435210817-159,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.870870870870871,48.552552552552555,6.157250881195068\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,160,20261111-1435210817-160,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.921921921921922,47.73573573573574,6.076684236526489\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,161,20261111-1435210817-161,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.7807807807807805,47.57957957957958,5.58656120300293\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,162,20261111-1435210817-162,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.867867867867868,47.651651651651655,6.679186820983887\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,163,20261111-1435210817-163,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.990990990990991,47.72372372372372,5.490961790084839\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,164,20261111-1435210817-164,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.243243243243244,48.606606606606604,6.398802995681763\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,165,20261111-1435210817-165,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.924924924924925,48.5945945945946,5.6756837368011475\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,166,20261111-1435210817-166,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.912912912912913,47.873873873873876,5.461431503295898\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,167,20261111-1435210817-167,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.951951951951952,48.51651651651652,6.178370237350464\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,168,20261111-1435210817-168,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.8558558558558556,47.92192192192192,6.301705837249756\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,169,20261111-1435210817-169,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.126126126126126,48.072072072072075,6.718021869659424\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,170,20261111-1435210817-170,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,7.903903903903904,48.33633633633634,5.962810516357422\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,171,20261111-1435210817-171,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.63963963963964,47.81981981981982,6.651499032974243\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,172,20261111-1435210817-172,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.921921921921922,48.348348348348345,6.630477666854858\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,173,20261111-1435210817-173,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.882882882882883,47.387387387387385,5.733321905136108\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,174,20261111-1435210817-174,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.63063063063063,48.42042042042042,5.9660539627075195\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,175,20261111-1435210817-175,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.756756756756757,48.44444444444444,6.614933252334595\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,176,20261111-1435210817-176,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.858858858858859,48.150150150150154,6.755162715911865\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,177,20261111-1435210817-177,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.921921921921922,48.270270270270274,6.1439597606658936\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,178,20261111-1435210817-178,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.951951951951952,48.62462462462462,5.903035640716553\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,179,20261111-1435210817-179,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.888888888888889,47.351351351351354,6.048346519470215\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,180,20261111-1435210817-180,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.0990990990991,47.55555555555556,5.4682512283325195\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,181,20261111-1435210817-181,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.078078078078079,48.25825825825826,6.577225923538208\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,182,20261111-1435210817-182,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,8.198198198198199,48.42042042042042,5.829090118408203\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,183,20261111-1435210817-183,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.861861861861862,48.486486486486484,5.747101068496704\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,184,20261111-1435210817-184,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.870870870870871,47.8978978978979,5.919707536697388\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,185,20261111-1435210817-185,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.873873873873874,48.12012012012012,6.28936767578125\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,186,20261111-1435210817-186,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.903903903903904,47.67567567567568,5.793895959854126\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,187,20261111-1435210817-187,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.87987987987988,48.0,5.853530168533325\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,188,20261111-1435210817-188,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.960960960960961,48.054054054054056,6.193209648132324\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,189,20261111-1435210817-189,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.900900900900901,48.294294294294296,6.043206214904785\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,190,20261111-1435210817-190,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.984984984984985,47.73573573573574,5.809067010879517\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,191,20261111-1435210817-191,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.900900900900901,48.25825825825826,6.7691192626953125\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,192,20261111-1435210817-192,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.927927927927928,47.32132132132132,5.944222927093506\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,193,20261111-1435210817-193,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.888888888888889,47.411411411411414,6.157660484313965\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,194,20261111-1435210817-194,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.867867867867868,47.87987987987988,5.6530444622039795\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,195,20261111-1435210817-195,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.012012012012011,48.04204204204204,5.820996999740601\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,196,20261111-1435210817-196,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.753753753753754,48.006006006006004,6.236378192901611\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,197,20261111-1435210817-197,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.693693693693693,48.51651651651652,5.778885126113892\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,198,20261111-1435210817-198,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.81981981981982,48.02402402402402,5.693967342376709\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,199,20261111-1435210817-199,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.02102102102102,47.42342342342342,6.066854000091553\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,200,20261111-1435210817-200,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.066066066066066,47.813813813813816,5.683706521987915\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,201,20261111-1435210817-201,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,1,0,0.5,7.8408408408408405,48.246246246246244,6.195084095001221\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,202,20261111-1435210817-202,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.978978978978979,48.4024024024024,6.231878042221069\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,203,20261111-1435210817-203,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.903903903903904,47.747747747747745,6.42156982421875\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,204,20261111-1435210817-204,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.243243243243244,48.114114114114116,6.532282114028931\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,205,20261111-1435210817-205,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.84984984984985,48.75675675675676,5.975554704666138\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,206,20261111-1435210817-206,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.963963963963964,47.789789789789786,6.034339904785156\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,207,20261111-1435210817-207,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.963963963963964,48.090090090090094,7.168707847595215\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,208,20261111-1435210817-208,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.14114114114114,47.969969969969966,5.561611175537109\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,209,20261111-1435210817-209,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.7807807807807805,47.369369369369366,6.496201276779175\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,210,20261111-1435210817-210,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.834834834834835,47.729729729729726,5.63167405128479\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,211,20261111-1435210817-211,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.891891891891892,48.62462462462462,6.153656482696533\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,212,20261111-1435210817-212,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.615615615615615,47.525525525525524,6.75047755241394\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,213,20261111-1435210817-213,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.156156156156156,48.22222222222222,5.652186632156372\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,214,20261111-1435210817-214,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,8.2012012012012,47.831831831831835,6.732041120529175\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,215,20261111-1435210817-215,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.885885885885886,48.07807807807808,6.004211187362671\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,216,20261111-1435210817-216,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.7957957957957955,47.95795795795796,5.82837986946106\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,217,20261111-1435210817-217,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.02102102102102,47.267267267267265,6.794722080230713\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,218,20261111-1435210817-218,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.714714714714715,47.945945945945944,6.943414926528931\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,219,20261111-1435210817-219,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.23123123123123,48.192192192192195,5.890117645263672\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,220,20261111-1435210817-220,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.7957957957957955,47.747747747747745,5.349165201187134\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,221,20261111-1435210817-221,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.915915915915916,47.549549549549546,6.452896595001221\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,222,20261111-1435210817-222,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.8558558558558556,47.369369369369366,6.811060905456543\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,223,20261111-1435210817-223,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.837837837837838,47.18318318318318,6.12285590171814\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,224,20261111-1435210817-224,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.06906906906907,47.669669669669666,5.319133043289185\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,225,20261111-1435210817-225,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.963963963963964,47.873873873873876,5.9544336795806885\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,226,20261111-1435210817-226,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.783783783783784,47.393393393393396,6.314759969711304\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,227,20261111-1435210817-227,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.681681681681682,48.528528528528525,5.907843351364136\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,228,20261111-1435210817-228,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.930930930930931,48.38438438438438,6.8420939445495605\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,229,20261111-1435210817-229,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.93993993993994,47.73573573573574,6.463092803955078\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,230,20261111-1435210817-230,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.993993993993994,47.669669669669666,5.808816432952881\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,231,20261111-1435210817-231,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.09009009009009,47.73573573573574,5.71403431892395\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,232,20261111-1435210817-232,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.8258258258258255,48.468468468468465,6.890076398849487\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,233,20261111-1435210817-233,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.846846846846847,47.63963963963964,6.456515550613403\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,234,20261111-1435210817-234,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.903903903903904,48.58258258258258,6.114256381988525\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,235,20261111-1435210817-235,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.8258258258258255,47.705705705705704,5.733224153518677\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,236,20261111-1435210817-236,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.828828828828829,47.74174174174174,6.061519384384155\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,237,20261111-1435210817-237,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.990990990990991,48.4984984984985,5.9490437507629395\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,238,20261111-1435210817-238,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.954954954954955,48.234234234234236,6.162338495254517\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,239,20261111-1435210817-239,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.012012012012011,48.01801801801802,5.563582181930542\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,240,20261111-1435210817-240,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.60960960960961,47.393393393393396,6.124359369277954\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,241,20261111-1435210817-241,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.7957957957957955,47.57957957957958,6.204588174819946\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,242,20261111-1435210817-242,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.5675675675675675,47.549549549549546,5.507436037063599\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,243,20261111-1435210817-243,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.948948948948949,48.408408408408405,5.659980058670044\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,244,20261111-1435210817-244,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.921921921921922,47.813813813813816,6.620581150054932\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,245,20261111-1435210817-245,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.933933933933934,48.36036036036036,6.461511850357056\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,246,20261111-1435210817-246,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.297297297297296,47.591591591591595,5.804655313491821\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,247,20261111-1435210817-247,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,1,1,1.5,7.636636636636637,48.612612612612615,6.314169645309448\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,248,20261111-1435210817-248,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.237237237237236,48.432432432432435,6.809608697891235\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,249,20261111-1435210817-249,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.873873873873874,47.78378378378378,5.859004497528076\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,250,20261111-1435210817-250,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.624624624624625,48.3963963963964,6.305419921875\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,251,20261111-1435210817-251,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,7.837837837837838,48.1021021021021,6.215040922164917\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,252,20261111-1435210817-252,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.813813813813814,47.813813813813816,6.010951995849609\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,253,20261111-1435210817-253,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.960960960960961,47.95795795795796,5.612712621688843\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,254,20261111-1435210817-254,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,7.615615615615615,47.87987987987988,6.5081751346588135\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,255,20261111-1435210817-255,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.966966966966967,48.072072072072075,6.400984287261963\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,256,20261111-1435210817-256,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.027027027027026,47.693693693693696,6.222062110900879\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,257,20261111-1435210817-257,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.93993993993994,47.91591591591592,6.03780460357666\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,258,20261111-1435210817-258,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.0,48.192192192192195,6.471143007278442\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,259,20261111-1435210817-259,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.0,48.13813813813814,6.335650682449341\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,260,20261111-1435210817-260,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.852852852852853,47.729729729729726,5.945566415786743\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,261,20261111-1435210817-261,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.924924924924925,47.92192192192192,6.262528657913208\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,262,20261111-1435210817-262,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.7357357357357355,48.06006006006006,6.187853813171387\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,263,20261111-1435210817-263,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.32132132132132,48.234234234234236,6.177850008010864\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,264,20261111-1435210817-264,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.003003003003004,48.048048048048045,6.4112465381622314\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,265,20261111-1435210817-265,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.684684684684685,48.234234234234236,5.874013662338257\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,266,20261111-1435210817-266,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.867867867867868,48.1981981981982,6.888898134231567\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,267,20261111-1435210817-267,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,1,1,1.5,7.837837837837838,48.012012012012015,7.007516384124756\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,268,20261111-1435210817-268,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.912912912912913,48.08408408408408,6.58802056312561\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,269,20261111-1435210817-269,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.981981981981982,47.72372372372372,6.4700987339019775\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,270,20261111-1435210817-270,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,0,2,2.0,8.0990990990991,48.53453453453454,6.079029560089111\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,271,20261111-1435210817-271,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.903903903903904,47.993993993993996,6.156412839889526\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,272,20261111-1435210817-272,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.831831831831832,48.228228228228225,6.114177942276001\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,273,20261111-1435210817-273,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,8.015015015015015,48.732732732732735,6.531035661697388\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,274,20261111-1435210817-274,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,1,0,0.5,7.792792792792793,47.765765765765764,6.3445446491241455\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,275,20261111-1435210817-275,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.807807807807808,47.33933933933934,6.3161773681640625\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,276,20261111-1435210817-276,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.012012012012011,48.13813813813814,5.90293288230896\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,277,20261111-1435210817-277,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.861861861861862,48.21621621621622,6.10105299949646\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,278,20261111-1435210817-278,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.837837837837838,47.38138138138138,6.2436113357543945\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,279,20261111-1435210817-279,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.984984984984985,47.92192192192192,6.496246814727783\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,280,20261111-1435210817-280,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.753753753753754,48.21621621621622,6.044466733932495\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,281,20261111-1435210817-281,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.978978978978979,48.270270270270274,6.970586776733398\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,282,20261111-1435210817-282,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.003003003003004,47.46546546546546,6.561285018920898\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,283,20261111-1435210817-283,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.945945945945946,48.14414414414414,6.28628945350647\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,284,20261111-1435210817-284,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.957957957957958,48.306306306306304,5.62499213218689\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,285,20261111-1435210817-285,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.252252252252251,48.34234234234234,6.310560941696167\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,286,20261111-1435210817-286,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.954954954954955,48.390390390390394,6.501194953918457\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,287,20261111-1435210817-287,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.684684684684685,47.83783783783784,6.759967565536499\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,288,20261111-1435210817-288,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.972972972972973,47.6996996996997,6.026196002960205\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,289,20261111-1435210817-289,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,8.243243243243244,47.927927927927925,5.8993988037109375\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,290,20261111-1435210817-290,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,331,1,1,1.5,7.837837837837838,47.453453453453456,6.785837411880493\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,291,20261111-1435210817-291,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.771771771771772,47.65765765765766,5.735947132110596\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,292,20261111-1435210817-292,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.930930930930931,47.993993993993996,6.098903179168701\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,293,20261111-1435210817-293,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.951951951951952,48.552552552552555,5.687715291976929\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,294,20261111-1435210817-294,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.882882882882883,47.765765765765764,6.563294410705566\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,295,20261111-1435210817-295,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.7957957957957955,48.13813813813814,6.470998287200928\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,296,20261111-1435210817-296,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.903903903903904,48.510510510510514,6.3454272747039795\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,297,20261111-1435210817-297,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,0,1,1.0,7.867867867867868,47.78378378378378,6.398402690887451\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,298,20261111-1435210817-298,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,1,0,0.5,7.828828828828829,47.73573573573574,6.722909212112427\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,299,20261111-1435210817-299,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,332,1,0,0.5,7.672672672672673,47.4954954954955,6.705659866333008\npilot-20261110T165235Z-e8cfd1,B|noisy|d5|p0.0003|K3|g1,300,20261111-1435210817-300,B,noisy,5,0.0003,0.0,3,1,5,5,1,5,2,333,333,0,0,0.0,7.915915915915916,47.987987987987985,6.410538911819458\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,0,20261111-145983582-0,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,0,2,2.0,33.356,35.664,10.807128667831421\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,1,20261111-145983582-1,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.576,35.272,11.08299994468689\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,2,20261111-145983582-2,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,1,1,1.5,33.044,35.92,10.619194746017456\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,3,20261111-145983582-3,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,34.256,35.984,10.760090827941895\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,4,20261111-145983582-4,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.92,36.232,9.428962230682373\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,5,20261111-145983582-5,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.544,35.984,8.883015871047974\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,6,20261111-145983582-6,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,34.216,36.416,9.561334133148193\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,7,20261111-145983582-7,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.504,36.24,8.153175592422485\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,8,20261111-145983582-8,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.664,36.008,9.25232219696045\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,9,20261111-145983582-9,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.36,35.872,10.110263109207153\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,10,20261111-145983582-10,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.912,36.128,8.182586908340454\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,11,20261111-145983582-11,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.84,36.824,8.219094514846802\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,12,20261111-145983582-12,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,0,2,2.0,33.992,35.84,9.378405094146729\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,13,20261111-145983582-13,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.184,36.216,8.810002088546753\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,14,20261111-145983582-14,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.912,36.192,9.405403852462769\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,15,20261111-145983582-15,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.556,36.216,9.12093734741211\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,16,20261111-145983582-16,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.48,35.664,9.31459927558899\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,17,20261111-145983582-17,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.608,36.584,8.73509168624878\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,18,20261111-145983582-18,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.632,35.808,10.158956289291382\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,19,20261111-145983582-19,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.844,35.936,9.251139879226685\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,20,20261111-145983582-20,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,34.004,35.968,8.607017517089844\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,21,20261111-145983582-21,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.72,35.76,8.793516159057617\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,22,20261111-145983582-22,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.448,35.408,8.89590334892273\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,23,20261111-145983582-23,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,34.284,36.104,9.700596809387207\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,24,20261111-145983582-24,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.54,36.112,8.533596515655518\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,25,20261111-145983582-25,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.612,35.712,9.069790840148926\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,26,20261111-145983582-26,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.44,35.968,8.708341121673584\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,27,20261111-145983582-27,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,33.472,35.328,9.455133199691772\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,28,20261111-145983582-28,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,2,0,1.0,33.592,36.144,8.023256063461304\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,29,20261111-145983582-29,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.528,36.104,8.391974449157715\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,30,20261111-145983582-30,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.192,35.568,9.187777042388916\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,31,20261111-145983582-31,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,0,2,2.0,33.932,36.28,9.709742784500122\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,32,20261111-145983582-32,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,0,2,2.0,33.856,36.296,8.22073483467102\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,33,20261111-145983582-33,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,0,2,2.0,33.472,36.088,9.459059476852417\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,34,20261111-145983582-34,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,34.056,35.952,8.867882013320923\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,35,20261111-145983582-35,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.836,36.752,8.498610734939575\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,36,20261111-145983582-36,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,0,2,2.0,33.336,35.152,9.056784868240356\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,37,20261111-145983582-37,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,248,0,2,2.0,34.364,35.344,8.636391639709473\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,38,20261111-145983582-38,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,250,0,0,0.0,34.536,36.184,8.448586463928223\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K1|g0,39,20261111-145983582-39,B,noisy,7,0.001,0.0,1,0,7,7,7,7,2,250,249,0,1,1.0,33.452,36.272,8.573723077774048\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,0,20261111-2221060627-0,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,59.488,71.856,12.396960735321045\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,1,20261111-2221060627-1,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,1,1,1.5,58.536,72.848,11.079006671905518\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,2,20261111-2221060627-2,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,1,0,0.5,58.304,72.624,12.26176929473877\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,3,20261111-2221060627-3,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,59.672,71.248,10.82428240776062\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,4,20261111-2221060627-4,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,57.504,72.608,7.027613401412964\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,5,20261111-2221060627-5,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.904,72.4,8.232376098632812\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,6,20261111-2221060627-6,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,59.176,71.408,7.610985040664673\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,7,20261111-2221060627-7,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,59.168,72.304,8.281522989273071\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,8,20261111-2221060627-8,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,57.792,72.432,7.649503946304321\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,9,20261111-2221060627-9,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.0,71.568,8.270527124404907\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,10,20261111-2221060627-10,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.336,72.528,8.570039987564087\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,11,20261111-2221060627-11,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,59.136,70.208,8.102689981460571\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,12,20261111-2221060627-12,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,59.216,72.864,6.907424688339233\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,13,20261111-2221060627-13,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.288,72.624,8.018204927444458\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,14,20261111-2221060627-14,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,122,0,3,3.0,57.504,72.016,8.785351991653442\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,15,20261111-2221060627-15,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,58.24,71.696,8.647518396377563\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,16,20261111-2221060627-16,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,57.768,70.224,8.43190622329712\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,17,20261111-2221060627-17,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,59.128,70.16,7.7277631759643555\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,18,20261111-2221060627-18,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,59.088,72.848,8.15854549407959\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,19,20261111-2221060627-19,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.736,71.12,7.847981929779053\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,20,20261111-2221060627-20,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.632,72.512,8.54573941230774\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,21,20261111-2221060627-21,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.216,73.392,8.254563331604004\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,22,20261111-2221060627-22,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,57.704,72.16,8.46498155593872\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,23,20261111-2221060627-23,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,58.52,71.056,7.790431499481201\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,24,20261111-2221060627-24,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,1,1,1.5,58.592,72.192,8.460160732269287\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,25,20261111-2221060627-25,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.688,72.368,7.4884419441223145\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,26,20261111-2221060627-26,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.88,71.12,9.291227340698242\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,27,20261111-2221060627-27,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,59.152,72.112,7.7909629344940186\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,28,20261111-2221060627-28,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.848,72.464,7.717857599258423\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,29,20261111-2221060627-29,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.176,72.272,7.95756983757019\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,30,20261111-2221060627-30,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,122,0,3,3.0,59.136,72.96,7.5876381397247314\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,31,20261111-2221060627-31,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,59.072,72.336,7.686314344406128\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,32,20261111-2221060627-32,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,57.816,72.304,8.134201526641846\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,33,20261111-2221060627-33,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,122,0,3,3.0,58.456,72.64,7.67363166809082\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,34,20261111-2221060627-34,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,58.88,71.856,9.009185075759888\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,35,20261111-2221060627-35,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,122,0,3,3.0,58.232,70.496,6.688402891159058\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,36,20261111-2221060627-36,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,59.616,72.912,8.771970748901367\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,37,20261111-2221060627-37,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,59.968,71.072,8.68322491645813\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,38,20261111-2221060627-38,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.592,71.792,7.2216362953186035\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,39,20261111-2221060627-39,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,59.192,71.296,7.4444286823272705\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,40,20261111-2221060627-40,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,57.4,71.648,8.87839674949646\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,41,20261111-2221060627-41,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.44,71.696,7.729538202285767\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,42,20261111-2221060627-42,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.088,71.232,8.528196573257446\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,43,20261111-2221060627-43,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.816,71.392,7.767406940460205\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,44,20261111-2221060627-44,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,57.472,71.92,7.829871892929077\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,45,20261111-2221060627-45,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.224,72.688,8.407607316970825\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,46,20261111-2221060627-46,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,57.208,71.888,8.301179647445679\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,47,20261111-2221060627-47,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.296,71.776,7.7845234870910645\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,48,20261111-2221060627-48,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.224,72.912,8.114281177520752\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,49,20261111-2221060627-49,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.176,72.496,8.90170693397522\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,50,20261111-2221060627-50,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,57.792,72.096,6.7499003410339355\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,51,20261111-2221060627-51,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,57.92,72.672,9.201512813568115\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,52,20261111-2221060627-52,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,59.272,72.368,8.631720781326294\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,53,20261111-2221060627-53,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,58.128,72.176,8.934592485427856\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,54,20261111-2221060627-54,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,122,1,2,2.5,57.56,72.128,7.34833288192749\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,55,20261111-2221060627-55,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,59.016,71.68,8.309859037399292\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,56,20261111-2221060627-56,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,1,0,0.5,59.144,71.728,8.626654386520386\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,57,20261111-2221060627-57,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,59.808,72.032,7.92455792427063\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,58,20261111-2221060627-58,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,57.664,72.016,8.156847953796387\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,59,20261111-2221060627-59,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.736,71.648,8.98040509223938\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,60,20261111-2221060627-60,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,122,0,3,3.0,57.912,70.704,7.407604217529297\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,61,20261111-2221060627-61,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.136,70.8,6.805827379226685\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,62,20261111-2221060627-62,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,1,1,1.5,59.032,72.192,8.175206661224365\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,63,20261111-2221060627-63,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.232,70.88,7.582279920578003\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,64,20261111-2221060627-64,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,59.056,72.544,6.632120132446289\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,65,20261111-2221060627-65,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,57.08,71.184,9.131330966949463\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,66,20261111-2221060627-66,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,57.992,71.44,7.103707313537598\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,67,20261111-2221060627-67,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,57.728,71.68,8.165410041809082\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,68,20261111-2221060627-68,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,121,0,4,4.0,58.696,72.592,6.954899549484253\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,69,20261111-2221060627-69,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.448,71.664,8.924198865890503\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,70,20261111-2221060627-70,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,60.528,72.064,9.397202730178833\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,71,20261111-2221060627-71,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,60.032,72.624,8.845330953598022\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,72,20261111-2221060627-72,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,60.44,70.496,7.359485149383545\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,73,20261111-2221060627-73,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,58.304,72.88,8.305870056152344\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,74,20261111-2221060627-74,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,58.456,71.872,9.390882730484009\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,75,20261111-2221060627-75,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,123,0,2,2.0,57.824,72.672,7.535872459411621\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,76,20261111-2221060627-76,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.264,71.376,7.55413556098938\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,77,20261111-2221060627-77,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,124,0,1,1.0,59.32,71.44,8.83694052696228\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,78,20261111-2221060627-78,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,58.368,72.464,7.542027950286865\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g7,79,20261111-2221060627-79,B,noisy,7,0.001,0.0,2,7,7,7,7,7,2,125,125,0,0,0.0,57.112,72.448,7.793015241622925\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,0,20261111-1828788006-0,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.536,72.416,10.639257192611694\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,1,20261111-1828788006-1,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.056,72.816,9.882039070129395\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,2,20261111-1828788006-2,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.8,71.008,9.322644710540771\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,3,20261111-1828788006-3,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.904,71.68,10.81910252571106\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,4,20261111-1828788006-4,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,121,1,3,3.5,51.168,70.48,7.5098114013671875\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,5,20261111-1828788006-5,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.952,71.68,5.887703895568848\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,6,20261111-1828788006-6,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.0,74.16,6.805276870727539\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,7,20261111-1828788006-7,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,52.064,72.384,7.865192413330078\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,8,20261111-1828788006-8,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,51.768,72.512,7.074888467788696\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,9,20261111-1828788006-9,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,122,0,3,3.0,52.352,71.984,6.836745500564575\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,10,20261111-1828788006-10,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.504,72.72,6.694904088973999\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,11,20261111-1828788006-11,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.8,71.824,6.562605857849121\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,12,20261111-1828788006-12,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.616,72.88,6.847451448440552\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,13,20261111-1828788006-13,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.28,70.992,7.327570915222168\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,14,20261111-1828788006-14,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,50.8,72.272,5.926545858383179\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,15,20261111-1828788006-15,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,50.608,71.568,6.036679744720459\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,16,20261111-1828788006-16,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,52.576,70.416,6.740655899047852\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,17,20261111-1828788006-17,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,50.376,72.448,6.903067350387573\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,18,20261111-1828788006-18,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.08,73.328,6.6938018798828125\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,19,20261111-1828788006-19,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.112,71.776,6.438526630401611\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,20,20261111-1828788006-20,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.12,72.656,6.470821380615234\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,21,20261111-1828788006-21,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,52.4,71.584,7.48861837387085\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,22,20261111-1828788006-22,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,52.176,71.584,7.07277774810791\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,23,20261111-1828788006-23,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.256,71.744,6.47100043296814\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,24,20261111-1828788006-24,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.088,72.0,6.505679607391357\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,25,20261111-1828788006-25,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,52.592,70.64,6.441573619842529\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,26,20261111-1828788006-26,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,122,1,2,2.5,51.016,72.352,6.094216585159302\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,27,20261111-1828788006-27,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,50.504,71.552,6.6987011432647705\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,28,20261111-1828788006-28,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,51.152,72.048,6.665577411651611\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,29,20261111-1828788006-29,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.616,71.392,6.677500486373901\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,30,20261111-1828788006-30,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.48,72.176,7.373351573944092\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,31,20261111-1828788006-31,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.984,72.384,5.7251667976379395\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,32,20261111-1828788006-32,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.168,70.288,7.29051947593689\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,33,20261111-1828788006-33,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,50.28,72.352,6.618893146514893\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,34,20261111-1828788006-34,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.216,71.584,7.293932914733887\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,35,20261111-1828788006-35,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,51.64,72.096,6.130577325820923\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,36,20261111-1828788006-36,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,52.048,71.28,6.039278745651245\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,37,20261111-1828788006-37,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,52.192,72.256,6.387213468551636\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,38,20261111-1828788006-38,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.752,71.168,6.971131801605225\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,39,20261111-1828788006-39,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,51.488,72.032,6.941193580627441\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,40,20261111-1828788006-40,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,122,0,3,3.0,51.192,71.712,6.155554294586182\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,41,20261111-1828788006-41,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.488,69.968,8.13609766960144\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,42,20261111-1828788006-42,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.64,72.688,6.251787185668945\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,43,20261111-1828788006-43,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,50.808,70.768,5.862555265426636\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,44,20261111-1828788006-44,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,50.856,72.544,6.9419121742248535\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,45,20261111-1828788006-45,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.192,71.648,8.076829671859741\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,46,20261111-1828788006-46,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,50.808,71.136,6.70021915435791\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,47,20261111-1828788006-47,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,52.104,72.496,6.915733814239502\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,48,20261111-1828788006-48,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,49.696,72.608,7.320234537124634\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,49,20261111-1828788006-49,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.944,71.296,6.832566499710083\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,50,20261111-1828788006-50,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,50.232,71.856,5.960986852645874\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,51,20261111-1828788006-51,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,50.248,71.76,7.158156633377075\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,52,20261111-1828788006-52,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.032,72.352,7.583962678909302\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,53,20261111-1828788006-53,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.096,72.832,6.600456953048706\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,54,20261111-1828788006-54,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,52.488,72.528,7.19256854057312\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,55,20261111-1828788006-55,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,50.688,71.648,6.669500112533569\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,56,20261111-1828788006-56,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,52.024,71.536,6.1160500049591064\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,57,20261111-1828788006-57,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.272,70.592,6.601440191268921\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,58,20261111-1828788006-58,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,50.792,72.176,7.365511178970337\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,59,20261111-1828788006-59,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,51.44,72.08,7.092514514923096\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,60,20261111-1828788006-60,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,50.472,71.408,6.2536163330078125\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,61,20261111-1828788006-61,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,50.888,70.72,5.9874022006988525\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,62,20261111-1828788006-62,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.864,71.936,6.706884860992432\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,63,20261111-1828788006-63,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.528,71.952,7.6277711391448975\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,64,20261111-1828788006-64,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,51.568,72.848,5.7686522006988525\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,65,20261111-1828788006-65,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.704,71.76,6.354208469390869\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,66,20261111-1828788006-66,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,122,0,3,3.0,50.728,70.944,7.273425579071045\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,67,20261111-1828788006-67,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.536,72.48,6.180383920669556\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,68,20261111-1828788006-68,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,51.808,71.312,6.582302093505859\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,69,20261111-1828788006-69,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,52.344,70.528,5.589423656463623\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,70,20261111-1828788006-70,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,50.72,72.048,6.148344993591309\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,71,20261111-1828788006-71,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,0,2,2.0,51.064,73.36,6.552459478378296\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,72,20261111-1828788006-72,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,52.208,71.952,7.647691011428833\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,73,20261111-1828788006-73,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,51.68,71.456,5.584348678588867\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,74,20261111-1828788006-74,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,0,1,1.0,51.328,73.872,6.143366575241089\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,75,20261111-1828788006-75,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,124,1,0,0.5,50.376,71.184,6.090883731842041\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,76,20261111-1828788006-76,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,50.264,72.64,6.324463129043579\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,77,20261111-1828788006-77,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,1,1,1.5,50.96,72.912,8.048109531402588\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,78,20261111-1828788006-78,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,125,0,0,0.0,51.376,72.4,6.168633222579956\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K2|g1,79,20261111-1828788006-79,B,noisy,7,0.001,0.0,2,1,7,7,1,7,2,125,123,1,1,1.5,51.008,71.488,6.2029547691345215\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,0,20261111-1021236598-0,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,84.27710843373494,109.1566265060241,13.714700698852539\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,1,20261111-1021236598-1,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.50602409638554,106.144578313253,13.037209272384644\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,2,20261111-1021236598-2,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,85.144578313253,110.04819277108433,13.982632875442505\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,3,20261111-1021236598-3,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,80,0,3,3.0,83.80722891566265,106.09638554216868,14.067827939987183\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,4,20261111-1021236598-4,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.46987951807229,107.66265060240964,6.871701002120972\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,5,20261111-1021236598-5,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.3855421686747,107.87951807228916,8.097792387008667\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,6,20261111-1021236598-6,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.3012048192771,108.07228915662651,8.458323955535889\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,7,20261111-1021236598-7,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,83.27710843373494,108.67469879518072,7.326568603515625\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,8,20261111-1021236598-8,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,1,1,1.5,82.90361445783132,109.34939759036145,6.392374753952026\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,9,20261111-1021236598-9,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,79,2,2,3.0,84.55421686746988,107.73493975903614,7.54834508895874\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,10,20261111-1021236598-10,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,84.86746987951807,106.67469879518072,7.189454078674316\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,11,20261111-1021236598-11,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,84.51807228915662,105.97590361445783,6.6778998374938965\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,12,20261111-1021236598-12,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.33734939759036,108.09638554216868,6.5123231410980225\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,13,20261111-1021236598-13,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,84.71084337349397,106.09638554216868,8.356696128845215\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,14,20261111-1021236598-14,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,81.19277108433735,108.0,8.22389554977417\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,15,20261111-1021236598-15,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,84.42168674698796,108.21686746987952,7.292025327682495\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,16,20261111-1021236598-16,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.3012048192771,107.78313253012048,7.419626474380493\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,17,20261111-1021236598-17,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,84.27710843373494,108.16867469879519,7.24725079536438\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,18,20261111-1021236598-18,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.25301204819277,108.67469879518072,8.125147581100464\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,19,20261111-1021236598-19,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,84.75903614457832,110.28915662650603,6.5929951667785645\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,20,20261111-1021236598-20,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.144578313253,108.07228915662651,8.317792177200317\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,21,20261111-1021236598-21,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.6144578313253,108.53012048192771,8.062721967697144\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,22,20261111-1021236598-22,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.96385542168674,108.57831325301204,6.720492124557495\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,23,20261111-1021236598-23,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.6987951807229,108.19277108433735,7.166577577590942\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,24,20261111-1021236598-24,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,80,0,3,3.0,83.2289156626506,107.90361445783132,8.428479433059692\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,25,20261111-1021236598-25,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.43373493975903,110.74698795180723,7.0660624504089355\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,26,20261111-1021236598-26,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,80.56626506024097,105.13253012048193,7.7039759159088135\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,27,20261111-1021236598-27,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,1,0,0.5,85.03614457831326,109.32530120481928,6.861081123352051\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,28,20261111-1021236598-28,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,81.96385542168674,106.48192771084338,7.111428499221802\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,29,20261111-1021236598-29,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.49397590361446,107.3012048192771,7.691486358642578\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,30,20261111-1021236598-30,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.59036144578313,109.49397590361446,6.257231712341309\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,31,20261111-1021236598-31,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,83.43373493975903,109.6867469879518,7.871894836425781\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,32,20261111-1021236598-32,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.72289156626506,107.97590361445783,6.9304046630859375\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,33,20261111-1021236598-33,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.6987951807229,108.21686746987952,6.232240676879883\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,34,20261111-1021236598-34,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,80,1,2,2.5,81.96385542168674,108.81927710843374,6.863245964050293\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,35,20261111-1021236598-35,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.08433734939759,106.93975903614458,8.429997444152832\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,36,20261111-1021236598-36,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,84.07228915662651,106.55421686746988,8.319310188293457\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,37,20261111-1021236598-37,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,82.67469879518072,108.36144578313252,7.397504091262817\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,38,20261111-1021236598-38,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,80.855421686747,106.96385542168674,6.314171314239502\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,39,20261111-1021236598-39,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,85.44578313253012,109.1566265060241,7.399871110916138\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,40,20261111-1021236598-40,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,84.20481927710843,106.7710843373494,8.845563650131226\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,41,20261111-1021236598-41,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.65060240963855,107.13253012048193,7.385668039321899\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,42,20261111-1021236598-42,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.79518072289157,106.16867469879519,7.505143165588379\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,43,20261111-1021236598-43,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,1,0,0.5,81.98795180722891,108.74698795180723,8.69837999343872\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,44,20261111-1021236598-44,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.855421686747,107.01204819277109,7.165202856063843\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,45,20261111-1021236598-45,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.34939759036145,109.75903614457832,7.753890037536621\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,46,20261111-1021236598-46,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.03614457831326,108.16867469879519,6.338200807571411\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,47,20261111-1021236598-47,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.72289156626506,107.56626506024097,8.78958535194397\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,48,20261111-1021236598-48,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.95180722891567,106.19277108433735,6.365411281585693\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,49,20261111-1021236598-49,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.39759036144578,108.16867469879519,6.560557126998901\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,50,20261111-1021236598-50,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.74698795180723,108.7710843373494,8.244393348693848\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,51,20261111-1021236598-51,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.59036144578313,107.6144578313253,7.411210536956787\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,52,20261111-1021236598-52,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.25301204819277,106.24096385542168,7.041736602783203\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,53,20261111-1021236598-53,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,83.75903614457832,107.1566265060241,6.714879512786865\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,54,20261111-1021236598-54,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.3855421686747,108.81927710843374,6.860506772994995\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,55,20261111-1021236598-55,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.97590361445783,108.07228915662651,8.680051326751709\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,56,20261111-1021236598-56,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.0,107.73493975903614,7.9503257274627686\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,57,20261111-1021236598-57,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,1,0,0.5,83.63855421686748,107.75903614457832,8.695252418518066\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,58,20261111-1021236598-58,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.5421686746988,106.72289156626506,6.689401626586914\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,59,20261111-1021236598-59,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,83.04819277108433,107.25301204819277,7.348247528076172\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,60,20261111-1021236598-60,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.3012048192771,110.19277108433735,8.066791772842407\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,61,20261111-1021236598-61,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,83.95180722891567,107.97590361445783,7.684471845626831\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,62,20261111-1021236598-62,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.19277108433735,108.33734939759036,6.862226963043213\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,63,20261111-1021236598-63,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,83.73493975903614,106.62650602409639,7.597875595092773\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,64,20261111-1021236598-64,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,82.66265060240964,109.32530120481928,6.7291505336761475\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,65,20261111-1021236598-65,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,81.91566265060241,107.46987951807229,7.334931373596191\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,66,20261111-1021236598-66,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,84.08433734939759,106.79518072289157,8.245423555374146\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,67,20261111-1021236598-67,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,81.37349397590361,108.0,8.250667572021484\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,68,20261111-1021236598-68,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,80,1,2,2.5,83.13253012048193,109.01204819277109,7.334888219833374\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,69,20261111-1021236598-69,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.44578313253012,108.02409638554217,8.130941390991211\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,70,20261111-1021236598-70,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.89156626506023,109.25301204819277,7.952243804931641\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,71,20261111-1021236598-71,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.03614457831326,111.08433734939759,8.45760726928711\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,72,20261111-1021236598-72,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,84.65060240963855,109.10843373493977,8.333675622940063\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,73,20261111-1021236598-73,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,1,1,1.5,84.91566265060241,107.90361445783132,7.421769142150879\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,74,20261111-1021236598-74,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,79,0,4,4.0,84.3132530120482,108.07228915662651,7.899752378463745\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,75,20261111-1021236598-75,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.75903614457832,107.18072289156626,7.841481447219849\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,76,20261111-1021236598-76,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.93975903614458,108.48192771084338,9.103179931640625\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,77,20261111-1021236598-77,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,84.78313253012048,108.8433734939759,8.336235046386719\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,78,20261111-1021236598-78,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.43373493975903,107.73493975903614,8.64533281326294\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,79,20261111-1021236598-79,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.75903614457832,107.39759036144578,8.652490854263306\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,80,20261111-1021236598-80,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.5421686746988,107.90361445783132,8.369765758514404\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,81,20261111-1021236598-81,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,84.5421686746988,106.48192771084338,7.522026777267456\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,82,20261111-1021236598-82,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.44578313253012,109.2289156626506,8.312666654586792\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,83,20261111-1021236598-83,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,81.46987951807229,110.4578313253012,8.081355094909668\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,84,20261111-1021236598-84,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.144578313253,108.04819277108433,7.2345616817474365\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,85,20261111-1021236598-85,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.87951807228916,107.73493975903614,8.277163028717041\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,86,20261111-1021236598-86,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,84.39759036144578,107.34939759036145,7.672237157821655\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,87,20261111-1021236598-87,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.18072289156626,107.5421686746988,8.127012968063354\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,88,20261111-1021236598-88,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.8433734939759,108.04819277108433,7.8401196002960205\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,89,20261111-1021236598-89,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,84.55421686746988,108.07228915662651,8.848767280578613\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,90,20261111-1021236598-90,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,80.78313253012048,110.86746987951807,6.6299660205841064\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,91,20261111-1021236598-91,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,1,1,1.5,83.3012048192771,109.32530120481928,7.1149821281433105\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,92,20261111-1021236598-92,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.08433734939759,107.5421686746988,8.104467630386353\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,93,20261111-1021236598-93,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,80,0,3,3.0,83.72289156626506,108.12048192771084,8.493549108505249\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,94,20261111-1021236598-94,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,81.8433734939759,105.66265060240964,6.862515687942505\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,95,20261111-1021236598-95,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,84.57831325301204,108.81927710843374,6.905070543289185\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,96,20261111-1021236598-96,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,78,0,5,5.0,83.73493975903614,108.12048192771084,8.385545253753662\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,97,20261111-1021236598-97,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.98795180722891,108.79518072289157,7.569000244140625\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,98,20261111-1021236598-98,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.96385542168674,108.81927710843374,8.868189096450806\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,99,20261111-1021236598-99,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.46987951807229,108.8433734939759,8.59152889251709\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,100,20261111-1021236598-100,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,84.3132530120482,109.87951807228916,7.195034503936768\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,101,20261111-1021236598-101,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.01204819277109,108.43373493975903,9.218066692352295\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,102,20261111-1021236598-102,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,83.56626506024097,109.49397590361446,8.099248170852661\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,103,20261111-1021236598-103,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,83.10843373493977,106.57831325301204,8.448613405227661\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,104,20261111-1021236598-104,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,82.98795180722891,109.63855421686748,7.066652059555054\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,105,20261111-1021236598-105,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.12048192771084,106.96385542168674,9.058393955230713\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,106,20261111-1021236598-106,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.33734939759036,107.5421686746988,8.175813674926758\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,107,20261111-1021236598-107,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,82.49397590361446,106.48192771084338,7.158408164978027\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,108,20261111-1021236598-108,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.34939759036145,108.09638554216868,7.351325988769531\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,109,20261111-1021236598-109,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.96385542168674,108.53012048192771,7.594547510147095\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,110,20261111-1021236598-110,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,81.62650602409639,108.36144578313252,9.150112628936768\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,111,20261111-1021236598-111,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,82.56626506024097,107.49397590361446,7.672010660171509\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,112,20261111-1021236598-112,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,79,1,3,3.5,83.87951807228916,107.5421686746988,8.597673416137695\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,113,20261111-1021236598-113,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.60240963855422,107.92771084337349,7.828266859054565\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,114,20261111-1021236598-114,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,85.43373493975903,108.3855421686747,8.956374406814575\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,115,20261111-1021236598-115,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.40963855421687,107.44578313253012,7.283268451690674\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,116,20261111-1021236598-116,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,83.02409638554217,107.01204819277109,7.379128932952881\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,117,20261111-1021236598-117,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,1,0,0.5,83.62650602409639,106.3855421686747,7.665519952774048\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,118,20261111-1021236598-118,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,81,0,2,2.0,84.21686746987952,109.39759036144578,7.416950225830078\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,119,20261111-1021236598-119,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,82,0,1,1.0,82.10843373493977,105.59036144578313,7.6404197216033936\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g7,120,20261111-1021236598-120,B,noisy,7,0.001,0.0,3,7,7,7,7,7,2,83,83,0,0,0.0,83.44578313253012,106.79518072289157,6.750934362411499\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,0,20261111-3585962051-0,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,80,0,3,3.0,69.74698795180723,107.46987951807229,11.601804494857788\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,1,20261111-3585962051-1,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.56626506024097,107.59036144578313,10.885034799575806\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,2,20261111-3585962051-2,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.19277108433735,109.13253012048193,11.01976227760315\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,3,20261111-3585962051-3,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,80,0,3,3.0,68.51807228915662,107.49397590361446,10.911508560180664\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,4,20261111-3585962051-4,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.57831325301204,108.93975903614458,6.220204830169678\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,5,20261111-3585962051-5,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,71.19277108433735,107.92771084337349,5.537891864776611\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,6,20261111-3585962051-6,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.8433734939759,106.21686746987952,5.745996475219727\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,7,20261111-3585962051-7,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.6867469879518,109.2289156626506,5.954575538635254\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,8,20261111-3585962051-8,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.50602409638554,107.39759036144578,6.192839622497559\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,9,20261111-3585962051-9,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,70.37349397590361,108.144578313253,6.708048582077026\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,10,20261111-3585962051-10,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.98795180722891,106.33734939759036,6.054677486419678\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,11,20261111-3585962051-11,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,70.1566265060241,107.44578313253012,6.72764253616333\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,12,20261111-3585962051-12,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.144578313253,107.34939759036145,6.0894739627838135\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,13,20261111-3585962051-13,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.66265060240964,108.19277108433735,6.233684062957764\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,14,20261111-3585962051-14,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,80,0,3,3.0,70.09638554216868,107.34939759036145,7.2612268924713135\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,15,20261111-3585962051-15,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,1,1,1.5,68.6867469879518,106.72289156626506,5.6381754875183105\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,16,20261111-3585962051-16,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,67.71084337349397,107.92771084337349,6.995299339294434\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,17,20261111-3585962051-17,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.10843373493977,107.27710843373494,5.251298666000366\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,18,20261111-3585962051-18,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,1,0,0.5,68.46987951807229,108.40963855421687,5.795100212097168\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,19,20261111-3585962051-19,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,79,0,4,4.0,68.49397590361446,107.37349397590361,7.098185062408447\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,20,20261111-3585962051-20,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,67.65060240963855,106.43373493975903,5.891756534576416\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,21,20261111-3585962051-21,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.74698795180723,107.87951807228916,5.532298564910889\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,22,20261111-3585962051-22,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,1,1,1.5,68.86746987951807,106.43373493975903,6.0356621742248535\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,23,20261111-3585962051-23,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.33734939759036,109.42168674698796,7.148324012756348\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,24,20261111-3585962051-24,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.42168674698796,109.18072289156626,5.439296722412109\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,25,20261111-3585962051-25,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,70.8433734939759,110.0,6.911522626876831\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,26,20261111-3585962051-26,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,67.6867469879518,108.09638554216868,6.651987075805664\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,27,20261111-3585962051-27,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,67.65060240963855,108.3855421686747,6.476557493209839\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,28,20261111-3585962051-28,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.25301204819277,108.4578313253012,6.887126684188843\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,29,20261111-3585962051-29,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,67.63855421686748,108.0,6.02405858039856\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,30,20261111-3585962051-30,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,80,0,3,3.0,68.93975903614458,109.95180722891567,6.490706920623779\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,31,20261111-3585962051-31,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.07228915662651,107.56626506024097,7.040859699249268\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,32,20261111-3585962051-32,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,1,0,0.5,69.02409638554217,110.16867469879519,7.383462905883789\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,33,20261111-3585962051-33,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.83132530120481,108.74698795180723,6.122244834899902\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,34,20261111-3585962051-34,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,67.73493975903614,108.12048192771084,6.2242348194122314\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,35,20261111-3585962051-35,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,67.79518072289157,107.10843373493977,6.030447721481323\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,36,20261111-3585962051-36,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.95180722891567,107.66265060240964,6.527234315872192\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,37,20261111-3585962051-37,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.855421686747,107.92771084337349,6.557464361190796\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,38,20261111-3585962051-38,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.59036144578313,107.10843373493977,6.377852439880371\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,39,20261111-3585962051-39,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.36144578313252,108.0,5.897634983062744\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,40,20261111-3585962051-40,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.24096385542168,108.86746987951807,6.823487758636475\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,41,20261111-3585962051-41,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.73493975903614,106.4578313253012,7.167141914367676\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,42,20261111-3585962051-42,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.62650602409639,108.91566265060241,5.856732130050659\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,43,20261111-3585962051-43,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.63855421686748,106.40963855421687,6.16133975982666\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,44,20261111-3585962051-44,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.56626506024097,106.65060240963855,6.401745796203613\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,45,20261111-3585962051-45,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,66.3132530120482,108.89156626506023,6.723207950592041\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,46,20261111-3585962051-46,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.86746987951807,108.98795180722891,6.636919736862183\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,47,20261111-3585962051-47,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.73493975903614,106.86746987951807,6.054293394088745\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,48,20261111-3585962051-48,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.80722891566265,107.27710843373494,7.375811576843262\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,49,20261111-3585962051-49,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,80,0,3,3.0,69.1566265060241,110.0,6.361705541610718\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,50,20261111-3585962051-50,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,69.49397590361446,109.80722891566265,6.94792914390564\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,51,20261111-3585962051-51,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.96385542168674,108.24096385542168,6.624255657196045\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,52,20261111-3585962051-52,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.56626506024097,107.03614457831326,6.097639799118042\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,53,20261111-3585962051-53,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.92771084337349,108.91566265060241,5.963423490524292\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,54,20261111-3585962051-54,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,80,0,3,3.0,70.55421686746988,108.55421686746988,6.569671154022217\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,55,20261111-3585962051-55,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.43373493975903,107.83132530120481,5.703358888626099\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,56,20261111-3585962051-56,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.1566265060241,108.8433734939759,6.270430088043213\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,57,20261111-3585962051-57,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,1,0,0.5,67.56626506024097,108.8433734939759,6.983698844909668\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,58,20261111-3585962051-58,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,1,0,0.5,70.10843373493977,109.51807228915662,5.826482772827148\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,59,20261111-3585962051-59,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.96385542168674,107.37349397590361,6.050462484359741\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,60,20261111-3585962051-60,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,1,1,1.5,70.04819277108433,109.97590361445783,6.148580074310303\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,61,20261111-3585962051-61,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.46987951807229,105.92771084337349,5.710839033126831\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,62,20261111-3585962051-62,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.33734939759036,107.59036144578313,6.432349681854248\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,63,20261111-3585962051-63,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,69.39759036144578,109.34939759036145,6.224836826324463\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,64,20261111-3585962051-64,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,67.91566265060241,110.24096385542168,6.399097442626953\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,65,20261111-3585962051-65,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.2289156626506,107.01204819277109,6.111397981643677\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,66,20261111-3585962051-66,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,69.26506024096386,108.0,6.1837756633758545\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,67,20261111-3585962051-67,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,70.6144578313253,106.98795180722891,6.28734827041626\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,68,20261111-3585962051-68,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.33734939759036,107.13253012048193,5.807375907897949\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,69,20261111-3585962051-69,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,66.42168674698796,109.63855421686748,7.243627309799194\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,70,20261111-3585962051-70,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,67.25301204819277,106.4578313253012,5.849996566772461\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,71,20261111-3585962051-71,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.34939759036145,109.6144578313253,6.548372268676758\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,72,20261111-3585962051-72,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.13253012048193,107.5421686746988,5.711275577545166\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,73,20261111-3585962051-73,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.21686746987952,108.16867469879519,6.074089288711548\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,74,20261111-3585962051-74,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,66.81927710843374,105.75903614457832,6.268818616867065\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,75,20261111-3585962051-75,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.63855421686748,108.55421686746988,6.266990661621094\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,76,20261111-3585962051-76,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.48192771084338,108.81927710843374,6.867742538452148\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,77,20261111-3585962051-77,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,67.855421686747,108.36144578313252,7.715810775756836\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,78,20261111-3585962051-78,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.24096385542168,109.1566265060241,6.445286750793457\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,79,20261111-3585962051-79,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.96385542168674,108.144578313253,6.833669900894165\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,80,20261111-3585962051-80,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.87951807228916,107.1566265060241,5.865469932556152\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,81,20261111-3585962051-81,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.37349397590361,107.3012048192771,5.9528961181640625\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,82,20261111-3585962051-82,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,70.40963855421687,107.01204819277109,7.482294321060181\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,83,20261111-3585962051-83,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.81927710843374,106.89156626506023,6.365692853927612\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,84,20261111-3585962051-84,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.10843373493977,109.78313253012048,6.729279041290283\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,85,20261111-3585962051-85,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.42168674698796,107.6144578313253,6.895400524139404\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,86,20261111-3585962051-86,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.08433734939759,104.67469879518072,6.233327627182007\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,87,20261111-3585962051-87,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,70.33734939759036,108.40963855421687,6.428955078125\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,88,20261111-3585962051-88,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.3132530120482,106.144578313253,7.0761353969573975\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,89,20261111-3585962051-89,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,1,0,0.5,68.90361445783132,108.3855421686747,5.605217456817627\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,90,20261111-3585962051-90,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.63855421686748,108.81927710843374,6.0630762577056885\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,91,20261111-3585962051-91,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,67.40963855421687,107.90361445783132,6.345900058746338\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,92,20261111-3585962051-92,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.04819277108433,107.83132530120481,6.36247444152832\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,93,20261111-3585962051-93,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,1,0,0.5,67.36144578313252,110.40963855421687,5.978049039840698\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,94,20261111-3585962051-94,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,70.07228915662651,107.75903614457832,6.958355188369751\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,95,20261111-3585962051-95,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.08433734939759,106.0,5.70795202255249\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,96,20261111-3585962051-96,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.10843373493977,109.37349397590361,5.534628629684448\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,97,20261111-3585962051-97,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,80,0,3,3.0,68.65060240963855,108.62650602409639,6.8997485637664795\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,98,20261111-3585962051-98,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,1,1,1.5,69.6867469879518,108.7710843373494,6.557712078094482\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,99,20261111-3585962051-99,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.144578313253,108.89156626506023,5.669890880584717\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,100,20261111-3585962051-100,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,76,0,7,7.0,66.8433734939759,108.144578313253,5.862570524215698\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,101,20261111-3585962051-101,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,80,0,3,3.0,67.86746987951807,105.51807228915662,6.3796117305755615\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,102,20261111-3585962051-102,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.24096385542168,108.02409638554217,7.1679112911224365\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,103,20261111-3585962051-103,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,67.89156626506023,109.13253012048193,7.078591346740723\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,104,20261111-3585962051-104,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,69.42168674698796,107.03614457831326,6.478460311889648\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,105,20261111-3585962051-105,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.72289156626506,107.855421686747,5.470648288726807\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,106,20261111-3585962051-106,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.09638554216868,109.71084337349397,7.050098419189453\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,107,20261111-3585962051-107,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,79,1,3,3.5,67.73493975903614,108.60240963855422,6.654257535934448\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,108,20261111-3585962051-108,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.51807228915662,107.63855421686748,5.733085632324219\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,109,20261111-3585962051-109,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.78313253012048,109.97590361445783,5.942399978637695\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,110,20261111-3585962051-110,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,67.855421686747,107.08433734939759,6.426345109939575\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,111,20261111-3585962051-111,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,67.97590361445783,108.60240963855422,6.365670204162598\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,112,20261111-3585962051-112,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,70.02409638554217,108.04819277108433,6.684394598007202\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,113,20261111-3585962051-113,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,68.57831325301204,110.6987951807229,6.680379629135132\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,114,20261111-3585962051-114,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,70.96385542168674,109.10843373493977,6.2701685428619385\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,115,20261111-3585962051-115,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,69.8433734939759,106.26506024096386,7.55367112159729\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,116,20261111-3585962051-116,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,83,0,0,0.0,67.0,108.86746987951807,5.867010116577148\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,117,20261111-3585962051-117,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.53012048192771,108.3855421686747,7.1555399894714355\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,118,20261111-3585962051-118,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,81,0,2,2.0,68.71084337349397,108.04819277108433,6.115718603134155\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,119,20261111-3585962051-119,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,69.43373493975903,109.34939759036145,6.086655139923096\npilot-20261110T165235Z-e8cfd1,B|noisy|d7|p0.001|K3|g1,120,20261111-3585962051-120,B,noisy,7,0.001,0.0,3,1,7,7,1,7,2,83,82,0,1,1.0,68.53012048192771,108.6987951807229,6.321585416793823\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,0,20261111-1038133639-0,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.3,4.039,7.113783121109009\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,1,20261111-1038133639-1,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.30625,4.026,6.9262940883636475\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,2,20261111-1038133639-2,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.27725,3.9885,7.187121629714966\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,3,20261111-1038133639-3,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.3005,4.0155,7.261599063873291\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,4,20261111-1038133639-4,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.2885,3.958,7.306259870529175\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,5,20261111-1038133639-5,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.27675,3.964,6.986668109893799\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,6,20261111-1038133639-6,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.3,3.9705,7.097956418991089\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,7,20261111-1038133639-7,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.29,3.961,6.856576204299927\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,8,20261111-1038133639-8,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.2845,3.9965,7.099084377288818\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,9,20261111-1038133639-9,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.293,3.9775,6.673975229263306\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,10,20261111-1038133639-10,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.2845,4.0145,7.246981382369995\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,11,20261111-1038133639-11,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.29075,4.0275,6.838487863540649\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,12,20261111-1038133639-12,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.27925,3.987,6.822108030319214\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,13,20261111-1038133639-13,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.29875,4.0055,7.232238531112671\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,14,20261111-1038133639-14,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.28825,4.0165,7.0316481590271\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,15,20261111-1038133639-15,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.2915,3.967,7.133784294128418\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,16,20261111-1038133639-16,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.27575,3.978,6.810234546661377\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,17,20261111-1038133639-17,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.292,4.022,7.241919994354248\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,18,20261111-1038133639-18,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.28475,3.974,6.500377893447876\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,19,20261111-1038133639-19,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.28875,3.9785,7.095423221588135\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,20,20261111-1038133639-20,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.2965,3.949,6.672593116760254\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,21,20261111-1038133639-21,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3996,1,3,3.5,0.29,3.9875,6.761139869689941\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,22,20261111-1038133639-22,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.30025,4.04,7.008666515350342\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,23,20261111-1038133639-23,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.29625,4.069,6.900769233703613\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,24,20261111-1038133639-24,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.28325,4.016,7.014970779418945\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,25,20261111-1038133639-25,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.30075,4.017,6.958107233047485\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,26,20261111-1038133639-26,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.2835,4.023,6.828428030014038\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,27,20261111-1038133639-27,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.29275,4.019,7.297892808914185\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,28,20261111-1038133639-28,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.303,3.9765,7.087994813919067\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,29,20261111-1038133639-29,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.292,4.038,7.063389778137207\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,30,20261111-1038133639-30,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.295,3.962,6.833781003952026\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,31,20261111-1038133639-31,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.29,3.9895,7.0271806716918945\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,32,20261111-1038133639-32,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.2835,3.9765,6.898428201675415\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,33,20261111-1038133639-33,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.3035,4.053,7.372539520263672\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,34,20261111-1038133639-34,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.2955,4.0285,7.3363518714904785\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,35,20261111-1038133639-35,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.301,4.013,6.882911920547485\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,36,20261111-1038133639-36,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.30425,3.9565,7.340765953063965\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,37,20261111-1038133639-37,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.29425,3.993,6.9980809688568115\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,38,20261111-1038133639-38,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.28775,3.9815,6.676346063613892\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,39,20261111-1038133639-39,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.27575,4.006,7.2573323249816895\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,40,20261111-1038133639-40,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.29075,3.9935,6.707056283950806\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,41,20261111-1038133639-41,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,1,1,1.5,0.29625,3.979,7.065142869949341\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,42,20261111-1038133639-42,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.2955,3.9835,6.889766216278076\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,43,20261111-1038133639-43,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3996,0,4,4.0,0.27,3.944,6.931970834732056\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,44,20261111-1038133639-44,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,4000,0,0,0.0,0.289,3.9945,6.881850957870483\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,45,20261111-1038133639-45,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.31275,4.0105,7.177759647369385\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,46,20261111-1038133639-46,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.29475,3.996,6.863913297653198\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,47,20261111-1038133639-47,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3998,0,2,2.0,0.26875,4.013,6.9265477657318115\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,48,20261111-1038133639-48,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3999,0,1,1.0,0.28675,3.9685,6.692227602005005\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K1|g0,49,20261111-1038133639-49,B,noisy,3,0.0001,0.0,1,0,3,3,3,3,2,4000,3997,0,3,3.0,0.289,4.039,7.069692134857178\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,0,20261111-3059505107-0,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.529,8.059,5.394423723220825\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,1,20261111-3059505107-1,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.493,8.047,5.161057233810425\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,2,20261111-3059505107-2,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5195,8.02,5.366432428359985\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,3,20261111-3059505107-3,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,0.507,7.959,5.326747417449951\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,4,20261111-3059505107-4,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5075,8.068,5.456502676010132\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,5,20261111-3059505107-5,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,0.536,8.019,5.597627401351929\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,6,20261111-3059505107-6,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5135,7.99,4.985423803329468\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,7,20261111-3059505107-7,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.502,8.097,4.925457000732422\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,8,20261111-3059505107-8,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.533,8.09,5.32184362411499\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,9,20261111-3059505107-9,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5125,7.927,5.1991798877716064\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,10,20261111-3059505107-10,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.511,8.041,4.936739921569824\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,11,20261111-3059505107-11,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,0.483,7.935,5.277693510055542\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,12,20261111-3059505107-12,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5095,8.067,4.788039922714233\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,13,20261111-3059505107-13,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5295,7.981,4.998041868209839\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,14,20261111-3059505107-14,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5185,8.04,5.33200478553772\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,15,20261111-3059505107-15,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5125,7.929,5.090320587158203\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,16,20261111-3059505107-16,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.519,7.956,5.213493347167969\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,17,20261111-3059505107-17,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5015,7.975,5.435063362121582\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,18,20261111-3059505107-18,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5235,8.043,5.128832578659058\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,19,20261111-3059505107-19,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.524,7.922,5.400330543518066\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,20,20261111-3059505107-20,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.4835,8.104,5.218412399291992\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,21,20261111-3059505107-21,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.499,7.949,4.959066390991211\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,22,20261111-3059505107-22,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5135,8.05,5.30415678024292\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,23,20261111-3059505107-23,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.522,7.989,5.383991718292236\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,24,20261111-3059505107-24,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5185,7.938,5.000244617462158\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,25,20261111-3059505107-25,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.4875,7.941,5.098474979400635\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,26,20261111-3059505107-26,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.517,7.947,5.021147966384888\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,27,20261111-3059505107-27,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5285,8.142,4.892546653747559\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,28,20261111-3059505107-28,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.511,7.921,5.197875499725342\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,29,20261111-3059505107-29,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.516,7.829,5.266655206680298\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,30,20261111-3059505107-30,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5065,7.901,4.980405330657959\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,31,20261111-3059505107-31,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.498,8.041,5.346353054046631\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,32,20261111-3059505107-32,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5205,8.091,5.382628440856934\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,33,20261111-3059505107-33,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.5115,8.079,5.400924444198608\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,34,20261111-3059505107-34,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.498,7.962,5.163684844970703\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,35,20261111-3059505107-35,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.4915,8.087,5.103107690811157\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,36,20261111-3059505107-36,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.52,7.956,5.0478599071502686\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,37,20261111-3059505107-37,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.524,8.008,5.187542676925659\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,38,20261111-3059505107-38,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.5245,8.057,5.139185905456543\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,39,20261111-3059505107-39,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1994,1,5,5.5,0.4955,7.94,5.042250633239746\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,40,20261111-3059505107-40,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.477,8.093,5.409762859344482\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,41,20261111-3059505107-41,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5205,8.039,4.928255081176758\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,42,20261111-3059505107-42,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.529,7.982,4.932063817977905\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,43,20261111-3059505107-43,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.499,7.992,4.827569007873535\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,44,20261111-3059505107-44,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,1,0,0.5,0.494,8.067,5.070047855377197\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,45,20261111-3059505107-45,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5045,7.93,4.97255802154541\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,46,20261111-3059505107-46,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.492,8.005,5.406946420669556\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,47,20261111-3059505107-47,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5285,7.917,4.926811456680298\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,48,20261111-3059505107-48,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,0.522,8.094,4.73151159286499\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,49,20261111-3059505107-49,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.488,8.0,5.430415630340576\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,50,20261111-3059505107-50,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.517,8.056,5.015755891799927\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,51,20261111-3059505107-51,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5365,8.116,5.250073194503784\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,52,20261111-3059505107-52,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.545,8.083,5.231955528259277\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,53,20261111-3059505107-53,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.511,7.913,5.341850996017456\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,54,20261111-3059505107-54,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,0.4995,8.047,4.958176612854004\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,55,20261111-3059505107-55,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.533,7.925,4.9859325885772705\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,56,20261111-3059505107-56,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.529,8.014,4.903115272521973\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,57,20261111-3059505107-57,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,0.53,7.979,5.024775743484497\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,58,20261111-3059505107-58,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.5295,8.069,5.062821865081787\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,59,20261111-3059505107-59,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.551,8.015,5.3735973834991455\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,60,20261111-3059505107-60,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5245,7.842,5.408279657363892\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,61,20261111-3059505107-61,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.528,8.02,4.97914981842041\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,62,20261111-3059505107-62,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5145,8.045,5.200634717941284\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,63,20261111-3059505107-63,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.491,8.016,5.284428596496582\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,64,20261111-3059505107-64,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,0.5105,7.981,5.33634090423584\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,65,20261111-3059505107-65,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,1,2,2.5,0.5235,8.068,4.925570964813232\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,66,20261111-3059505107-66,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.512,8.118,5.23133659362793\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,67,20261111-3059505107-67,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1994,0,6,6.0,0.499,8.027,4.941206216812134\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,68,20261111-3059505107-68,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.5305,7.931,5.363759517669678\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,69,20261111-3059505107-69,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,1,0,0.5,0.5105,8.032,5.198491811752319\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,70,20261111-3059505107-70,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5065,8.103,5.427495718002319\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,71,20261111-3059505107-71,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.5045,8.008,5.271918296813965\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,72,20261111-3059505107-72,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.491,8.048,5.078596591949463\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,73,20261111-3059505107-73,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.5165,8.014,5.249787330627441\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,74,20261111-3059505107-74,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1995,0,5,5.0,0.51,7.927,4.961817741394043\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,75,20261111-3059505107-75,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.497,8.009,5.241116762161255\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,76,20261111-3059505107-76,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.4915,8.041,5.144818067550659\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,77,20261111-3059505107-77,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.508,7.973,5.369523763656616\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,78,20261111-3059505107-78,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.4985,8.054,4.697810411453247\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,79,20261111-3059505107-79,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.492,8.024,4.878112316131592\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,80,20261111-3059505107-80,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.4995,7.987,5.221813678741455\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,81,20261111-3059505107-81,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.5295,7.967,4.941330432891846\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,82,20261111-3059505107-82,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.515,8.088,5.420749187469482\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,83,20261111-3059505107-83,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.5015,7.974,5.029665470123291\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,84,20261111-3059505107-84,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.526,8.058,5.160342454910278\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,85,20261111-3059505107-85,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1996,0,4,4.0,0.5365,8.093,4.9845969676971436\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,86,20261111-3059505107-86,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.525,7.9,5.271512746810913\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,87,20261111-3059505107-87,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.524,7.897,5.021352529525757\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,88,20261111-3059505107-88,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.4995,8.012,5.070201873779297\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,89,20261111-3059505107-89,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5145,7.919,5.02084755897522\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,90,20261111-3059505107-90,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.509,8.022,5.137798070907593\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,91,20261111-3059505107-91,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5285,8.097,4.973261833190918\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,92,20261111-3059505107-92,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.4855,8.0,4.847386121749878\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,93,20261111-3059505107-93,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.5115,7.989,4.897999048233032\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,94,20261111-3059505107-94,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,2000,0,0,0.0,0.514,8.04,5.09738826751709\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,95,20261111-3059505107-95,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1998,0,2,2.0,0.529,8.002,5.268584251403809\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,96,20261111-3059505107-96,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1997,0,3,3.0,0.4755,8.047,5.282458066940308\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,97,20261111-3059505107-97,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5095,7.947,4.736604452133179\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,98,20261111-3059505107-98,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.5095,8.052,5.204516649246216\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g3,99,20261111-3059505107-99,B,noisy,3,0.0001,0.0,2,3,3,3,3,3,2,2000,1999,0,1,1.0,0.514,8.02,5.3031744956970215\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,0,20261111-1481783039-0,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.4595,8.125,5.303401947021484\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,1,20261111-1481783039-1,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.4635,8.042,5.002955436706543\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,2,20261111-1481783039-2,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.474,8.034,4.856138229370117\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,3,20261111-1481783039-3,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.5005,8.073,5.205702066421509\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,4,20261111-1481783039-4,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.478,8.05,4.810672283172607\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,5,20261111-1481783039-5,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4745,8.073,4.632953405380249\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,6,20261111-1481783039-6,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.473,8.071,4.973963022232056\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,7,20261111-1481783039-7,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.462,8.017,5.131878614425659\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,8,20261111-1481783039-8,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.482,8.003,4.5875444412231445\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,9,20261111-1481783039-9,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.462,8.02,4.987613201141357\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,10,20261111-1481783039-10,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.462,8.056,4.853238105773926\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,11,20261111-1481783039-11,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.459,8.03,4.9631028175354\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,12,20261111-1481783039-12,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.447,8.05,4.773823261260986\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,13,20261111-1481783039-13,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.4765,7.974,4.716959476470947\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,14,20261111-1481783039-14,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.447,7.974,4.972272634506226\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,15,20261111-1481783039-15,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.447,7.942,4.629175424575806\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,16,20261111-1481783039-16,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.482,7.966,4.846322059631348\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,17,20261111-1481783039-17,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4475,8.005,4.845733880996704\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,18,20261111-1481783039-18,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4685,7.968,4.615207195281982\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,19,20261111-1481783039-19,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.443,8.147,4.456528425216675\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,20,20261111-1481783039-20,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.4505,7.953,4.897151947021484\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,21,20261111-1481783039-21,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,0.463,7.894,4.405999183654785\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,22,20261111-1481783039-22,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4425,8.022,4.769731283187866\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,23,20261111-1481783039-23,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.472,8.033,4.735006809234619\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,24,20261111-1481783039-24,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.484,8.119,4.606740951538086\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,25,20261111-1481783039-25,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.455,7.898,5.016704797744751\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,26,20261111-1481783039-26,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.427,8.004,4.575379848480225\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,27,20261111-1481783039-27,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4725,7.889,4.84204626083374\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,28,20261111-1481783039-28,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.488,8.056,4.760149002075195\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,29,20261111-1481783039-29,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.479,7.904,5.026334285736084\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,30,20261111-1481783039-30,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.484,7.925,4.765933990478516\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,31,20261111-1481783039-31,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4425,8.023,4.692593336105347\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,32,20261111-1481783039-32,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.4505,8.076,5.039229154586792\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,33,20261111-1481783039-33,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.462,8.023,5.167158842086792\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,34,20261111-1481783039-34,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.4715,7.935,5.162993669509888\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,35,20261111-1481783039-35,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4775,7.947,5.080199241638184\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,36,20261111-1481783039-36,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4735,7.875,5.027097702026367\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,37,20261111-1481783039-37,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4645,8.046,5.050498962402344\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,38,20261111-1481783039-38,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.473,7.976,4.671715259552002\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,39,20261111-1481783039-39,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.503,8.036,4.997922897338867\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,40,20261111-1481783039-40,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.5105,8.042,5.04888129234314\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,41,20261111-1481783039-41,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.4485,8.028,4.795273542404175\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,42,20261111-1481783039-42,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.465,8.017,4.604368209838867\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,43,20261111-1481783039-43,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.478,8.025,5.038837432861328\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,44,20261111-1481783039-44,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.4535,7.986,4.793211936950684\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,45,20261111-1481783039-45,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.487,7.977,4.6755242347717285\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,46,20261111-1481783039-46,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.457,7.989,4.954705476760864\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,47,20261111-1481783039-47,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.461,8.06,4.908020496368408\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,48,20261111-1481783039-48,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4805,7.94,5.103386402130127\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,49,20261111-1481783039-49,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.465,7.949,4.83927321434021\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,50,20261111-1481783039-50,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.4275,8.0,4.970656156539917\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,51,20261111-1481783039-51,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.486,8.091,4.8109636306762695\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,52,20261111-1481783039-52,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4785,8.019,5.17581582069397\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,53,20261111-1481783039-53,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4775,7.954,4.9817795753479\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,54,20261111-1481783039-54,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4655,8.05,4.934480905532837\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,55,20261111-1481783039-55,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.478,8.094,4.9417173862457275\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,56,20261111-1481783039-56,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4355,7.985,5.087020397186279\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,57,20261111-1481783039-57,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4685,7.988,4.90403151512146\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,58,20261111-1481783039-58,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.472,7.998,4.7087860107421875\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,59,20261111-1481783039-59,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.4855,7.882,4.9034764766693115\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,60,20261111-1481783039-60,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4495,7.935,4.937631130218506\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,61,20261111-1481783039-61,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.414,8.054,4.947885036468506\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,62,20261111-1481783039-62,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.413,8.062,4.850460767745972\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,63,20261111-1481783039-63,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4715,8.026,5.077484607696533\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,64,20261111-1481783039-64,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.4675,7.893,4.742406129837036\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,65,20261111-1481783039-65,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.482,7.992,4.861149072647095\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,66,20261111-1481783039-66,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.445,7.897,4.818537473678589\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,67,20261111-1481783039-67,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4645,8.047,5.18947958946228\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,68,20261111-1481783039-68,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1995,0,5,5.0,0.489,8.055,5.078858137130737\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,69,20261111-1481783039-69,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.464,7.992,4.625827312469482\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,70,20261111-1481783039-70,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4455,8.08,4.83783221244812\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,71,20261111-1481783039-71,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.473,7.959,5.17239236831665\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,72,20261111-1481783039-72,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4435,8.136,4.898078680038452\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,73,20261111-1481783039-73,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.466,7.915,4.731573581695557\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,74,20261111-1481783039-74,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.4635,7.985,4.828644514083862\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,75,20261111-1481783039-75,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4715,7.962,4.993203401565552\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,76,20261111-1481783039-76,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.473,8.059,5.134222984313965\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,77,20261111-1481783039-77,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.4595,8.047,5.001718044281006\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,78,20261111-1481783039-78,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.4915,8.046,5.194854259490967\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,79,20261111-1481783039-79,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.4655,8.027,5.1920695304870605\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,80,20261111-1481783039-80,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4855,7.885,5.0702431201934814\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,81,20261111-1481783039-81,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.458,7.921,4.7835376262664795\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,82,20261111-1481783039-82,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4765,7.956,4.816884756088257\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,83,20261111-1481783039-83,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.492,7.993,4.860964775085449\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,84,20261111-1481783039-84,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.4865,8.019,4.844927549362183\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,85,20261111-1481783039-85,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4725,7.934,4.900557518005371\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,86,20261111-1481783039-86,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4555,8.009,4.821748733520508\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,87,20261111-1481783039-87,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4635,8.007,4.936118841171265\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,88,20261111-1481783039-88,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4785,7.999,4.463882684707642\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,89,20261111-1481783039-89,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.474,7.988,4.932776927947998\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,90,20261111-1481783039-90,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4525,8.05,4.466146230697632\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,91,20261111-1481783039-91,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4655,7.93,4.695059061050415\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,92,20261111-1481783039-92,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.466,8.128,4.995003938674927\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,93,20261111-1481783039-93,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4885,7.89,5.127569198608398\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,94,20261111-1481783039-94,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1997,0,3,3.0,0.4685,8.107,4.705050945281982\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,95,20261111-1481783039-95,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.4495,8.068,4.953335285186768\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,96,20261111-1481783039-96,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1999,0,1,1.0,0.4685,8.171,4.707426309585571\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,97,20261111-1481783039-97,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1996,0,4,4.0,0.462,8.057,5.06773829460144\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,98,20261111-1481783039-98,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,2000,0,0,0.0,0.476,7.923,4.774912118911743\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K2|g1,99,20261111-1481783039-99,B,noisy,3,0.0001,0.0,2,1,3,3,1,3,2,2000,1998,0,2,2.0,0.5095,7.971,5.004986524581909\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,0,20261111-249573558-0,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.719429857464366,12.022505626406602,4.690972805023193\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,1,20261111-249573558-1,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7336834208552138,11.82595648912228,4.657138347625732\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,2,20261111-249573558-2,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.68192048012003,11.983495873968492,4.975622892379761\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,3,20261111-249573558-3,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7314328582145536,11.992498124531133,4.652697563171387\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,4,20261111-249573558-4,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7231807951987997,12.037509377344335,4.747301816940308\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,5,20261111-249573558-5,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7441860465116279,12.070517629407352,4.354578256607056\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,6,20261111-249573558-6,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7126781695423856,11.911477869467367,4.405118465423584\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,7,20261111-249573558-7,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7456864216054013,11.927981995498875,4.8399059772491455\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,8,20261111-249573558-8,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7651912978244562,12.027006751687923,4.307784080505371\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,9,20261111-249573558-9,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7299324831207802,11.849962490622655,4.629757404327393\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,10,20261111-249573558-10,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7254313578394599,12.016504126031508,4.430638074874878\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,11,20261111-249573558-11,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7471867966991748,11.944486121530383,4.206396818161011\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,12,20261111-249573558-12,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7396849212303076,12.031507876969242,4.623412609100342\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,13,20261111-249573558-13,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7299324831207802,12.085521380345087,4.525524377822876\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,14,20261111-249573558-14,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.6969242310577645,11.993998499624906,4.26656436920166\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,15,20261111-249573558-15,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7501875468867217,11.836459114778695,4.581304311752319\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,16,20261111-249573558-16,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7606901725431358,11.900975243810953,4.623481035232544\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,17,20261111-249573558-17,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7051762940735183,12.069017254313579,4.498897314071655\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,18,20261111-249573558-18,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7044261065266316,12.057014253563391,4.272122621536255\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,19,20261111-249573558-19,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7929482370592649,11.969992498124531,4.141656875610352\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,20,20261111-249573558-20,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7164291072768192,11.921980495123782,4.668051242828369\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,21,20261111-249573558-21,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.7809452363090773,12.069017254313579,4.634783506393433\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,22,20261111-249573558-22,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7156789197299325,12.138034508627157,4.462775230407715\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,23,20261111-249573558-23,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7419354838709677,11.896474118529632,4.186728477478027\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,24,20261111-249573558-24,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7074268567141786,11.915978994748688,4.472101449966431\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,25,20261111-249573558-25,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.746436609152288,12.040510127531883,4.528456449508667\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,26,20261111-249573558-26,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7404351087771943,12.018004501125281,4.3466637134552\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,27,20261111-249573558-27,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.795198799699925,11.944486121530383,4.237121105194092\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,28,20261111-249573558-28,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7434358589647412,11.944486121530383,4.711134433746338\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,29,20261111-249573558-29,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.6721680420105026,12.123030757689422,4.2462546825408936\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,30,20261111-249573558-30,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.6691672918229558,12.105026256564141,4.730952739715576\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,31,20261111-249573558-31,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7284321080270068,11.90847711927982,4.470158576965332\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,32,20261111-249573558-32,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.6856714178544636,11.99549887471868,4.6874120235443115\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,33,20261111-249573558-33,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.6946736684171043,12.168042010502626,4.384212493896484\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,34,20261111-249573558-34,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.705926481620405,12.150037509377345,4.579725742340088\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,35,20261111-249573558-35,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7456864216054013,12.069017254313579,4.7187159061431885\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,36,20261111-249573558-36,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7494373593398349,12.027006751687923,4.625568628311157\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,37,20261111-249573558-37,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.746436609152288,12.064516129032258,4.742283821105957\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,38,20261111-249573558-38,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.7344336084021005,11.94748687171793,4.279417514801025\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,39,20261111-249573558-39,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7501875468867217,12.024006001500375,4.281289339065552\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,40,20261111-249573558-40,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7014253563390848,11.972993248312077,4.3511433601379395\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,41,20261111-249573558-41,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7396849212303076,12.015003750937735,4.410401821136475\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,42,20261111-249573558-42,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7186796699174793,12.024006001500375,4.456793546676636\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,43,20261111-249573558-43,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7434358589647412,11.971492873218304,4.791803598403931\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,44,20261111-249573558-44,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7419354838709677,11.980495123780946,4.514675140380859\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,45,20261111-249573558-45,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.7149287321830458,11.897974493623407,4.719435691833496\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,46,20261111-249573558-46,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7561890472618155,12.0,4.63478422164917\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,47,20261111-249573558-47,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7449362340585146,11.974493623405852,4.379082679748535\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,48,20261111-249573558-48,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.741185296324081,11.993998499624906,4.522672414779663\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,49,20261111-249573558-49,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,0.7471867966991748,11.96549137284321,4.780423402786255\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,50,20261111-249573558-50,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7156789197299325,11.795948987246812,4.316520929336548\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,51,20261111-249573558-51,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7029257314328582,11.82145536384096,4.52680516242981\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,52,20261111-249573558-52,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7479369842460615,12.12153038259565,4.434532403945923\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,53,20261111-249573558-53,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7374343585896475,12.090022505626406,4.4088029861450195\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,54,20261111-249573558-54,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7216804201050263,11.831957989497374,4.712177038192749\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,55,20261111-249573558-55,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.735933983495874,12.105026256564141,4.601010084152222\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,56,20261111-249573558-56,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7366841710427607,12.102025506376593,4.36023736000061\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,57,20261111-249573558-57,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7576894223555889,12.037509377344335,4.775272369384766\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,58,20261111-249573558-58,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7749437359339835,12.007501875468867,4.812760353088379\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,59,20261111-249573558-59,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7434358589647412,12.055513878469617,4.311079025268555\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,60,20261111-249573558-60,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.6999249812453113,12.063015753938485,4.241014719009399\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,61,20261111-249573558-61,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7081770442610653,11.94748687171793,4.483846664428711\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,62,20261111-249573558-62,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.746436609152288,12.103525881470368,4.7041168212890625\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,63,20261111-249573558-63,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7591897974493623,11.824456114028507,4.555490732192993\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,64,20261111-249573558-64,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7164291072768192,12.217554388597149,4.505600452423096\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,65,20261111-249573558-65,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7501875468867217,11.88597149287322,4.513898849487305\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,66,20261111-249573558-66,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.6916729182295573,11.94748687171793,4.755541563034058\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,67,20261111-249573558-67,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.714178544636159,11.902475618904726,4.646981954574585\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,68,20261111-249573558-68,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7509377344336085,11.91297824456114,4.724064350128174\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,69,20261111-249573558-69,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7119279819954989,12.00450112528132,4.59037971496582\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,70,20261111-249573558-70,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7156789197299325,11.924981245311328,4.802788257598877\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,71,20261111-249573558-71,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7291822955738935,12.117029257314329,4.638923406600952\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,72,20261111-249573558-72,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7644411102775694,12.166541635408851,4.641143560409546\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,73,20261111-249573558-73,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7254313578394599,11.944486121530383,4.774835109710693\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,74,20261111-249573558-74,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7524381095273819,12.123030757689422,4.376231670379639\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,75,20261111-249573558-75,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.7351837959489872,12.088522130532633,4.688539266586304\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,76,20261111-249573558-76,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7036759189797449,11.96549137284321,4.736990928649902\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,77,20261111-249573558-77,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7284321080270068,12.061515378844712,4.6572911739349365\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,78,20261111-249573558-78,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7389347336834209,11.980495123780946,4.1261773109436035\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,79,20261111-249573558-79,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7404351087771943,11.854463615903976,4.695735454559326\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,80,20261111-249573558-80,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7156789197299325,12.051012753188298,4.6621413230896\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,81,20261111-249573558-81,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.735933983495874,11.911477869467367,4.53728461265564\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,82,20261111-249573558-82,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,0.7306826706676669,12.066016504126031,4.199656963348389\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,83,20261111-249573558-83,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,0.7239309827456865,12.061515378844712,4.693217754364014\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,84,20261111-249573558-84,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7426856714178545,12.123030757689422,4.436810493469238\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,85,20261111-249573558-85,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7261815453863466,11.990997749437359,4.643677234649658\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,86,20261111-249573558-86,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7111777944486122,12.12153038259565,4.3793652057647705\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,87,20261111-249573558-87,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.714178544636159,12.04801200300075,4.728261947631836\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,88,20261111-249573558-88,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1327,0,6,6.0,0.7449362340585146,12.070517629407352,4.5414206981658936\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,89,20261111-249573558-89,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7284321080270068,11.750937734433608,4.653814315795898\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,90,20261111-249573558-90,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.754688672168042,12.105026256564141,4.326698303222656\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,91,20261111-249573558-91,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7134283570892723,12.115528882220556,4.692420482635498\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,92,20261111-249573558-92,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7291822955738935,12.08702175543886,4.563449859619141\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,93,20261111-249573558-93,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.735933983495874,11.981995498874719,4.496384620666504\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,94,20261111-249573558-94,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7179294823705926,12.036009002250562,4.76280951499939\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,95,20261111-249573558-95,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.735933983495874,11.896474118529632,4.769629240036011\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,96,20261111-249573558-96,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7254313578394599,12.028507126781696,4.707444906234741\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,97,20261111-249573558-97,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7119279819954989,12.001500375093773,4.685977220535278\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,98,20261111-249573558-98,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.687171792948237,11.921980495123782,4.593673467636108\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,99,20261111-249573558-99,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.714178544636159,11.96099024756189,4.573399782180786\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,100,20261111-249573558-100,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.6879219804951238,12.066016504126031,4.532005310058594\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,101,20261111-249573558-101,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.7246811702925732,12.172543135783947,4.277010202407837\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,102,20261111-249573558-102,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7179294823705926,11.96099024756189,4.4449639320373535\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,103,20261111-249573558-103,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.7636909227306826,12.037509377344335,4.37246561050415\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,104,20261111-249573558-104,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7794448612153039,12.018004501125281,4.429696798324585\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,105,20261111-249573558-105,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.6984246061515379,12.00450112528132,4.4826507568359375\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,106,20261111-249573558-106,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7561890472618155,12.157539384846212,4.809857606887817\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,107,20261111-249573558-107,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7516879219804952,12.022505626406602,4.511857986450195\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,108,20261111-249573558-108,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7516879219804952,12.112528132033008,4.68765115737915\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,109,20261111-249573558-109,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7126781695423856,12.102025506376593,4.689774751663208\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,110,20261111-249573558-110,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7344336084021005,11.972993248312077,4.6790547370910645\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,111,20261111-249573558-111,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7396849212303076,12.024006001500375,4.687317848205566\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,112,20261111-249573558-112,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.6976744186046512,11.935483870967742,4.5713279247283936\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,113,20261111-249573558-113,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.7344336084021005,11.91747936984246,4.629489421844482\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,114,20261111-249573558-114,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7404351087771943,12.049512378094523,4.612891435623169\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,115,20261111-249573558-115,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7809452363090773,12.016504126031508,4.696878671646118\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,116,20261111-249573558-116,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7321830457614403,11.894973743435859,4.566633224487305\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,117,20261111-249573558-117,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7449362340585146,11.923480870217555,4.647388935089111\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,118,20261111-249573558-118,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.6976744186046512,12.031507876969242,4.671947240829468\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,119,20261111-249573558-119,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7419354838709677,12.20855213803451,4.696235656738281\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,120,20261111-249573558-120,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.6969242310577645,12.043510877719429,4.719456911087036\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,121,20261111-249573558-121,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7239309827456865,12.075018754688672,4.708934307098389\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,122,20261111-249573558-122,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.705926481620405,12.12153038259565,4.49822473526001\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,123,20261111-249573558-123,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1329,0,4,4.0,0.7351837959489872,12.192048012003001,4.6964499950408936\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,124,20261111-249573558-124,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.708927231807952,12.040510127531883,4.706935405731201\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,125,20261111-249573558-125,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.722430607651913,11.915978994748688,4.8309032917022705\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,126,20261111-249573558-126,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.6789197299324832,11.971492873218304,4.7627716064453125\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,127,20261111-249573558-127,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1328,0,5,5.0,0.7081770442610653,12.097524381095274,4.5942840576171875\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,128,20261111-249573558-128,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7209302325581395,11.924981245311328,4.735851049423218\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,129,20261111-249573558-129,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7524381095273819,11.91297824456114,4.571453809738159\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,130,20261111-249573558-130,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7569392348087022,11.941485371342836,4.807582139968872\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,131,20261111-249573558-131,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.6931732933233309,12.067516879219806,4.975153207778931\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,132,20261111-249573558-132,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7336834208552138,12.085521380345087,4.670950651168823\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,133,20261111-249573558-133,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.722430607651913,11.86496624156039,4.786683082580566\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,134,20261111-249573558-134,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7501875468867217,12.01350337584396,4.915223598480225\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,135,20261111-249573558-135,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7404351087771943,12.00450112528132,4.8776328563690186\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,136,20261111-249573558-136,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7111777944486122,11.959489872468117,4.707624673843384\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,137,20261111-249573558-137,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.6916729182295573,11.942985746436609,4.638073682785034\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,138,20261111-249573558-138,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7389347336834209,12.136534133533383,4.438734769821167\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,139,20261111-249573558-139,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7029257314328582,11.87396849212303,4.73132061958313\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,140,20261111-249573558-140,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7314328582145536,12.093023255813954,4.801026105880737\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,141,20261111-249573558-141,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7524381095273819,11.86496624156039,4.743844509124756\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,142,20261111-249573558-142,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7104276069017255,11.980495123780946,4.652360200881958\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,143,20261111-249573558-143,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.719429857464366,12.003000750187548,4.607435464859009\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,144,20261111-249573558-144,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1330,0,3,3.0,0.7119279819954989,11.722430607651914,4.765290021896362\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,145,20261111-249573558-145,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1332,0,1,1.0,0.7351837959489872,12.019504876219054,4.37368369102478\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,146,20261111-249573558-146,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7096774193548387,11.834958739684922,4.596688985824585\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,147,20261111-249573558-147,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1333,0,0,0.0,0.7396849212303076,12.012003000750187,4.585553169250488\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,148,20261111-249573558-148,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7119279819954989,11.950487621905477,4.713768720626831\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,149,20261111-249573558-149,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7351837959489872,12.075018754688672,4.299386262893677\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g3,150,20261111-249573558-150,B,noisy,3,0.0001,0.0,3,3,3,3,3,3,2,1333,1331,0,2,2.0,0.7516879219804952,11.867966991747936,4.869868993759155\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,0,20261111-3773714842-0,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6099024756189048,11.981995498874719,4.620364665985107\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,1,20261111-3773714842-1,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6474118529632408,11.936984246061515,4.581066608428955\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,2,20261111-3773714842-2,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6661665416354089,11.882970742685671,4.738271713256836\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,3,20261111-3773714842-3,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.622655663915979,11.858964741185297,4.747523784637451\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,4,20261111-3773714842-4,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6324081020255063,12.265566391597899,4.33301305770874\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,5,20261111-3773714842-5,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6676669167291823,12.151537884471118,4.039375305175781\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,6,20261111-3773714842-6,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.627906976744186,11.975993998499625,4.37483811378479\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,7,20261111-3773714842-7,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6609152288072018,12.117029257314329,4.305181980133057\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,8,20261111-3773714842-8,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.614403600900225,12.03900975243811,4.435456991195679\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,9,20261111-3773714842-9,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6181545386346586,11.809452363090772,4.506200790405273\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,10,20261111-3773714842-10,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,0.6286571642910728,12.123030757689422,4.389573812484741\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,11,20261111-3773714842-11,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6639159789947486,12.168042010502626,4.266971826553345\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,12,20261111-3773714842-12,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6624156039009752,11.932483120780194,4.080774545669556\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,13,20261111-3773714842-13,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6346586646661665,12.249062265566392,4.262879371643066\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,14,20261111-3773714842-14,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6534133533383346,12.040510127531883,4.124245643615723\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,15,20261111-3773714842-15,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6369092273068268,11.974493623405852,4.482431173324585\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,16,20261111-3773714842-16,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6489122280570142,12.022505626406602,4.179247856140137\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,17,20261111-3773714842-17,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6009002250562641,11.959489872468117,4.311826229095459\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,18,20261111-3773714842-18,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6106526631657915,11.870967741935484,4.448821544647217\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,19,20261111-3773714842-19,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6586646661665416,11.924981245311328,4.446896076202393\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,20,20261111-3773714842-20,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6234058514628658,11.767441860465116,4.400348901748657\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,21,20261111-3773714842-21,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6519129782445612,12.055513878469617,4.538553237915039\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,22,20261111-3773714842-22,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6241560390097525,12.040510127531883,4.0364179611206055\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,23,20261111-3773714842-23,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6504126031507877,11.971492873218304,4.305986166000366\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,24,20261111-3773714842-24,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6121530382595649,11.918979744936234,4.166714191436768\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,25,20261111-3773714842-25,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,0.6159039759939985,12.064516129032258,4.329671621322632\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,26,20261111-3773714842-26,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6399099774943736,11.894973743435859,4.119239568710327\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,27,20261111-3773714842-27,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.627906976744186,12.055513878469617,4.291040897369385\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,28,20261111-3773714842-28,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6466616654163541,11.980495123780946,4.00426721572876\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,29,20261111-3773714842-29,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6541635408852213,11.978994748687171,4.0955212116241455\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,30,20261111-3773714842-30,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6316579144786196,12.045011252813204,4.246922731399536\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,31,20261111-3773714842-31,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.63615903975994,11.894973743435859,4.221137285232544\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,32,20261111-3773714842-32,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6526631657914479,11.962490622655665,4.228860139846802\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,33,20261111-3773714842-33,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.5746436609152288,11.911477869467367,4.315321683883667\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,34,20261111-3773714842-34,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6249062265566392,11.815453863465866,4.145958423614502\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,35,20261111-3773714842-35,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6264066016504126,12.021005251312829,4.441422700881958\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,36,20261111-3773714842-36,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6744186046511628,11.840960240060015,4.396690607070923\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,37,20261111-3773714842-37,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6489122280570142,11.924981245311328,4.386947154998779\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,38,20261111-3773714842-38,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.5986496624156039,11.977494373593398,4.404759407043457\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,39,20261111-3773714842-39,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6166541635408852,12.019504876219054,4.161354303359985\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,40,20261111-3773714842-40,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6219054763690923,11.909977494373594,4.03641676902771\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,41,20261111-3773714842-41,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6196549137284321,12.153038259564891,4.104783296585083\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,42,20261111-3773714842-42,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6541635408852213,12.064516129032258,4.063289403915405\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,43,20261111-3773714842-43,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.5978994748687172,12.171042760690172,3.8713173866271973\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,44,20261111-3773714842-44,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6541635408852213,12.051012753188298,4.52181077003479\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,45,20261111-3773714842-45,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.622655663915979,11.969992498124531,4.312045574188232\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,46,20261111-3773714842-46,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6324081020255063,12.067516879219806,4.149049997329712\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,47,20261111-3773714842-47,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6436609152288072,11.944486121530383,4.015373945236206\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,48,20261111-3773714842-48,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6444111027756939,11.9264816204051,4.191537141799927\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,49,20261111-3773714842-49,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6249062265566392,11.843960990247561,3.9836466312408447\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,50,20261111-3773714842-50,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6459114778694673,12.072018004501125,4.209102153778076\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,51,20261111-3773714842-51,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6354088522130532,12.067516879219806,3.7893385887145996\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,52,20261111-3773714842-52,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,0.6189047261815454,11.90847711927982,4.226799726486206\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,53,20261111-3773714842-53,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,1,3,3.5,0.641410352588147,11.89947486871718,4.163883924484253\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,54,20261111-3773714842-54,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6399099774943736,12.0960240060015,4.467004776000977\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,55,20261111-3773714842-55,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6286571642910728,12.0735183795949,4.181048154830933\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,56,20261111-3773714842-56,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6451612903225806,11.96549137284321,4.310868501663208\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,57,20261111-3773714842-57,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6114028507126782,12.192048012003001,3.916560649871826\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,58,20261111-3773714842-58,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6436609152288072,11.933983495873969,4.300390720367432\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,59,20261111-3773714842-59,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6369092273068268,12.054013503375844,4.35085129737854\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,60,20261111-3773714842-60,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1326,0,7,7.0,0.6654163540885222,11.906976744186046,4.408168077468872\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,61,20261111-3773714842-61,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6024006001500375,11.887471867966992,4.637317419052124\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,62,20261111-3773714842-62,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1328,0,5,5.0,0.6466616654163541,11.953488372093023,4.3028342723846436\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,63,20261111-3773714842-63,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6526631657914479,12.063015753938485,4.2310919761657715\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,64,20261111-3773714842-64,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6489122280570142,11.971492873218304,4.422630310058594\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,65,20261111-3773714842-65,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6564141035258815,11.941485371342836,4.5266101360321045\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,66,20261111-3773714842-66,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6211552888222055,12.003000750187548,4.024378538131714\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,67,20261111-3773714842-67,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6294073518379595,12.045011252813204,4.164128541946411\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,68,20261111-3773714842-68,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.641410352588147,11.969992498124531,4.107241868972778\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,69,20261111-3773714842-69,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6234058514628658,12.105026256564141,4.359692811965942\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,70,20261111-3773714842-70,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6564141035258815,12.001500375093773,4.063483476638794\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,71,20261111-3773714842-71,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6399099774943736,11.87396849212303,4.315906286239624\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,72,20261111-3773714842-72,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,1,1,1.5,0.6391597899474869,12.08252063015754,4.295517206192017\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,73,20261111-3773714842-73,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6204051012753188,11.927981995498875,4.435418367385864\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,74,20261111-3773714842-74,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6181545386346586,12.109527381845462,4.362281322479248\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,75,20261111-3773714842-75,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6234058514628658,12.105026256564141,4.067070484161377\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,76,20261111-3773714842-76,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6166541635408852,11.897974493623407,4.1856396198272705\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,77,20261111-3773714842-77,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6339084771192798,12.046511627906977,4.387627363204956\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,78,20261111-3773714842-78,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6316579144786196,11.79144786196549,4.27276611328125\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,79,20261111-3773714842-79,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6534133533383346,11.95198799699925,4.003611087799072\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,80,20261111-3773714842-80,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6339084771192798,11.890472618154538,4.089156150817871\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,81,20261111-3773714842-81,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6174043510877719,11.786946736684172,4.310761451721191\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,82,20261111-3773714842-82,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6264066016504126,12.145536384096024,4.314597845077515\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,83,20261111-3773714842-83,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6436609152288072,12.04801200300075,4.4999237060546875\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,84,20261111-3773714842-84,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6691672918229558,11.929482370592648,4.165826797485352\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,85,20261111-3773714842-85,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6466616654163541,11.852963240810203,4.134026765823364\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,86,20261111-3773714842-86,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,1,3,3.5,0.6774193548387096,11.822955738934734,4.333495378494263\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,87,20261111-3773714842-87,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6136534133533383,11.88597149287322,4.34114670753479\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,88,20261111-3773714842-88,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.5993998499624906,11.813953488372093,4.2568700313568115\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,89,20261111-3773714842-89,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6541635408852213,11.902475618904726,4.202432632446289\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,90,20261111-3773714842-90,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,0.6579144786196549,12.027006751687923,4.416967153549194\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,91,20261111-3773714842-91,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6376594148537135,11.968492123030758,4.650363922119141\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,92,20261111-3773714842-92,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6249062265566392,11.987996999249813,4.793020486831665\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,93,20261111-3773714842-93,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6489122280570142,11.8304576144036,4.317135810852051\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,94,20261111-3773714842-94,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,0.6384096024006002,12.037509377344335,4.26700758934021\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,95,20261111-3773714842-95,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6541635408852213,12.007501875468867,4.663090944290161\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,96,20261111-3773714842-96,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,0.6436609152288072,11.983495873968492,4.240038871765137\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,97,20261111-3773714842-97,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6234058514628658,12.057014253563391,4.134000062942505\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,98,20261111-3773714842-98,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6444111027756939,11.87396849212303,4.647273778915405\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,99,20261111-3773714842-99,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6354088522130532,12.210052513128282,4.069535970687866\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,100,20261111-3773714842-100,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6466616654163541,12.133533383345837,4.1031105518341064\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,101,20261111-3773714842-101,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6234058514628658,12.055513878469617,4.372136831283569\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,102,20261111-3773714842-102,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,0.6691672918229558,12.147036759189797,4.363330364227295\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,103,20261111-3773714842-103,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6256564141035259,11.989497374343586,4.224909067153931\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,104,20261111-3773714842-104,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6654163540885222,11.956489122280571,4.172492504119873\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,105,20261111-3773714842-105,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.5986496624156039,12.019504876219054,4.388404369354248\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,106,20261111-3773714842-106,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,0.5941485371342836,12.0,4.24922776222229\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,107,20261111-3773714842-107,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6301575393848462,11.972993248312077,4.155283689498901\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,108,20261111-3773714842-108,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6789197299324832,12.132033008252064,4.330148935317993\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,109,20261111-3773714842-109,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.5918979744936234,12.118529632408102,4.4228997230529785\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,110,20261111-3773714842-110,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6106526631657915,11.996999249812452,4.179093837738037\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,111,20261111-3773714842-111,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6166541635408852,11.929482370592648,4.33243203163147\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,112,20261111-3773714842-112,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6256564141035259,11.85146286571643,4.357421159744263\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,113,20261111-3773714842-113,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6316579144786196,12.08252063015754,4.492516756057739\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,114,20261111-3773714842-114,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6384096024006002,11.914478619654913,4.40847373008728\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,115,20261111-3773714842-115,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,0,4,4.0,0.6384096024006002,12.105026256564141,4.2856268882751465\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,116,20261111-3773714842-116,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6646661665416355,12.072018004501125,4.254983901977539\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,117,20261111-3773714842-117,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.63615903975994,11.989497374343586,4.365162134170532\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,118,20261111-3773714842-118,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6436609152288072,11.870967741935484,4.323742389678955\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,119,20261111-3773714842-119,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6406601650412603,12.123030757689422,4.445818901062012\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,120,20261111-3773714842-120,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,1,2,2.5,0.6151537884471118,12.055513878469617,4.06887149810791\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,121,20261111-3773714842-121,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6061515378844711,11.792948237059266,4.180146932601929\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,122,20261111-3773714842-122,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6369092273068268,11.983495873968492,4.358354806900024\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,123,20261111-3773714842-123,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6331582895723931,11.929482370592648,4.351407051086426\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,124,20261111-3773714842-124,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6354088522130532,11.998499624906227,4.105993270874023\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,125,20261111-3773714842-125,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6556639159789948,12.120030007501876,4.1479480266571045\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,126,20261111-3773714842-126,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6399099774943736,11.927981995498875,4.46712589263916\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,127,20261111-3773714842-127,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,1,2,2.5,0.6481620405101275,12.030007501875469,4.007235527038574\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,128,20261111-3773714842-128,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6339084771192798,11.927981995498875,4.031183958053589\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,129,20261111-3773714842-129,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6106526631657915,12.26106526631658,4.316917419433594\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,130,20261111-3773714842-130,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6421605401350338,11.870967741935484,3.9692983627319336\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,131,20261111-3773714842-131,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1329,1,3,3.5,0.6811702925731433,12.016504126031508,4.460334777832031\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,132,20261111-3773714842-132,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6346586646661665,11.858964741185297,3.914597988128662\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,133,20261111-3773714842-133,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6489122280570142,12.001500375093773,4.357218027114868\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,134,20261111-3773714842-134,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.654913728432108,11.927981995498875,4.147501468658447\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,135,20261111-3773714842-135,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,1,2,2.5,0.63615903975994,12.007501875468867,4.22986912727356\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,136,20261111-3773714842-136,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6316579144786196,11.840960240060015,3.966158151626587\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,137,20261111-3773714842-137,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6391597899474869,12.021005251312829,4.429201364517212\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,138,20261111-3773714842-138,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6249062265566392,11.956489122280571,4.4085853099823\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,139,20261111-3773714842-139,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6646661665416355,11.950487621905477,4.409478187561035\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,140,20261111-3773714842-140,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6609152288072018,11.972993248312077,4.040082216262817\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,141,20261111-3773714842-141,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.622655663915979,12.085521380345087,4.356531381607056\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,142,20261111-3773714842-142,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.609152288072018,12.030007501875469,4.358279466629028\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,143,20261111-3773714842-143,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6879219804951238,12.145536384096024,4.228656768798828\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,144,20261111-3773714842-144,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6534133533383346,12.153038259564891,3.985903024673462\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,145,20261111-3773714842-145,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6331582895723931,11.953488372093023,4.341010332107544\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,146,20261111-3773714842-146,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1330,0,3,3.0,0.6376594148537135,12.036009002250562,4.301799774169922\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,147,20261111-3773714842-147,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1333,0,0,0.0,0.6511627906976745,12.027006751687923,4.492542028427124\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,148,20261111-3773714842-148,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6466616654163541,12.030007501875469,4.3337907791137695\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,149,20261111-3773714842-149,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1332,0,1,1.0,0.6136534133533383,11.846961740435109,4.5161871910095215\npilot-20261110T165235Z-e8cfd1,B|noisy|d3|p0.0001|K3|g1,150,20261111-3773714842-150,B,noisy,3,0.0001,0.0,3,1,3,3,1,3,2,1333,1331,0,2,2.0,0.6459114778694673,12.196549137284322,4.41646933555603\n'
print(len(STUDY3), '条件（研究 3 の比較表）,', len(REF3), '点（研究 3 の出力不忠実度）,', '試行データ', 'あり' if EMBED_PILOT_MC else 'なし')

## 1. 自己検査

In [ ]:
# ==================================================================================================
# 自己検査（1〜2 分）。詳しい検証は 研究4_validation.ipynb。
# ==================================================================================================
def selfcheck():
    ok = True
    # (1) K = 1 は研究 3 の D4Experiment と 1 ショットごとに同じ結果（同じ seed）
    a = D4Experiment(3, 3, 3, 3, NoiseModel(2e-3, p_idle=0.0))
    b = SeqExperiment(3, 3, 3, 3, 3, 1, NoiseModel(2e-3, p_idle=0.0))
    ia, _ = a.run(300, np.random.default_rng(5)); ib, _ = b.run(300, np.random.default_rng(5))
    r = bool(np.array_equal(ia, ib)); ok &= r
    print('[%s] K=1 が研究 3 の回路・復号器と 1 ショットごとに一致（300 ショット、失敗 %d 件）' % ('PASS' if r else 'FAIL', int((ia > 0).sum())))
    # (2) 雑音なしなら K 回連続でも不忠実度 0
    worst = 0.0
    for d, K in [(2, 2), (3, 2), (3, 3), (4, 4)]:
        inf, _ = SeqExperiment(d, 1, d, 1, 1, K, NoiseModel(0.0)).run(10, np.random.default_rng(K))
        worst = max(worst, float(inf.max()))
    r = worst == 0.0; ok &= r
    print('[%s] 雑音なしで不忠実度 0（d=2..4, K=2..4）' % ('PASS' if r else 'FAIL'))
    # (3) 真の非 Clifford 回路（厳密な疎状態ベクトル）との照合：L=2, K=2 の全測定と最後の <H>
    rng = np.random.default_rng(3)
    ex = SeqExperiment(2, 1, 1, 1, 1, 2, NoiseModel(0.02))
    bad = nm = 0
    for _ in range(3):
        res = lockstep_full(2, 1, 1, NoiseModel(0.02), rng, ex=ex)
        bad += not res['agree']; nm += res['n_meas']
    r = bad == 0; ok &= r
    print('[%s] L=2, K=2：Pauli proxy と真の非 Clifford 回路が一致（3 軌道、測定 %d 個）' % ('PASS' if r else 'FAIL', nm))
    assert ok, '自己検査に失敗しました'


if RUN_SELFCHECK and RUN_MODE != 'analysis_only':
    selfcheck()

## 2. モンテカルロ

定義（次のセル）と実行（その次のセル）。途中で止めても、`RESUME = True` のまま同じセルを実行すれば続きから計算する。

In [ ]:
# ==================================================================================================
# モンテカルロ本体
#   1 点 = (シナリオ, p, d, K, gap, 窓)。1 点のショットをチャンクに分け、チャンクごとに CSV に 1 行を追記する。
#   ショット数は固定（失敗数や時間では止めない）。seed は (BASE_SEED, 点, チャンク番号) だけで決まる。
# ==================================================================================================
import time, zlib, uuid, datetime, multiprocessing as _mp

MC_CSV = f'{OUT_DIR}/study4_mc.csv'
MC_COLUMNS = ['run_id', 'point_id', 'chunk_id', 'seed_key', 'scenario', 'window', 'd', 'p', 'p_idle', 'K', 'gap',
              'R0', 'Rin', 'Rgap', 'Rout', 'n_pre_u', 'shots', 'n0', 'n_half', 'n_one', 'sum_infidelity',
              'mean_errors', 'mean_so_cz', 'seconds']


def gap_rounds(gap, d):
    return d if gap == 'd' else int(np.ceil(d / 2)) if gap == 'half' else int(gap)


def make_points(plan):
    pts = []
    for scen, p, d, shots in plan:
        for quiet in ([False, True] if QUIET_WINDOW else [False]):
            for K in K_LIST:
                for gap in (GAP_LIST if K > 1 else GAP_LIST[:1]):       # K = 1 には間のラウンドが無い
                    rg = gap_rounds(gap, d)
                    pid = f"{scen}|{'quiet' if quiet else 'noisy'}|d{d}|p{p:g}|K{K}|g{rg if K > 1 else 0}"
                    if any(q['point_id'] == pid for q in pts):
                        continue
                    pts.append(dict(point_id=pid, scenario=scen, p=p, p_idle=SCENARIOS[scen] * p, d=d, K=K,
                                    Rgap=rg, quiet=quiet, shots=shots))
    return pts


def chunk_size(d, K):
    return max(50, int({3: 4000, 5: 1000, 7: 250}.get(d, 100) / K))


_EX = {}


def _experiment(pt):
    if pt['point_id'] not in _EX:
        d = pt['d']
        R0, Rin, Rout = [max(1, int(np.ceil(k * d))) for k in KAPPA]
        _EX.clear()                                     # 1 プロセスが同時に持つ実験は 1 つ（メモリ節約）
        _EX[pt['point_id']] = SeqExperiment(d, R0, Rin, pt['Rgap'], Rout, pt['K'],
                                            NoiseModel(pt['p'], p_idle=pt['p_idle']), n_pre_u=N_PRE_U,
                                            quiet_window=pt['quiet'])
    return _EX[pt['point_id']]


def run_chunk(task):
    pt, chunk_id, n = task
    t0 = time.time()
    ex = _experiment(pt)
    key = (int(BASE_SEED), zlib.crc32(pt['point_id'].encode()), int(chunk_id))
    rng = np.random.default_rng(np.random.SeedSequence(key))
    inf, st = ex.run(n, rng)
    c = ex.circ
    return dict(point_id=pt['point_id'], chunk_id=chunk_id, seed_key='-'.join(map(str, key)), scenario=pt['scenario'],
                window='quiet' if pt['quiet'] else 'noisy', d=pt['d'], p=pt['p'], p_idle=pt['p_idle'], K=pt['K'],
                gap=pt['Rgap'] if pt['K'] > 1 else 0, R0=c.R0, Rin=c.Rin, Rgap=c.Rgap, Rout=c.Rout, n_pre_u=c.n_pre_u,
                shots=n, n0=int((inf == 0).sum()), n_half=int((inf == 0.5).sum()), n_one=int((inf == 1).sum()),
                sum_infidelity=float(inf.sum()), mean_errors=float(np.mean([s['n_err'] for s in st])),
                mean_so_cz=float(np.mean([s['so_cz'] for s in st])), seconds=time.time() - t0)


def run_monte_carlo(plan):
    os.makedirs(OUT_DIR, exist_ok=True)
    run_id = RUN_MODE + '-' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:6]
    done = {}
    if os.path.exists(MC_CSV):
        if not RESUME:
            raise FileExistsError(f'{MC_CSV} が既にあります。RESUME=True にするか OUT_DIR を変えてください')
        old = pd.read_csv(MC_CSV)
        assert list(old.columns) == MC_COLUMNS, '既存の CSV の列がこのノートと違います'
        done = {(r.point_id, int(r.chunk_id)): int(r.shots) for r in old.itertuples()}
    tasks = []
    for pt in make_points(plan):
        cs = chunk_size(pt['d'], pt['K'])
        for k in range(int(np.ceil(pt['shots'] / cs))):       # ショット数はチャンクの倍数に切り上げる
            if (pt['point_id'], k) in done:                   # （後からショット数を増やしても既存のチャンクと整合する）
                assert done[(pt['point_id'], k)] == cs, f"{pt['point_id']} のチャンクの大きさが既存の CSV と違います"
                continue
            tasks.append((pt, k, cs))
    # 軽い点から順に（途中で止めても軽い点は揃う）。同じ点のチャンクは連続させる（回路の作り直しを減らす）
    order = {pt['point_id']: i for i, pt in enumerate(make_points(plan))}
    tasks.sort(key=lambda t: (order[t[0]['point_id']], t[1]))
    print(f'run_id = {run_id} | 残り {len(tasks)} チャンク / {sum(t[2] for t in tasks)} ショット | 出力 {MC_CSV}')
    if not tasks:
        return
    t0, n_done = time.time(), 0
    total = sum(t[2] for t in tasks)

    def save(row):
        nonlocal n_done
        row = dict(run_id=run_id, **row)
        pd.DataFrame([row])[MC_COLUMNS].to_csv(MC_CSV, mode='a', header=not os.path.exists(MC_CSV), index=False)
        n_done += row['shots']
    last = time.time()
    if WORKERS > 1:
        with _mp.get_context('fork').Pool(WORKERS) as pool:
            for row in pool.imap(run_chunk, tasks, chunksize=1):
                save(row)
                if time.time() - last > 60:
                    last = time.time()
                    print(f'  {n_done}/{total} ショット, {time.time() - t0:.0f} 秒, 現在 {row["point_id"]}', flush=True)
    else:
        for task in tasks:
            save(run_chunk(task))
    print(f'完了：{n_done} ショット, {time.time() - t0:.0f} 秒')

In [ ]:
if RUN_MODE != 'analysis_only':
    run_monte_carlo(PLAN[RUN_MODE])
else:
    print('analysis_only：モンテカルロは実行しません')

## 3. 回路の規模と償却係数

In [ ]:
# ==================================================================================================
# 回路の規模と「償却係数」
#   V_single : 研究 3 と同じ 1 回の魔法状態準備（S0 + S1 + U + SO + S4）の時空間体積
#   V_cycle  : 連続実行で T ゲート 1 個が増えるごとに増える体積（S1 + U + SO + 間の FSC ラウンド）
#   alpha    : V_cycle / V_single（1 より小さいほど、連続実行で 1 個あたりが安くなる）
# ==================================================================================================
def cost_table(d_list, gaps):
    rows = []
    for d in d_list:
        c1 = seq_counts(d, 1, n_pre_u=N_PRE_U)
        for gap in gaps:
            rg = gap_rounds(gap, d)
            c2 = seq_counts(d, 2, Rgap=rg, n_pre_u=N_PRE_U)
            rows.append(dict(d=d, gap=rg, qubits=c1['qubits'], steps_single=c1['time_steps'], V_single=c1['volume_total'],
                             steps_cycle=c2['time_steps'] - c1['time_steps'], V_cycle=c2['volume_total'] - c1['volume_total']))
    df = pd.DataFrame(rows)
    df['alpha'] = df.V_cycle / df.V_single
    df['alpha_time'] = df.steps_cycle / df.steps_single
    return df


COST = cost_table([3, 5, 7, 9, 11], ['d', 'half', 1])
_c = seq_counts(5, 1)
assert _c['volume_total'] == 632 * 125 + 454 * 25 + 8 * 5 - 14 and _c['time_steps'] == 62 * 5 + 21    # 研究 3 の式と一致
COST.round(4)

## 4. 出力不忠実度の集計

In [ ]:
# ==================================================================================================
# 集計：点ごとの不忠実度
#   1 ショットの不忠実度は 0, 1/2, 1 のどれか。eps = (n_one + n_half / 2) / N。
#   標準誤差は多項分布の分散から。区間は n_one と n_half のそれぞれの Clopper–Pearson 区間を Bonferroni で
#   合成したもの（ショット数が固定なので有効。保守的）。
# ==================================================================================================
def cp_interval(k, n, alpha):
    lo = 0.0 if k == 0 else _st.beta.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else _st.beta.ppf(1 - alpha / 2, k + 1, n - k)
    return lo, hi


def summarize_mc(mc):
    g = mc.groupby(['scenario', 'window', 'p', 'd', 'K', 'gap'], as_index=False).agg(
        shots=('shots', 'sum'), n_half=('n_half', 'sum'), n_one=('n_one', 'sum'), seconds=('seconds', 'sum'))
    g['eps'] = (g.n_one + g.n_half / 2) / g.shots
    p1, ph = g.n_one / g.shots, g.n_half / g.shots
    g['se'] = np.sqrt(np.maximum(p1 + ph / 4 - g.eps ** 2, 0) / g.shots)
    a = (1 - CONFIDENCE) / 2                      # Bonferroni：2 成分に半分ずつ
    lo, hi = [], []
    for r in g.itertuples():
        l1, h1 = cp_interval(r.n_one, r.shots, a)
        l2, h2 = cp_interval(r.n_half, r.shots, a)
        lo.append(l1 + l2 / 2); hi.append(h1 + h2 / 2)
    g['eps_lo'], g['eps_hi'] = lo, hi
    return g


def load_mc():
    path = f'{OUT_DIR}/study4_mc.csv'
    if os.path.exists(path):
        print('読み込み：', path)
        return pd.read_csv(path), 'this_run'
    print('OUT_DIR に CSV が無いので、ノートに埋め込んだ試行データ（results_pilot）を使います')
    return pd.read_csv(io.StringIO(EMBED_PILOT_MC)), 'embedded_pilot'


MC, MC_SOURCE = load_mc()
SUM = summarize_mc(MC)
SUM.to_csv(f'{OUT_DIR}/study4_summary.csv', index=False)
print('合計ショット数：', int(MC.shots.sum()))
SUM[SUM.window == 'noisy'][['scenario', 'p', 'd', 'K', 'gap', 'shots', 'n_half', 'n_one', 'eps', 'se', 'eps_lo', 'eps_hi']]

## 5. 1 サイクルあたりの誤り（加法性の検査）

In [ ]:
# ==================================================================================================
# 1 サイクルあたりの誤り
#   eps_cycle = (eps(K=K_hi) - eps(K=1)) / (K_hi - 1)   … S0 と最後の評価の寄与は差で消える
#   m         = eps_cycle / eps(K=1)                     … 1 に近ければ「誤りはサイクル数に比例し、連続実行で
#                                                           1 個あたりの誤りは減りも増えもしない」
#   K_hi は奇数の最大値を使う（K が奇数の出力は T 型の状態で、論理 X・Y 誤りに対する感度が K=1 と同じ。
#   偶数の K は感度が違うので参考値として別に示す）。区間は正規近似（独立な標本の差）。
# ==================================================================================================
def per_cycle(sumdf, window='noisy'):
    z = _st.norm.ppf(1 - (1 - CONFIDENCE) / 2)
    s = sumdf[sumdf.window == window]
    rows = []
    for (scen, p, d), grp in s.groupby(['scenario', 'p', 'd']):
        base = grp[grp.K == 1]
        if base.empty:
            continue
        b = base.iloc[0]
        ref = REF3[(REF3.scenario == scen) & np.isclose(REF3.p, p) & (REF3.d == d)]
        for gap, gg in grp[grp.K > 1].groupby('gap'):
            row = dict(scenario=scen, p=p, d=d, gap=int(gap), eps1=b.eps, se1=b.se, shots1=int(b.shots),
                       eps1_study3=float(ref.eps.iloc[0]) if len(ref) else np.nan)
            for K in sorted(gg.K):
                r = gg[gg.K == K].iloc[0]
                row[f'eps{K}'], row[f'se{K}'] = r.eps, r.se
            odd = [K for K in sorted(gg.K) if K % 2 == 1]
            if odd:
                Kh = odd[-1]
                r = gg[gg.K == Kh].iloc[0]
                ec = (r.eps - b.eps) / (Kh - 1)
                sec = np.sqrt(r.se ** 2 + b.se ** 2) / (Kh - 1)
                m = ec / b.eps if b.eps > 0 else np.nan
                # デルタ法：m = (eps_K/eps_1 - 1)/(K-1)
                sem = (np.sqrt((r.se / b.eps) ** 2 + (r.eps * b.se / b.eps ** 2) ** 2) / (Kh - 1)) if b.eps > 0 else np.nan
                row.update(K_hi=Kh, eps_cycle=ec, se_cycle=sec, eps_cycle_lo=ec - z * sec, eps_cycle_hi=ec + z * sec,
                           m=m, se_m=sem, m_lo=m - z * sem, m_hi=m + z * sem)
            rows.append(row)
    return pd.DataFrame(rows)


PC = per_cycle(SUM)
PC.to_csv(f'{OUT_DIR}/study4_per_cycle.csv', index=False)
_cols = [c for c in ['scenario', 'p', 'd', 'gap', 'shots1', 'eps1', 'eps1_study3', 'eps2', 'eps3', 'eps_cycle', 'se_cycle', 'm', 'm_lo', 'm_hi'] if c in PC.columns]
PC[_cols]

## 6. 間の FSC ラウンドを短くした効果

In [ ]:
# 間の FSC ラウンドを短くしたときの誤りの変化：eps_cycle(gap) / eps_cycle(gap = d)
def gap_effect(pc):
    rows = []
    if 'eps_cycle' not in pc.columns:
        return pd.DataFrame(rows)
    for (scen, p, d), grp in pc.groupby(['scenario', 'p', 'd']):
        full = grp[(grp.gap == d) & grp.eps_cycle.notna()]
        if full.empty:
            continue
        f = full.iloc[0]
        for r in grp[(grp.gap != d) & grp.eps_cycle.notna()].itertuples():
            ratio = r.eps_cycle / f.eps_cycle if f.eps_cycle > 0 else np.nan
            se = abs(ratio) * np.sqrt((r.se_cycle / r.eps_cycle) ** 2 + (f.se_cycle / f.eps_cycle) ** 2) if f.eps_cycle > 0 and r.eps_cycle > 0 else np.nan
            a = COST[(COST.d == d)]
            rows.append(dict(scenario=scen, p=p, d=d, gap=r.gap, eps_cycle_gap=r.eps_cycle, eps_cycle_full=f.eps_cycle,
                             error_ratio=ratio, se_error_ratio=se,
                             volume_ratio=float(a[a.gap == r.gap].V_cycle.iloc[0] / a[a.gap == d].V_cycle.iloc[0]) if (a.gap == r.gap).any() else np.nan))
    return pd.DataFrame(rows)


GAP = gap_effect(PC)
GAP.to_csv(f'{OUT_DIR}/study4_gap_effect.csv', index=False)
GAP

## 7. 費用の比較

In [ ]:
# ==================================================================================================
# 費用の比較（T ゲート 1 個あたりの時空間体積：連続実行の code switching ÷ 15-to-1 蒸留）
#
# 研究 3 の比（1 回の魔法状態準備の体積 ÷ 蒸留工場の体積）を出発点に、連続実行による変化を掛ける：
#     比_連続 = 比_研究3 × alpha(d, gap) ÷ (1 + c)
#   alpha : 償却係数（上のセル）。d は研究 3 と同じサイズを使う。
#           これは「1 サイクルあたりの誤りが研究 3 の 1 回の誤り以下（m <= 1）」のときに成り立つ。
#           m > 1 なら必要なサイズは研究 3 以上になり、比はこれより大きくなる（＝この表は code switching に有利な側）。
#   c     : 蒸留側で魔法状態を消費する格子手術の体積 ÷ 工場の体積（CONSUME_OVERHEAD）。研究 3 は c = 0。
# 研究 3 の列の意味：status = data_certified ならデータで保証された比、それ以外は「データで棄却されない最小の
# サイズ」での比（＝体積の下限。保証ではない）。
# ==================================================================================================
def alpha_of(d, gap):
    rg = gap_rounds(gap, d)
    hit = COST[(COST.d == d) & (COST.gap == rg)]
    if len(hit):
        return float(hit.alpha.iloc[0])
    c1, c2 = seq_counts(d, 1, n_pre_u=N_PRE_U), seq_counts(d, 2, Rgap=rg, n_pre_u=N_PRE_U)
    return (c2['volume_total'] - c1['volume_total']) / c1['volume_total']


def amortized_table(targets, gaps, overheads):
    rows = []
    for r in STUDY3[STUDY3.target.isin(targets)].itertuples():
        d = int(r.kobayashi_d_low)
        row = dict(scenario=r.scenario, p=r.p, target=r.target, study3_status=r.kobayashi_status, d=d,
                   d_simulated=d <= 9, ratio_study3=r.ratio_kobayashi_low_over_litinski)
        for gap in gaps:
            for c in overheads:
                row[f'ratio_seq_gap{gap}_c{c:g}'] = r.ratio_kobayashi_low_over_litinski * alpha_of(d, gap) / (1 + c)
        # 損益分岐に必要な消費の体積（工場の体積に対する比）：比_研究3 × alpha - 1
        row['c_needed_gap_d'] = r.ratio_kobayashi_low_over_litinski * alpha_of(d, 'd') - 1
        row['c_needed_gap_1'] = r.ratio_kobayashi_low_over_litinski * alpha_of(d, 1) - 1
        rows.append(row)
    return pd.DataFrame(rows)


AMORT = amortized_table(TARGETS, ['d', 1], CONSUME_OVERHEAD)
AMORT.to_csv(f'{OUT_DIR}/study4_amortized_cost.csv', index=False)
_t = AMORT[AMORT.target == TARGETS[0]]
print(f'要求 {TARGETS[0]:g}：比_連続（c = 0）の最小値  gap = d : {_t["ratio_seq_gapd_c0"].min():.2f}   gap = 1 : {_t["ratio_seq_gap1_c0"].min():.2f}'
      f'   （研究 3 の比の最小値 {_t.ratio_study3.min():.2f}）')
AMORT[AMORT.target == TARGETS[0]].round(3)

## 8. 判定の要約

In [ ]:
# ==================================================================================================
# 判定の要約（文章）。数値は上の表から機械的に作る。
# ==================================================================================================
def verdict():
    out = []
    t = AMORT[AMORT.target == TARGETS[0]]
    out.append(f'[費用] 要求 {TARGETS[0]:g} の {len(t)} 条件：連続実行の比（c=0）は gap=d で {t["ratio_seq_gapd_c0"].min():.2f}〜{t["ratio_seq_gapd_c0"].max():.2f}、'
               f'gap=1 で {t["ratio_seq_gap1_c0"].min():.2f}〜{t["ratio_seq_gap1_c0"].max():.2f}。1 を下回る条件は '
               f'gap=d で {int((t["ratio_seq_gapd_c0"] < 1).sum())} 個、gap=1 で {int((t["ratio_seq_gap1_c0"] < 1).sum())} 個。')
    if len(PC) and 'm' in PC.columns:
        full = PC[(PC.gap == PC.d) & PC.m.notna()]
        out.append('[誤り] m = eps_cycle / eps(K=1)（gap = d）： ' + '、'.join(
            f'{r.scenario},p={r.p:g},d={r.d}: {r.m:.2f} [{r.m_lo:.2f}, {r.m_hi:.2f}]' for r in full.itertuples()))
        if len(full) > 1:                       # 逆分散重みの平均（点の間で m が共通と仮定した参考値）と一様性の chi^2
            w = 1 / full.se_m ** 2
            mp = float((w * full.m).sum() / w.sum())
            out.append(f'      全点の重み付き平均 m = {mp:.3f} ± {float(w.sum() ** -0.5):.3f}'
                       f'（chi^2 = {float((((full.m - mp) / full.se_m) ** 2).sum()):.1f}, 自由度 {len(full) - 1}）')
        lowm = full[full.m_hi < 1]
        out.append(f'      m が 1 より有意に小さい点（連続実行で 1 個あたりの誤りが減る点）：{len(lowm)} / {len(full)}')
    if len(GAP):
        out.append('[間のラウンド] eps_cycle(gap) / eps_cycle(gap=d)： ' + '、'.join(
            f'{r.scenario},p={r.p:g},d={r.d},gap={r.gap}: {r.error_ratio:.2f}±{r.se_error_ratio:.2f}（体積 {r.volume_ratio:.2f} 倍）' for r in GAP.itertuples()))
    if len(GAP) > 1:
        w = 1 / GAP.se_error_ratio ** 2
        rp = float((w * GAP.error_ratio).sum() / w.sum())
        out.append(f'      全点の重み付き平均 = {rp:.3f} ± {float(w.sum() ** -0.5):.3f}'
                   f'（chi^2 = {float((((GAP.error_ratio - rp) / GAP.se_error_ratio) ** 2).sum()):.1f}, 自由度 {len(GAP) - 1}）')
    out.append('[注意] データの出所：' + ('このノートの実行結果' if MC_SOURCE == 'this_run' else '埋め込みの試行データ') +
               f'（合計 {int(MC.shots.sum())} ショット）。費用の表は研究 3 の本番データ（554 万ショット）の判定に償却係数を掛けたもの。')
    return '\n'.join(out)


VERDICT = verdict()
open(f'{OUT_DIR}/study4_verdict.txt', 'w').write(VERDICT + '\n')
print(VERDICT)

## 9. 図

In [ ]:
# ==================================================================================================
# 図（PNG を OUT_DIR に保存）。図の中の文字は英語（Colab に日本語フォントが無いため）。
# ==================================================================================================
_C = {'d': '#2a78d6', 'short': '#eb6834', 'ref': '#52514e'}
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#e6e5e0',
                     'grid.linewidth': 0.6, 'font.size': 9, 'axes.titlesize': 9.5, 'legend.frameon': False})


def fig_eps_vs_K(sumdf):
    s = sumdf[sumdf.window == 'noisy']
    keys = sorted({(r.scenario, r.p, r.d) for r in s.itertuples()}, key=lambda k: (k[0], -k[1], k[2]))
    if not keys:
        return
    ncol = min(4, len(keys)); nrow = int(np.ceil(len(keys) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 2.7 * nrow), squeeze=False)
    for ax in axes.ravel()[len(keys):]:
        ax.axis('off')
    for ax, (scen, p, d) in zip(axes.ravel(), keys):
        g = s[(s.scenario == scen) & np.isclose(s.p, p) & (s.d == d)]
        b = g[g.K == 1]
        kmax = int(g.K.max())
        if len(b):
            ax.plot([1, kmax], [b.eps.iloc[0], kmax * b.eps.iloc[0]], color=_C['ref'], lw=1, ls=':', label='K x eps(K=1)')
        for gap in sorted(g[g.K > 1].gap.unique(), reverse=True):
            gg = pd.concat([b, g[(g.K > 1) & (g.gap == gap)]]).sort_values('K')
            full = gap == d
            ax.errorbar(gg.K + (0 if full else 0.06), gg.eps, yerr=[gg.eps - gg.eps_lo, gg.eps_hi - gg.eps], lw=2, capsize=2,
                        color=_C['d'] if full else _C['short'], marker='o' if full else 's', ms=5,
                        label=f'gap = {int(gap)} round' + ('s' if gap > 1 else '') + (' (= d)' if full else ''))
        ax.set_title(f'scenario {scen}, p = {p:g}, d = {d}')
        ax.set_xticks(range(1, kmax + 1)); ax.set_ylim(bottom=0)
        ax.set_xlabel('number of consecutive cycles K'); ax.set_ylabel('output infidelity')
        ax.legend(fontsize=7.5, loc='upper left')
    fig.tight_layout()
    fig.savefig(f'{OUT_DIR}/fig_eps_vs_K.png', dpi=200)
    plt.show()


def fig_ratio(amort, target):
    t = amort[amort.target == target].sort_values(['scenario', 'p']).reset_index(drop=True)
    if t.empty:
        return
    fig, ax = plt.subplots(figsize=(7.6, 3.8))
    y = np.arange(len(t))
    ax.axvline(1, color='#0b0b0b', lw=1)
    ax.scatter(t.ratio_study3, y, color=_C['ref'], marker='D', s=36, zorder=3, label='single preparation (study 3)')
    ax.scatter(t['ratio_seq_gapd_c0'], y, color=_C['d'], marker='o', s=40, zorder=3, label='consecutive, gap = d')
    ax.scatter(t['ratio_seq_gap1_c0'], y, color=_C['short'], marker='s', s=36, zorder=3, label='consecutive, gap = 1 (if no extra error)')
    ax.hlines(y, t['ratio_seq_gap1_c0'], t.ratio_study3, color='#c3c2b7', lw=1, zorder=1)
    ax.set_yticks(y); ax.set_yticklabels([f'{r.scenario}, p = {r.p:g}, d = {r.d}' + ('' if r.d_simulated else '*') for r in t.itertuples()])
    ax.set_xscale('log'); ax.invert_yaxis()
    ax.set_xlabel('volume per T gate: code switching / 15-to-1 distillation   (< 1: code switching cheaper)')
    ax.set_xlim(left=min(0.8, 0.9 * t['ratio_seq_gap1_c0'].min()))
    ax.set_title(f'target error per T gate = {target:g}   (* size not simulated: weak lower bound)', pad=26)
    ax.legend(fontsize=7.5, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=3, borderaxespad=0.2)
    fig.tight_layout()
    fig.savefig(f'{OUT_DIR}/fig_cost_ratio.png', dpi=200)
    plt.show()


fig_eps_vs_K(SUM)
fig_ratio(AMORT, TARGETS[0])